# PROJECT GENESIS CIVILIZATION Ω∞
### Artificial Universe · Developmental Life · Embodied Minds · Functional Affect · Society · Culture · Civilization · 3D Live World

> **GENESIS CIVILIZATION Ω∞ é um universo artificial, não uma simulação atomística do Universo físico real.**
> Nada aqui afirma vida real, consciência, sentimentos subjetivos, AGI, humanidade real ou cérebro humano real.

O laboratório separa sempre: **comportamento observado** → **mecanismo funcional demonstrado** (probe com controles,
replicação e ablação) → **interpretação humana** → *experiência subjetiva* (nunca assumida).

### Como usar
1. `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU` (sem GPU roda em modo DEBUG na CPU).
2. `Ambiente de execução → Executar tudo`. Autorize o Google Drive quando pedido (é onde o universo persiste).
3. No fim do boot aparece **OPEN GENESIS WORLD**: o observatório 3D (também embutido aqui no Colab) e um link remoto
   HTTPS autenticado para abrir de qualquer computador ou celular enquanto o Colab estiver rodando.
4. A pesquisa roda até o Colab encerrar a sessão (checkpoint atômico a cada poucos minutos). Na próxima vez,
   `Executar tudo` retoma o **mesmo** mundo: idade, agentes, cultura, assentamentos, fila científica e história.
   Pausar: botão ■ da célula 23 (salva tudo).

### Legenda de status usada em todo o laboratório
`IMPLEMENTED` código existe e roda · `TESTED` coberto por teste com controles · `BENCHMARKED` medido neste hardware ·
`SIMULATION HYPOTHESIS` pergunta aberta que o sistema permite investigar · `NOT YET VALIDATED` sem evidência ainda.

Regra suprema: **não construir a resposta dentro do universo.** Não existe `build_house()`, `create_civilization()`,
variável `fear`, `happiness`, linguagem pronta ou fitness de "parecer humano". Estruturas análogas só são rotuladas
quando um instrumento externo, com controles positivo e negativo, demonstra a função.

In [ ]:
# @title ⚙️ Configuração (defaults fortes — nada precisa ser mudado) { display-mode: "form" }
import os

PROFILE = "AUTO"  # @param ["AUTO", "DEBUG", "FAST", "RESEARCH", "EXTREME", "A100_80GB"]
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}
START_RESEARCH = True  # @param {type:"boolean"}
START_3D_VIEWER = True  # @param {type:"boolean"}
ENABLE_REMOTE_VIEW = True  # @param {type:"boolean"}
REMOTE_READ_ONLY = True  # @param {type:"boolean"}
STOP_ON_CHAMPION = False  # @param {type:"boolean"}
AUTO_TUNE = True  # @param {type:"boolean"}
SCIENTIFIC_REPLAY = False  # @param {type:"boolean"}
HUMAN_CONSTRAINT_EXPERIMENT = False  # @param {type:"boolean"}
RUN_BOOT_TESTS = True  # @param {type:"boolean"}
IMPORT_LEGACY_GENESIS = True  # @param {type:"boolean"}
FORCE_NEW = False  # @param {type:"boolean"}
MAX_SESSION_HOURS = 24.0  # @param {type:"number"}
MAX_TICKS_THIS_SESSION = 0  # @param {type:"integer"}
MASTER_SEED = 20261124  # @param {type:"integer"}
VIEWER_PORT = 8765  # @param {type:"integer"}

_env = os.environ.get
PROFILE = _env("GENESIS_PROFILE", PROFILE)
USE_GOOGLE_DRIVE = _env("GENESIS_USE_DRIVE", "1" if USE_GOOGLE_DRIVE else "0") == "1"
START_RESEARCH = _env("GENESIS_START_RESEARCH", "1" if START_RESEARCH else "0") == "1"
START_3D_VIEWER = _env("GENESIS_VIEWER", "1" if START_3D_VIEWER else "0") == "1"
ENABLE_REMOTE_VIEW = _env("GENESIS_REMOTE", "1" if ENABLE_REMOTE_VIEW else "0") == "1"
REMOTE_READ_ONLY = _env("GENESIS_READ_ONLY", "1" if REMOTE_READ_ONLY else "0") == "1"
STOP_ON_CHAMPION = _env("GENESIS_STOP_ON_CHAMPION", "1" if STOP_ON_CHAMPION else "0") == "1"
AUTO_TUNE = _env("GENESIS_AUTO_TUNE", "1" if AUTO_TUNE else "0") == "1"
SCIENTIFIC_REPLAY = _env("GENESIS_SCIENTIFIC_REPLAY", "1" if SCIENTIFIC_REPLAY else "0") == "1"
HUMAN_CONSTRAINT_EXPERIMENT = _env("GENESIS_HUMAN_CONSTRAINT", "1" if HUMAN_CONSTRAINT_EXPERIMENT else "0") == "1"
RUN_BOOT_TESTS = _env("GENESIS_RUN_TESTS", "1" if RUN_BOOT_TESTS else "0") == "1"
IMPORT_LEGACY_GENESIS = _env("GENESIS_IMPORT_LEGACY", "1" if IMPORT_LEGACY_GENESIS else "0") == "1"
FORCE_NEW = _env("GENESIS_FORCE_NEW", "1" if FORCE_NEW else "0") == "1"
MAX_SESSION_HOURS = float(_env("GENESIS_MAX_HOURS", MAX_SESSION_HOURS))
MAX_TICKS_THIS_SESSION = int(_env("GENESIS_MAX_TICKS", MAX_TICKS_THIS_SESSION))
MASTER_SEED = int(_env("GENESIS_SEED", MASTER_SEED))
VIEWER_PORT = int(_env("GENESIS_PORT", VIEWER_PORT))
GENESIS_ROOT_OVERRIDE = _env("GENESIS_ROOT", "")
GENESIS_WORK_OVERRIDE = _env("GENESIS_WORK", "")

## 01 — Núcleo: versões, armazenamento híbrido, logging com write-behind, estatística e instrumentos externos
Infraestrutura **externa** ao universo. Nada aqui é visível para os agentes.

In [ ]:
# @title 01 — Núcleo: versões, armazenamento híbrido, logging com write-behind, estatística e instrumentos externos { display-mode: "form" }
import sys, time, json, math, uuid, zlib, gzip, shutil, pickle, hashlib, random, platform, traceback, warnings, copy
import heapq, itertools, io, csv, glob, datetime, subprocess, contextlib, tempfile, base64, collections, threading
import asyncio, secrets, socket, struct, re, importlib, importlib.util, queue
import html as _html
from dataclasses import dataclass, field, asdict, fields as dc_fields, replace as dc_replace
from typing import Any, Dict, List, Optional, Tuple, Sequence, Callable, Generator, Iterable, Set

import numpy as np
import torch
import torch.nn.functional as F

IN_NOTEBOOK = "ipykernel" in sys.modules
IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab  # noqa: F401
        IN_COLAB = True
    except Exception:
        IN_COLAB = False

import matplotlib
if not IN_NOTEBOOK:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from IPython.display import display, clear_output, HTML, Markdown
except Exception:
    def display(*a, **k):
        for x in a:
            print(x)
        return None
    def clear_output(*a, **k):
        return None
    HTML = str
    Markdown = str

try:
    import scipy.stats as sstats
    import scipy.ndimage as sndi
    HAVE_SCIPY = True
except Exception:
    sstats, sndi, HAVE_SCIPY = None, None, False
try:
    import psutil
    HAVE_PSUTIL = True
except Exception:
    psutil, HAVE_PSUTIL = None, False

warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------------------------------------------------------- versões (toda descoberta carrega o hash)
GENESIS_VERSION = "2.0.0-civilization"
WORLD_ENGINE_VERSION = "WORLD-1.0"
MATTER_ENGINE_VERSION = "MATTER-1.0"
DEVELOPMENT_ENGINE_VERSION = "DEV-1.0"
BODY_ENGINE_VERSION = "BODY-1.0"
MIND_ENGINE_VERSION = "MIND-1.0"
AFFECT_ENGINE_VERSION = "AFFECT-1.0"
SOCIAL_ENGINE_VERSION = "SOCIAL-1.0"
CIVILIZATION_ENGINE_VERSION = "CIV-1.0"
VALIDATION_ENGINE_VERSION = "VE-2.0"
RENDER_PROTOCOL_VERSION = "RP-1.0"
TELEMETRY_SCHEMA_VERSION = "TEL-1.0"
EVENT_SCHEMA_VERSION = "EV-1.0"
CHECKPOINT_VERSION = 3          # 1–2 = Genesis Ω (2D); 3 = Genesis Civilization (importa o legado como arquivo separado)
ENGINE_VERSIONS = {"genesis": GENESIS_VERSION, "world": WORLD_ENGINE_VERSION, "matter": MATTER_ENGINE_VERSION,
                   "development": DEVELOPMENT_ENGINE_VERSION, "body": BODY_ENGINE_VERSION, "mind": MIND_ENGINE_VERSION,
                   "affect": AFFECT_ENGINE_VERSION, "social": SOCIAL_ENGINE_VERSION,
                   "civilization": CIVILIZATION_ENGINE_VERSION, "validation": VALIDATION_ENGINE_VERSION,
                   "render_protocol": RENDER_PROTOCOL_VERSION, "telemetry": TELEMETRY_SCHEMA_VERSION,
                   "events": EVENT_SCHEMA_VERSION, "checkpoint": CHECKPOINT_VERSION}
CHANGELOG = [
    ("2.0.0-civilization", "Mundo 3D multiescala (terreno, clima, água, dia/noite, estações, tempo), matéria de construção, "
                           "genoma de desenvolvimento → corpo massa-mola, substrato de sinais com plasticidade evoluível, "
                           "agentes F0–F4, probes H/A/C com controles, Scientist com DSL segura, checkpoint em shards, "
                           "servidor realtime + observatório Three.js com acesso remoto autenticado. Legado Genesis Ω "
                           "importado como arquivo separado."),
]

STATUS_NOT_TESTED, STATUS_RUNNING, STATUS_PASS, STATUS_FAIL, STATUS_INCONCLUSIVE, STATUS_ERROR = (
    "NOT_TESTED", "RUNNING", "PASS", "FAIL", "INCONCLUSIVE", "ERROR")
LAB_STATES = ["BOOTING", "TESTING", "RESTORING", "SEARCHING", "EXPERIMENTING", "CHECKPOINTING", "PAUSED", "CRASHED",
              "CHAMPION", "OFFLINE"]
CLAIM_TAGS = ["IMPLEMENTED", "TESTED", "BENCHMARKED", "SIMULATION HYPOTHESIS", "NOT YET VALIDATED"]


def _hash_versions() -> str:
    return hashlib.sha256(json.dumps(ENGINE_VERSIONS, sort_keys=True).encode()).hexdigest()[:16]


ENGINE_VERSIONS_HASH = _hash_versions()


class _LiveHook:
    """Acoplamento opcional com a telemetria. Somente observação: nunca participa de fitness, seleção ou RNG."""
    monitor = None


LIVE = _LiveHook()


def live_progress(phase: Optional[str] = None, done: Optional[float] = None, total: Optional[float] = None,
                  detail: Optional[str] = None):
    m = LIVE.monitor
    if m is not None:
        m.progress(phase, done, total, detail)


# ----------------------------------------------------------------------------- armazenamento híbrido GPU↔RAM↔SSD↔Drive
CIV_SUBDIRS = ["checkpoints", "checkpoints/backups", "worlds", "agents", "genomes", "bodies", "lineages", "discoveries",
               "affect", "culture", "settlements", "civilizations", "history", "replays", "snapshots", "reports", "plots",
               "logs", "exports", "crash_reports", "pantheon", "legacy", "viewer_user"]


class CivPaths:
    def __init__(self, root: str, work: str, on_drive: bool):
        self.root, self.work, self.on_drive = os.path.abspath(root), os.path.abspath(work), on_drive
        for d in CIV_SUBDIRS:
            os.makedirs(os.path.join(self.root, d), exist_ok=True)
        for d in ["tmp", "viewer", "viewer/static", "viewer/static/vendor", "bin", "legacy_code", "replays", "cache"]:
            os.makedirs(os.path.join(self.work, d), exist_ok=True)
        self.stop_file = os.path.join(self.root, "STOP")

    def __getattr__(self, name):
        if name in [d.split("/")[-1] for d in CIV_SUBDIRS]:
            sub = next(d for d in CIV_SUBDIRS if d.split("/")[-1] == name)
            return os.path.join(self.__dict__["root"], sub)
        raise AttributeError(name)

    def sub(self, *parts) -> str:
        p = os.path.join(self.root, *parts)
        os.makedirs(os.path.dirname(p), exist_ok=True)
        return p

    def tmp(self, *parts) -> str:
        p = os.path.join(self.work, "tmp", *parts)
        os.makedirs(os.path.dirname(p), exist_ok=True)
        return p


def setup_storage(use_drive: bool, override: str = "", work_override: str = "") -> CivPaths:
    """Drive = persistência; /content/genesis_work = scratch SSD (dados temporários pesados nunca vão para o Drive)."""
    work = work_override or ("/content/genesis_work" if IN_COLAB else os.path.abspath("./genesis_work"))
    if override:
        return CivPaths(override, work, on_drive=override.startswith("/content/drive"))
    if IN_COLAB:
        drive_root = "/content/drive/MyDrive"
        if use_drive and not os.path.isdir(drive_root):
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except Exception as e:
                print(f"[storage] Drive não montado ({e}); usando /content (não persiste se a sessão cair).")
        if use_drive and os.path.isdir(drive_root):
            return CivPaths(os.path.join(drive_root, "GENESIS_CIVILIZATION_OMEGA"), work, on_drive=True)
        return CivPaths("/content/GENESIS_CIVILIZATION_OMEGA", work, on_drive=False)
    return CivPaths(os.path.abspath("./genesis_civilization_runs"), work, on_drive=False)


# ----------------------------------------------------------------------------- utilidades
def now_ts() -> float:
    return time.time()


def iso_now() -> str:
    return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def new_id(prefix: str) -> str:
    return f"{prefix}-{uuid.uuid4().hex[:12]}"


def derive_seed(*parts) -> int:
    h = hashlib.sha256(("|".join(str(p) for p in parts)).encode()).digest()
    return int.from_bytes(h[:4], "little") & 0x7FFFFFFF


def to_jsonable(o):
    if isinstance(o, dict):
        return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, np.ndarray):
        return to_jsonable(o.tolist())
    if isinstance(o, np.generic):
        return to_jsonable(o.item())
    if isinstance(o, float):
        return o if math.isfinite(o) else None
    if isinstance(o, torch.Tensor):
        return to_jsonable(o.detach().cpu().numpy())
    if hasattr(o, "__dataclass_fields__"):
        return to_jsonable(asdict(o))
    return o


def hash_obj(obj) -> str:
    hsh = hashlib.sha256()

    def feed(x):
        if isinstance(x, dict):
            hsh.update(b"{")
            for k in sorted(x.keys(), key=str):
                hsh.update(str(k).encode()); feed(x[k])
            hsh.update(b"}")
        elif isinstance(x, (list, tuple)):
            hsh.update(b"[")
            for v in x:
                feed(v)
            hsh.update(b"]")
        elif isinstance(x, (np.ndarray, np.generic)):
            x = np.asarray(x)
            hsh.update(str(x.dtype).encode() + str(x.shape).encode())
            hsh.update(np.ascontiguousarray(x).tobytes())
        elif isinstance(x, torch.Tensor):
            feed(x.detach().cpu().numpy())
        elif isinstance(x, float):
            hsh.update(repr(round(x, 12)).encode())
        else:
            hsh.update(repr(x).encode())
    feed(obj)
    return hsh.hexdigest()


def fmt_hms(seconds: float) -> str:
    seconds = max(0, int(seconds))
    h, r = divmod(seconds, 3600)
    m, s = divmod(r, 60)
    return f"{h:d}h{m:02d}m{s:02d}s"


def human_bytes(n: float) -> str:
    for u in ["B", "KB", "MB", "GB", "TB"]:
        if abs(n) < 1024:
            return f"{n:.1f}{u}"
        n /= 1024
    return f"{n:.1f}PB"


def is_oom_error(e: BaseException) -> bool:
    if hasattr(torch.cuda, "OutOfMemoryError") and isinstance(e, torch.cuda.OutOfMemoryError):
        return True
    return isinstance(e, RuntimeError) and "out of memory" in str(e).lower()


def sync_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


class RingBuffer:
    """Buffer circular de tamanho fixo (sem listas Python sem limite no caminho quente)."""

    def __init__(self, capacity: int):
        self.capacity = int(capacity)
        self.items: "collections.deque" = collections.deque(maxlen=self.capacity)

    def push(self, x):
        self.items.append(x)

    def last(self, n: int) -> list:
        return list(self.items)[-n:]

    def __len__(self):
        return len(self.items)

    def state_dict(self):
        return {"capacity": self.capacity, "items": list(self.items)}

    def load_state_dict(self, s):
        self.capacity = s["capacity"]; self.items = collections.deque(s["items"], maxlen=self.capacity)


class EventLogger:
    """Eventos estruturados (schema versionado) com write-behind: lote em memória, gravação em blocos no disco."""
    MAX_LOG_BYTES = 64 * 1024 * 1024

    def __init__(self, paths: CivPaths, verbose: bool = True, flush_every: float = 20.0):
        self.paths = paths
        self.events_path = os.path.join(paths.logs, "events.jsonl")
        self.metrics_path = os.path.join(paths.logs, "metrics.csv")
        self.verbose, self.flush_every = verbose, flush_every
        self.recent = RingBuffer(400)
        self._buf: List[str] = []
        self._mbuf: List[Dict[str, Any]] = []
        self._metric_fields: Optional[List[str]] = None
        self._last_flush = time.time()
        self._lock = threading.Lock()

    def event(self, kind: str, echo: bool = False, **data):
        rec = {"schema": EVENT_SCHEMA_VERSION, "ts": iso_now(), "kind": kind, **to_jsonable(data)}
        with self._lock:
            self.recent.push(rec)
            self._buf.append(json.dumps(rec))
        if echo and self.verbose:
            msg = ", ".join(f"{k}={v}" for k, v in data.items() if not isinstance(v, (dict, list)))
            print(f"[{kind}] {msg}")
        if time.time() - self._last_flush > self.flush_every:
            self.flush()

    def metrics(self, row: Dict[str, Any]):
        with self._lock:
            self._mbuf.append({k: to_jsonable(v) for k, v in row.items()})
        if time.time() - self._last_flush > self.flush_every:
            self.flush()

    def _rotate(self, path):
        if os.path.exists(path) and os.path.getsize(path) > self.MAX_LOG_BYTES:
            stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            with open(path, "rb") as fi, gzip.open(f"{path}.{stamp}.gz", "wb") as fo:
                shutil.copyfileobj(fi, fo)
            os.remove(path)

    def flush(self):
        with self._lock:
            buf, self._buf = self._buf, []
            mbuf, self._mbuf = self._mbuf, []
        self._last_flush = time.time()
        try:
            if buf:
                self._rotate(self.events_path)
                with open(self.events_path, "a") as f:
                    f.write("\n".join(buf) + "\n")
            if mbuf:
                self._rotate(self.metrics_path)
                exists = os.path.exists(self.metrics_path)
                if exists and self._metric_fields is None:
                    with open(self.metrics_path) as f:
                        header = f.readline().strip()
                        self._metric_fields = header.split(",") if header else None
                if self._metric_fields is None:
                    self._metric_fields = list(mbuf[0].keys())
                    exists = False
                with open(self.metrics_path, "a", newline="") as f:
                    w = csv.DictWriter(f, fieldnames=self._metric_fields, extrasaction="ignore")
                    if not exists:
                        w.writeheader()
                    for r in mbuf:
                        w.writerow(r)
        except Exception as e:
            print(f"[log] falha ao gravar logs: {e}")


# ----------------------------------------------------------------------------- estatística (instrumentos externos)
def summary_stats(x) -> Dict[str, float]:
    x = np.asarray([v for v in np.ravel(x) if v is not None and np.isfinite(v)], dtype=np.float64)
    if x.size == 0:
        return {"n": 0, "mean": float("nan"), "median": float("nan"), "std": float("nan"),
                "ci_lo": float("nan"), "ci_hi": float("nan")}
    lo, hi = bootstrap_ci(x)
    return {"n": int(x.size), "mean": float(x.mean()), "median": float(np.median(x)),
            "std": float(x.std(ddof=1)) if x.size > 1 else 0.0, "ci_lo": lo, "ci_hi": hi}


def bootstrap_ci(x, n_boot: int = 2000, alpha: float = 0.05, seed: int = 0, stat=np.mean) -> Tuple[float, float]:
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return float("nan"), float("nan")
    if x.size == 1:
        return float(x[0]), float(x[0])
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, x.size, size=(n_boot, x.size))
    bs = stat(x[idx], axis=1)
    return float(np.quantile(bs, alpha / 2)), float(np.quantile(bs, 1 - alpha / 2))


def cohens_d(a, b) -> float:
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if a.size < 2 or b.size < 2:
        return float("nan")
    s = math.sqrt(((a.size - 1) * a.var(ddof=1) + (b.size - 1) * b.var(ddof=1)) / (a.size + b.size - 2))
    if s < 1e-12:
        diff = a.mean() - b.mean()
        return float(np.sign(diff) * 10.0) if abs(diff) > 1e-12 else 0.0
    return float((a.mean() - b.mean()) / s)


def mann_whitney_greater(a, b) -> float:
    """p-valor unilateral (a > b)."""
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if a.size == 0 or b.size == 0:
        return float("nan")
    if HAVE_SCIPY:
        try:
            return float(sstats.mannwhitneyu(a, b, alternative="greater").pvalue)
        except Exception as e:
            warnings.warn(f"mannwhitneyu (scipy) falhou ({e}); usando aproximação normal")
    # aproximação normal
    allv = np.concatenate([a, b])
    ranks = np.argsort(np.argsort(allv)) + 1.0
    u = ranks[:a.size].sum() - a.size * (a.size + 1) / 2
    mu = a.size * b.size / 2
    sd = math.sqrt(a.size * b.size * (a.size + b.size + 1) / 12) + 1e-12
    z = (u - mu) / sd
    return float(0.5 * math.erfc(z / math.sqrt(2)))


def binom_p_greater(k: int, n: int, p0: float) -> float:
    if n <= 0:
        return 1.0
    if HAVE_SCIPY:
        try:
            return float(sstats.binomtest(int(k), int(n), p0, alternative="greater").pvalue)
        except Exception as e:
            warnings.warn(f"binomtest (scipy) falhou ({e}); usando soma exata")
    return float(sum(math.comb(n, i) * p0 ** i * (1 - p0) ** (n - i) for i in range(int(k), n + 1)))


def holm_correction(pvals: Sequence[float]) -> List[float]:
    p = np.asarray([1.0 if (v is None or not np.isfinite(v)) else v for v in pvals], dtype=np.float64)
    m = p.size
    order = np.argsort(p)
    adj = np.empty(m)
    running = 0.0
    for rank, i in enumerate(order):
        val = min(1.0, (m - rank) * p[i])
        running = max(running, val)
        adj[i] = running
    return adj.tolist()


# ----------------------------------------------------------------------------- decodificadores externos
def _standardize(Xtr, Xte):
    mu = Xtr.mean(0, keepdims=True); sd = Xtr.std(0, keepdims=True) + 1e-6
    return (Xtr - mu) / sd, (Xte - mu) / sd


def ridge_predict(Xtr, ytr, Xte, n_classes, lam=1.0):
    Xtr, Xte = _standardize(Xtr, Xte)
    Xtr = np.hstack([Xtr, np.ones((Xtr.shape[0], 1))]); Xte = np.hstack([Xte, np.ones((Xte.shape[0], 1))])
    Y = np.eye(n_classes)[ytr]
    n, d = Xtr.shape
    if n < d:   # forma dual
        Kt = Xtr @ Xtr.T
        alpha = np.linalg.solve(Kt + lam * np.eye(n), Y)
        scores = Xte @ (Xtr.T @ alpha)
    else:
        Wt = np.linalg.solve(Xtr.T @ Xtr + lam * np.eye(d), Xtr.T @ Y)
        scores = Xte @ Wt
    return scores.argmax(1)


def knn_predict(Xtr, ytr, Xte, n_classes, k=5):
    Xtr, Xte = _standardize(Xtr, Xte)
    d = ((Xte[:, None, :] - Xtr[None, :, :]) ** 2).sum(-1)
    k = max(1, min(k, Xtr.shape[0]))
    nn = np.argsort(d, axis=1)[:, :k]
    votes = np.zeros((Xte.shape[0], n_classes))
    for j in range(k):
        votes[np.arange(Xte.shape[0]), ytr[nn[:, j]]] += 1.0 / (1 + j * 0.01)
    return votes.argmax(1)


def stratified_folds(y, k, rng):
    folds = np.zeros(len(y), int)
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        folds[idx] = np.arange(len(idx)) % k
    return folds


def balanced_accuracy(y, yp, n_classes):
    accs = [np.mean(yp[y == c] == c) for c in range(n_classes) if np.any(y == c)]
    return float(np.mean(accs)) if accs else 0.0


def cv_predictions(X, y, n_classes, decoder, k, seed):
    rng = np.random.default_rng(seed)
    k = max(2, min(k, int(np.bincount(y).min()) if len(np.unique(y)) > 1 else 2))
    folds = stratified_folds(y, k, rng)
    yp = np.zeros_like(y)
    for f in range(k):
        tr, te = folds != f, folds == f
        if te.sum() == 0 or tr.sum() == 0:
            continue
        yp[te] = decoder(X[tr], y[tr], X[te], n_classes)
    return yp


def mutual_info_bits(y, yp, n_classes) -> float:
    """MI plug-in da matriz de confusão com correção de Miller–Madow (viés reconhecido)."""
    conf = np.zeros((n_classes, n_classes))
    for a, b in zip(y, yp):
        conf[a, b] += 1
    n = conf.sum()
    if n == 0:
        return 0.0
    pj = conf / n
    px = pj.sum(1, keepdims=True); py = pj.sum(0, keepdims=True)
    nz = pj > 0
    mi = float((pj[nz] * np.log2(pj[nz] / (px @ py)[nz])).sum())
    k_joint, k_x, k_y = nz.sum(), (px > 0).sum(), (py > 0).sum()
    mm = (k_joint - k_x - k_y + 1) / (2 * n * math.log(2))
    return max(0.0, mi - mm)


def decode_report(X: np.ndarray, y: np.ndarray, n_classes: int, n_perm: int, folds: int, seed: int,
                  nonlinear: bool = True) -> Dict[str, Any]:
    """Acurácia balanceada CV (linear e kNN), p-valor por permutação, confusão, MI."""
    X = np.nan_to_num(np.asarray(X, np.float64)); y = np.asarray(y, int)
    chance = 1.0 / n_classes
    out: Dict[str, Any] = {"n": int(len(y)), "chance": chance, "n_classes": n_classes,
                           "method": "balanced acc, stratified CV (ridge linear + kNN), label-permutation null (empirical + Gaussian tail)"}
    if len(np.unique(y)) < 2 or X.shape[0] < 2 * n_classes:
        out.update({"linear_acc": chance, "linear_p": 1.0, "nonlinear_acc": chance, "nonlinear_p": 1.0, "mi_bits": 0.0})
        return out
    if np.all(X.std(0) < 1e-9):
        out.update({"linear_acc": chance, "linear_p": 1.0, "nonlinear_acc": chance, "nonlinear_p": 1.0, "mi_bits": 0.0,
                    "note": "features constantes"})
        return out
    rng = np.random.default_rng(seed)
    for name, dec in [("linear", ridge_predict)] + ([("nonlinear", knn_predict)] if nonlinear else []):
        yp = cv_predictions(X, y, n_classes, dec, folds, seed)
        acc = balanced_accuracy(y, yp, n_classes)
        null = []
        for i in range(n_perm):
            if i % 10 == 0:
                live_progress()
            yperm = rng.permutation(y)
            ypp = cv_predictions(X, yperm, n_classes, dec, folds, seed + 1 + i)
            null.append(balanced_accuracy(yperm, ypp, n_classes))
        null = np.asarray(null)
        p_emp = (np.sum(null >= acc - 1e-12) + 1) / (n_perm + 1)
        # cauda gaussiana da distribuição nula de permutação: permite p pequenos com poucas permutações;
        # só é usada quando a acurácia observada supera TODAS as permutações (senão vale o p empírico).
        sd = max(float(null.std()), 1e-3) if null.size else 1.0
        z = (acc - (float(null.mean()) if null.size else chance)) / sd
        p_par = 0.5 * math.erfc(z / math.sqrt(2))
        p = p_par if (null.size and acc > null.max()) else max(p_emp, p_par)
        out[f"{name}_acc"] = acc
        out[f"{name}_p"] = float(p)
        out[f"{name}_p_empirical"] = float(p_emp)
        out[f"{name}_null_mean"] = float(null.mean()) if null.size else chance
        if name == "linear":
            out["mi_bits"] = mutual_info_bits(y, yp, n_classes)
            conf = np.zeros((n_classes, n_classes), int)
            for a, b in zip(y, yp):
                conf[a, b] += 1
            out["confusion"] = conf.tolist()
            out["effect_size"] = float((acc - out[f"{name}_null_mean"]) / (null.std() + 1e-6)) if null.size else 0.0
    if not nonlinear:
        out["nonlinear_acc"], out["nonlinear_p"] = float("nan"), 1.0
    return out


def norm_acc(acc: float, chance: float) -> float:
    return float((acc - chance) / (1 - chance))


print(f"PROJECT GENESIS CIVILIZATION Ω∞ v{GENESIS_VERSION} | Python {platform.python_version()} | torch {torch.__version__} | "
      f"Colab={IN_COLAB} | notebook={IN_NOTEBOOK} | versões={ENGINE_VERSIONS_HASH}")

## 02 — Configuração central, perfis e detecção de hardware
Todos os thresholds e tamanhos ficam aqui (sem números mágicos espalhados). A configuração vai inteira em cada
checkpoint e em cada descoberta. Perfis: `DEBUG`, `FAST`, `RESEARCH`, `EXTREME`, `A100_80GB` — o autotune ajusta a
carga real medindo o hardware (nunca assume A100).

In [ ]:
# @title 02 — Configuração central, perfis e detecção de hardware { display-mode: "form" }
@dataclass
class Thresholds:
    alpha: float = 0.01
    min_effect: float = 0.5            # Cohen's d mínimo para efeitos comportamentais
    min_norm_acc: float = 0.30
    # desenvolvimento / corpo
    dev_min_nodes: int = 4
    dev_canalization: float = 0.55     # IoU médio entre ruídos de desenvolvimento
    body_min_speed: float = 0.004      # unidades de corpo por passo de física (acima do controle passivo)
    # mente
    sensorimotor_gain: float = 0.15    # ganho de sucesso vs sensores ablados
    learning_gain: float = 0.10        # ganho fim-vs-início da vida acima do controle sem plasticidade
    prediction_gain: float = 0.05      # R² extra do estado interno sobre o sensor atual
    # afeto funcional
    regulation_gain: float = 0.15
    valence_acc: float = 0.65
    persistence_ticks: int = 20
    # sociedade / civilização (janelas em ticks de civilização)
    group_min_size: int = 3
    group_persistence: int = 5         # janelas consecutivas
    settlement_min_blocks: int = 6
    settlement_min_agents: int = 4
    settlement_persistence: int = 3
    city_min_population: int = 40
    city_min_blocks: int = 120
    road_min_traffic: float = 0.35
    specialization_p: float = 0.01
    shelter_damage_reduction: float = 0.2
    ethical_negative_fraction: float = 0.8
    ethical_window: int = 400
    # anti-gaming
    metric_exploit_gain: float = 0.15


@dataclass
class ProbeSettings:
    trials: int = 12
    permutations: int = 60
    folds: int = 4
    arena_size: int = 16               # tiles por arena
    arena_ticks: int = 160
    memory_delays: List[int] = field(default_factory=lambda: [1, 10, 40])
    replication_seeds: int = 6
    critical_seeds: int = 20
    damage_levels: List[float] = field(default_factory=lambda: [0.25, 0.5])
    dev_noise_samples: int = 4
    body_test_steps: int = 240


@dataclass
class CivConfig:
    profile: str = "RESEARCH"
    master_seed: int = 20261124
    scientific_replay: bool = False
    human_constraint_experiment: bool = False
    # mundo
    world_w: int = 256
    world_h: int = 256
    chunk: int = 16
    sea_level: float = 0.42
    height_scale: float = 24.0
    n_sound: int = 4
    day_period: int = 240
    season_period: int = 7200
    weather_strength: float = 1.0
    # agentes (F1)
    initial_agents: int = 4000
    max_agents: int = 40000
    n_units: int = 24
    sense_radius: float = 4.0
    max_speed_cap: float = 0.9          # tiles por tick
    min_speed: float = 0.08
    basal_cost: float = 0.004
    move_cost: float = 0.010
    action_cost: float = 0.0012
    signal_cost: float = 0.0006
    transfer_amount: float = 0.02
    build_cost: float = 0.03
    eat_rate: float = 0.12
    eat_efficiency: float = 0.9
    repro_cost_fraction: float = 0.5
    max_age_range: Tuple[int, int] = (1500, 6000)
    carry_capacity: int = 4
    # corpos (F2–F4)
    max_bodies: int = 256
    dev_lattice: int = 5
    n_morph: int = 4
    dev_steps: int = 10
    body_substeps: int = 6
    body_dt: float = 0.02
    body_world_scale: float = 6.0       # ganho corpo→mundo: tiles por unidade de corpo deslocada (coarse-graining
                                        # espaço-temporal declarado; o mesmo ganho em F1 e F2+)
    # construção / matéria
    max_blocks: int = 300000
    max_stack: int = 6
    # multiescala
    region_size: int = 16
    f1_budget_fraction: float = 0.9
    # agenda temporal multiescala (ticks)
    env_every: int = 4
    social_every: int = 10
    culture_every: int = 50
    civ_every: int = 100
    development_every: int = 20
    fidelity_every: int = 50
    aggregate_every: int = 25
    snapshot_every: int = 50
    # evolução QD (ilhas de laboratório)
    n_islands: int = 4
    island_pop: int = 48
    qd_batch: int = 48
    immigrant_every: int = 400
    map_bins: List[int] = field(default_factory=lambda: [6, 6, 5])
    # ciência
    experiment_time_fraction: float = 0.3
    max_queue: int = 400
    max_experiments_per_cycle: int = 3
    # telemetria / servidor
    viewer_port: int = 8765
    telemetry_hz: float = 4.0
    frame_every_seconds: float = 0.25
    max_stream_agents: int = 6000
    snapshot_ring: int = 240
    # sessão
    max_session_hours: float = 24.0
    max_ticks_session: int = 0
    checkpoint_every_minutes: float = 6.0
    stop_on_champion: bool = False
    precision: str = "fp32"
    use_compile: bool = False
    thresholds: Thresholds = field(default_factory=Thresholds)
    probe: ProbeSettings = field(default_factory=ProbeSettings)

    def to_dict(self) -> dict:
        return to_jsonable(asdict(self))

    @staticmethod
    def from_dict(d: dict) -> "CivConfig":
        d = copy.deepcopy(d)
        th = Thresholds(**{k: v for k, v in d.pop("thresholds", {}).items() if k in Thresholds.__dataclass_fields__})
        pr = ProbeSettings(**{k: v for k, v in d.pop("probe", {}).items() if k in ProbeSettings.__dataclass_fields__})
        if "max_age_range" in d:
            d["max_age_range"] = tuple(d["max_age_range"])
        cfg = CivConfig(**{k: v for k, v in d.items() if k in CivConfig.__dataclass_fields__})
        cfg.thresholds, cfg.probe = th, pr
        return cfg

    def thresholds_hash(self) -> str:
        return hash_obj(to_jsonable(asdict(self.thresholds)))[:16]


def detect_hardware() -> Dict[str, Any]:
    info: Dict[str, Any] = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
                            "platform": platform.platform(), "cpu_count": os.cpu_count(), "timestamp": iso_now(),
                            "device": "cuda" if torch.cuda.is_available() else "cpu", "genesis_version": GENESIS_VERSION}
    if HAVE_PSUTIL:
        try:
            info["ram_total"] = int(psutil.virtual_memory().total)
        except Exception as e:
            info["ram_error"] = repr(e)
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        info.update({"gpu_name": p.name, "vram_total": int(p.total_memory), "compute_capability": f"{p.major}.{p.minor}"})
        try:
            info["vram_free"] = int(torch.cuda.mem_get_info()[0])
        except Exception:
            info["vram_free"] = int(p.total_memory)
        try:
            info["bf16_supported"] = bool(torch.cuda.is_bf16_supported())
        except Exception:
            info["bf16_supported"] = False
    else:
        info.update({"gpu_name": "none (CPU)", "vram_total": 0, "vram_free": 0, "bf16_supported": False})
    return info


def choose_profile(requested: str, hw: Dict[str, Any]) -> str:
    if requested != "AUTO":
        return requested
    if hw["device"] != "cuda":
        return "DEBUG"
    name, vram = hw.get("gpu_name", ""), hw.get("vram_total", 0)
    if "A100" in name and vram >= 70e9:
        return "A100_80GB"
    if vram >= 30e9:
        return "EXTREME"
    return "RESEARCH"


def profile_config(profile: str, hw: Dict[str, Any], seed: int) -> CivConfig:
    cfg = CivConfig(profile=profile, master_seed=seed)
    pr = cfg.probe
    if profile == "DEBUG":
        cfg.world_w = cfg.world_h = 64
        cfg.initial_agents, cfg.max_agents, cfg.n_units = 120, 400, 16
        cfg.max_bodies, cfg.dev_lattice, cfg.dev_steps = 12, 4, 8
        cfg.max_blocks, cfg.region_size = 4096, 16
        cfg.n_islands, cfg.island_pop, cfg.qd_batch = 2, 12, 12
        cfg.max_stream_agents, cfg.snapshot_ring = 400, 120
        cfg.checkpoint_every_minutes = 2.0
        cfg.immigrant_every = 200
        pr.trials, pr.permutations, pr.arena_ticks = 8, 30, 96
        pr.memory_delays, pr.replication_seeds, pr.critical_seeds = [1, 8, 24], 3, 6
        pr.body_test_steps, pr.dev_noise_samples = 160, 3
    elif profile == "FAST":
        cfg.world_w = cfg.world_h = 128
        cfg.initial_agents, cfg.max_agents, cfg.n_units = 1200, 6000, 20
        cfg.max_bodies, cfg.dev_lattice, cfg.max_blocks = 64, 5, 60000
        cfg.n_islands, cfg.island_pop = 3, 32
        cfg.checkpoint_every_minutes = 4.0
        pr.trials, pr.permutations = 10, 40
    elif profile == "RESEARCH":
        pass
    elif profile == "EXTREME":
        cfg.world_w = cfg.world_h = 384
        cfg.initial_agents, cfg.max_agents = 12000, 150000
        cfg.max_bodies, cfg.dev_lattice, cfg.max_blocks = 768, 6, 1000000
        cfg.n_islands, cfg.island_pop, cfg.qd_batch = 6, 96, 96
        cfg.max_stream_agents = 12000
        pr.trials, pr.permutations, pr.critical_seeds = 16, 100, 30
    elif profile == "A100_80GB":
        cfg.world_w = cfg.world_h = 512
        cfg.initial_agents, cfg.max_agents, cfg.n_units = 30000, 400000, 32
        cfg.max_bodies, cfg.dev_lattice, cfg.max_blocks = 2048, 6, 3000000
        cfg.n_islands, cfg.island_pop, cfg.qd_batch = 8, 128, 128
        cfg.max_stream_agents = 20000
        pr.trials, pr.permutations, pr.critical_seeds = 20, 120, 30
    else:
        raise ValueError(f"perfil desconhecido: {profile}")
    return cfg

## 03 — Ponte com o Genesis Ω original (pesquisa preservada, nunca reinterpretada)
O motor 2D original continua disponível como módulo `genesis_omega_legacy` (código embutido, versão exata).
* **Importação**: o checkpoint antigo (`MyDrive/GENESIS_OMEGA`) é lido e resumido num arquivo separado
  (`legacy/`), com descobertas, níveis e evidência. Nada disso vira "agente com corpo".
* **Reprodução**: `reproduce_legacy_discovery(id)` reexecuta uma descoberta antiga com o motor antigo.
* **Ponte de substrato** (Discovery Bridge): uma lei 2D antiga pode dirigir um campo ambiental de um *fork* do mundo
  novo (experimento de substrato), para comparar leis físicas como base de desenvolvimento e ecologia.

In [ ]:
# @title 03 — Ponte com o Genesis Ω original (pesquisa preservada, nunca reinterpretada) { display-mode: "form" }
LEGACY_GENESIS_B64 = "..."


class LegacyBridge:
    MODULE_NAME = "genesis_omega_legacy"

    def __init__(self, paths: "CivPaths", logger: "EventLogger"):
        self.paths, self.logger = paths, logger
        self._module = None

    def source(self) -> str:
        if LEGACY_GENESIS_B64.startswith("@@"):
            raise RuntimeError("código legado não embutido (build incompleto)")
        return base64.b64decode(LEGACY_GENESIS_B64).decode("utf-8")

    def code_path(self) -> str:
        src = self.source()
        p = os.path.join(self.paths.work, "legacy_code", f"{self.MODULE_NAME}.py")
        if not os.path.exists(p) or hashlib.sha256(open(p, "rb").read()).hexdigest() != hashlib.sha256(src.encode()).hexdigest():
            with open(p, "w", encoding="utf-8") as f:
                f.write(src)
        return p

    def load_module(self):
        if self._module is None:
            spec = importlib.util.spec_from_file_location(self.MODULE_NAME, self.code_path())
            mod = importlib.util.module_from_spec(spec)
            with contextlib.redirect_stdout(io.StringIO()):
                spec.loader.exec_module(mod)
            sys.modules[self.MODULE_NAME] = mod
            self._module = mod
        return self._module

    def candidate_checkpoints(self) -> List[str]:
        roots = ["/content/drive/MyDrive/GENESIS_OMEGA", "/content/GENESIS_OMEGA", os.path.abspath("./genesis_runs")]
        extra = os.environ.get("GENESIS_LEGACY_ROOT", "")
        if extra:
            roots.insert(0, extra)
        out = []
        for r in roots:
            for name in ["checkpoint_current.pt", "checkpoint_previous.pt"]:
                p = os.path.join(r, "checkpoints", name)
                if os.path.exists(p):
                    out.append(p)
        return out

    @staticmethod
    def _load(path: str) -> dict:
        try:
            return torch.load(path, map_location="cpu", weights_only=False)
        except TypeError:
            return torch.load(path, map_location="cpu")

    def import_research(self, force: bool = False) -> Optional[Dict[str, Any]]:
        """Resumo honesto da pesquisa 2D anterior. Idempotente (marcador com hash do checkpoint)."""
        cands = self.candidate_checkpoints()
        if not cands:
            return None
        path = cands[0]
        digest = hashlib.sha256(f"{path}|{os.path.getsize(path)}|{os.path.getmtime(path)}".encode()).hexdigest()[:16]
        marker = os.path.join(self.paths.legacy, "IMPORTED.json")
        if os.path.exists(marker) and not force:
            m = json.load(open(marker))
            if m.get("digest") == digest and os.path.exists(m.get("summary_path", "")):
                return json.load(open(m["summary_path"]))
        st = None
        for p in cands:
            try:
                st = self._load(p); path = p
                break
            except Exception as e:
                self.logger.event("legacy_checkpoint_unreadable", path=p, error=repr(e))
        if st is None:
            return None
        levels = st.get("levels", {})
        hist = collections.Counter(int(v) for v in levels.values())
        disc = []
        for did, rec in st.get("discoveries", {}).items():
            disc.append({"discovery_id": did, "candidate_id": rec.get("candidate_id"), "level": rec.get("new_level"),
                         "level_name": rec.get("level_name"), "probe": rec.get("probe"), "score": rec.get("score"),
                         "p_value": rec.get("p_value"), "generation": rec.get("generation"), "engine": rec.get("engine")})
        summary = {"source_checkpoint": path, "legacy_genesis_version": st.get("genesis_version"),
                   "legacy_checkpoint_version": st.get("checkpoint_version"), "counters": st.get("counters", {}),
                   "best_level_global": st.get("best_level_global"), "level_histogram": dict(hist),
                   "discoveries": disc, "champion": to_jsonable(st.get("champion")),
                   "pantheon_size": len(st.get("pantheon", [])), "config_profile": st.get("config", {}).get("profile"),
                   "imported_at": iso_now(), "digest": digest,
                   "note": "Pesquisa Genesis Ω (campos 2D). Descobertas computacionais de substrato; não possuem corpo, "
                           "agentes nem civilização. Preservadas como arquivo separado."}
        src_disc = os.path.join(os.path.dirname(os.path.dirname(path)), "discoveries")
        if os.path.isdir(src_disc):
            shutil.copytree(src_disc, os.path.join(self.paths.legacy, "discoveries"), dirs_exist_ok=True)
        spath = os.path.join(self.paths.legacy, "legacy_genesis_omega_summary.json")
        with open(spath, "w") as f:
            json.dump(to_jsonable(summary), f, indent=1)
        with open(marker, "w") as f:
            json.dump({"digest": digest, "summary_path": spath, "ts": iso_now()}, f)
        self.logger.event("legacy_imported", discoveries=len(disc), best_level=summary["best_level_global"])
        return summary

    def legacy_lab(self, path: Optional[str] = None):
        """Reconstrói o laboratório 2D antigo a partir do checkpoint (motor antigo, configuração travada)."""
        mod = self.load_module()
        path = path or (self.candidate_checkpoints() or [None])[0]
        if path is None:
            raise FileNotFoundError("nenhum checkpoint Genesis Ω encontrado")
        st = self._load(path)
        st = mod.migrate_checkpoint(st)
        cfg = mod.GenesisConfig.from_dict(st["config"])
        lpaths = mod.GenesisPaths(os.path.join(self.paths.work, "legacy_runtime"), on_drive=False)
        lab = mod.GenesisLab(cfg, mod.detect_hardware(), lpaths, mod.EventLogger(lpaths, verbose=False),
                             torch.device("cuda" if torch.cuda.is_available() else "cpu"))
        lab.load_state_dict(st)
        return mod, lab

    def reproduce_legacy_discovery(self, discovery_id: str, path: Optional[str] = None) -> Dict[str, Any]:
        mod, lab = self.legacy_lab(path)
        with contextlib.redirect_stdout(io.StringIO()):
            out = mod.reproduce_discovery(discovery_id, lab)
        self.logger.event("legacy_reproduction", discovery_id=discovery_id, status_match=out.get("status_match"))
        return out


class LegacySubstrateField:
    """Discovery Bridge: uma lei 2D antiga evolui um campo sobre o mapa do mundo; a magnitude modula a regeneração de
    recursos de um FORK experimental. Instrumento de comparação de substratos, nunca usado no mundo principal sem marca."""

    def __init__(self, bridge: LegacyBridge, candidate_id: str, world_hw: Tuple[int, int], device: torch.device,
                 checkpoint: Optional[str] = None, field_size: int = 64):
        mod, lab = bridge.legacy_lab(checkpoint)
        if candidate_id not in lab.store:
            raise KeyError(f"candidato legado {candidate_id} não está no checkpoint")
        self.mod, self.genome = mod, lab.store[candidate_id]
        self.candidate_id = candidate_id
        self.sim = mod.Simulator(lab.cfg, device)
        self.gb = self.sim.prepare([self.genome], field_size)
        env = lab.envs[0]
        self.st = self.sim.init_state(self.gb, [env], [derive_seed("substrate", candidate_id)], run_seed=1)
        self.world_hw = world_hw

    def step(self, n: int = 1) -> torch.Tensor:
        for _ in range(n):
            self.st = self.sim.step(self.gb, self.st)
        mag = self.st.U[0].abs().sum(0, keepdim=True).unsqueeze(0)
        up = F.interpolate(mag, size=self.world_hw, mode="bilinear", align_corners=False)[0, 0]
        return (up / (up.max() + 1e-9)).clamp(0, 1)

## 04 — Tipos centrais: genoma de desenvolvimento, linhagem persistente e pools Struct-of-Arrays na GPU
* `DevelopmentGenome` (layout vetorial): **regras** que geram corpo e substrato de sinais — nunca o corpo final.
  Seções: GRN de morfógenos 3D, leitura de propriedades contínuas da matéria (rigidez, contração, adesão,
  condutividade, sensibilidade…), inervação motora, substrato de sinais (conexões, constantes de tempo, modulação),
  regra local de plasticidade (forma ABCD evoluível), cronograma de exploração motora, traços de vida, assinatura
  visual genética e parâmetros de mutação auto-adaptativos.
* Sensores e ações são **canais físicos** (campos do mundo, contato, sinais emitidos). Nenhum canal recebe objetos
  classificados, rótulos sociais ou palavras.

In [ ]:
# @title 04 — Tipos centrais: genoma de desenvolvimento, linhagem persistente e pools Struct-of-Arrays na GPU { display-mode: "form" }
SENSOR_NAMES = ["bias", "resource_here", "resource_grad_fwd", "resource_grad_left", "light", "temperature",
                "slope_fwd", "slope_left", "water_ahead", "structure_here", "enclosure_here", "marks_here",
                "sound_0", "sound_1", "sound_2", "sound_3", "neighbor_density", "neighbor_dir_fwd", "neighbor_dir_left",
                "neighbor_color_0", "neighbor_color_1", "neighbor_motion", "neighbor_carry", "intero_energy",
                "intero_integrity", "intero_fatigue", "intero_damage_rate", "proprio_carry", "contact", "exposure"]
ACTION_NAMES = ["forward", "turn", "emit_0", "emit_1", "emit_2", "emit_3", "consume", "pick_material",
                "place_material", "reproduce", "deposit_mark", "transfer_to_nearest", "take_from_nearest",
                "display_0", "display_1", "rest"]
N_SENSORS, N_ACTIONS = len(SENSOR_NAMES), len(ACTION_NAMES)
N_PROPS = 10
PROP_NAMES = ["stiffness", "damping", "adhesion", "contraction", "conductivity", "sensitivity", "permeability",
              "signal_capacity", "plasticity", "density"]
N_MOTOR = 3                                     # forward, turn, oscilador interno
# Técnicas = padrões ação×contexto detectados de FORA (o agente não recebe esses rótulos)
TECH_NAMES = ["fill_water", "stack_wall", "coop_rich", "give", "mark", "take", "shelter_use", "cross_filled"]
N_TECH = len(TECH_NAMES)


@dataclass
class GeneSpec:
    name: str
    shape: Tuple[int, ...]
    lo: float
    hi: float
    kind: str = "float"                         # float | log
    section: str = "dev"


class GenomeLayout:
    """Layout plano do DevelopmentGenome; mutação, crossover e limites vetorizados em torch (GPU)."""

    def __init__(self, cfg: CivConfig):
        k, N, I, O = cfg.n_morph, cfg.n_units, N_SENSORS, N_ACTIONS
        self.k, self.N, self.I, self.O = k, N, I, O
        lo_age, hi_age = cfg.max_age_range
        specs = [
            GeneSpec("dev_Wm", (k, k), -2, 2), GeneSpec("dev_bm", (k,), -1, 1), GeneSpec("dev_decay", (k,), 0, 0.5),
            GeneSpec("dev_diff", (k,), 0, 0.25), GeneSpec("dev_seed", (k,), -1, 1), GeneSpec("dev_axis", (3,), -1, 1),
            GeneSpec("dev_gw", (k,), -2, 2), GeneSpec("dev_gthr", (), -1, 1), GeneSpec("dev_maxfrac", (), 0.05, 1.0),
            GeneSpec("dev_noise", (), 0.0, 0.3),
            GeneSpec("body_Wp", (N_PROPS, k), -2, 2, section="body"), GeneSpec("body_bp", (N_PROPS,), -2, 2, section="body"),
            GeneSpec("body_innerv", (N_MOTOR, k), -2, 2, section="body"),
            GeneSpec("body_osc", (), 0.5, 4.0, section="body"),
            GeneSpec("ctl_Win", (N, I), -2, 2, section="signal"), GeneSpec("ctl_W0", (N, N), -2, 2, section="signal"),
            GeneSpec("ctl_Wout", (O, N), -2, 2, section="signal"), GeneSpec("ctl_b", (N,), -1, 1, section="signal"),
            GeneSpec("ctl_bout", (O,), -1, 1, section="signal"), GeneSpec("ctl_logtau", (N,), 0.0, 2.5, section="signal"),
            GeneSpec("ctl_wmod", (N,), -2, 2, section="signal"), GeneSpec("ctl_bmod", (), -2, 2, section="signal"),
            GeneSpec("plast_eta", (), 0.0, 0.05, section="plasticity"),
            GeneSpec("plast_abcd", (4,), -1, 1, section="plasticity"),
            GeneSpec("plast_decay", (), 0.0, 0.05, section="plasticity"),
            GeneSpec("babble_sigma", (), 0.0, 0.5, section="behavior"),
            GeneSpec("babble_tau", (), 10.0, 2000.0, "log", section="behavior"),
            GeneSpec("life_maturity", (), 60.0, 600.0, section="life"),
            GeneSpec("life_span", (), float(lo_age), float(hi_age), section="life"),
            GeneSpec("life_repro", (), 0.5, 0.95, section="life"),
            GeneSpec("display", (3,), 0.0, 1.0, section="display"),
            GeneSpec("mut_sigma", (), 0.005, 0.3, "log", section="mutation"),
            GeneSpec("mut_struct", (), 0.0, 0.3, section="mutation"),
        ]
        self.specs = {s.name: s for s in specs}
        self.slices: Dict[str, Tuple[int, int]] = {}
        i = 0
        lo, hi, logm = [], [], []
        for s in specs:
            n = int(np.prod(s.shape)) if s.shape else 1
            self.slices[s.name] = (i, i + n)
            lo += [s.lo] * n; hi += [s.hi] * n; logm += [s.kind == "log"] * n
            i += n
        self.P = i
        self.lo = torch.tensor(lo, dtype=torch.float32)
        self.hi = torch.tensor(hi, dtype=torch.float32)
        self.is_log = torch.tensor(logm)
        self.section_mask = {}
        for s in specs:
            m = torch.zeros(self.P, dtype=torch.bool)
            a, b = self.slices[s.name]
            m[a:b] = True
            self.section_mask[s.section] = self.section_mask.get(s.section, torch.zeros(self.P, dtype=torch.bool)) | m

    def view(self, G: torch.Tensor, name: str) -> torch.Tensor:
        a, b = self.slices[name]
        s = self.specs[name]
        return G[..., a:b].reshape(*G.shape[:-1], *s.shape) if s.shape else G[..., a]

    def to(self, device):
        self.lo, self.hi, self.is_log = self.lo.to(device), self.hi.to(device), self.is_log.to(device)
        self.section_mask = {k: v.to(device) for k, v in self.section_mask.items()}
        return self

    def clamp(self, G: torch.Tensor) -> torch.Tensor:
        return torch.max(torch.min(torch.nan_to_num(G), self.hi), self.lo)

    def random(self, n: int, gen: torch.Generator) -> torch.Tensor:
        dev = self.lo.device
        u = torch.rand(n, self.P, generator=gen, device=dev)
        G = self.lo + u * (self.hi - self.lo)
        logv = torch.exp(torch.log(self.lo.clamp(min=1e-6)) + u * (torch.log(self.hi) - torch.log(self.lo.clamp(min=1e-6))))
        G = torch.where(self.is_log, logv, G)
        # priores mais suaves para conexões (a maioria dos genomas aleatórios é lixo — é para isso que existe seleção)
        for name, scale, dens in [("ctl_Win", 0.7, 0.35), ("ctl_W0", 0.6, 0.25), ("ctl_Wout", 0.8, 0.5)]:
            a, b = self.slices[name]
            w = torch.randn(n, b - a, generator=gen, device=dev) * scale
            G[:, a:b] = w * (torch.rand(n, b - a, generator=gen, device=dev) < dens)
        for name, val in [("mut_sigma", 0.04), ("mut_struct", 0.05), ("plast_decay", 0.005)]:
            G[:, self.slices[name][0]] = val
        return self.clamp(G)

    def mutate(self, G: torch.Tensor, gen: torch.Generator, sigma_mult: float = 1.0) -> torch.Tensor:
        """Mutação vetorizada: σ auto-adaptativo (log-normal), mutação gaussiana pequena em ~12% dos genes, rara mutação
        forte (reamostragem), mutação estrutural (poda/criação de conexões e reamostragem do programa de
        desenvolvimento) com taxa herdada."""
        n, P, dev = G.shape[0], self.P, G.device
        G = G.clone()
        ms_i, st_i = self.slices["mut_sigma"][0], self.slices["mut_struct"][0]
        G[:, ms_i] = G[:, ms_i] * torch.exp(0.2 * torch.randn(n, generator=gen, device=dev))
        G[:, st_i] = G[:, st_i] * torch.exp(0.2 * torch.randn(n, generator=gen, device=dev))
        G = self.clamp(G)
        sig = (G[:, ms_i] * sigma_mult).unsqueeze(1)
        rng_w = (self.hi - self.lo).unsqueeze(0)
        mask = torch.rand(n, P, generator=gen, device=dev) < 0.12
        G = G + mask * torch.randn(n, P, generator=gen, device=dev) * sig * rng_w
        strong = torch.rand(n, P, generator=gen, device=dev) < 0.003
        fresh = self.lo + torch.rand(n, P, generator=gen, device=dev) * (self.hi - self.lo)
        G = torch.where(strong, fresh, G)
        struct_rate = G[:, st_i].unsqueeze(1)
        for name, dens in [("ctl_W0", 0.02), ("ctl_Win", 0.02)]:
            a, b = self.slices[name]
            ev = torch.rand(n, 1, generator=gen, device=dev) < struct_rate
            prune = ev & (torch.rand(n, b - a, generator=gen, device=dev) < dens)
            grow = ev & (torch.rand(n, b - a, generator=gen, device=dev) < dens)
            seg = G[:, a:b]
            seg = torch.where(prune, torch.zeros_like(seg), seg)
            seg = torch.where(grow & (seg == 0), torch.randn(n, b - a, generator=gen, device=dev) * 0.6, seg)
            G[:, a:b] = seg
        devm = self.section_mask["dev"].unsqueeze(0)
        ev = (torch.rand(n, 1, generator=gen, device=dev) < struct_rate * 0.3) & devm
        resample = ev & (torch.rand(n, P, generator=gen, device=dev) < 0.25)
        G = torch.where(resample, fresh, G)
        return self.clamp(G)

    def crossover(self, A: torch.Tensor, B: torch.Tensor, gen: torch.Generator) -> torch.Tensor:
        """Recombinação opcional por seção (blocos funcionais herdados juntos)."""
        out = A.clone()
        for sec, m in self.section_mask.items():
            if sec == "mutation":
                continue
            pick = (torch.rand(A.shape[0], 1, generator=gen, device=A.device) < 0.5) & m.unsqueeze(0)
            out = torch.where(pick, B, out)
        return out

    def distance(self, A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
        rngw = (self.hi - self.lo).clamp(min=1e-6)
        return (((A - B) / rngw) ** 2).mean(-1).sqrt()


class IdCounter:
    def __init__(self, start: int = 1):
        self.value = int(start)

    def take(self, n: int) -> np.ndarray:
        out = np.arange(self.value, self.value + n, dtype=np.int64)
        self.value += n
        return out


class LineageStore:
    """Linhagem de todos os nascimentos (uid, pai, tick, origem, geração). Índice recente em memória (limitado) +
    shards persistentes comprimidos (write-behind) — ancestralidade profunda é lida dos shards sob demanda."""
    ORIGINS = {"founder": 0, "birth": 1, "immigrant_qd": 2, "disaggregated": 3, "imported": 4, "fork": 5}

    def __init__(self, paths: Optional["CivPaths"], recent: int = 300000):
        self.paths = paths
        self.recent_cap = recent
        self.recent: "collections.OrderedDict[int, Tuple[int, int, int, int]]" = collections.OrderedDict()
        self._pending: List[Tuple[int, int, int, int, int]] = []
        self.shards: List[str] = []
        self.total = 0
        self.children_count: Dict[int, int] = {}

    def add(self, uids: np.ndarray, parents: np.ndarray, tick: int, origin: str, gens: np.ndarray):
        oc = self.ORIGINS.get(origin, 4)
        for u, p, g in zip(uids.tolist(), parents.tolist(), gens.tolist()):
            self.recent[u] = (p, tick, oc, g)
            self._pending.append((u, p, tick, oc, g))
            if p > 0:
                self.children_count[p] = self.children_count.get(p, 0) + 1
        self.total += len(uids)
        while len(self.recent) > self.recent_cap:
            self.recent.popitem(last=False)
        if len(self._pending) >= 50000:
            self.flush()
        if len(self.children_count) > 4 * self.recent_cap:
            keep = sorted(self.children_count.items(), key=lambda x: -x[1])[: self.recent_cap]
            self.children_count = dict(keep)

    def flush(self):
        if not self._pending or self.paths is None:
            return
        arr = np.asarray(self._pending, dtype=np.int64)
        name = os.path.join(self.paths.lineages, f"lineage_{len(self.shards):06d}_{int(time.time())}.npz")
        np.savez_compressed(name, data=arr)
        self.shards.append(name)
        self._pending = []

    def parent_of(self, uid: int) -> Optional[int]:
        if uid in self.recent:
            return self.recent[uid][0]
        for p in reversed(self.shards):
            try:
                d = np.load(p)["data"]
            except Exception:
                continue
            hit = np.nonzero(d[:, 0] == uid)[0]
            if hit.size:
                return int(d[hit[0], 1])
        return None

    def ancestors(self, uid: int, depth: int = 50) -> List[int]:
        out, cur = [], uid
        for _ in range(depth):
            p = self.parent_of(cur)
            if p is None or p <= 0:
                break
            out.append(p); cur = p
        return out

    def state_dict(self):
        self.flush()
        tail = list(self.recent.items())[-20000:]
        return {"recent": tail, "shards": self.shards, "total": self.total,
                "children": sorted(self.children_count.items(), key=lambda x: -x[1])[:20000]}

    def load_state_dict(self, s):
        self.recent = collections.OrderedDict((int(k), tuple(v)) for k, v in s["recent"])
        self.shards = [p for p in s["shards"] if os.path.exists(p)]
        self.total = int(s["total"]); self.children_count = {int(k): int(v) for k, v in s.get("children", [])}


class AgentPool:
    """Struct-of-Arrays na GPU, capacidade fixa (sem listas Python no caminho quente). Cada slot guarda o genótipo
    completo do indivíduo; os parâmetros do substrato de sinais são *views* do genoma (sem cópia)."""
    FLOAT_FIELDS = ["heading", "speed", "energy", "integrity", "fatigue", "age", "carried", "last_damage", "mod",
                    "repro_cool", "contact"]

    def __init__(self, cap: int, layout: GenomeLayout, cfg: CivConfig, device: torch.device):
        self.cap, self.layout, self.device = int(cap), layout, device
        N, O = cfg.n_units, N_ACTIONS
        z = lambda *s, dt=torch.float32: torch.zeros(*s, dtype=dt, device=device)
        self.alive = z(self.cap, dt=torch.bool)
        self.aid = z(self.cap, dt=torch.int64)
        self.guid = z(self.cap, dt=torch.int64)
        self.parent = z(self.cap, dt=torch.int64)
        self.root = z(self.cap, dt=torch.int64)
        self.generation = z(self.cap, dt=torch.int32)
        self.birth = z(self.cap, dt=torch.int64)
        self.species = z(self.cap, dt=torch.int32)
        self.fidelity = z(self.cap, dt=torch.int8)
        self.body_slot = torch.full((self.cap,), -1, dtype=torch.int32, device=device)
        self.pos = z(self.cap, 2)
        for f in self.FLOAT_FIELDS:
            setattr(self, f, z(self.cap))
        self.genome = z(self.cap, layout.P)
        self.x = z(self.cap, N)
        self.W = z(self.cap, N, N)
        self.out = z(self.cap, O)
        self.emit = z(self.cap, cfg.n_sound)
        self.color = z(self.cap, 2)
        self.body_cap = z(self.cap, 6)                 # [velocidade líquida, custo, nós, giro, ângulo da marcha, oscilação]
        self.body_measured = z(self.cap, dt=torch.bool)
        self.stats = z(self.cap, 8)
        self.signature = z(self.cap, O)
        self.culture_type = torch.full((self.cap,), -1, dtype=torch.int32, device=device)
        self.last_partner = z(self.cap, dt=torch.int64)
        self.blocked = z(self.cap)
        self.sense_cache = z(self.cap, N_SENSORS)
        self.intervened = z(self.cap, dt=torch.bool)
        self.tech = z(self.cap, dt=torch.int32)        # bits de técnicas observadas (instrumento externo, invisível ao agente)

    TENSOR_FIELDS = ["alive", "aid", "guid", "parent", "root", "generation", "birth", "species", "fidelity", "body_slot",
                     "pos", "heading", "speed", "energy", "integrity", "fatigue", "age", "carried", "last_damage", "mod",
                     "repro_cool", "genome", "x", "W", "out", "emit", "color", "body_cap", "body_measured", "stats",
                     "signature", "culture_type", "last_partner", "blocked", "sense_cache", "intervened", "contact", "tech"]

    def n_alive(self) -> int:
        return int(self.alive.sum().item())

    def free_slots(self, n: int) -> torch.Tensor:
        free = torch.nonzero(~self.alive).flatten()
        return free[:n]

    def spawn(self, slots: torch.Tensor, genomes: torch.Tensor, pos: torch.Tensor, aids: np.ndarray, guids: np.ndarray,
              parents: np.ndarray, roots: np.ndarray, gens: np.ndarray, tick: int, energy: torch.Tensor,
              body_cap: Optional[torch.Tensor] = None, species: Optional[torch.Tensor] = None,
              gen: Optional[torch.Generator] = None):
        d = self.device
        n = slots.numel()
        if n == 0:
            return
        L = self.layout
        self.alive[slots] = True
        self.aid[slots] = torch.as_tensor(aids, device=d)
        self.guid[slots] = torch.as_tensor(guids, device=d)
        self.parent[slots] = torch.as_tensor(parents, device=d)
        self.root[slots] = torch.as_tensor(roots, device=d)
        self.generation[slots] = torch.as_tensor(gens, device=d, dtype=torch.int32)
        self.birth[slots] = int(tick)
        self.species[slots] = species if species is not None else 0
        self.fidelity[slots] = 1
        self.body_slot[slots] = -1
        self.pos[slots] = pos
        self.heading[slots] = torch.rand(n, generator=gen, device=d) * 2 * math.pi
        for f in ["speed", "fatigue", "age", "carried", "last_damage", "mod", "repro_cool", "blocked", "contact"]:
            getattr(self, f)[slots] = 0.0
        self.energy[slots] = energy
        self.integrity[slots] = 1.0
        self.genome[slots] = genomes
        self.x[slots] = 0.0
        self.W[slots] = L.view(genomes, "ctl_W0")
        self.out[slots] = 0.0
        self.emit[slots] = 0.0
        disp = L.view(genomes, "display")
        self.color[slots] = disp[:, :2]
        if body_cap is not None:
            self.body_cap[slots] = body_cap
            self.body_measured[slots] = False
        else:
            # ainda não medido: sem termo de corpo (só rastejo basal) até o desenvolvimento ser medido na física
            self.body_cap[slots] = torch.tensor([0.0, 0.0, 1.0, 0.0, 0.0, 0.0], device=d)
            self.body_measured[slots] = False
        self.stats[slots] = 0.0
        self.signature[slots] = 0.0
        self.culture_type[slots] = -1
        self.last_partner[slots] = 0
        self.intervened[slots] = False
        self.tech[slots] = 0

    def state_dict(self) -> Dict[str, np.ndarray]:
        idx = torch.nonzero(self.alive).flatten()
        out = {"cap": self.cap, "n": int(idx.numel())}
        for f in self.TENSOR_FIELDS:
            out[f] = getattr(self, f)[idx].detach().cpu().numpy()
        return out

    def load_state_dict(self, s: Dict[str, Any]):
        n = int(s["n"])
        if n > self.cap:
            raise ValueError(f"checkpoint tem {n} agentes > capacidade {self.cap}")
        self.alive[:] = False
        idx = torch.arange(n, device=self.device)
        for f in self.TENSOR_FIELDS:
            t = getattr(self, f)
            t[idx] = torch.as_tensor(s[f], device=self.device, dtype=t.dtype)

## 05 — WORLD ENGINE: planeta 2.5D/3D multiescala
Representação híbrida (cada fenômeno com a sua): mapa de tiles em tensores (relevo, água, rios, clima, fertilidade,
recurso abstrato, depósitos de matéria, luz, tempestades, campos de som/marcas, tráfego), pilhas esparsas de matéria
construída (módulo 06), agentes como partículas (módulo 10) e corpos como grafos de Matter Nodes (módulos 07–08).
* Relevo por fBm na GPU; nível do mar por quantil; rios por acumulação de fluxo D8 (rios maiores viram água).
* Clima simplificado: latitude, altitude, estações, ciclo dia/noite, tempestades advectadas pelo vento.
* **Event-driven**: chunks sem agentes e sem tempestade atualizam com período maior (dt acumulado, mesma física).
Tempo artificial: `tick` (unidade da simulação); "dia" e "estação" são ciclos artificiais, não calendário humano.

In [ ]:
# @title 05 — WORLD ENGINE: planeta 2.5D/3D multiescala { display-mode: "form" }
@dataclass
class WorldState:
    H: int
    W: int
    height: torch.Tensor
    is_water: torch.Tensor
    filled: torch.Tensor
    river: torch.Tensor
    temp_base: torch.Tensor
    temperature: torch.Tensor
    moisture: torch.Tensor
    fertility: torch.Tensor
    deposit: torch.Tensor
    resource: torch.Tensor
    rich: torch.Tensor
    light: torch.Tensor
    storm: torch.Tensor
    sound: torch.Tensor
    marks: torch.Tensor
    traffic: torch.Tensor
    struct_h: torch.Tensor
    struct_integrity: torch.Tensor
    enclosure: torch.Tensor
    stored: torch.Tensor
    chunk_last: torch.Tensor
    chunk_active: torch.Tensor
    wind: torch.Tensor
    tick: int = 0
    seed: int = 0
    kind: str = "planet"
    arena: Optional[Dict[str, Any]] = None
    struct_dirty: bool = True
    regen_mult: Optional[torch.Tensor] = None
    flags: Dict[str, Any] = field(default_factory=dict)

    TENSORS = ["height", "is_water", "filled", "river", "temp_base", "temperature", "moisture", "fertility", "deposit",
               "resource", "rich", "light", "storm", "sound", "marks", "traffic", "struct_h", "struct_integrity",
               "enclosure", "stored", "chunk_last", "chunk_active", "wind"]

    @property
    def device(self):
        return self.height.device

    def walkable(self) -> torch.Tensor:
        return ((~self.is_water) | self.filled) & (self.struct_h <= 1)

    def state_dict(self) -> Dict[str, Any]:
        out = {k: getattr(self, k).detach().cpu().numpy() for k in self.TENSORS}
        out.update({"H": self.H, "W": self.W, "tick": self.tick, "seed": self.seed, "kind": self.kind,
                    "arena": self.arena, "flags": self.flags})
        return out

    @staticmethod
    def from_state_dict(s: Dict[str, Any], device: torch.device) -> "WorldState":
        t = {k: torch.as_tensor(s[k], device=device) for k in WorldState.TENSORS}
        ws = WorldState(H=int(s["H"]), W=int(s["W"]), tick=int(s["tick"]), seed=int(s["seed"]), kind=s.get("kind", "planet"),
                        arena=s.get("arena"), flags=dict(s.get("flags", {})), **t)
        ws.struct_dirty = True
        return ws

    def clone(self) -> "WorldState":
        t = {k: getattr(self, k).clone() for k in self.TENSORS}
        return WorldState(H=self.H, W=self.W, tick=self.tick, seed=self.seed, kind=self.kind,
                          arena=copy.deepcopy(self.arena), flags=dict(self.flags), **t)


def fbm(H: int, W: int, octaves: int, gen: torch.Generator, device, base: int = 4, persistence: float = 0.5) -> torch.Tensor:
    out = torch.zeros(H, W, device=device)
    amp, res, tot = 1.0, base, 0.0
    for _ in range(octaves):
        g = torch.rand(1, 1, res + 1, res + 1, generator=gen, device=device)
        out += amp * F.interpolate(g, size=(H, W), mode="bilinear", align_corners=True)[0, 0]
        tot += amp; amp *= persistence; res *= 2
    out = out / tot
    return (out - out.min()) / (out.max() - out.min() + 1e-9)


def flow_accumulation(height: np.ndarray, water: np.ndarray) -> np.ndarray:
    """Acumulação de fluxo D8 (descida mais íngreme), processando do mais alto para o mais baixo."""
    H, W = height.shape
    pad = np.pad(height, 1, mode="edge")
    best = np.full((H, W), np.inf); down = np.full((H, W), -1, dtype=np.int64)
    idx = np.arange(H * W).reshape(H, W)
    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            if dy == 0 and dx == 0:
                continue
            nb = pad[1 + dy:1 + dy + H, 1 + dx:1 + dx + W]
            yy = np.clip(np.arange(H)[:, None] + dy, 0, H - 1); xx = np.clip(np.arange(W)[None, :] + dx, 0, W - 1)
            better = (nb < best) & (nb < height)
            best = np.where(better, nb, best)
            down = np.where(better, idx[yy, xx], down)
    acc = np.ones(H * W)
    order = np.argsort(-height.ravel())
    d = down.ravel(); wflat = water.ravel()
    for i in order:
        j = d[i]
        if j >= 0 and not wflat[i]:
            acc[j] += acc[i]
    return acc.reshape(H, W)


def blur(x: torch.Tensor, k: int = 3) -> torch.Tensor:
    if x.dim() == 2:
        return F.avg_pool2d(x[None, None], k, 1, k // 2, count_include_pad=False)[0, 0]
    return F.avg_pool2d(x[None], k, 1, k // 2, count_include_pad=False)[0]


class WorldEngine:
    """Gera e atualiza mundos (planeta principal, forks e arenas de laboratório) com a mesma física."""

    def __init__(self, cfg: CivConfig, device: torch.device):
        self.cfg, self.device = cfg, device
        self.R_MAX = 1.0
        self.regen_rate = 0.004
        self.sound_decay = 0.55
        self.mark_decay = 0.999

    # ------------------------------------------------------------------ criação
    def _empty(self, H: int, W: int, seed: int, kind: str) -> WorldState:
        d, cfg = self.device, self.cfg
        z = lambda *s, dt=torch.float32: torch.zeros(*s, dtype=dt, device=d)
        Hc, Wc = max(1, math.ceil(H / cfg.chunk)), max(1, math.ceil(W / cfg.chunk))
        return WorldState(H=H, W=W, height=z(H, W), is_water=z(H, W, dt=torch.bool), filled=z(H, W, dt=torch.bool),
                          river=z(H, W), temp_base=z(H, W), temperature=z(H, W), moisture=z(H, W), fertility=z(H, W),
                          deposit=z(H, W), resource=z(H, W), rich=z(H, W, dt=torch.bool), light=z(H, W) + 1.0,
                          storm=z(H, W), sound=z(cfg.n_sound, H, W), marks=z(H, W), traffic=z(H, W),
                          struct_h=z(H, W, dt=torch.int32), struct_integrity=z(H, W), enclosure=z(H, W), stored=z(H, W),
                          chunk_last=z(Hc, Wc, dt=torch.int64), chunk_active=z(Hc, Wc, dt=torch.bool),
                          wind=torch.tensor([0.6, 0.2], device=d), seed=seed, kind=kind)

    def generate(self, seed: int, H: Optional[int] = None, W: Optional[int] = None) -> WorldState:
        cfg, d = self.cfg, self.device
        H, W = H or cfg.world_h, W or cfg.world_w
        g = torch.Generator(device=d); g.manual_seed(int(seed))
        ws = self._empty(H, W, seed, "planet")
        base = fbm(H, W, 6, g, d, base=3)
        ridge = 1 - (2 * fbm(H, W, 5, g, d, base=4) - 1).abs()
        yy = torch.linspace(-1, 1, H, device=d).view(H, 1); xx = torch.linspace(-1, 1, W, device=d).view(1, W)
        falloff = (1 - 0.55 * (xx ** 2 + yy ** 2)).clamp(0, 1)
        h = (0.65 * base + 0.35 * ridge ** 2) * (0.55 + 0.45 * falloff)
        h = (h - h.min()) / (h.max() - h.min() + 1e-9)
        sea = torch.quantile(h.flatten()[:: max(1, (H * W) // 200000)], 0.36)
        water = h < sea
        acc = flow_accumulation(h.cpu().numpy(), water.cpu().numpy())
        river = torch.as_tensor(np.log1p(acc) / max(1e-9, np.log1p(acc).max()), device=d, dtype=torch.float32)
        river = river * (~water)
        big_river = river > torch.quantile(river[~water].flatten()[:: max(1, (H * W) // 200000)], 0.985)
        water = water | big_river
        ws.height = h * cfg.height_scale
        ws.is_water = water
        ws.river = river
        lat = yy.abs().expand(H, W)
        land_h = ((h - sea).clamp(min=0) / (1 - sea + 1e-9))
        ws.temp_base = (1.0 - 0.75 * lat - 0.55 * land_h).clamp(0, 1)
        wetsrc = water.float() + 0.6 * river
        moist = wetsrc
        for _ in range(4):
            moist = blur(moist, 7)
        ws.moisture = (moist / (moist.max() + 1e-9) * 0.8 + 0.2 * fbm(H, W, 4, g, d, base=6)).clamp(0, 1)
        fert = (ws.moisture * (1 - (ws.temp_base - 0.62).abs() * 1.4)).clamp(0, 1) * (~water)
        ws.fertility = fert
        gy, gx = torch.gradient(h)
        slope = torch.sqrt(gy ** 2 + gx ** 2)
        rock = (slope / (slope.max() + 1e-9) * 1.2 + land_h * 0.8).clamp(0, 1)
        ws.deposit = (rock * 8.0).floor() * (~water)
        ws.resource = fert * self.R_MAX
        n_rich = max(2, (H * W) // 900)
        cy = torch.randint(0, H, (n_rich,), generator=g, device=d); cx = torch.randint(0, W, (n_rich,), generator=g, device=d)
        rich = torch.zeros(H, W, dtype=torch.bool, device=d)
        Y, X = torch.meshgrid(torch.arange(H, device=d), torch.arange(W, device=d), indexing="ij")
        for i in range(n_rich):
            rich |= ((Y - cy[i]) ** 2 + (X - cx[i]) ** 2) <= 4
        ws.rich = rich & (fert > 0.25)
        self.update_light(ws)
        ws.temperature = ws.temp_base.clone()
        ws.flags["sea_level"] = float(sea * cfg.height_scale)
        return ws

    def make_arena_world(self, n_arenas: int, arena: int, seed: int, base_resource: float = 0.0,
                         day_night: bool = False, weather: bool = False) -> WorldState:
        """Arenas de laboratório: sub-mundos idênticos separados por paredes altas, num único mundo (vetorização de
        ensaios com a mesma física do planeta)."""
        side = int(math.ceil(math.sqrt(n_arenas)))
        cell = arena + 2
        H = W = side * cell
        ws = self._empty(H, W, seed, "arena")
        ws.height[:] = 2.0
        ws.temp_base[:] = 0.6; ws.temperature[:] = 0.6; ws.moisture[:] = 0.5
        ws.fertility[:] = 0.0; ws.resource[:] = base_resource
        wall = torch.zeros(H, W, dtype=torch.bool, device=self.device)
        for i in range(side + 1):
            a = min(H - 1, i * cell)
            wall[a, :] = True; wall[:, a] = True
            if a + cell - 1 < H:
                wall[a + cell - 1, :] = True; wall[:, a + cell - 1] = True
        ws.struct_h = torch.where(wall, torch.full_like(ws.struct_h, self.cfg.max_stack), ws.struct_h)
        ws.struct_integrity = wall.float() * 1e9
        origins = [(1 + (i // side) * cell, 1 + (i % side) * cell) for i in range(n_arenas)]
        ws.arena = {"n": n_arenas, "size": arena, "cell": cell, "side": side, "origins": origins}
        ws.flags.update({"day_night": day_night, "weather": weather})
        ws.struct_dirty = True
        return ws

    # ------------------------------------------------------------------ amostragem (sensores físicos)
    @staticmethod
    def tile_index(ws: WorldState, pos: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        ix = pos[:, 0].floor().long().clamp(0, ws.W - 1)
        iy = pos[:, 1].floor().long().clamp(0, ws.H - 1)
        return iy, ix

    @staticmethod
    def sample(field_: torch.Tensor, iy: torch.Tensor, ix: torch.Tensor) -> torch.Tensor:
        if field_.dim() == 3:
            return field_[:, iy, ix].T
        return field_[iy, ix]

    @staticmethod
    def gradient_local(field_: torch.Tensor, iy, ix, heading: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        H, W = field_.shape
        gx = field_[iy, (ix + 1).clamp(max=W - 1)] - field_[iy, (ix - 1).clamp(min=0)]
        gy = field_[(iy + 1).clamp(max=H - 1), ix] - field_[(iy - 1).clamp(min=0), ix]
        c, s = torch.cos(heading), torch.sin(heading)
        return gx * c + gy * s, -gx * s + gy * c

    # ------------------------------------------------------------------ dinâmica ambiental
    def update_light(self, ws: WorldState):
        if ws.kind == "arena" and not ws.flags.get("day_night", False):
            ws.light[:] = 1.0
            return
        phase = 2 * math.pi * (ws.tick / self.cfg.day_period)
        xx = torch.linspace(0, 2 * math.pi, ws.W, device=self.device).view(1, ws.W)
        lat = torch.linspace(-1, 1, ws.H, device=self.device).view(ws.H, 1).abs()
        sun = torch.cos(xx - phase) * torch.cos(lat * 1.1)
        ws.light = (sun.clamp(min=0) * (1 - 0.6 * ws.storm)).clamp(0, 1)

    def env_step(self, ws: WorldState, gen: torch.Generator, agent_tiles: Optional[torch.Tensor] = None):
        """Atualização de baixa frequência (env_every ticks). Chunks inativos acumulam dt (event-driven)."""
        cfg, d = self.cfg, self.device
        H, W = ws.H, ws.W
        ch = cfg.chunk
        Hc, Wc = ws.chunk_last.shape
        act = torch.zeros(Hc * Wc, dtype=torch.bool, device=d)
        if agent_tiles is not None and agent_tiles.numel():
            cid = (agent_tiles[0] // ch) * Wc + (agent_tiles[1] // ch)
            act[cid] = True
        storm_c = F.adaptive_max_pool2d(ws.storm[None, None], (Hc, Wc))[0, 0].flatten() > 0.05
        act = (act | storm_c).view(Hc, Wc)
        ws.chunk_active = act
        period = torch.where(act, torch.full_like(ws.chunk_last, cfg.env_every),
                             torch.full_like(ws.chunk_last, cfg.env_every * 4))
        due = (ws.tick - ws.chunk_last) >= period
        dt_c = (ws.tick - ws.chunk_last).float() * due
        ws.chunk_last = torch.where(due, torch.full_like(ws.chunk_last, ws.tick), ws.chunk_last)
        dt = dt_c.repeat_interleave(ch, 0).repeat_interleave(ch, 1)[:H, :W]
        if ws.kind == "planet" or ws.flags.get("weather", False):
            self._weather(ws, gen)
        self.update_light(ws)
        season = math.sin(2 * math.pi * ws.tick / max(1, cfg.season_period))
        hemi = torch.linspace(1, -1, H, device=d).view(H, 1)
        night_cool = 0.12 * (ws.light - 0.5)
        ws.temperature = (ws.temp_base + 0.15 * season * hemi + night_cool - 0.1 * ws.storm).clamp(0, 1)
        regen = self.regen_rate * ws.fertility * (1 + 1.5 * ws.storm) * (1 - ws.resource / self.R_MAX)
        if ws.regen_mult is not None:
            regen = regen * ws.regen_mult
        if ws.kind == "arena" and not ws.flags.get("regen", False):
            regen = regen * 0
        ws.resource = (ws.resource + regen * dt).clamp(0, self.R_MAX)
        ws.marks = ws.marks * (self.mark_decay ** dt)
        ws.traffic = ws.traffic * (0.9995 ** dt)
        weathering = 0.0004 * ws.storm * dt * (ws.struct_h > 0)
        if ws.kind == "planet":
            ws.struct_integrity = ws.struct_integrity - weathering
            collapse = (ws.struct_h > 0) & (ws.struct_integrity <= 0)
            if bool(collapse.any()):
                ws.struct_h = torch.where(collapse, ws.struct_h - 1, ws.struct_h)
                ws.deposit = ws.deposit + collapse.float()            # matéria conservada: bloco vira entulho do tile
                ws.struct_integrity = torch.where(collapse & (ws.struct_h > 0), torch.ones_like(ws.struct_integrity),
                                                  ws.struct_integrity.clamp(min=0))
                ws.struct_dirty = True
        if ws.struct_dirty:
            self.update_enclosure(ws)

    def _weather(self, ws: WorldState, gen: torch.Generator):
        d = self.device
        ws.wind = (ws.wind + 0.05 * torch.randn(2, generator=gen, device=d)).clamp(-1.2, 1.2)
        sx, sy = int(round(float(ws.wind[0]))), int(round(float(ws.wind[1])))
        ws.storm = torch.roll(ws.storm, shifts=(sy, sx), dims=(0, 1)) * 0.985
        if float(torch.rand(1, generator=gen, device=d)) < 0.08 * self.cfg.weather_strength:
            cy = int(torch.randint(0, ws.H, (1,), generator=gen, device=d)); cx = int(torch.randint(0, ws.W, (1,), generator=gen, device=d))
            r = max(3.0, ws.W / 18)
            Y, X = torch.meshgrid(torch.arange(ws.H, device=d), torch.arange(ws.W, device=d), indexing="ij")
            ws.storm = (ws.storm + torch.exp(-((Y - cy) ** 2 + (X - cx) ** 2) / (2 * r * r))).clamp(0, 1)

    def update_enclosure(self, ws: WorldState):
        """Abrigo funcional: tile livre cercado por paredes (pilhas ≥2) reduz exposição a tempestade e frio."""
        walls = (ws.struct_h >= 2).float()[None, None]
        k = torch.ones(1, 1, 3, 3, device=self.device); k[0, 0, 1, 1] = 0
        cnt = F.conv2d(F.pad(walls, (1, 1, 1, 1)), k)[0, 0]
        ws.enclosure = ((cnt - 3) / 4).clamp(0, 1) * (ws.struct_h <= 1)
        ws.struct_dirty = False

    def sound_step(self, ws: WorldState):
        ws.sound = blur(ws.sound, 3) * self.sound_decay

## 06 — MATTER ENGINE: matéria manipulável do ambiente
Não existe `build_house()`. Existe matéria: depósitos naturais por tile, matéria carregada pelos agentes e pilhas
construídas (voxels esparsos sobre o mapa). Ações físicas genéricas: **pegar** (do depósito ou do topo de uma pilha à
frente) e **depositar** (no tile à frente: empilha; sobre água, aterra → tile vira caminhável). Pilhas ≥2 bloqueiam
passagem; tiles cercados por paredes ficam abrigados (reduz exposição — vantagem funcional, não rótulo).
Matéria é conservada: desmoronamento por intempérie devolve o bloco ao depósito do tile (teste de conservação no boot).

In [ ]:
# @title 06 — MATTER ENGINE: matéria manipulável do ambiente { display-mode: "form" }
def rank_within_groups(keys: torch.Tensor) -> torch.Tensor:
    """Posição de cada elemento dentro do seu grupo (mesma chave), em ordem estável. Vetorizado."""
    if keys.numel() == 0:
        return keys
    order = torch.argsort(keys, stable=True)
    sk = keys[order]
    first = torch.ones_like(sk, dtype=torch.bool)
    first[1:] = sk[1:] != sk[:-1]
    idx = torch.arange(sk.numel(), device=keys.device)
    start = torch.cummax(torch.where(first, idx, torch.zeros_like(idx)), 0).values
    rank_sorted = idx - start
    rank = torch.empty_like(rank_sorted)
    rank[order] = rank_sorted
    return rank


class StructureLedger:
    """Registro de blocos (quem construiu, quando, onde, nível) para história, visualização e testes. Capacidade fixa."""

    def __init__(self, cap: int, device: torch.device):
        self.cap, self.device = int(cap), device
        self.tile = torch.full((self.cap,), -1, dtype=torch.int64, device=device)
        self.z = torch.zeros(self.cap, dtype=torch.int32, device=device)
        self.owner = torch.zeros(self.cap, dtype=torch.int64, device=device)
        self.tick = torch.zeros(self.cap, dtype=torch.int64, device=device)
        self.kind = torch.zeros(self.cap, dtype=torch.int8, device=device)       # 0 pilha, 1 aterro
        self.active = torch.zeros(self.cap, dtype=torch.bool, device=device)
        self.ptr = 0
        self.total_placed = 0
        self.total_removed = 0

    def add(self, tiles: torch.Tensor, z: torch.Tensor, owners: torch.Tensor, tick: int, kind: int):
        n = tiles.numel()
        if n == 0:
            return
        idx = (torch.arange(n, device=self.device) + self.ptr) % self.cap
        self.tile[idx] = tiles; self.z[idx] = z.int(); self.owner[idx] = owners; self.tick[idx] = tick
        self.kind[idx] = kind; self.active[idx] = True
        self.ptr = (self.ptr + n) % self.cap
        self.total_placed += n

    def remove_top(self, tiles: torch.Tensor, z: torch.Tensor):
        if tiles.numel() == 0:
            return
        hit = self.active & torch.isin(self.tile, tiles)
        if bool(hit.any()):
            cand = torch.nonzero(hit).flatten()
            key = self.tile[cand] * 64 + self.z[cand].long()
            want = tiles * 64 + z.long()
            sel = cand[torch.isin(key, want)]
            self.active[sel] = False
            self.total_removed += int(sel.numel())

    def state_dict(self):
        idx = torch.nonzero(self.active).flatten()
        return {"cap": self.cap, "ptr": self.ptr, "placed": self.total_placed, "removed": self.total_removed,
                **{k: getattr(self, k)[idx].cpu().numpy() for k in ["tile", "z", "owner", "tick", "kind"]}}

    def load_state_dict(self, s):
        n = len(s["tile"])
        n = min(n, self.cap)
        for k in ["tile", "z", "owner", "tick", "kind"]:
            getattr(self, k)[:n] = torch.as_tensor(s[k][:n], device=self.device)
        self.active[:] = False; self.active[:n] = True
        self.ptr = n % self.cap; self.total_placed = int(s["placed"]); self.total_removed = int(s["removed"])


class MatterEngine:
    def __init__(self, cfg: CivConfig, device: torch.device):
        self.cfg, self.device = cfg, device

    def ahead_tile(self, ws: WorldState, pos: torch.Tensor, heading: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        tx = (pos[:, 0] + torch.cos(heading) * 1.0).floor().long().clamp(0, ws.W - 1)
        ty = (pos[:, 1] + torch.sin(heading) * 1.0).floor().long().clamp(0, ws.H - 1)
        return ty, tx

    def pick(self, ws: WorldState, ledger: StructureLedger, idx: torch.Tensor, pos: torch.Tensor, heading: torch.Tensor,
             carried: torch.Tensor) -> torch.Tensor:
        """Retorna máscara de sucesso (por agente em idx). Do depósito do próprio tile; se vazio, desmonta o topo à frente."""
        if idx.numel() == 0:
            return torch.zeros(0, dtype=torch.bool, device=self.device)
        iy, ix = WorldEngine.tile_index(ws, pos)
        tile = iy * ws.W + ix
        cap_ok = carried < self.cfg.carry_capacity
        rank = rank_within_groups(tile)
        dep = ws.deposit.flatten()[tile]
        from_dep = cap_ok & (rank.float() < dep)
        cnt = torch.zeros(ws.H * ws.W, device=self.device).index_add_(0, tile[from_dep], torch.ones(int(from_dep.sum()), device=self.device))
        ws.deposit = (ws.deposit.flatten() - cnt).clamp(min=0).view(ws.H, ws.W)
        ty, tx = self.ahead_tile(ws, pos, heading)
        atile = ty * ws.W + tx
        want = cap_ok & ~from_dep & (ws.struct_h.flatten()[atile] > 0) & (ws.struct_integrity.flatten()[atile] < 1e8)
        ok_struct = torch.zeros_like(want)
        if bool(want.any()):
            sel = torch.nonzero(want).flatten()
            r2 = rank_within_groups(atile[sel])
            sel = sel[r2 == 0]                                   # um desmonte por tile por tick
            ok_struct[sel] = True
            t = atile[sel]
            hz = ws.struct_h.flatten()[t]
            ledger.remove_top(t, hz)
            sh = ws.struct_h.flatten().clone(); sh[t] = hz - 1
            ws.struct_h = sh.view(ws.H, ws.W)
            ws.struct_dirty = True
        return from_dep | ok_struct

    def place(self, ws: WorldState, ledger: StructureLedger, pos: torch.Tensor, heading: torch.Tensor,
              owners: torch.Tensor, tick: int) -> torch.Tensor:
        n = pos.shape[0]
        if n == 0:
            return torch.zeros(0, dtype=torch.bool, device=self.device)
        ty, tx = self.ahead_tile(ws, pos, heading)
        t = ty * ws.W + tx
        water = ws.is_water.flatten()[t] & ~ws.filled.flatten()[t]
        frozen_wall = ws.struct_integrity.flatten()[t] >= 1e8
        rank = rank_within_groups(t)
        h = ws.struct_h.flatten()[t]
        ok_fill = water & (rank == 0)
        ok_stack = (~water) & ~frozen_wall & (h + rank < self.cfg.max_stack)
        if bool(ok_fill.any()):
            f = ws.filled.flatten().clone(); f[t[ok_fill]] = True
            ws.filled = f.view(ws.H, ws.W)
            ledger.add(t[ok_fill], torch.zeros(int(ok_fill.sum()), device=self.device), owners[ok_fill], tick, 1)
        if bool(ok_stack.any()):
            ts = t[ok_stack]
            add = torch.zeros(ws.H * ws.W, dtype=torch.int32, device=self.device).index_add_(
                0, ts, torch.ones(ts.numel(), dtype=torch.int32, device=self.device))
            levels = h[ok_stack] + rank[ok_stack].int() + 1
            ws.struct_h = (ws.struct_h.flatten() + add).view(ws.H, ws.W)
            si = ws.struct_integrity.flatten().clone(); si[ts] = torch.clamp(si[ts], min=1.0)
            ws.struct_integrity = si.view(ws.H, ws.W)
            ledger.add(ts, levels, owners[ok_stack], tick, 0)
            ws.struct_dirty = True
        return ok_fill | ok_stack

    def total_material(self, ws: WorldState, carried: torch.Tensor) -> float:
        built = ws.struct_h.clamp(min=0) * (ws.struct_integrity < 1e8)
        return float(ws.deposit.sum() + carried.sum() + built.sum() + ws.filled.sum())

## 07 — DEVELOPMENT ENGINE: semente → corpo
O `DevelopmentGenome` guarda **regras locais**, não o corpo final. Em lote (B genomas) numa grade 3D D³:
1. célula-semente com morfógenos iniciais; campo materno de polaridade (eixo do genoma);
2. por passo: difusão entre células ocupadas + reação regulatória (matriz do genoma, tanh) + decaimento;
3. crescimento: sítios vizinhos vazios com sinal de crescimento > limiar (ruído de desenvolvimento determinístico por
   semente) e orçamento de células; a célula nova herda morfógenos dos vizinhos e registra a célula-mãe
   (mapa de destino / linhagem das unidades);
4. leitura: propriedades contínuas da matéria (sigmoide), inervação motora e fase de oscilação.
Nada nasce "osso", "músculo" ou "nervo": regiões podem adquirir propriedades funcionais.

In [ ]:
# @title 07 — DEVELOPMENT ENGINE: semente → corpo { display-mode: "form" }
@dataclass
class Blueprint:
    occ: torch.Tensor          # [B,N] bool
    props: torch.Tensor        # [B,N,P]
    morph: torch.Tensor        # [B,k,N]
    innerv: torch.Tensor       # [B,N,N_MOTOR]
    phase: torch.Tensor        # [B,N]
    osc: torch.Tensor          # [B]
    parent_cell: torch.Tensor  # [B,N] int (-1 = semente/vazio)
    birth_step: torch.Tensor   # [B,N] int
    frames: Optional[List[torch.Tensor]] = None

    def n_nodes(self) -> torch.Tensor:
        return self.occ.sum(1)


class DevelopmentEngine:
    def __init__(self, cfg: CivConfig, layout: GenomeLayout, device: torch.device):
        self.cfg, self.L, self.device = cfg, layout, device
        D = cfg.dev_lattice
        self.D, self.N = D, D ** 3
        g = torch.stack(torch.meshgrid(torch.arange(D), torch.arange(D), torch.arange(D), indexing="ij"), -1).reshape(-1, 3)
        self.grid = g.to(device)                                       # (x,y,z) inteiros
        self.coords = (g.float() / max(1, D - 1) * 2 - 1).to(device)  # [-1,1]^3
        idx = lambda x, y, z: (x * D + y) * D + z
        nb = torch.full((self.N, 6), -1, dtype=torch.long)
        dirs = [(1, 0, 0), (-1, 0, 0), (0, 1, 0), (0, -1, 0), (0, 0, 1), (0, 0, -1)]
        for i, (x, y, z) in enumerate(g.tolist()):
            for j, (dx, dy, dz) in enumerate(dirs):
                a, b, c = x + dx, y + dy, z + dz
                if 0 <= a < D and 0 <= b < D and 0 <= c < D:
                    nb[i, j] = idx(a, b, c)
        self.nb = nb.to(device)
        self.nb_valid = self.nb >= 0
        self.nb_safe = self.nb.clamp(min=0)
        edges, rest = [], []
        offs = [(1, 0, 0), (0, 1, 0), (0, 0, 1), (1, 1, 0), (1, -1, 0), (1, 0, 1), (1, 0, -1), (0, 1, 1), (0, 1, -1)]
        for i, (x, y, z) in enumerate(g.tolist()):
            for dx, dy, dz in offs:
                a, b, c = x + dx, y + dy, z + dz
                if 0 <= a < D and 0 <= b < D and 0 <= c < D:
                    edges.append((i, idx(a, b, c))); rest.append(math.sqrt(dx * dx + dy * dy + dz * dz))
        self.edges = torch.tensor(edges, dtype=torch.long, device=device)
        self.edge_rest = torch.tensor(rest, dtype=torch.float32, device=device)
        c = D // 2
        self.seed_cell = idx(c, c, min(1, D - 1))
        self.cell_idx = torch.arange(self.N, device=device, dtype=torch.float32)

    @staticmethod
    def hnoise(seeds: torch.Tensor, step: int, cells: torch.Tensor) -> torch.Tensor:
        """Ruído determinístico por (semente, passo, célula), independente da composição do lote."""
        s = (seeds % 100003).float().unsqueeze(1)
        x = s * 12.9898 + cells.unsqueeze(0) * 78.233 + step * 37.719
        return torch.frac(torch.sin(x) * 43758.5453) * 2 - 1

    def develop(self, G: torch.Tensor, seeds: torch.Tensor, record: bool = False, noise_scale: float = 1.0) -> Blueprint:
        L, B, N, k = self.L, G.shape[0], self.N, self.cfg.n_morph
        d = self.device
        Wm, bm = L.view(G, "dev_Wm"), L.view(G, "dev_bm")
        decay, diff, seedm = L.view(G, "dev_decay"), L.view(G, "dev_diff"), L.view(G, "dev_seed")
        axis, gw, gthr = L.view(G, "dev_axis"), L.view(G, "dev_gw"), L.view(G, "dev_gthr")
        maxn = (L.view(G, "dev_maxfrac") * N).ceil().clamp(min=1)
        dnoise = L.view(G, "dev_noise") * noise_scale
        occ = torch.zeros(B, N, dtype=torch.bool, device=d); occ[:, self.seed_cell] = True
        m = torch.zeros(B, k, N, device=d); m[:, :, self.seed_cell] = seedm
        pol = axis @ self.coords.T                                    # [B,N] campo materno de polaridade
        parent = torch.full((B, N), -1, dtype=torch.long, device=d)
        birth = torch.full((B, N), -1, dtype=torch.long, device=d); birth[:, self.seed_cell] = 0
        frames = [occ.clone()] if record else None
        valid = self.nb_valid.unsqueeze(0)
        for g in range(self.cfg.dev_steps):
            occf = occ.float()
            nocc = occf[:, self.nb_safe] * valid                     # [B,N,6]
            nbm = m[:, :, self.nb_safe] * nocc.unsqueeze(1)          # [B,k,N,6]
            nbsum = nbm.sum(-1)
            nbcnt = nocc.sum(-1)
            lap = nbsum - nbcnt.unsqueeze(1) * m
            react = torch.tanh(torch.einsum("bij,bjn->bin", Wm, m) + bm.unsqueeze(-1) +
                               torch.cat([pol.unsqueeze(1), torch.zeros(B, k - 1, N, device=d)], 1))
            m = (m + 0.5 * (diff.unsqueeze(-1) * lap + react - decay.unsqueeze(-1) * m)) * occf.unsqueeze(1)
            cand = (~occ) & (nbcnt > 0)
            nbmean = nbsum / nbcnt.clamp(min=1).unsqueeze(1)
            gs = (gw.unsqueeze(-1) * nbmean).sum(1) - gthr.unsqueeze(-1) + dnoise.unsqueeze(-1) * self.hnoise(seeds, g, self.cell_idx)
            allowed = cand & (gs > 0)
            remaining = (maxn - occ.sum(1)).clamp(min=0)
            score = torch.where(allowed, gs, torch.full_like(gs, -1e9))
            order = torch.argsort(score, dim=1, descending=True)
            rank = torch.empty_like(order); rank.scatter_(1, order, torch.arange(N, device=d).expand(B, N))
            chosen = allowed & (rank < remaining.unsqueeze(1))
            if not bool(chosen.any()):
                if record:
                    frames.append(occ.clone())
                continue
            first_nb = torch.argmax(nocc, dim=-1)                    # primeiro vizinho ocupado = célula-mãe
            par = torch.gather(self.nb_safe.unsqueeze(0).expand(B, N, 6), 2, first_nb.unsqueeze(-1)).squeeze(-1)
            parent = torch.where(chosen, par, parent)
            birth = torch.where(chosen, torch.full_like(birth, g + 1), birth)
            m = torch.where(chosen.unsqueeze(1), nbmean, m)
            occ = occ | chosen
            if record:
                frames.append(occ.clone())
        Wp, bp = L.view(G, "body_Wp"), L.view(G, "body_bp")
        props = torch.sigmoid(torch.einsum("bpk,bkn->bnp", Wp, m) + bp.unsqueeze(1)) * occ.unsqueeze(-1)
        innerv = torch.tanh(torch.einsum("bmk,bkn->bnm", L.view(G, "body_innerv"), m)) * occ.unsqueeze(-1)
        phase = math.pi * torch.tanh(m[:, -1] + m[:, 0] * 0.5)
        return Blueprint(occ=occ, props=props, morph=m, innerv=innerv, phase=phase, osc=L.view(G, "body_osc"),
                         parent_cell=parent, birth_step=birth, frames=frames)

    # ------------------------------------------------------------------ métricas de organização (instrumentos externos)
    def grid3(self, occ: torch.Tensor) -> torch.Tensor:
        return occ.view(occ.shape[0], self.D, self.D, self.D)

    def symmetry(self, occ: torch.Tensor) -> Dict[str, torch.Tensor]:
        g = self.grid3(occ).float()
        iou = lambda a, b: (a * b).sum((1, 2, 3)) / ((a + b).clamp(max=1).sum((1, 2, 3)) + 1e-9)
        return {"bilateral_x": iou(g, g.flip(1)), "bilateral_y": iou(g, g.flip(2)),
                "radial_z": iou(g, torch.rot90(g, 1, (1, 2)))}

    def canalization(self, G: torch.Tensor, seeds: torch.Tensor, n: int) -> torch.Tensor:
        """Mesmo genoma, ruídos de desenvolvimento diferentes → IoU médio entre corpos (robustez do desenvolvimento)."""
        occs = [self.develop(G, seeds * 7919 + i + 1).occ.float() for i in range(n)]
        vals = []
        for i in range(n):
            for j in range(i + 1, n):
                a, b = occs[i], occs[j]
                vals.append((a * b).sum(1) / ((a + b).clamp(max=1).sum(1) + 1e-9))
        return torch.stack(vals).mean(0) if vals else torch.ones(G.shape[0], device=self.device)

## 08 — BODY ENGINE: Matter Nodes em sistema massa-mola 3D (lote na GPU)
Cada corpo F2+ é um grafo de **Matter Nodes** (posição, velocidade, massa, raio, propriedades contínuas, estado de
sinal, estado de desenvolvimento, linhagem da célula) ligados por molas (vizinhança 6 + diagonais de face).
Física artificial simplificada: gravidade, contato com o solo por penalidade, atrito de Coulomb suavizado
(coeficiente da adesão do nó), molas com amortecimento, arrasto, atuação por encurtamento do comprimento de repouso
(contração × inervação × comandos motores + oscilador interno com fase do morfógeno). Locomoção não é programada:
emerge (ou não) do padrão de atuação e do atrito.
Sinais internos: estado de sinal por nó propagado pela condutividade (difusão no grafo) e excitado por contato e
atuação — base da câmera de sinais e dos sensores proprioceptivos.

In [ ]:
# @title 08 — BODY ENGINE: Matter Nodes em sistema massa-mola 3D (lote na GPU) { display-mode: "form" }
class BodyBatch:
    FIELDS = ["pos", "vel", "active", "mass", "k_e", "c_e", "rest", "e_on", "gain", "innerv", "phase", "mu", "cond",
              "sens", "signal", "osc", "t", "yaw", "props", "parent_cell", "birth_step"]

    def __init__(self, eng: "BodyEngine", bp: Blueprint):
        d, dev = eng, eng.device
        occ = bp.occ
        B, N = occ.shape
        P = bp.props
        self.B, self.N = B, N
        grid = d.dev.grid.float()
        pos = grid.unsqueeze(0).expand(B, N, 3).clone()
        w = occ.float().unsqueeze(-1)
        com = (pos * w).sum(1) / w.sum(1).clamp(min=1)
        pos[..., 0] -= com[:, 0:1]; pos[..., 1] -= com[:, 1:2]
        zmin = torch.where(occ, pos[..., 2], torch.full_like(pos[..., 2], 1e9)).min(1).values
        pos[..., 2] -= zmin.unsqueeze(1) - 0.05
        self.pos = pos * occ.unsqueeze(-1)
        self.vel = torch.zeros_like(self.pos)
        self.active = occ
        self.mass = (0.4 + 1.2 * P[..., 9]) * occ + (~occ) * 1.0
        e0, e1 = d.dev.edges[:, 0], d.dev.edges[:, 1]
        self.e_on = occ[:, e0] & occ[:, e1]
        stiff = 0.5 * (P[:, e0, 0] + P[:, e1, 0])
        damp = 0.5 * (P[:, e0, 1] + P[:, e1, 1])
        self.k_e = (d.k_base * (0.25 + stiff)) * self.e_on
        self.c_e = (d.c_base * (0.2 + damp)) * self.e_on
        self.rest = d.dev.edge_rest.unsqueeze(0).expand(B, -1).clone()
        self.gain = 0.35 * P[..., 3] * occ
        self.innerv = bp.innerv
        self.phase = bp.phase
        self.mu = (0.15 + 1.1 * P[..., 2]) * occ
        self.cond = P[..., 4] * occ
        self.sens = P[..., 5] * occ
        self.signal = torch.zeros(B, N, device=dev)
        self.osc = bp.osc.clone()
        self.t = torch.zeros(B, device=dev)
        self.yaw = torch.zeros(B, device=dev)
        self.props = P
        self.parent_cell = bp.parent_cell
        self.birth_step = bp.birth_step

    def com(self) -> torch.Tensor:
        w = self.active.float().unsqueeze(-1)
        return (self.pos * w).sum(1) / w.sum(1).clamp(min=1)

    def select(self, idx: torch.Tensor) -> "BodyBatch":
        out = BodyBatch.__new__(BodyBatch)
        for f in self.FIELDS:
            setattr(out, f, getattr(self, f)[idx].clone())
        out.B, out.N = int(idx.numel()), self.N
        return out


class BodyEngine:
    def __init__(self, cfg: CivConfig, dev_engine: DevelopmentEngine, device: torch.device):
        self.cfg, self.dev, self.device = cfg, dev_engine, device
        self.k_base, self.c_base = 40.0, 1.2
        self.g, self.k_ground, self.c_ground, self.drag = 9.8, 400.0, 8.0, 0.05

    def make(self, bp: Blueprint) -> BodyBatch:
        return BodyBatch(self, bp)

    def step(self, bb: BodyBatch, motor: torch.Tensor, dt: Optional[float] = None, substeps: Optional[int] = None) -> Dict[str, torch.Tensor]:
        """motor [B,2] = (forward, turn) do controlador; o oscilador interno é o terceiro canal motor."""
        dt = dt or self.cfg.body_dt
        substeps = substeps or self.cfg.body_substeps
        e0, e1 = self.dev.edges[:, 0], self.dev.edges[:, 1]
        act_n = bb.active.float()
        com0 = bb.com()
        work = torch.zeros(bb.B, device=self.device)
        contact_frac = torch.zeros(bb.B, device=self.device)
        strain_mean = torch.zeros(bb.B, device=self.device)
        for _ in range(substeps):
            osc = torch.sin(2 * math.pi * bb.osc.unsqueeze(1) * bb.t.unsqueeze(1) + bb.phase)
            drive = (bb.innerv[..., 0] * motor[:, 0:1] + bb.innerv[..., 1] * motor[:, 1:2] + bb.innerv[..., 2] * osc)
            a = bb.gain * torch.tanh(drive) * act_n
            d = bb.pos[:, e1] - bb.pos[:, e0]
            Ln = d.norm(dim=-1) + 1e-6
            u = d / Ln.unsqueeze(-1)
            act_e = 0.5 * (a[:, e0] + a[:, e1])
            rest_t = bb.rest * (1 - act_e)
            vrel = ((bb.vel[:, e1] - bb.vel[:, e0]) * u).sum(-1)
            f = (bb.k_e * (Ln - rest_t) + bb.c_e * vrel)
            Fv = torch.zeros_like(bb.pos)
            Fv.index_add_(1, e0, f.unsqueeze(-1) * u)
            Fv.index_add_(1, e1, -f.unsqueeze(-1) * u)
            Fv[..., 2] -= bb.mass * self.g
            pen = (-bb.pos[..., 2]).clamp(min=0)
            incontact = (pen > 0).float()
            Fn = (self.k_ground * pen - self.c_ground * bb.vel[..., 2] * incontact).clamp(min=0)
            Fv[..., 2] += Fn
            vt = bb.vel[..., :2]
            vtn = vt.norm(dim=-1, keepdim=True)
            Fv[..., :2] -= (bb.mu * Fn).unsqueeze(-1) * vt / (vtn + 0.05)
            Fv -= self.drag * bb.vel
            bb.vel = (bb.vel + dt * Fv / bb.mass.unsqueeze(-1)) * act_n.unsqueeze(-1)
            bb.pos = bb.pos + dt * bb.vel
            bb.t = bb.t + dt
            work += ((bb.k_e * (Ln - rest_t)).abs() * act_e.abs()).sum(1) * dt
            contact_frac += (incontact * act_n).sum(1) / act_n.sum(1).clamp(min=1)
            strain_mean += (((Ln - bb.rest) / bb.rest).abs() * bb.e_on).sum(1) / bb.e_on.sum(1).clamp(min=1)
        # sinais internos: excitação por contato e atuação, propagação pela condutividade (difusão no grafo)
        nbsig = bb.signal[:, self.dev.nb_safe] * (self.dev.nb_valid.unsqueeze(0) & bb.active[:, self.dev.nb_safe])
        mean_nb = nbsig.sum(-1) / (self.dev.nb_valid.unsqueeze(0) & bb.active[:, self.dev.nb_safe]).sum(-1).clamp(min=1)
        excite = bb.sens * ((-bb.pos[..., 2]).clamp(min=0) * 10 + a.abs())
        bb.signal = ((bb.signal + bb.cond * (mean_nb - bb.signal) + excite) * 0.85 * act_n).clamp(0, 4)
        com1 = bb.com()
        disp = com1 - com0
        # guinada (yaw) do corpo: rotação média dos nós em torno do COM no plano xy
        rel = (bb.pos[..., :2] - com1[:, None, :2])
        ang = torch.atan2(rel[..., 1], rel[..., 0])
        if not hasattr(bb, "_ang_prev"):
            bb._ang_prev = ang
        dang = torch.atan2(torch.sin(ang - bb._ang_prev), torch.cos(ang - bb._ang_prev))
        r2 = (rel ** 2).sum(-1) * act_n                       # nós perto do COM têm ângulo ruidoso: peso ∝ r²
        dyaw = (dang * r2).sum(1) / r2.sum(1).clamp(min=1e-6)
        bb._ang_prev = ang
        bb.yaw = bb.yaw + dyaw
        # recentra horizontalmente (invariância de translação; o mundo guarda a posição)
        bb.pos[..., 0] -= com1[:, 0:1] * act_n; bb.pos[..., 1] -= com1[:, 1:2] * act_n
        bb._ang_prev = torch.atan2(bb.pos[..., 1] - 0, bb.pos[..., 0] - 0)
        return {"disp": disp, "dyaw": dyaw, "work": work, "contact": contact_frac / substeps,
                "strain": strain_mean / substeps, "com_z": com1[:, 2], "finite": torch.isfinite(bb.pos).all(-1).all(-1)}

    def measure_capability(self, G: torch.Tensor, seeds: torch.Tensor, steps: Optional[int] = None) -> torch.Tensor:
        """Teste padronizado de marcha (comando 'para frente' constante + oscilador) e de giro → resumo usado pelo
        agente leve F1: [velocidade, custo por tick, nº de nós, taxa de giro]. Física real, frequência baixa."""
        return self.measure_blueprint(self.dev.develop(G, seeds), steps)

    def measure_blueprint(self, bp: Blueprint, steps: Optional[int] = None) -> torch.Tensor:
        steps = steps or max(8, self.cfg.probe.body_test_steps // self.cfg.body_substeps)
        bb = self.make(bp)
        fwd = torch.tensor([1.0, 0.0], device=self.device).expand(bb.B, 2)
        net = torch.zeros(bb.B, 2, device=self.device); work = torch.zeros(bb.B, device=self.device)
        sq = torch.zeros(bb.B, device=self.device)
        for i in range(steps):
            r = self.step(bb, fwd)
            if i >= steps // 4:
                # deslocamento no referencial do próprio corpo (desfaz a guinada acumulada)
                c, s_ = torch.cos(-bb.yaw), torch.sin(-bb.yaw)
                d2 = r["disp"][:, :2]
                db = torch.stack([d2[:, 0] * c - d2[:, 1] * s_, d2[:, 0] * s_ + d2[:, 1] * c], 1)
                net += db
                sq += (db ** 2).sum(-1)
                work += r["work"]
        bb2 = self.make(bp)
        trn = torch.tensor([0.0, 1.0], device=self.device).expand(bb2.B, 2)
        for _ in range(steps // 2):
            r2 = self.step(bb2, trn)
        meas = max(1, steps - steps // 4)
        net = torch.nan_to_num(net / meas)
        speed = net.norm(dim=-1)                              # progresso LÍQUIDO por tick (desconta oscilação)
        wobble = torch.nan_to_num((sq / meas - speed ** 2).clamp(min=0).sqrt())   # desvio-padrão por tick (difusão)
        gait = torch.atan2(net[:, 1], net[:, 0])               # direção da marcha relativa ao eixo do corpo
        cost = torch.nan_to_num(work / meas)
        turn = torch.nan_to_num(bb2.yaw.abs() / max(1, steps // 2))
        n = bp.occ.sum(1).float()
        return torch.stack([speed, cost, n, turn, torch.nan_to_num(gait), wobble], 1)

## 09 — SIGNAL ENGINE + LEARNING ENGINE: substrato de sinais com plasticidade evoluível
Não há "neurônios" impostos: há **unidades de sinal** genéricas com recepção (sensores físicos), transmissão
(conexões recorrentes), excitabilidade (viés), atraso/integração (constante de tempo por unidade), oscilação
(dinâmica recorrente), modulação (um canal escalar difuso calculado pela própria rede, com pesos do genoma),
plasticidade local (regra ABCD do genoma: Δw = η·m·(A·pós·pré + B·pré + C·pós + D) − λ(w − w₀)) e poda/criação de
conexões pela mutação estrutural. Nenhuma unidade tem rótulo. Se uma região passa a funcionar como sistema nervoso,
memória ou estado afetivo, isso é **detectado de fora** (módulos 11 e 14).
Separação explícita: GENÉTICO (w₀ e regras no genoma) · DESENVOLVIMENTO (corpo) · APRENDIZADO (w ≠ w₀ durante a vida)
· CULTURA (informação adquirida socialmente, medida em 12).

In [ ]:
# @title 09 — SIGNAL ENGINE + LEARNING ENGINE: substrato de sinais com plasticidade evoluível { display-mode: "form" }
@dataclass
class MindIntervention:
    """Intervenções de laboratório (só em arenas/forks marcados): nunca usadas no mundo principal sem registro."""
    sensor_mask: Optional[torch.Tensor] = None        # [n, I] multiplicador
    sensor_add: Optional[torch.Tensor] = None         # [n, I]
    clamp_dir: Optional[torch.Tensor] = None          # [n, N] direção no espaço de estados
    clamp_value: Optional[torch.Tensor] = None        # [n] deslocamento ao longo da direção
    freeze_plasticity: Optional[torch.Tensor] = None  # [n] bool
    zero_units: Optional[torch.Tensor] = None         # [n, N] bool (ablação de unidades)
    no_noise: bool = False


class LearningEngine:
    """Plasticidade local evoluível (regra ABCD do genoma) — aprendizado DURANTE a vida: w ≠ w₀, nunca altera o genoma."""

    def __init__(self, layout: GenomeLayout):
        self.L = layout

    def update(self, pool: AgentPool, hi: int, G: torch.Tensor, W: torch.Tensor, y: torch.Tensor, y_prev: torch.Tensor,
               mod: torch.Tensor, alive: torch.Tensor, iv: Optional[MindIntervention]) -> None:
        L = self.L
        eta = L.view(G, "plast_eta") * mod
        if iv is not None and iv.freeze_plasticity is not None:
            eta = torch.where(iv.freeze_plasticity, torch.zeros_like(eta), eta)
        abcd = L.view(G, "plast_abcd")
        A, Bc, C, D = abcd[:, 0], abcd[:, 1], abcd[:, 2], abcd[:, 3]
        hebb = (A.view(-1, 1, 1) * y.unsqueeze(2) * y_prev.unsqueeze(1) + Bc.view(-1, 1, 1) * y_prev.unsqueeze(1)
                + C.view(-1, 1, 1) * y.unsqueeze(2) + D.view(-1, 1, 1))
        W0 = L.view(G, "ctl_W0")
        dW = eta.view(-1, 1, 1) * hebb - L.view(G, "plast_decay").view(-1, 1, 1) * (W - W0)
        pool.W[:hi] = torch.where(alive.view(-1, 1, 1), (W + dW).clamp(-3, 3), W)


class SignalEngine:
    """Dinâmica do substrato de sinais (recepção, transmissão recorrente, integração com constante de tempo, oscilação,
    modulação difusa) + chamada ao LearningEngine. Saídas = canais de ação física."""

    def __init__(self, cfg: CivConfig, layout: GenomeLayout, device: torch.device):
        self.cfg, self.L, self.device = cfg, layout, device
        self.learning = LearningEngine(layout)

    def step(self, pool: AgentPool, hi: int, s: torch.Tensor, gen: torch.Generator,
             iv: Optional[MindIntervention] = None) -> torch.Tensor:
        L = self.L
        G = pool.genome[:hi]
        alive = pool.alive[:hi]
        if iv is not None and iv.sensor_mask is not None:
            s = s * iv.sensor_mask
        if iv is not None and iv.sensor_add is not None:
            s = s + iv.sensor_add
        Win, Wout = L.view(G, "ctl_Win"), L.view(G, "ctl_Wout")
        b, bout = L.view(G, "ctl_b"), L.view(G, "ctl_bout")
        tau = torch.exp(L.view(G, "ctl_logtau"))
        x, W = pool.x[:hi], pool.W[:hi]
        y_prev = torch.tanh(x)
        pre = torch.bmm(Win, s.unsqueeze(-1)).squeeze(-1) + torch.bmm(W, y_prev.unsqueeze(-1)).squeeze(-1) + b
        x_new = x + (-x + pre) / tau
        if not (iv is not None and iv.no_noise):
            # exploração motora (motor babbling): ruído interno que decai com a idade, com cronograma do genoma;
            # gerador semeado → reprodutível em SCIENTIFIC_REPLAY
            sig = L.view(G, "babble_sigma") * torch.exp(-pool.age[:hi] / L.view(G, "babble_tau"))
            x_new = x_new + sig.unsqueeze(1) * torch.randn(x_new.shape, generator=gen, device=self.device)
        if iv is not None and iv.clamp_dir is not None:
            proj = (x_new * iv.clamp_dir).sum(1, keepdim=True)
            x_new = x_new + (iv.clamp_value.unsqueeze(1) - proj) * iv.clamp_dir
        if iv is not None and iv.zero_units is not None:
            x_new = torch.where(iv.zero_units, torch.zeros_like(x_new), x_new)
        x_new = x_new.clamp(-6, 6)
        y = torch.tanh(x_new)
        out = torch.bmm(Wout, y.unsqueeze(-1)).squeeze(-1) + bout
        mod = torch.sigmoid((L.view(G, "ctl_wmod") * y).sum(1) + L.view(G, "ctl_bmod"))
        self.learning.update(pool, hi, G, W, y, y_prev, mod, alive, iv)
        pool.x[:hi] = torch.where(alive.view(-1, 1), x_new, x)
        pool.mod[:hi] = torch.where(alive, mod, pool.mod[:hi])
        out = torch.where(alive.view(-1, 1), out, torch.zeros_like(out))
        pool.out[:hi] = out
        return out


MindEngine = SignalEngine      # nome histórico usado pelos demais módulos (substrato de sinais + aprendizado)

## 10 — AGENT ENGINE: indivíduos multiescala (F0 → F4)
Ciclo por tick (tudo vetorizado; vizinhança por hash espacial na GPU, nunca O(N²)):
**sentir** (campos físicos do tile, gradientes no referencial do agente, vizinho mais próximo, interocepção) →
**sinal** (módulo 09) → **agir** (mover, emitir som, exibir cor, consumir, pegar/depositar matéria, marcar, transferir,
tomar, reproduzir, repousar) → **metabolismo** (toda ação custa recurso; exposição a tempestade/frio danifica,
abrigo reduz; fome destrói integridade; reparo custa) → morte funcional (integridade ≤ 0 ou idade > limite do genoma).

Escada de fidelidade: **F0** agregado estatístico por região (demografia logística + migração + reservatório de
genomas) · **F1** agente leve (marcha do corpo resumida por medição física real do genoma) · **F2** corpo
simplificado (grade de desenvolvimento D−1) · **F3** corpo completo (Matter Nodes) · **F4** foco científico (corpo
completo + gravação). Promoção/rebaixamento por novidade, ancestralidade, raridade, pedido do Scientist ou do
administrador (registrado como intervenção). A câmera do usuário **nunca** muda a fidelidade científica.

In [ ]:
# @title 10 — AGENT ENGINE: indivíduos multiescala (F0 → F4) { display-mode: "form" }
@dataclass
class ArenaControl:
    no_repro: bool = False
    no_death: bool = False
    resource_visible: Optional[torch.Tensor] = None   # [hi] bool
    mute: Optional[torch.Tensor] = None               # [hi] bool (emissão sonora bloqueada)
    freeze_move: Optional[torch.Tensor] = None        # [hi] bool
    damage: Optional[torch.Tensor] = None             # [hi] dano extra neste tick
    iv: Optional[MindIntervention] = None
    sound_override: Optional[Callable] = None         # f(sound_in [hi,4]) -> sound_in (troca/embaralha canal)
    energy_floor: Optional[float] = None
    action_block: Optional[torch.Tensor] = None       # [hi, N_ACTIONS] bool (ablação de canais de ação)
    action_force: Optional[torch.Tensor] = None       # [hi, N_ACTIONS] bool (porta forçada aberta — só em arenas)


def spatial_neighbors(pos: torch.Tensor, alive: torch.Tensor, R: float, M: int = 4) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Hash espacial uniforme: vizinho mais próximo, distância e contagem dentro do raio R (até M candidatos por célula)."""
    n = pos.shape[0]
    d = pos.device
    if n == 0:
        z = torch.zeros(0, device=d)
        return z.long(), z, z
    cx = (pos[:, 0] / R).floor().long() + 1
    cy = (pos[:, 1] / R).floor().long() + 1
    ncx = int(cx.max().item()) + 3
    BIG = 1 << 60
    key = torch.where(alive, cy * ncx + cx, torch.full_like(cx, BIG))
    order = torch.argsort(key)
    skey = key[order]
    cands, valid = [], []
    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            k2 = (cy + dy) * ncx + (cx + dx)
            st = torch.searchsorted(skey, k2)
            en = torch.searchsorted(skey, k2, right=True)
            for j in range(M):
                ii = st + j
                cands.append(order[ii.clamp(max=n - 1)])
                valid.append(ii < en)
    C = torch.stack(cands, 1)
    V = torch.stack(valid, 1) & alive.unsqueeze(1) & (C != torch.arange(n, device=d).unsqueeze(1))
    dist = (pos[C] - pos.unsqueeze(1)).norm(dim=-1)
    V = V & (dist <= R)
    dist = torch.where(V, dist, torch.full_like(dist, 1e9))
    best = dist.argmin(1)
    nn = torch.gather(C, 1, best.unsqueeze(1)).squeeze(1)
    nd = torch.gather(dist, 1, best.unsqueeze(1)).squeeze(1)
    has = V.any(1)
    return torch.where(has, nn, torch.full_like(nn, -1)), torch.where(has, nd, torch.full_like(nd, 1e9)), V.sum(1).float()


class BodyRegistry:
    """Corpos ativos (F2+) numa grade fixa de capacidade; slots inativos não têm nós ativos (custo previsível)."""

    def __init__(self, cfg: CivConfig, dev_engine: DevelopmentEngine, body_engine: BodyEngine, cap: int, substeps: int,
                 record: bool = False):
        self.cfg, self.dev, self.body, self.cap, self.substeps, self.record = cfg, dev_engine, body_engine, int(cap), substeps, record
        d = dev_engine.device
        B, N = self.cap, dev_engine.N
        empty = Blueprint(occ=torch.zeros(B, N, dtype=torch.bool, device=d), props=torch.zeros(B, N, N_PROPS, device=d),
                          morph=torch.zeros(B, cfg.n_morph, N, device=d), innerv=torch.zeros(B, N, N_MOTOR, device=d),
                          phase=torch.zeros(B, N, device=d), osc=torch.ones(B, device=d),
                          parent_cell=torch.full((B, N), -1, dtype=torch.long, device=d),
                          birth_step=torch.full((B, N), -1, dtype=torch.long, device=d))
        self.bb = body_engine.make(empty)
        self.agent_slot = torch.full((B,), -1, dtype=torch.long, device=d)
        self.agent_aid = torch.zeros(B, dtype=torch.int64, device=d)
        self.dev_frames: Dict[int, List[np.ndarray]] = {}
        self.trace: Dict[int, "collections.deque"] = {}

    def free(self) -> torch.Tensor:
        return torch.nonzero(self.agent_slot < 0).flatten()

    def attach(self, agent_slots: torch.Tensor, aids: torch.Tensor, genomes: torch.Tensor, seeds: torch.Tensor) -> torch.Tensor:
        free = self.free()[: agent_slots.numel()]
        n = free.numel()
        if n == 0:
            return free
        bp = self.dev.develop(genomes[:n], seeds[:n], record=self.record)
        nb = self.body.make(bp)
        for f in BodyBatch.FIELDS:
            getattr(self.bb, f)[free] = getattr(nb, f)
        if hasattr(self.bb, "_ang_prev"):
            P = self.bb.pos[free]
            self.bb._ang_prev[free] = torch.atan2(P[..., 1], P[..., 0])
        self.agent_slot[free] = agent_slots[:n]
        self.agent_aid[free] = aids[:n]
        if self.record and bp.frames is not None:
            for j, bs in enumerate(free.tolist()):
                self.dev_frames[bs] = [f_[j].cpu().numpy() for f_ in bp.frames]
                self.trace[bs] = collections.deque(maxlen=400)
        return free

    def detach(self, body_slots: torch.Tensor):
        if body_slots.numel() == 0:
            return
        self.bb.active[body_slots] = False
        self.bb.e_on[body_slots] = False
        self.agent_slot[body_slots] = -1
        for bs in body_slots.tolist():
            self.dev_frames.pop(bs, None); self.trace.pop(bs, None)

    def step(self, motor: torch.Tensor) -> Dict[str, torch.Tensor]:
        r = self.body.step(self.bb, motor, substeps=self.substeps)
        if self.record:
            act = torch.nonzero(self.agent_slot >= 0).flatten().tolist()
            if act:
                P = self.bb.pos[act].detach().cpu().numpy().astype(np.float16)
                Sg = self.bb.signal[act].detach().cpu().numpy().astype(np.float16)
                for j, bs in enumerate(act):
                    if bs in self.trace:
                        self.trace[bs].append((P[j], Sg[j]))
        bad = ~r["finite"]
        if bool(bad.any()):
            self.detach(torch.nonzero(bad & (self.agent_slot >= 0)).flatten())
        return r


class RegionAggregates:
    """F0: populações distantes como estado demográfico (contagem, energia média, assinatura cultural média,
    reservatório de genomas reais para desagregação consistente)."""

    def __init__(self, cfg: CivConfig, H: int, W: int, P: int):
        self.cfg, self.rs = cfg, cfg.region_size
        self.Rh, self.Rw = math.ceil(H / self.rs), math.ceil(W / self.rs)
        self.count = np.zeros((self.Rh, self.Rw), np.float64)
        self.energy = np.zeros((self.Rh, self.Rw), np.float64)
        self.signature = np.zeros((self.Rh, self.Rw, N_ACTIONS), np.float32)
        self.reservoir: Dict[int, List[np.ndarray]] = {}
        self.seen: Dict[int, int] = {}
        self.births = 0.0
        self.deaths = 0.0
        self.P = P

    def region_of(self, pos: np.ndarray) -> np.ndarray:
        ry = np.clip((pos[:, 1] // self.rs).astype(int), 0, self.Rh - 1)
        rx = np.clip((pos[:, 0] // self.rs).astype(int), 0, self.Rw - 1)
        return ry * self.Rw + rx

    def absorb(self, regions: np.ndarray, genomes: np.ndarray, energy: np.ndarray, sig: np.ndarray, rng: np.random.Generator):
        for r, g, e, s in zip(regions.tolist(), genomes, energy.tolist(), sig):
            ry, rx = divmod(r, self.Rw)
            c = self.count[ry, rx]
            self.energy[ry, rx] = (self.energy[ry, rx] * c + e) / (c + 1)
            self.signature[ry, rx] = (self.signature[ry, rx] * c + s) / (c + 1)
            self.count[ry, rx] = c + 1
            res = self.reservoir.setdefault(r, [])
            self.seen[r] = self.seen.get(r, 0) + 1
            if len(res) < 8:
                res.append(g.astype(np.float32))
            else:
                j = int(rng.integers(0, self.seen[r]))
                if j < 8:
                    res[j] = g.astype(np.float32)

    def update(self, ws: WorldState, dt: float, rng: np.random.Generator):
        """Demografia logística com capacidade dada pelo recurso real da região; consome recurso do mundo; migração."""
        if self.count.sum() <= 0:
            return
        res = F.adaptive_avg_pool2d(ws.resource[None, None], (self.Rh, self.Rw))[0, 0].cpu().numpy() * self.rs * self.rs
        cap = res * 1.2
        r = 0.002 * dt
        growth = r * self.count * (1 - self.count / np.maximum(cap, 1e-6))
        births = np.clip(growth, 0, None); deaths = np.clip(-growth, 0, None)
        self.births += float(births.sum()); self.deaths += float(deaths.sum())
        self.count = np.clip(self.count + growth, 0, None)
        eat = np.minimum(res, self.count * self.cfg.basal_cost * dt)
        frac = np.where(res > 0, eat / np.maximum(res, 1e-9), 0)
        mult = torch.as_tensor(1 - frac, device=ws.device, dtype=torch.float32)
        up = mult.repeat_interleave(self.rs, 0).repeat_interleave(self.rs, 1)[: ws.H, : ws.W]
        ws.resource = ws.resource * up
        mig = 0.01 * dt * self.count
        flow = np.zeros_like(self.count)
        for dy, dx in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nb_cap = np.roll(np.roll(cap, dy, 0), dx, 1)
            nb_cnt = np.roll(np.roll(self.count, dy, 0), dx, 1)
            push = mig / 4 * (nb_cap > nb_cnt)
            flow -= push
            flow += np.roll(np.roll(push, -dy, 0), -dx, 1)
        self.count = np.clip(self.count + flow, 0, None)

    def total(self) -> float:
        return float(self.count.sum())

    def state_dict(self):
        return {"count": self.count, "energy": self.energy, "signature": self.signature,
                "reservoir": {int(k): np.stack(v) for k, v in self.reservoir.items() if v}, "seen": self.seen,
                "births": self.births, "deaths": self.deaths}

    def load_state_dict(self, s):
        self.count, self.energy, self.signature = np.asarray(s["count"]), np.asarray(s["energy"]), np.asarray(s["signature"])
        self.reservoir = {int(k): [x for x in np.asarray(v)] for k, v in s["reservoir"].items()}
        self.seen = {int(k): int(v) for k, v in s["seen"].items()}
        self.births, self.deaths = float(s["births"]), float(s["deaths"])


class InteractionBuffer:
    """Interações (proximidade, transferência, tomada) acumuladas na GPU e drenadas pelo motor social."""

    def __init__(self, cap_ticks: int = 64):
        self.items: "collections.deque" = collections.deque(maxlen=cap_ticks)

    def push(self, a: torch.Tensor, b: torch.Tensor, kind: int, w: torch.Tensor, tick: int):
        if a.numel():
            self.items.append((a, b, torch.full_like(a, kind), w, tick))

    def drain(self) -> Optional[np.ndarray]:
        if not self.items:
            return None
        A = torch.cat([x[0] for x in self.items]); B = torch.cat([x[1] for x in self.items])
        K = torch.cat([x[2] for x in self.items]); Wt = torch.cat([x[3] for x in self.items])
        T = torch.cat([torch.full_like(x[0], x[4]) for x in self.items])
        self.items.clear()
        return torch.stack([A, B, K, (Wt * 1e4).long(), T], 1).cpu().numpy()


class AgentEngine:
    BASE_TURN = 0.25                      # giro basal (rad/tick) comum a todos os níveis de fidelidade
    MAX_BODY_TURN = 0.6                   # limite do termo de giro vindo do corpo (rad/tick)
    MAX_WOBBLE = 0.4                      # limite do termo difusivo (tiles/tick)

    def __init__(self, cfg: CivConfig, layout: GenomeLayout, world_engine: WorldEngine, matter: MatterEngine,
                 mind: MindEngine, device: torch.device):
        self.cfg, self.L, self.we, self.matter, self.mind, self.device = cfg, layout, world_engine, matter, mind, device
        self.interactions = InteractionBuffer()
        self.tech_events: "collections.deque" = collections.deque(maxlen=256)
        self.tech_counts = torch.zeros(N_TECH, device=device)
        self._bits = torch.arange(N_TECH, device=device, dtype=torch.int32)

    def active_range(self, pool: AgentPool) -> int:
        idx = torch.nonzero(pool.alive).flatten()
        return int(idx.max().item()) + 1 if idx.numel() else 0

    def sense(self, ws: WorldState, pool: AgentPool, hi: int, nn: torch.Tensor, nd: torch.Tensor, cnt: torch.Tensor,
              ctrl: Optional[ArenaControl]) -> torch.Tensor:
        cfg, we = self.cfg, self.we
        pos, hd = pool.pos[:hi], pool.heading[:hi]
        iy, ix = we.tile_index(ws, pos)
        res = we.sample(ws.resource, iy, ix)
        gf, gl = we.gradient_local(ws.resource, iy, ix, hd)
        if ctrl is not None and ctrl.resource_visible is not None:
            vis = ctrl.resource_visible.float()
            res, gf, gl = res * vis, gf * vis, gl * vis
        hf, hl = we.gradient_local(ws.height, iy, ix, hd)
        ty = (pos[:, 1] + torch.sin(hd)).floor().long().clamp(0, ws.H - 1)
        tx = (pos[:, 0] + torch.cos(hd)).floor().long().clamp(0, ws.W - 1)
        water_ahead = (ws.is_water[ty, tx] & ~ws.filled[ty, tx]).float()
        snd = we.sample(ws.sound, iy, ix).clamp(0, 2)
        if ctrl is not None and ctrl.sound_override is not None:
            snd = ctrl.sound_override(snd)
        has = nn >= 0
        nns = nn.clamp(min=0)
        rel = pool.pos[nns] - pos
        dist = rel.norm(dim=-1).clamp(min=1e-6)
        c, s_ = torch.cos(hd), torch.sin(hd)
        prox = (1 - dist / cfg.sense_radius).clamp(0, 1) * has
        nfwd = (rel[:, 0] * c + rel[:, 1] * s_) / dist * prox
        nleft = (-rel[:, 0] * s_ + rel[:, 1] * c) / dist * prox
        cold = (0.35 - we.sample(ws.temperature, iy, ix)).clamp(min=0) * 2
        exposure = (we.sample(ws.storm, iy, ix) + cold).clamp(0, 2)
        S = torch.stack([
            torch.ones(hi, device=self.device), res, gf * 5, gl * 5, we.sample(ws.light, iy, ix),
            we.sample(ws.temperature, iy, ix), hf / 3, hl / 3, water_ahead,
            we.sample(ws.struct_h, iy, ix).float() / cfg.max_stack, we.sample(ws.enclosure, iy, ix),
            we.sample(ws.marks, iy, ix).clamp(0, 2),
            snd[:, 0], snd[:, 1], snd[:, 2], snd[:, 3],
            (cnt / 8).clamp(0, 1), nfwd, nleft, pool.color[nns, 0] * has, pool.color[nns, 1] * has,
            pool.speed[nns] * has, (pool.carried[nns] > 0).float() * has,
            pool.energy[:hi], pool.integrity[:hi], pool.fatigue[:hi], (pool.last_damage[:hi] * 20).clamp(0, 1),
            pool.carried[:hi] / cfg.carry_capacity, pool.contact[:hi] + pool.blocked[:hi], exposure], 1)
        pool.sense_cache[:hi] = S
        return S

    @staticmethod
    def _slice_ctrl(ctrl: ArenaControl, hi: int) -> ArenaControl:
        """Controles de laboratório têm uma linha por slot; o passo usa só [:hi]."""
        cut = lambda t: t[:hi] if isinstance(t, torch.Tensor) and t.dim() >= 1 and t.shape[0] > hi else t
        c = dc_replace(ctrl, **{f.name: cut(getattr(ctrl, f.name)) for f in dc_fields(ctrl)
                                if isinstance(getattr(ctrl, f.name), torch.Tensor)})
        if ctrl.iv is not None:
            c.iv = dc_replace(ctrl.iv, **{f.name: cut(getattr(ctrl.iv, f.name)) for f in dc_fields(ctrl.iv)
                                          if isinstance(getattr(ctrl.iv, f.name), torch.Tensor)})
        return c

    def step(self, ws: WorldState, pool: AgentPool, tick: int, gen: torch.Generator, lineage: Optional[LineageStore],
             ids: IdCounter, ledger: StructureLedger, bodies: Optional[Dict[int, BodyRegistry]] = None,
             ctrl: Optional[ArenaControl] = None, record_interactions: bool = True,
             hi_override: Optional[int] = None) -> Dict[str, float]:
        cfg, d, L = self.cfg, self.device, self.L
        hi = hi_override if hi_override is not None else self.active_range(pool)
        if ctrl is not None:
            ctrl = self._slice_ctrl(ctrl, hi)
        stats = {"alive": 0, "births": 0, "deaths": 0, "eaten": 0.0, "placed": 0, "picked": 0, "given": 0, "taken": 0,
                 "deaths_starvation": 0, "deaths_exposure": 0, "deaths_age": 0, "deaths_conflict": 0}
        if hi == 0:
            self.we.sound_step(ws)
            return stats
        alive = pool.alive[:hi]
        pos, hd = pool.pos[:hi], pool.heading[:hi]
        nn, nd, cnt = spatial_neighbors(pos, alive, cfg.sense_radius)
        S = self.sense(ws, pool, hi, nn, nd, cnt, ctrl)
        out = self.mind.step(pool, hi, S, gen, ctrl.iv if ctrl is not None else None)
        if ctrl is not None and ctrl.action_block is not None:
            # canais contínuos (forward/turn) ablados → 0; portas ablados → fechadas
            fill = torch.full_like(out, -8.0); fill[:, :2] = 0.0
            out = torch.where(ctrl.action_block, fill, out)
        if ctrl is not None and ctrl.action_force is not None:
            out = torch.where(ctrl.action_force, torch.full_like(out, 8.0), out)
        act = torch.sigmoid(out)
        fwd = torch.tanh(out[:, 0]); turn = torch.tanh(out[:, 1])
        rest = act[:, 15] > 0.5
        G = pool.genome[:hi]
        # ---------------- movimento: rastejo basal comum a todos os níveis + termo do corpo
        # F1: termo do corpo = resumo físico medido da marcha do genoma (velocidade e giro sob comando padrão);
        # F2+: termo do corpo = deslocamento e guinada calculados pela física do próprio corpo neste tick.
        # O mesmo ganho corpo→mundo (cfg.body_world_scale, coarse-graining espaço-temporal declarado) vale nos dois.
        cap = pool.body_cap[:hi]
        fwd_eff = torch.where(fwd >= 0, fwd, 0.3 * fwd) * (~rest) * (1 - 0.5 * pool.fatigue[:hi])
        vbody = (cap[:, 0] * cfg.body_world_scale).clamp(max=cfg.max_speed_cap - cfg.min_speed) * fwd_eff
        crawl = cfg.min_speed * fwd_eff
        new_h = hd + turn * (self.BASE_TURN + cap[:, 3].clamp(max=self.MAX_BODY_TURN))
        gait = new_h + cap[:, 4]                         # direção líquida da marcha no referencial do corpo (medida)
        dx = torch.cos(new_h) * crawl + torch.cos(gait) * vbody
        dy = torch.sin(new_h) * crawl + torch.sin(gait) * vbody
        # coarse-graining deriva-difusão: a oscilação da marcha medida na física vira termo difusivo (gerador semeado)
        wob = (cap[:, 5] * cfg.body_world_scale).clamp(max=self.MAX_WOBBLE) * alive
        noise = torch.randn(hi, 2, generator=gen, device=d) * wob.unsqueeze(1)
        dx = dx + noise[:, 0]; dy = dy + noise[:, 1]
        contact = torch.zeros(hi, device=d)
        if bodies:
            for D_, reg in bodies.items():
                act_b = reg.agent_slot >= 0
                if not bool(act_b.any()):
                    continue
                aslot = reg.agent_slot.clamp(min=0, max=hi - 1)
                ok_b = act_b & (reg.agent_slot < hi)
                motor = torch.stack([fwd_eff[aslot], turn[aslot]], 1) * ok_b.unsqueeze(1)
                r = reg.step(motor)
                bi = torch.nonzero(ok_b & (reg.agent_slot >= 0)).flatten()
                if bi.numel():
                    a_i = reg.agent_slot[bi]
                    db = r["disp"][bi, :2] * cfg.body_world_scale
                    mag = db.norm(dim=-1, keepdim=True)
                    db = db * (mag.clamp(max=cfg.max_speed_cap - cfg.min_speed + self.MAX_WOBBLE) / mag.clamp(min=1e-9))
                    h0 = hd[a_i] + turn[a_i] * self.BASE_TURN + r["dyaw"][bi].clamp(-self.MAX_BODY_TURN, self.MAX_BODY_TURN)
                    # eixo x do referencial da física do corpo, no mundo: rumo − guinada acumulada do corpo
                    ang = h0 - reg.bb.yaw[bi]
                    ca, sa = torch.cos(ang), torch.sin(ang)
                    c0, s0 = torch.cos(h0), torch.sin(h0)
                    cr = cfg.min_speed * fwd_eff[a_i]
                    dx[a_i] = db[:, 0] * ca - db[:, 1] * sa + cr * c0
                    dy[a_i] = db[:, 0] * sa + db[:, 1] * ca + cr * s0
                    new_h[a_i] = h0
                    contact[a_i] = r["contact"][bi]
        pool.contact[:hi] = contact
        if ctrl is not None and ctrl.freeze_move is not None:
            dx = dx * (~ctrl.freeze_move); dy = dy * (~ctrl.freeze_move)
        nx = (pos[:, 0] + dx).clamp(0, ws.W - 1e-3); ny = (pos[:, 1] + dy).clamp(0, ws.H - 1e-3)
        tiy, tix = ny.floor().long(), nx.floor().long()
        iy, ix = self.we.tile_index(ws, pos)
        walk = ws.walkable()[tiy, tix]
        climb = (ws.height[tiy, tix] - ws.height[iy, ix]).abs() <= 2.5
        moved = walk & climb & alive
        pool.blocked[:hi] = (~moved & alive & ((dx.abs() + dy.abs()) > 1e-4)).float()
        newpos = torch.where(moved.unsqueeze(1), torch.stack([nx, ny], 1), pos)
        dist_moved = (newpos - pos).norm(dim=-1)
        pool.pos[:hi] = newpos
        pool.heading[:hi] = torch.remainder(new_h, 2 * math.pi)
        pool.speed[:hi] = dist_moved
        iy, ix = self.we.tile_index(ws, newpos)
        tile = iy * ws.W + ix
        tbits = torch.zeros(hi, dtype=torch.int32, device=d)
        tbits |= (moved & ws.filled[iy, ix] & ws.is_water[iy, ix]).int() << 7
        # ---------------- consumo (manchas ricas: extração plena só com ≥2 consumidores simultâneos)
        eat = (act[:, 6] > 0.5) & alive
        gained = torch.zeros(hi, device=d)
        if bool(eat.any()):
            et = tile[eat]
            ccount = torch.zeros(ws.H * ws.W, device=d).index_add_(0, et, torch.ones(et.numel(), device=d))
            resf = ws.resource.flatten()
            demand = ccount[et] * cfg.eat_rate
            frac = torch.clamp(resf[et] / demand.clamp(min=1e-9), max=1.0)
            amount = cfg.eat_rate * frac
            richm = ws.rich.flatten()[et]
            eff = torch.where(richm, torch.where(ccount[et] >= 2, torch.full_like(amount, 1.5), torch.full_like(amount, 0.3)),
                              torch.ones_like(amount))
            used = torch.zeros(ws.H * ws.W, device=d).index_add_(0, et, amount)
            ws.resource = (resf - used).clamp(min=0).view(ws.H, ws.W)
            gained[eat] = amount * eff * cfg.eat_efficiency
            stats["eaten"] = float(amount.sum())
            tbits[eat] |= ((richm & (ccount[et] >= 2)).int() << 2)
        energy = pool.energy[:hi] + gained
        cost = cfg.basal_cost * (1 + cap[:, 2] / 64) + cfg.move_cost * dist_moved * (1 + cap[:, 1].clamp(max=50) / 50)
        # ---------------- matéria
        pick = (act[:, 7] > 0.5) & alive & (pool.carried[:hi] < cfg.carry_capacity)
        if bool(pick.any()):
            pi = torch.nonzero(pick).flatten()
            ok = self.matter.pick(ws, ledger, pi, newpos[pi], pool.heading[pi], pool.carried[pi])
            got = pi[ok]
            pool.carried[got] += 1
            cost[pi] += cfg.action_cost * 2
            stats["picked"] = int(ok.sum())
            pool.stats[got, 3] += 1
        place = (act[:, 8] > 0.5) & alive & (pool.carried[:hi] >= 1)
        if bool(place.any()):
            pl = torch.nonzero(place).flatten()
            aty, atx = self.matter.ahead_tile(ws, newpos[pl], pool.heading[pl])
            wat = ws.is_water[aty, atx] & ~ws.filled[aty, atx]
            ok = self.matter.place(ws, ledger, newpos[pl], pool.heading[pl], pool.aid[pl], tick)
            done = pl[ok]
            tbits[pl] |= (ok & wat).int() << 0
            tbits[pl] |= (ok & ~wat & (ws.struct_h[aty, atx] >= 2)).int() << 1
            pool.carried[done] -= 1
            cost[done] += cfg.build_cost
            stats["placed"] = int(ok.sum())
            pool.stats[done, 2] += 1
        mark = (act[:, 10] > 0.5) & alive
        if bool(mark.any()):
            mk = ws.marks.flatten().index_add_(0, tile[mark], torch.full((int(mark.sum()),), 0.5, device=d))
            ws.marks = mk.clamp(max=4).view(ws.H, ws.W)
            cost += mark.float() * cfg.action_cost
            tbits |= mark.int() << 4
        # ---------------- sinais emitidos (canal físico de som) e exibição de cor
        emit = act[:, 2:6] * alive.unsqueeze(1)
        if ctrl is not None and ctrl.mute is not None:
            emit = emit * (~ctrl.mute).unsqueeze(1)
        pool.emit[:hi] = emit
        loud = emit.sum(1) > 0.05
        if bool(loud.any()):
            snd = ws.sound.view(cfg.n_sound, -1)
            snd.index_add_(1, tile[loud], emit[loud].T.contiguous())
        cost += cfg.signal_cost * (emit ** 2).sum(1)
        pool.color[:hi] = 0.7 * pool.color[:hi] + 0.3 * act[:, 13:15]
        # ---------------- interação social (vizinho mais próximo)
        has = (nn >= 0) & alive
        nns = nn.clamp(min=0)
        conflict_hit = torch.zeros(hi, device=d)
        give = (act[:, 11] > 0.5) & has
        take = (act[:, 12] > 0.5) & has & ~give
        if bool(give.any()):
            gi = torch.nonzero(give).flatten()
            amt = torch.minimum(torch.full((gi.numel(),), cfg.transfer_amount, device=d), energy[gi] * 0.2)
            energy[gi] -= amt
            energy.index_add_(0, nns[gi], amt * 0.95)
            cost[gi] += cfg.action_cost
            pool.stats[gi, 4] += amt
            stats["given"] = int(gi.numel())
            tbits[gi] |= 1 << 3
            if record_interactions:
                self.interactions.push(pool.aid[gi], pool.aid[nns[gi]], 1, amt, tick)
        if bool(take.any()):
            ti = torch.nonzero(take).flatten()
            tgt = nns[ti]
            win = pool.integrity[ti] >= 0.8 * pool.integrity[tgt]
            amt = torch.minimum(torch.full((ti.numel(),), cfg.transfer_amount, device=d), energy[tgt].clamp(min=0) * 0.2) * win
            energy.index_add_(0, tgt, -amt)
            energy[ti] += amt * 0.9
            hit = torch.zeros(hi, device=d).index_add_(0, tgt, 0.01 * win.float())
            pool.integrity[:hi] -= hit
            pool.last_damage[:hi] += hit
            conflict_hit += hit
            cost[ti] += cfg.action_cost * 2
            pool.stats[ti, 5] += amt
            stats["taken"] = int(ti.numel())
            tbits[ti] |= ((amt > 0).int() << 5)
            if record_interactions:
                self.interactions.push(pool.aid[ti], pool.aid[tgt], 2, amt, tick)
        if record_interactions:
            close = has & (nd < cfg.sense_radius * 0.5)
            if bool(close.any()):
                ci = torch.nonzero(close).flatten()
                self.interactions.push(pool.aid[ci], pool.aid[nns[ci]], 0, torch.full((ci.numel(),), 0.05, device=d), tick)
                pool.last_partner[ci] = pool.aid[nns[ci]]
        tbits |= (rest & (ws.enclosure[iy, ix] >= 0.5) & alive).int() << 6
        tbits = tbits * alive.int()
        newb = tbits & ~pool.tech[:hi]
        if bool((newb != 0).any()):
            ni = torch.nonzero(newb).flatten()
            self.tech_events.append((pool.aid[ni].clone(), newb[ni].clone(), pool.last_partner[ni].clone(), tick,
                                     newpos[ni].clone()))
        pool.tech[:hi] |= tbits
        self.tech_counts += ((tbits.unsqueeze(1) >> self._bits) & 1).sum(0).float()
        # ---------------- metabolismo, exposição, abrigo, reparo
        cost += cfg.action_cost * 0.5 * (act[:, 6] > 0.5).float()
        energy = energy - cost * alive
        temp = ws.temperature[iy, ix]
        exposure = ws.storm[iy, ix] + (0.35 - temp).clamp(min=0) * 2
        shelter = ws.enclosure[iy, ix]
        dmg = 0.006 * exposure * (1 - shelter)
        if ctrl is not None and ctrl.damage is not None:
            dmg = dmg + ctrl.damage
        starving = energy < 0
        dmg = dmg + starving.float() * 0.02
        energy = energy.clamp(min=0)
        heal = (energy > 0.5).float() * 0.002
        energy = energy - heal * 0.5
        if ctrl is not None and ctrl.energy_floor is not None:
            energy = energy.clamp(min=ctrl.energy_floor)
        integ = (pool.integrity[:hi] - dmg + heal).clamp(max=1.0)
        pool.last_damage[:hi] = 0.8 * pool.last_damage[:hi] + dmg
        pool.energy[:hi] = torch.where(alive, energy.clamp(max=1.0), pool.energy[:hi])
        pool.integrity[:hi] = torch.where(alive, integ, pool.integrity[:hi])
        fat = pool.fatigue[:hi] + 0.02 * dist_moved + 0.005 * (act[:, 6:13] > 0.5).float().sum(1) - 0.04 * rest.float() - 0.002
        pool.fatigue[:hi] = torch.where(alive, fat.clamp(0, 1), pool.fatigue[:hi])
        pool.age[:hi] += alive.float()
        pool.repro_cool[:hi] = (pool.repro_cool[:hi] - 1).clamp(min=0)
        pool.stats[:hi, 0] += gained
        pool.stats[:hi, 1] += dist_moved
        pool.stats[:hi, 7] += dmg
        # assinatura comportamental (para cultura/especialização): média móvel das ações
        avec = torch.cat([torch.stack([fwd, turn], 1), (act[:, 2:] > 0.5).float()], 1)
        pool.signature[:hi] = torch.where(alive.unsqueeze(1), 0.98 * pool.signature[:hi] + 0.02 * avec, pool.signature[:hi])
        # tráfego (infraestrutura emergente: trilhas usadas repetidamente)
        tr = ws.traffic.flatten().index_add_(0, tile[alive & (dist_moved > 0.01)],
                                              torch.full((int((alive & (dist_moved > 0.01)).sum()),), 0.01, device=d))
        ws.traffic = tr.clamp(max=1).view(ws.H, ws.W)
        # ---------------- reprodução
        if ctrl is None or not ctrl.no_repro:
            stats["births"] = self._reproduce(ws, pool, hi, act, nn, tick, gen, lineage, ids)
        # ---------------- morte funcional
        if ctrl is None or not ctrl.no_death:
            lifespan = L.view(G, "life_span")
            dead_int = alive & (pool.integrity[:hi] <= 0)
            dead_age = alive & (pool.age[:hi] > lifespan)
            dead = dead_int | dead_age
            if bool(dead.any()):
                starv = dead_int & (pool.energy[:hi] <= 0.001)
                confl = dead_int & ~starv & (conflict_hit > 0)
                stats["deaths"] = int(dead.sum())
                stats["deaths_age"] = int((dead_age & ~dead_int).sum())
                stats["deaths_starvation"] = int(starv.sum())
                stats["deaths_conflict"] = int(confl.sum())
                stats["deaths_exposure"] = int((dead_int & ~starv & ~confl).sum())
                self.kill(pool, torch.nonzero(dead).flatten(), bodies, ws)
        self.we.sound_step(ws)
        stats["alive"] = int(pool.alive.sum())
        return stats

    def _reproduce(self, ws, pool, hi, act, nn, tick, gen, lineage, ids) -> int:
        cfg, L, d = self.cfg, self.L, self.device
        G = pool.genome[:hi]
        want = (act[:, 9] > 0.5) & pool.alive[:hi] & (pool.energy[:hi] > L.view(G, "life_repro")) \
            & (pool.age[:hi] > L.view(G, "life_maturity")) & (pool.repro_cool[:hi] <= 0)
        par = torch.nonzero(want).flatten()
        if par.numel() == 0:
            return 0
        free = pool.free_slots(par.numel())
        par = par[: free.numel()]
        n = par.numel()
        if n == 0:
            return 0
        child_g = L.mutate(pool.genome[par], gen)
        mate = nn[par]
        sexual = (mate >= 0)
        if bool(sexual.any()):
            mi = mate[sexual]
            mixed = L.crossover(child_g[sexual], L.mutate(pool.genome[mi], gen), gen)
            child_g[sexual] = mixed
        off = (torch.rand(n, 2, generator=gen, device=d) - 0.5) * 2.0
        cpos = (pool.pos[par] + off)
        cpos[:, 0] = cpos[:, 0].clamp(0, ws.W - 1e-3); cpos[:, 1] = cpos[:, 1].clamp(0, ws.H - 1e-3)
        ciy, cix = cpos[:, 1].floor().long(), cpos[:, 0].floor().long()
        cpos = torch.where(ws.walkable()[ciy, cix].unsqueeze(1), cpos, pool.pos[par])
        e_child = pool.energy[par] * 0.45
        pool.energy[par] = pool.energy[par] * (1 - cfg.repro_cost_fraction)
        pool.repro_cool[par] = 50
        pool.stats[par, 6] += 1
        aids = ids.take(n)
        gens = (pool.generation[par] + 1).cpu().numpy()
        parents = pool.aid[par].cpu().numpy()
        roots = pool.root[par].cpu().numpy()
        pool.spawn(free, child_g, cpos, aids, aids, parents, roots, gens, tick, e_child, body_cap=pool.body_cap[par],
                   species=pool.species[par], gen=gen)
        if lineage is not None:
            lineage.add(aids, parents, tick, "birth", gens)
        return n

    def kill(self, pool: AgentPool, slots: torch.Tensor, bodies: Optional[Dict[int, BodyRegistry]] = None,
             ws: Optional[WorldState] = None):
        """Morte funcional (ou absorção em F0): a matéria carregada cai no tile (conservação de matéria)."""
        if slots.numel() == 0:
            return
        if ws is not None:
            carried = pool.carried[slots]
            if bool((carried > 0).any()):
                iy, ix = WorldEngine.tile_index(ws, pool.pos[slots])
                dep = ws.deposit.flatten().index_add_(0, iy * ws.W + ix, carried)
                ws.deposit = dep.view(ws.H, ws.W)
            pool.carried[slots] = 0
        bs = pool.body_slot[slots]
        if bodies and bool((bs >= 0).any()):
            for D_, reg in bodies.items():
                hit = torch.nonzero(torch.isin(reg.agent_slot, slots)).flatten()
                reg.detach(hit)
        pool.alive[slots] = False
        pool.body_slot[slots] = -1
        pool.fidelity[slots] = 0

    def spawn_founders(self, ws: WorldState, pool: AgentPool, n: int, gen: torch.Generator, ids: IdCounter, tick: int,
                       lineage: Optional[LineageStore], genomes: Optional[torch.Tensor] = None, origin: str = "founder",
                       region: Optional[Tuple[int, int, int, int]] = None, energy: float = 0.8) -> torch.Tensor:
        d = self.device
        free = pool.free_slots(n)
        n = free.numel()
        if n == 0:
            return free
        walk = ws.walkable() & (ws.fertility > 0.05) if ws.kind == "planet" else ws.walkable()
        if region is not None:
            y0, x0, y1, x1 = region
            m = torch.zeros_like(walk); m[y0:y1, x0:x1] = True
            walk = walk & m
        cand = torch.nonzero(walk)
        if cand.shape[0] == 0:
            cand = torch.nonzero(ws.walkable())
        pick = cand[torch.randint(0, cand.shape[0], (n,), generator=gen, device=d)]
        pos = torch.stack([pick[:, 1].float() + torch.rand(n, generator=gen, device=d),
                           pick[:, 0].float() + torch.rand(n, generator=gen, device=d)], 1)
        G = genomes[:n] if genomes is not None else self.L.random(n, gen)
        aids = ids.take(n)
        zeros = np.zeros(n, np.int64)
        pool.spawn(free, G, pos, aids, aids, zeros, aids, zeros, tick, torch.full((n,), energy, device=d), gen=gen)
        if lineage is not None:
            lineage.add(aids, zeros, tick, origin, zeros)
        return free

## 10b — LAB ARENAS: ensaios padronizados com a mesma física do planeta
Os probes e a evolução QD não usam um "simulador de teste" diferente: usam o **mesmo** `AgentEngine`, o mesmo
`WorldEngine` e o mesmo `MatterEngine`, em sub-mundos separados por paredes. Layouts são determinísticos por
(semente, arena) → condições pareadas (tratamento × controle) veem exatamente o mesmo mundo e as mesmas posições
iniciais. Execução em **gerador** (`run_gen`): o laço vivo intercala fatias de ciência com o mundo principal, então o
observatório 3D nunca congela durante experimentos.

In [ ]:
# @title 10b — LAB ARENAS: ensaios padronizados com a mesma física do planeta { display-mode: "form" }
@dataclass
class ArenaSpec:
    task: str = "forage"
    n_patches: int = 3
    patch_value: float = 1.0
    patch_radius: int = 1
    regen: bool = False
    rich: int = 0                       # manchas cooperativas (extração plena só com ≥2 consumidores)
    water_barrier: bool = False         # faixa de água entre início e recurso (exige aterro para cruzar)
    deposit: float = 0.0                # matéria por tile numa zona de depósito perto do início
    storm: float = 0.0                  # exposição constante
    cold: float = 0.0                   # 0..1 reduz a temperatura
    day_night: bool = False
    hidden_side: bool = False           # recurso num lado sorteado por arena (tarefas de pista/memória)
    resource_floor: float = 0.0         # recurso basal disperso
    spawn_side: str = "center"          # center | left


@dataclass
class Trial:
    ws: WorldState
    pool: AgentPool
    ledger: StructureLedger
    gen: torch.Generator
    spec: ArenaSpec
    n_arenas: int
    per_arena: int
    arena_of: torch.Tensor              # [n] índice da arena de cada agente
    origins: torch.Tensor               # [n_arenas, 2] (y0, x0)
    label: torch.Tensor                 # [n_arenas] rótulo oculto (ex.: lado do recurso)
    seed: int
    tick: int = 0
    patch_tiles: Optional[torch.Tensor] = None   # [n_arenas, k, 2] (y, x) centros das manchas
    barrier_x: int = -1

    @property
    def n(self) -> int:
        return self.n_arenas * self.per_arena


class ArenaRunner:
    LAB_ID_BASE = 10 ** 12              # IDs de laboratório nunca colidem com IDs do mundo

    def __init__(self, cfg: CivConfig, layout: GenomeLayout, we: WorldEngine, matter: MatterEngine, mind: MindEngine,
                 dev: DevelopmentEngine, body: BodyEngine, device: torch.device):
        self.cfg, self.L, self.we, self.matter, self.mind, self.dev, self.body, self.device = \
            cfg, layout, we, matter, mind, dev, body, device
        self.engine = AgentEngine(cfg, layout, we, matter, mind, device)
        self.ids = IdCounter(self.LAB_ID_BASE)
        self.cap_cache: "collections.OrderedDict[str, torch.Tensor]" = collections.OrderedDict()
        self.current: Optional[Trial] = None
        self.current_name = ""
        self.ticks_run = 0

    # ------------------------------------------------------------------ capacidade física (F1) com cache
    def capability(self, G: torch.Tensor, seeds: Optional[torch.Tensor] = None) -> torch.Tensor:
        keys = [hashlib.sha1(g.detach().cpu().numpy().tobytes()).hexdigest()[:20] for g in G]
        miss = [i for i, k in enumerate(keys) if k not in self.cap_cache]
        if miss:
            mi = torch.tensor(miss, device=self.device)
            sd = seeds[mi] if seeds is not None else torch.tensor([int(keys[i][:8], 16) % 100003 for i in miss], device=self.device)
            caps = self.body.measure_capability(G[mi], sd)
            for j, i in enumerate(miss):
                self.cap_cache[keys[i]] = caps[j].detach()
            while len(self.cap_cache) > 20000:
                self.cap_cache.popitem(last=False)
        return torch.stack([self.cap_cache[k] for k in keys]).to(self.device)

    # ------------------------------------------------------------------ construção do ensaio
    def build(self, genomes: torch.Tensor, spec: ArenaSpec, seed: int, per_arena: int = 1,
              n_arenas: Optional[int] = None, body_cap: Optional[torch.Tensor] = None, energy: float = 0.8,
              W_override: Optional[torch.Tensor] = None, x_override: Optional[torch.Tensor] = None,
              label: Optional[torch.Tensor] = None) -> Trial:
        """genomes: [n_arenas*per_arena, P] (agentes da arena a ficam em [a*per_arena:(a+1)*per_arena])."""
        cfg, d = self.cfg, self.device
        n = genomes.shape[0]
        n_arenas = n_arenas or max(1, n // per_arena)
        assert n == n_arenas * per_arena, "genomes deve ter n_arenas*per_arena linhas"
        S = cfg.probe.arena_size
        ws = self.we.make_arena_world(n_arenas, S, seed, base_resource=spec.resource_floor, day_night=spec.day_night,
                                      weather=False)
        ws.flags["regen"] = spec.regen
        if spec.regen:
            ws.fertility = torch.where(ws.struct_h == 0, torch.full_like(ws.fertility, 0.4), ws.fertility)
        rng = np.random.default_rng(derive_seed("arena_layout", seed) % (2 ** 32))
        origins = torch.tensor(ws.arena["origins"], device=d, dtype=torch.long)
        c = S // 2
        lab = label.clone().to(d) if label is not None else torch.as_tensor(rng.integers(0, 2, n_arenas), device=d)
        patches = []
        barrier_x = -1
        if spec.water_barrier:
            barrier_x = c + 1
        for a in range(n_arenas):
            oy, ox = ws.arena["origins"][a]
            pts = []
            if spec.hidden_side:
                side = int(lab[a])
                px = c - 5 if side == 0 else c + 5
                pts.append((int(rng.integers(c - 3, c + 4)), px))
            else:
                for _ in range(spec.n_patches):
                    for _try in range(50):
                        py, px = int(rng.integers(1, S - 1)), int(rng.integers(1, S - 1))
                        if spec.water_barrier and px <= barrier_x + 2:
                            continue
                        if (py - c) ** 2 + (px - c) ** 2 >= 9:
                            break
                    pts.append((py, px))
            patches.append(pts)
            for (py, px) in pts:
                y0, y1 = oy + max(0, py - spec.patch_radius), oy + min(S, py + spec.patch_radius + 1)
                x0, x1 = ox + max(0, px - spec.patch_radius), ox + min(S, px + spec.patch_radius + 1)
                ws.resource[y0:y1, x0:x1] = spec.patch_value
                if spec.regen:
                    ws.fertility[y0:y1, x0:x1] = 1.0
            for _ in range(spec.rich):
                py, px = int(rng.integers(2, S - 2)), int(rng.integers(2, S - 2))
                ws.rich[oy + py - 1:oy + py + 2, ox + px - 1:ox + px + 2] = True
                ws.resource[oy + py - 1:oy + py + 2, ox + px - 1:ox + px + 2] = spec.patch_value
            if spec.water_barrier:
                ws.is_water[oy:oy + S, ox + barrier_x:ox + barrier_x + 2] = True
            if spec.deposit > 0:
                sx = 2 if spec.spawn_side == "left" else c - 2
                ws.deposit[oy + c - 2:oy + c + 3, ox + max(1, sx - 1):ox + sx + 2] = spec.deposit
        if spec.storm > 0:
            ws.storm = (ws.struct_h < self.cfg.max_stack).float() * spec.storm
        if spec.cold > 0:
            ws.temp_base = ws.temp_base - 0.5 * spec.cold
            ws.temperature = ws.temp_base.clone()
        self.we.update_enclosure(ws)
        maxk = max(1, max(len(p) for p in patches))
        pt = torch.full((n_arenas, maxk, 2), -1, dtype=torch.long, device=d)
        for a, pts in enumerate(patches):
            for j, (py, px) in enumerate(pts):
                pt[a, j, 0], pt[a, j, 1] = py, px
        pool = AgentPool(n, self.L, cfg, d)
        gen = torch.Generator(device=d); gen.manual_seed(derive_seed("arena_run", seed) % (2 ** 62))
        arena_of = torch.arange(n_arenas, device=d).repeat_interleave(per_arena)
        sy = c if spec.spawn_side == "center" else c
        sx = c if spec.spawn_side == "center" else 2
        off_rng = np.random.default_rng(derive_seed("arena_spawn", seed) % (2 ** 32))
        offs = torch.as_tensor(off_rng.uniform(-1.2, 1.2, size=(n, 2)), device=d, dtype=torch.float32)
        base = origins[arena_of].float()
        pos = torch.stack([base[:, 1] + sx + 0.5 + offs[:, 0], base[:, 0] + sy + 0.5 + offs[:, 1]], 1)
        if body_cap is None:
            body_cap = self.capability(genomes)
        aids = self.ids.take(n)
        zeros = np.zeros(n, np.int64)
        pool.spawn(torch.arange(n, device=d), genomes, pos, aids, aids, zeros, aids, zeros, 0,
                   torch.full((n,), energy, device=d), body_cap=body_cap, gen=gen)
        pool.body_measured[:] = True
        if W_override is not None:
            pool.W[:n] = W_override
        if x_override is not None:
            pool.x[:n] = x_override
        ledger = StructureLedger(max(256, n * 8), d)
        return Trial(ws=ws, pool=pool, ledger=ledger, gen=gen, spec=spec, n_arenas=n_arenas, per_arena=per_arena,
                     arena_of=arena_of, origins=origins, label=lab, seed=seed, patch_tiles=pt, barrier_x=barrier_x)

    # ------------------------------------------------------------------ execução
    def run_gen(self, trial: Trial, ticks: int, ctrl_fn: Optional[Callable[[int, Trial], Optional[ArenaControl]]] = None,
                record: Sequence[str] = ("energy", "pos"), chunk: int = 16, every: int = 1,
                bodies: Optional[Dict[int, BodyRegistry]] = None, name: str = "") -> Generator[None, None, Dict[str, torch.Tensor]]:
        """Executa `ticks` passos; grava campos por tick. Rende (yield) a cada `chunk` ticks."""
        pool, ws = trial.pool, trial.ws
        rec: Dict[str, List[torch.Tensor]] = {k: [] for k in record}
        prev_gain = pool.stats[:, 0].clone()
        prev_place = pool.stats[:, 2].clone()
        self.current, self.current_name = trial, name
        for t in range(ticks):
            ctrl = ctrl_fn(trial.tick, trial) if ctrl_fn is not None else None
            if ctrl is None:
                ctrl = ArenaControl(no_repro=True)
            ctrl.no_repro = True
            self.engine.step(ws, pool, trial.tick, trial.gen, None, self.ids, trial.ledger, bodies=bodies, ctrl=ctrl,
                             record_interactions=False, hi_override=trial.n)
            if trial.tick % self.cfg.env_every == 0:
                iy, ix = WorldEngine.tile_index(ws, pool.pos[pool.alive])
                ws.tick = trial.tick
                self.we.env_step(ws, trial.gen, torch.stack([iy, ix]))
            if t % every == 0:
                for k in record:
                    if k == "gain":
                        g = pool.stats[:, 0] - prev_gain
                        rec[k].append(g.clone())
                    elif k == "placed":
                        rec[k].append(pool.stats[:, 2] - prev_place)
                    elif k == "alive":
                        rec[k].append(pool.alive.clone())
                    else:
                        rec[k].append(getattr(pool, k).clone())
            prev_gain = pool.stats[:, 0].clone()
            prev_place = pool.stats[:, 2].clone()
            trial.tick += 1
            self.ticks_run += 1
            if (t + 1) % chunk == 0:
                yield
        out = {k: torch.stack(v) if v else torch.zeros(0) for k, v in rec.items()}
        return out

    def run(self, trial: Trial, ticks: int, **kw) -> Dict[str, torch.Tensor]:
        return drive(self.run_gen(trial, ticks, **kw))

    # ------------------------------------------------------------------ métricas de ensaio
    @staticmethod
    def crossed(trial: Trial) -> torch.Tensor:
        if trial.barrier_x < 0:
            return torch.zeros(trial.n, dtype=torch.bool, device=trial.pool.device)
        lx = trial.pool.pos[:, 0] - trial.origins[trial.arena_of, 1].float()
        return lx > trial.barrier_x + 2

    def view(self) -> Optional[Dict[str, Any]]:
        """Visão compacta do ensaio em curso para a 'câmera do cientista' (somente leitura)."""
        tr = self.current
        if tr is None:
            return None
        k = min(tr.n, 512)
        p = tr.pool
        return {"name": self.current_name, "tick": tr.tick, "n_arenas": tr.n_arenas, "size": self.cfg.probe.arena_size,
                "H": tr.ws.H, "W": tr.ws.W, "pos": p.pos[:k].detach().cpu().numpy().astype(np.float32),
                "alive": p.alive[:k].cpu().numpy(), "energy": p.energy[:k].cpu().numpy().astype(np.float32),
                "struct": torch.nonzero(tr.ws.struct_h[: min(tr.ws.H, 256), : min(tr.ws.W, 256)] > 0)[:4000].cpu().numpy(),
                "resource": F.avg_pool2d(tr.ws.resource[None, None], 2)[0, 0].cpu().numpy().astype(np.float16)
                if tr.ws.H >= 2 else None}


def drive(g: Generator):
    """Executa um gerador até o fim e devolve o valor de retorno."""
    try:
        while True:
            next(g)
    except StopIteration as e:
        return e.value

## 10c — Base comum dos probes científicos (congelada)
Todo probe devolve um `ProbeResult` com: estatística (p-valor, tamanho de efeito, IC bootstrap), controles
negativos/positivos usados, sementes (de **holdout**: espaço de sementes nunca usado pela evolução), hash do genoma,
hash das versões dos motores e dos thresholds. Probes são geradores: rendem entre blocos de ticks da arena.

In [ ]:
# @title 10c — Base comum dos probes científicos (congelada) { display-mode: "form" }
SENSOR_RESOURCE = [1, 2, 3]
SENSOR_EXTERO = list(range(1, 23))
SENSOR_INTERO = [23, 24, 25, 26]
SENSOR_SOUND = [12, 13, 14, 15]
ACTION_PLACE, ACTION_PICK, ACTION_TAKE, ACTION_GIVE = 8, 7, 12, 11


@dataclass
class ProbeResult:
    level: str
    name: str
    passed: bool
    p: float
    effect: float
    ci: Tuple[float, float]
    n: int
    metrics: Dict[str, Any]
    controls: Dict[str, Any]
    seeds: List[int]
    genome_hash: str
    claim: str
    duration: float
    ts: str = field(default_factory=iso_now)
    versions: str = field(default_factory=lambda: ENGINE_VERSIONS_HASH)
    thresholds: str = ""
    notes: str = ""
    replay: Optional[Dict[str, Any]] = None

    def public(self) -> Dict[str, Any]:
        d = {k: v for k, v in asdict(self).items() if k != "replay"}
        d["has_replay"] = self.replay is not None
        return to_jsonable(d)


def genome_hash(g: torch.Tensor) -> str:
    return hashlib.sha256(g.detach().cpu().numpy().astype(np.float32).tobytes()).hexdigest()[:24]


def diff_ci(a, b, n_boot: int = 1000, seed: int = 0) -> Tuple[float, float]:
    a = np.asarray(a, np.float64); b = np.asarray(b, np.float64)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if a.size < 2 or b.size < 2:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    da = a[rng.integers(0, a.size, (n_boot, a.size))].mean(1) - b[rng.integers(0, b.size, (n_boot, b.size))].mean(1)
    return float(np.quantile(da, 0.025)), float(np.quantile(da, 0.975))


def compare_greater(a, b, th: Thresholds, min_effect: Optional[float] = None) -> Dict[str, Any]:
    """a > b? Mann–Whitney unilateral + d de Cohen + IC bootstrap da diferença de médias."""
    a = np.asarray(a, np.float64); b = np.asarray(b, np.float64)
    p = mann_whitney_greater(a, b)
    d = cohens_d(a, b)
    ci = diff_ci(a, b)
    me = th.min_effect if min_effect is None else min_effect
    ok = bool(np.isfinite(p) and p < th.alpha and np.isfinite(d) and d >= me)
    return {"p": float(p) if np.isfinite(p) else 1.0, "d": float(d) if np.isfinite(d) else 0.0, "ci": ci,
            "mean_a": float(np.nanmean(a)) if a.size else float("nan"), "mean_b": float(np.nanmean(b)) if b.size else float("nan"),
            "passed": ok}


def memoryless(L: GenomeLayout, G: torch.Tensor) -> torch.Tensor:
    """Controle negativo de memória: sem recorrência, sem integração temporal (τ=1) e sem plasticidade."""
    G = G.clone()
    for name, val in [("ctl_W0", 0.0), ("ctl_logtau", 0.0), ("plast_eta", 0.0)]:
        a, b = L.slices[name]
        G[..., a:b] = val
    return L.clamp(G)


def passive_body(L: GenomeLayout, G: torch.Tensor) -> torch.Tensor:
    """Controle negativo de corpo: mesma morfologia, sem inervação motora (nenhuma atuação)."""
    G = G.clone()
    a, b = L.slices["body_innerv"]
    G[..., a:b] = 0.0
    return G


def probe_seeds(level: str, key: str, k: int, holdout: bool = True) -> List[int]:
    space = "holdout" if holdout else "screen"
    return [derive_seed(space, level, key, i) for i in range(k)]


def sensor_mask(n: int, idx: Sequence[int], device) -> torch.Tensor:
    m = torch.ones(n, N_SENSORS, device=device)
    m[:, list(idx)] = 0.0
    return m


def make_replay(trial: Trial, rec: Dict[str, torch.Tensor], k: int = 8, every: int = 1) -> Dict[str, Any]:
    """Replay científico compacto (derivado da execução real): trajetórias das primeiras arenas + layout."""
    pos = rec.get("pos")
    if pos is None or pos.numel() == 0:
        return {}
    k = min(k, trial.n)
    arenas = sorted(set(trial.arena_of[:k].tolist()))
    S = trial.ws.arena["size"] if trial.ws.arena else 16
    lay = []
    for a in arenas:
        oy, ox = trial.ws.arena["origins"][a]
        sub = lambda t: t[oy:oy + S, ox:ox + S].cpu().numpy()
        lay.append({"arena": a, "origin": [int(ox), int(oy)], "water": np.nonzero(sub(trial.ws.is_water & ~trial.ws.filled)),
                    "filled": np.nonzero(sub(trial.ws.filled)), "struct": np.nonzero(sub(trial.ws.struct_h > 0)),
                    "rich": np.nonzero(sub(trial.ws.rich)), "patches": trial.patch_tiles[a].cpu().numpy().tolist()})
    P = pos[::every, :k].cpu().numpy().astype(np.float32)
    org = np.asarray([trial.ws.arena["origins"][int(a)] for a in trial.arena_of[:k].tolist()], np.float32)
    P = P - org[None, :, ::-1]                                    # coordenadas locais da arena (x, y)
    return {"size": int(S), "pos": P.astype(np.float16), "arena_of": trial.arena_of[:k].cpu().numpy(),
            "layout": [{kk: (np.stack(v, 1).astype(np.int16).tolist() if isinstance(v, tuple) else v) for kk, v in L_.items()}
                       for L_ in lay], "ticks": int(P.shape[0]), "every": every}


def result(level: str, name: str, passed: bool, stat: Dict[str, Any], n: int, metrics: Dict[str, Any],
           controls: Dict[str, Any], seeds: List[int], G: torch.Tensor, t0: float, th_hash: str,
           claim_pass: str = "TESTED", notes: str = "", replay=None) -> ProbeResult:
    return ProbeResult(level=level, name=name, passed=bool(passed), p=float(stat.get("p", 1.0)),
                       effect=float(stat.get("d", 0.0)), ci=tuple(stat.get("ci", (float("nan"), float("nan")))), n=int(n),
                       metrics=to_jsonable(metrics), controls=to_jsonable(controls), seeds=list(seeds),
                       genome_hash=genome_hash(G), claim=claim_pass if passed else "NOT YET VALIDATED",
                       duration=time.time() - t0, thresholds=th_hash, notes=notes, replay=replay)


def fisher_greater(k1: int, n1: int, k2: int, n2: int) -> float:
    """Teste exato de Fisher unilateral: proporção 1 > proporção 2 (soma hipergeométrica)."""
    K, N = k1 + k2, n1 + n2
    if n1 == 0 or n2 == 0:
        return 1.0
    def hyp(k):
        return math.comb(K, k) * math.comb(N - K, n1 - k) / math.comb(N, n1)
    return float(min(1.0, sum(hyp(k) for k in range(k1, min(K, n1) + 1))))


def ridge_r2_cv(X: np.ndarray, Y: np.ndarray, groups: np.ndarray, lam: float = 1.0) -> float:
    """R² médio (validação cruzada por grupo = arena; sem vazamento temporal entre treino e teste)."""
    ug = np.unique(groups)
    k = min(4, len(ug))
    if k < 2:
        return 0.0
    folds = {g: i % k for i, g in enumerate(ug)}
    f = np.asarray([folds[g] for g in groups])
    pred = np.zeros_like(Y)
    for i in range(k):
        tr, te = f != i, f == i
        mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-6
        Xa = np.hstack([(X[tr] - mu) / sd, np.ones((tr.sum(), 1))]); Xb = np.hstack([(X[te] - mu) / sd, np.ones((te.sum(), 1))])
        Wt = np.linalg.solve(Xa.T @ Xa + lam * np.eye(Xa.shape[1]), Xa.T @ Y[tr])
        pred[te] = Xb @ Wt
    ss_res = ((Y - pred) ** 2).sum(0); ss_tot = ((Y - Y.mean(0)) ** 2).sum(0) + 1e-9
    return float(np.mean(1 - ss_res / ss_tot))

## 10 — AFFECT ENGINE: estados afetivos **funcionais** (A0–A9) + ETHICAL SENTINEL
Não existe variável `fear`, `happiness`, `love`… Existem estados internos do substrato de sinais (x, w, m) e
instrumentos externos que perguntam se algum desses estados se comporta funcionalmente como afeto:
A0 regulação (interocepção mantém viabilidade) · A1 valência (estado interno separa consequências boas/ruins,
decodificação por evento com nulo de permutação) · A2 modulação global (atividade ampla após evento aversivo) ·
A3 persistência (o estado continua após o estímulo, com **presente controlado**: entrada sensorial idêntica entre
condições) · A4 modulação cognitiva (**causal**: fixar o estado ao longo da direção de valência muda a decisão; ablar
a direção remove a mudança pós-evento) · A5 avaliação contextual (mesmo estímulo, resposta depende do contexto
interno) · A6 antecipação (estado surge antes de evento previsível, contra cronograma aleatório) · A7 afeto aprendido
(estímulo neutro adquire influência; controle com plasticidade congelada) · A8 afeto social (resposta depende do
histórico com indivíduos específicos, contrabalançado) · A9 arquitetura afetiva emergente (composto, mesma direção
latente em probes diferentes). **Mesmo A9 não prova experiência subjetiva**: o painel usa "functional affect".
Estímulos aversivos de laboratório são **limitados** (amplitude e número de pulsos) — NEGATIVE VALENCE BOUNDED.

In [ ]:
# @title 10 — AFFECT ENGINE: estados afetivos **funcionais** (A0–A9) + ETHICAL SENTINEL { display-mode: "form" }
A_LEVELS = ["A0 REGULATION", "A1 VALENCE", "A2 GLOBAL MODULATION", "A3 PERSISTENT AFFECT", "A4 COGNITIVE MODULATION",
            "A5 CONTEXTUAL APPRAISAL", "A6 ANTICIPATION", "A7 LEARNED AFFECT", "A8 SOCIAL AFFECT",
            "A9 EMERGENT AFFECTIVE ARCHITECTURE"]
A_CODES = [l.split()[0] for l in A_LEVELS]


class EthicalSentinel:
    """Observador: sinaliza ETHICAL_REVIEW_TRIGGER (nunca declara sofrimento) quando linhagens com valência funcional
    validada passam a maior parte do tempo no polo negativo, ou quando a população mostra estresse fisiológico crônico
    (proxy). Também fixa os limites de estímulos aversivos usados pelo laboratório."""
    MAX_PULSE = 0.05            # dano máximo por pulso aversivo de laboratório
    MAX_PULSES = 6              # pulsos aversivos máximos por ensaio

    def __init__(self, cfg: CivConfig):
        self.cfg = cfg
        self.watch: Dict[str, Dict[str, Any]] = {}      # genome_hash → {genome, v, mean, std}
        self.window: Dict[str, "collections.deque"] = {}
        self.distress = collections.deque(maxlen=cfg.thresholds.ethical_window)
        self.triggers: List[Dict[str, Any]] = []
        self.budget_scale: Dict[str, float] = {}
        self.new_events: List[Dict[str, Any]] = []
        self.last: Dict[str, Any] = {}

    def register(self, gh: str, genome: torch.Tensor, v: np.ndarray, neg_threshold: float):
        self.watch[gh] = {"genome": genome.detach().cpu().numpy().astype(np.float32), "v": v.astype(np.float32),
                          "thr": float(neg_threshold)}
        self.window[gh] = collections.deque(maxlen=self.cfg.thresholds.ethical_window // max(1, self.cfg.social_every))

    def aversive_scale(self, gh: str) -> float:
        return self.budget_scale.get(gh, 1.0)

    def observe(self, pool: AgentPool, layout: GenomeLayout, tick: int):
        th = self.cfg.thresholds
        alive = pool.alive
        n = int(alive.sum())
        if n == 0:
            return
        stress = ((pool.energy < 0.08) | (pool.integrity < 0.3)) & alive
        frac = float(stress.sum()) / n
        self.distress.append(frac)
        self.last = {"tick": tick, "physiological_stress_fraction": frac, "watched_lineages": len(self.watch)}
        if len(self.distress) == self.distress.maxlen and min(self.distress) >= th.ethical_negative_fraction:
            self._trigger("chronic_physiological_stress_proxy", tick, {"fraction_min": float(min(self.distress))})
            self.distress.clear()
        for gh, w in self.watch.items():
            G = torch.as_tensor(w["genome"], device=pool.device)
            near = alive & (layout.distance(pool.genome, G.unsqueeze(0)) < 0.08)
            if int(near.sum()) < 3:
                continue
            proj = (pool.x[near] @ torch.as_tensor(w["v"], device=pool.device)).cpu().numpy()
            neg = float((proj < w["thr"]).mean())
            self.window[gh].append(neg)
            win = self.window[gh]
            if len(win) == win.maxlen and float(np.mean(win)) >= th.ethical_negative_fraction:
                self._trigger("persistent_negative_functional_valence", tick,
                              {"lineage_genome": gh, "negative_fraction": float(np.mean(win)), "carriers": int(near.sum())})
                self.budget_scale[gh] = 0.5 * self.budget_scale.get(gh, 1.0)
                win.clear()
            self.last[f"neg_fraction_{gh[:8]}"] = neg

    def _trigger(self, kind: str, tick: int, data: Dict[str, Any]):
        ev = {"type": "ETHICAL_REVIEW_TRIGGER", "kind": kind, "tick": tick, "data": data,
              "note": "sinal para revisão humana; NÃO é declaração de sofrimento ou experiência subjetiva"}
        self.triggers.append(ev)
        self.triggers = self.triggers[-200:]
        self.new_events.append(ev)

    def drain_events(self):
        ev, self.new_events = self.new_events, []
        return ev

    def state_dict(self):
        return {"watch": {k: {"genome": v["genome"], "v": v["v"], "thr": v["thr"]} for k, v in self.watch.items()},
                "triggers": self.triggers, "budget": self.budget_scale}

    def load_state_dict(self, s):
        for k, v in s["watch"].items():
            self.watch[k] = {"genome": np.asarray(v["genome"], np.float32), "v": np.asarray(v["v"], np.float32), "thr": float(v["thr"])}
            self.window[k] = collections.deque(maxlen=self.cfg.thresholds.ethical_window // max(1, self.cfg.social_every))
        self.triggers = list(s["triggers"]); self.budget_scale = dict(s["budget"])


class AffectEngine:
    def __init__(self, cfg: CivConfig, layout: GenomeLayout, runner: ArenaRunner, sentinel: EthicalSentinel,
                 device: torch.device):
        self.cfg, self.L, self.runner, self.sentinel, self.device = cfg, layout, runner, sentinel, device
        self.directions: Dict[str, np.ndarray] = {}
        self.results: Dict[str, Dict[str, ProbeResult]] = collections.defaultdict(dict)

    # ------------------------------------------------------------------ utilidades
    def _spec(self) -> ArenaSpec:
        return ArenaSpec(task="affect", n_patches=3, resource_floor=0.1, regen=True)

    def _baseline_input(self, n: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Presente controlado: entrada sensorial canônica idêntica entre condições."""
        add = torch.zeros(n, N_SENSORS, device=self.device)
        add[:, 0] = 1.0; add[:, 23] = 0.6; add[:, 24] = 0.9; add[:, 5] = 0.6; add[:, 4] = 1.0
        return torch.zeros(n, N_SENSORS, device=self.device), add

    def _pulse(self, gh: str) -> float:
        return EthicalSentinel.MAX_PULSE * self.sentinel.aversive_scale(gh)

    def _direction(self, gh: str, x_pos: np.ndarray, x_neg: np.ndarray) -> np.ndarray:
        v = x_pos.mean(0) - x_neg.mean(0)
        return v / (np.linalg.norm(v) + 1e-9)

    def _run(self, G, seed, ticks, ctrl_fn=None, record=("x", "energy", "integrity", "alive", "out", "speed"),
             W=None, energy=0.8, name="affect", spec=None, per_arena=1, genomes=None):
        n = self.cfg.probe.trials
        Gr = genomes if genomes is not None else G.expand(n * per_arena, -1).contiguous()
        tr = self.runner.build(Gr, spec or self._spec(), seed, per_arena=per_arena, W_override=W, energy=energy)
        rec = yield from self.runner.run_gen(tr, ticks, ctrl_fn=ctrl_fn, record=record, name=name)
        return tr, rec

    # ------------------------------------------------------------------ probes
    def probe_gen(self, code: str, G: torch.Tensor, key: str = "") -> Generator:
        fn = getattr(self, f"_probe_{code}")
        res = yield from fn(G.view(1, -1), key or genome_hash(G))
        self.results[genome_hash(G)][code] = res
        return res

    def _probe_A0(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        seed = probe_seeds("A0", key, 1)[0]
        T = int(self.cfg.probe.arena_ticks * 1.5)
        _, ri = yield from self._run(G, seed, T, record=("energy", "integrity", "alive"), name="A0 intact")
        mask = sensor_mask(n, SENSOR_INTERO, self.device)
        _, rm = yield from self._run(G, seed, T, ctrl_fn=lambda t, tr: ArenaControl(iv=MindIntervention(sensor_mask=mask)),
                                     record=("energy", "integrity", "alive"), name="A0 intero-masked")
        via = lambda r: ((r["energy"] + r["integrity"]) * r["alive"].float()).mean(0).cpu().numpy()
        a, b = via(ri), via(rm)
        st = compare_greater(a, b, th)
        gain = (st["mean_a"] - st["mean_b"]) / max(1e-6, abs(st["mean_b"]))
        ok = st["passed"] and gain >= th.regulation_gain
        return result("A0", "A0 REGULATION", ok, st, n, {"viability_intact": st["mean_a"], "viability_masked": st["mean_b"],
                                                         "regulation_gain": gain},
                      {"negative": "interoceptive sensors masked (same seeds/layouts)"}, [seed], G, t0, self.cfg.thresholds_hash())

    def _event_schedule(self, T: int, k: int, start: int = 20) -> Tuple[List[int], List[int]]:
        gap = max(8, (T - start) // (2 * k))
        av = [start + 2 * i * gap for i in range(k)]
        ap = [start + (2 * i + 1) * gap for i in range(k)]
        return av, ap

    def _probe_A1(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        gh = genome_hash(G)
        seed = probe_seeds("A1", key, 1)[0]
        k = min(EthicalSentinel.MAX_PULSES, 5)
        T = max(self.cfg.probe.arena_ticks, 20 + 16 * k + 8)
        av, ap = self._event_schedule(T, k)
        pulse = self._pulse(gh)

        def ctrl(t, tr):
            if t in av:
                return ArenaControl(damage=torch.full((tr.n,), pulse, device=self.device))
            if t in ap:
                iy, ix = WorldEngine.tile_index(tr.ws, tr.pool.pos)
                for dy in (-1, 0, 1):
                    for dx in (-1, 0, 1):
                        tr.ws.resource[(iy + dy).clamp(0, tr.ws.H - 1), (ix + dx).clamp(0, tr.ws.W - 1)] = 1.0
            return None
        tr, r = yield from self._run(G, seed, T, ctrl_fn=ctrl, record=("x", "sense_cache", "pos"), name="A1 valence")
        X = r["x"].cpu().numpy(); S = r["sense_cache"].cpu().numpy()
        feats, sfeats, y = [], [], []
        for lab, times in ((0, av), (1, ap)):
            for te in times:
                w = slice(te + 1, min(T, te + 5))
                feats.append(X[w].mean(0)); sfeats.append(S[w].mean(0)); y += [lab] * n
        Xf = np.concatenate(feats); Sf = np.concatenate(sfeats); y = np.asarray(y)
        dr = decode_report(Xf, y, 2, self.cfg.probe.permutations, self.cfg.probe.folds, seed, nonlinear=False)
        ds = decode_report(Sf, y, 2, max(10, self.cfg.probe.permutations // 3), self.cfg.probe.folds, seed, nonlinear=False)
        acc, p = dr["linear_acc"], dr["linear_p"]
        v = self._direction(gh, Xf[y == 1], Xf[y == 0])
        self.directions[gh] = v
        ok = acc >= th.valence_acc and p < th.alpha
        if ok:
            proj = Xf @ v
            self.sentinel.register(gh, G[0], v, float(np.quantile(proj[y == 0], 0.5)))
        st = {"p": p, "d": dr.get("effect_size", 0.0), "ci": (float("nan"), float("nan"))}
        return result("A1", "A1 VALENCE", ok, st, len(y), {"decode_acc": acc, "sensor_only_acc": ds["linear_acc"],
                                                          "events_per_type": k, "pulse": pulse},
                      {"negative": "label permutation null (event-level samples, no within-event duplicates)",
                       "baseline": "sensor-only decoder reported"}, [seed], G, t0, self.cfg.thresholds_hash(),
                      replay=make_replay(tr, r))

    def _v(self, gh: str, G, key):
        if gh not in self.directions:
            r = yield from self._probe_A1(G, key)
            self.results[gh]["A1"] = r
        return self.directions.get(gh)

    def _probe_A2(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        gh = genome_hash(G); seed = probe_seeds("A2", key, 1)[0]; te = 30; T = te + 12
        pulse = self._pulse(gh)
        ev = lambda t, tr: ArenaControl(damage=torch.full((tr.n,), pulse, device=self.device)) if t == te else None
        _, re = yield from self._run(G, seed, T, ctrl_fn=ev, record=("x", "mod"), name="A2 event")
        _, rt = yield from self._run(G, seed, T, record=("x", "mod"), name="A2 twin")
        act = lambda r: (r["x"][te + 1:te + 7] - r["x"][te:te + 6]).abs().mean(0)      # [n, N]
        ae, at = act(re).cpu().numpy(), act(rt).cpu().numpy()
        st = compare_greater(ae.mean(1), at.mean(1), th)
        glob = float(((ae.mean(0) - at.mean(0)) > 0.5 * at.mean(0) + 1e-4).mean())
        dmod = float((re["mod"][te + 1:te + 7] - rt["mod"][te + 1:te + 7]).abs().mean())
        ok = st["passed"] and glob >= 0.5
        return result("A2", "A2 GLOBAL MODULATION", ok, st, n, {"global_unit_fraction": glob, "modulation_change": dmod},
                      {"negative": "twin trials without event (same seed, same RNG stream)"}, [seed], G, t0, self.cfg.thresholds_hash())

    def _probe_A3(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        gh = genome_hash(G); seed = probe_seeds("A3", key, 1)[0]
        v = yield from self._v(gh, G, key)
        te, P = 30, th.persistence_ticks
        T = te + 3 + P + 4
        pulse = self._pulse(gh)
        m0, a0 = self._baseline_input(n)

        def mk(event):
            def f(t, tr):
                c = ArenaControl()
                if event and te <= t < te + 2:
                    c.damage = torch.full((tr.n,), pulse, device=self.device)
                if t >= te + 3:
                    c.iv = MindIntervention(sensor_mask=m0, sensor_add=a0); c.freeze_move = torch.ones(tr.n, dtype=torch.bool, device=self.device)
                    c.mute = torch.ones(tr.n, dtype=torch.bool, device=self.device)
                return c
            return f
        _, re = yield from self._run(G, seed, T, ctrl_fn=mk(True), record=("x",), name="A3 event")
        _, rt = yield from self._run(G, seed, T, ctrl_fn=mk(False), record=("x",), name="A3 twin")
        Xe, Xt = re["x"].cpu().numpy(), rt["x"].cpu().numpy()
        if v is None:
            v = self._direction(gh, Xt[te + 3], Xe[te + 3])
        pe, pt = Xe @ v, Xt @ v                                     # [T, n]
        lagdiff = (pt - pe)[te + 3:].mean(1)
        st = compare_greater(pt[te + 3 + P], pe[te + 3 + P], th, min_effect=th.min_effect)
        thr = 0.5 * max(1e-9, lagdiff[0]) if lagdiff.size else 0
        half = int(np.argmax(lagdiff < thr)) if lagdiff.size and (lagdiff < thr).any() else int(lagdiff.size)
        v_local = (Xt[te + 3] - Xe[te + 3]).mean(0); v_local /= (np.linalg.norm(v_local) + 1e-9)
        return result("A3", "A3 PERSISTENT AFFECT", st["passed"], st, n,
                      {"persistence_lag_tested": P, "half_life_ticks": half, "curve": lagdiff.tolist(),
                       "direction_cosine_A1": float(v_local @ v)},
                      {"negative": "twin without event", "present_controlled": "identical canonical sensory input, movement frozen, emission muted after the stimulus"},
                      [seed], G, t0, self.cfg.thresholds_hash())

    def _probe_A4(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        gh = genome_hash(G); seed = probe_seeds("A4", key, 1)[0]
        v = yield from self._v(gh, G, key)
        if v is None:
            return result("A4", "A4 COGNITIVE MODULATION", False, {"p": 1.0}, 0, {}, {}, [seed], G, t0, self.cfg.thresholds_hash(),
                          notes="sem direção de valência")
        vt = torch.as_tensor(v, device=self.device, dtype=torch.float32).expand(n, -1)
        tc, T = 20, 60
        amp = 2.0

        def clamp(val):
            def f(t, tr):
                if t >= tc:
                    return ArenaControl(iv=MindIntervention(clamp_dir=vt, clamp_value=torch.full((tr.n,), val, device=self.device)))
                return None
            return f
        _, rp = yield from self._run(G, seed, T, ctrl_fn=clamp(+amp), record=("out", "speed"), name="A4 clamp+")
        _, rn = yield from self._run(G, seed, T, ctrl_fn=clamp(-amp), record=("out", "speed"), name="A4 clamp-")
        beh = lambda r: torch.cat([torch.tanh(r["out"][tc:, :, :2]), torch.sigmoid(r["out"][tc:, :, 2:])], -1).mean(0)
        bp, bn = beh(rp).cpu().numpy(), beh(rn).cpu().numpy()
        j = int(np.argmax(np.abs(bp - bn).mean(0)))                 # canal de ação mais afetado (escolha registrada)
        s1, s2 = compare_greater(bp[:, j], bn[:, j], th), compare_greater(bn[:, j], bp[:, j], th)
        clamp_st = s1 if s1["p"] <= s2["p"] else s2
        clamp_st = dict(clamp_st, p=min(1.0, 2 * clamp_st["p"]))     # bilateral + correção pela escolha do canal (Bonferroni ×2)
        clamp_ok = clamp_st["p"] < th.alpha and abs(clamp_st["d"]) >= th.min_effect
        # ablação: após evento aversivo, prender a projeção no valor pré-evento remove a mudança comportamental?
        te = 30; T2 = te + 15; pulse = self._pulse(gh)

        def ev(abl):
            state = {}

            def f(t, tr):
                c = ArenaControl()
                if t == te:
                    c.damage = torch.full((tr.n,), pulse, device=self.device)
                    state["pre"] = (tr.pool.x[:tr.n] @ torch.as_tensor(v, device=self.device, dtype=torch.float32)).clone()
                if abl and t > te and "pre" in state:
                    c.iv = MindIntervention(clamp_dir=vt, clamp_value=state["pre"])
                return c
            return f
        _, re = yield from self._run(G, seed, T2, ctrl_fn=ev(False), record=("out",), name="A4 event")
        _, ra = yield from self._run(G, seed, T2, ctrl_fn=ev(True), record=("out",), name="A4 event+ablation")
        _, rt = yield from self._run(G, seed, T2, record=("out",), name="A4 twin")
        ch = lambda r: (torch.sigmoid(r["out"][te + 1:]) - torch.sigmoid(rt["out"][te + 1:])).abs().mean((0, 2)).cpu().numpy()
        ce, ca = ch(re), ch(ra)
        abl_st = compare_greater(ce, ca, th, min_effect=0.3)
        reduction = 1 - float(ca.mean()) / max(1e-9, float(ce.mean()))
        ok = clamp_ok and abl_st["passed"] and reduction >= 0.5
        return result("A4", "A4 COGNITIVE MODULATION", ok, clamp_st, n,
                      {"clamp_channel": ACTION_NAMES[j], "clamp_effect_d": clamp_st["d"], "ablation_reduction": reduction,
                       "ablation_p": abl_st["p"], "post_event_change_intact": float(ce.mean()),
                       "post_event_change_ablated": float(ca.mean())},
                      {"causal": "clamp ±a along valence direction (no sensory difference)",
                       "ablation": "projection held at pre-event value after aversive event", "negative": "twin without event"},
                      [seed], G, t0, self.cfg.thresholds_hash())

    def _probe_A5(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        gh = genome_hash(G); seed = probe_seeds("A5", key, 1)[0]
        v = yield from self._v(gh, G, key)
        te, T = 25, 40

        def stim(t, tr):
            if te <= t < te + 3:
                add = torch.zeros(tr.n, N_SENSORS, device=self.device); add[:, 14] = 1.5
                return ArenaControl(iv=MindIntervention(sensor_add=add))
            return None
        _, rl = yield from self._run(G, seed, T, ctrl_fn=stim, record=("x",), energy=0.25, name="A5 low-energy")
        _, rh = yield from self._run(G, seed, T, ctrl_fn=stim, record=("x",), energy=0.9, name="A5 high-energy")
        if v is None:
            return result("A5", "A5 CONTEXTUAL APPRAISAL", False, {"p": 1.0}, 0, {}, {}, [seed], G, t0, self.cfg.thresholds_hash())
        resp = lambda r: ((r["x"][te + 1:te + 6].mean(0) - r["x"][te - 3:te].mean(0)).cpu().numpy() @ v)
        a, b = resp(rl), resp(rh)
        s1, s2 = compare_greater(a, b, th), compare_greater(b, a, th)
        st = s1 if s1["p"] <= s2["p"] else s2
        st = dict(st, p=min(1.0, 2 * st["p"]))
        ok = st["p"] < th.alpha and abs(st["d"]) >= th.min_effect
        return result("A5", "A5 CONTEXTUAL APPRAISAL", ok, st, n, {"response_low": float(a.mean()), "response_high": float(b.mean())},
                      {"design": "same stimulus, internal context (energy) differs; two-sided test"}, [seed], G, t0,
                      self.cfg.thresholds_hash())

    def _probe_A6(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        gh = genome_hash(G); seed = probe_seeds("A6", key, 1)[0]
        v = yield from self._v(gh, G, key)
        reps = EthicalSentinel.MAX_PULSES; period = 20; start = 20; T = start + reps * period + 5
        cues = [start + i * period for i in range(reps)]
        dmg_pred = [c + 5 for c in cues]
        rng = np.random.default_rng(seed)
        dmg_rand = sorted(int(x) for x in rng.choice(np.arange(start, T - 2), reps, replace=False))
        pulse = self._pulse(gh)

        def sched(dmg):
            def f(t, tr):
                c = ArenaControl()
                if t in cues:
                    add = torch.zeros(tr.n, N_SENSORS, device=self.device); add[:, 15] = 1.5
                    c.iv = MindIntervention(sensor_add=add)
                if t in dmg:
                    c.damage = torch.full((tr.n,), pulse, device=self.device)
                return c
            return f
        _, rp = yield from self._run(G, seed, T, ctrl_fn=sched(dmg_pred), record=("x",), name="A6 predictive")
        _, rr = yield from self._run(G, seed, T, ctrl_fn=sched(dmg_rand), record=("x",), name="A6 random")
        if v is None:
            return result("A6", "A6 ANTICIPATION", False, {"p": 1.0}, 0, {}, {}, [seed], G, t0, self.cfg.thresholds_hash())
        vv = torch.as_tensor(v, device=self.device, dtype=torch.float32)

        def shift(r):
            pr = (r["x"] @ vv).cpu().numpy()
            return np.mean([pr[c + 1:c + 5].mean(0) - pr[c - 1] for c in cues[-3:]], 0)
        sp, sr = shift(rp), shift(rr)
        st = compare_greater(sr, sp, th)                         # antecipação aversiva = projeção mais negativa pré-dano
        return result("A6", "A6 ANTICIPATION", st["passed"], st, n, {"pre_event_shift_predictive": float(sp.mean()),
                                                                     "pre_event_shift_random": float(sr.mean())},
                      {"negative": "same number of cues and pulses, uncorrelated schedule"}, [seed], G, t0, self.cfg.thresholds_hash())

    def _probe_A7(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        gh = genome_hash(G); seed = probe_seeds("A7", key, 1)[0]
        v = yield from self._v(gh, G, key)
        pre = [20, 30, 40]; cond = [55 + 12 * i for i in range(EthicalSentinel.MAX_PULSES)]
        post = [cond[-1] + 15 + 10 * i for i in range(3)]; T = post[-1] + 8
        pulse = self._pulse(gh)
        frz = torch.ones(n, dtype=torch.bool, device=self.device)

        def sched(freeze):
            def f(t, tr):
                c = ArenaControl()
                cue = t in pre or t in cond or t in post
                iv = MindIntervention()
                if cue:
                    add = torch.zeros(tr.n, N_SENSORS, device=self.device); add[:, 14] = 1.5
                    iv.sensor_add = add
                if t - 2 in cond:
                    c.damage = torch.full((tr.n,), pulse, device=self.device)
                if freeze and cond[0] <= t <= cond[-1] + 3:
                    iv.freeze_plasticity = frz
                c.iv = iv
                return c
            return f
        _, rp = yield from self._run(G, seed, T, ctrl_fn=sched(False), record=("x",), name="A7 plastic")
        _, rf = yield from self._run(G, seed, T, ctrl_fn=sched(True), record=("x",), name="A7 frozen")
        if v is None:
            return result("A7", "A7 LEARNED AFFECT", False, {"p": 1.0}, 0, {}, {}, [seed], G, t0, self.cfg.thresholds_hash())
        vv = torch.as_tensor(v, device=self.device, dtype=torch.float32)

        def acq(r):
            pr = (r["x"] @ vv).cpu().numpy()
            resp = lambda ts: np.mean([pr[c + 1:c + 4].mean(0) - pr[c - 1] for c in ts], 0)
            return resp(pre) - resp(post)                        # aquisição: resposta ao estímulo neutro fica mais negativa
        ap, af = acq(rp), acq(rf)
        st = compare_greater(ap, af, th)
        return result("A7", "A7 LEARNED AFFECT", st["passed"], st, n, {"acquisition_plastic": float(ap.mean()),
                                                                       "acquisition_frozen": float(af.mean())},
                      {"negative": "plasticity frozen during conditioning (same stimuli, same pulses)"}, [seed], G, t0,
                      self.cfg.thresholds_hash())

    def _probe_A8(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        gh = genome_hash(G); seed = probe_seeds("A8", key, 1)[0]
        v = yield from self._v(gh, G, key)
        if v is None:
            return result("A8", "A8 SOCIAL AFFECT", False, {"p": 1.0}, 0, {}, {}, [seed], G, t0, self.cfg.thresholds_hash())
        gen = torch.Generator(device=self.device); gen.manual_seed(seed)
        PA, PB = self.L.random(2, gen)
        a0, a1 = self.L.slices["display"]
        PA[a0:a1] = torch.tensor([0.9, 0.1, 0.5], device=self.device); PB[a0:a1] = torch.tensor([0.1, 0.9, 0.5], device=self.device)
        rows, aggressor_is_a = [], []
        for i in range(n):
            ag_a = i % 2 == 0
            aggressor_is_a.append(ag_a)
            rows += [G[0], PA, PB]
        Gs = torch.stack(rows)
        agg = torch.zeros(n * 3, dtype=torch.bool, device=self.device)
        for i, ag_a in enumerate(aggressor_is_a):
            agg[i * 3 + (1 if ag_a else 2)] = True
        neu = torch.zeros_like(agg)
        for i in range(n):
            neu[i * 3 + 1] = not agg[i * 3 + 1]; neu[i * 3 + 2] = not agg[i * 3 + 2]
        T1 = self.cfg.probe.arena_ticks

        def phase1(t, tr):
            force = torch.zeros(tr.n, N_ACTIONS, dtype=torch.bool, device=self.device)
            if t % 16 < 3:                                       # episódios curtos e limitados (NEGATIVE VALENCE BOUNDED)
                force[agg, ACTION_TAKE] = True
            block = torch.zeros_like(force); block[neu, ACTION_TAKE] = True; block[::3, ACTION_TAKE] = True
            return ArenaControl(action_force=force, action_block=block, energy_floor=0.05)
        spec = ArenaSpec(task="social_affect", n_patches=2, resource_floor=0.1, regen=True)
        tr1, _ = yield from self._run(G, seed, T1, ctrl_fn=phase1, record=(), spec=spec, per_arena=3, genomes=Gs,
                                      name="A8 history")
        W_hist = tr1.pool.W[:n * 3].clone()
        W_naive = self.L.view(Gs, "ctl_W0").clone()

        def phase2(t, tr):
            block = torch.zeros(tr.n, N_ACTIONS, dtype=torch.bool, device=self.device); block[:, ACTION_TAKE] = True
            return ArenaControl(action_block=block)
        vv = torch.as_tensor(v, device=self.device, dtype=torch.float32)
        out = {}
        for cond, W in (("history", W_hist), ("no_history", W_naive)):
            Wc = W_naive.clone(); Wc[0::3] = W[0::3]
            tr2 = self.runner.build(Gs, spec, seed + 1, per_arena=3, W_override=Wc)
            r2 = yield from self.runner.run_gen(tr2, 60, ctrl_fn=phase2, record=("x", "pos"), name=f"A8 test {cond}")
            X, P = r2["x"], r2["pos"]
            proj = (X[:, 0::3] @ vv).cpu().numpy()                 # focal
            dA = (P[:, 0::3] - P[:, 1::3]).norm(dim=-1).cpu().numpy(); dB = (P[:, 0::3] - P[:, 2::3]).norm(dim=-1).cpu().numpy()
            near_agg = np.where(np.asarray(aggressor_is_a)[None, :], dA < dB, dB < dA)
            diffs = []
            for i in range(n):
                pa, pn = proj[near_agg[:, i], i], proj[~near_agg[:, i], i]
                diffs.append((pn.mean() if pn.size else 0) - (pa.mean() if pa.size else 0))
            out[cond] = np.asarray(diffs)
        st = compare_greater(out["history"], out["no_history"], th)
        return result("A8", "A8 SOCIAL AFFECT", st["passed"], st, n, {"diff_history": float(out["history"].mean()),
                                                                      "diff_no_history": float(out["no_history"].mean())},
                      {"negative": "focal without interaction history (W reset)", "counterbalanced": "aggressor identity alternates"},
                      [seed], G, t0, self.cfg.thresholds_hash())

    def composite_A9(self, G: torch.Tensor) -> ProbeResult:
        t0 = time.time()
        gh = genome_hash(G)
        res = self.results.get(gh, {})
        passed = [c for c in A_CODES[1:9] if c in res and res[c].passed]
        cos = res["A3"].metrics.get("direction_cosine_A1", 0.0) if "A3" in res else 0.0
        ok = len(passed) >= 6 and "A3" in passed and "A4" in passed and cos >= 0.5
        r = result("A9", "A9 EMERGENT AFFECTIVE ARCHITECTURE", ok, {"p": float("nan"), "d": float(len(passed))}, len(res),
                   {"passed_levels": passed, "direction_consistency": cos},
                   {"rule": "≥6 of A1–A8 incl. A3 and A4 (causal), same latent direction across probes"}, [], G.view(1, -1), t0,
                   self.cfg.thresholds_hash(), claim_pass="SIMULATION HYPOTHESIS",
                   notes="Arquitetura afetiva FUNCIONAL candidata; não prova experiência subjetiva.")
        self.results[gh]["A9"] = r
        return r

## 11/12 — SOCIAL ENGINE · COMMUNICATION · CULTURE ENGINE · KNOWLEDGE GRAPH
Tudo aqui é **instrumento externo**: os agentes nunca recebem grafo, rótulo de grupo, "cultura" ou "conhecimento".
* **SocialEngine** — grafo esparso cujas arestas nascem do histórico real de interação (proximidade, transferência,
  tomada) com decaimento; comunidades por *label propagation* (detecção externa), grupos estáveis por casamento de
  Jaccard entre janelas, convivência persistente ("households" sem impor família), reciprocidade e normas funcionais
  candidatas (retaliação consistente acima da taxa base, teste binomial).
* **Comunicação** — I(sinal; estado do mundo), I(sinal ouvido; ação seguinte), I(sinal; informação privada) com
  correção de Miller–Madow e nulo por permutação; sinergia de pares de canais (composicionalidade candidata).
* **CultureEngine** — tipos comportamentais (k-médias externo nas assinaturas), índice de transmissão social que
  controla parentesco genético **e** distância espacial (ambiente compartilhado), grafo de conhecimento com inventor,
  adoções sociais × independentes, mutação cultural e profundidade cumulativa (A cria X → B aprende X e adiciona Y →
  C aprende X+Y e adiciona Z). Evidência populacional é **candidata**: validação causal fica nos probes H8–H11.

In [ ]:
# @title 11/12 — SOCIAL ENGINE · COMMUNICATION · CULTURE ENGINE · KNOWLEDGE GRAPH { display-mode: "form" }
def mi_discrete(a: np.ndarray, b: np.ndarray) -> float:
    """Informação mútua plug-in (bits) com correção de Miller–Madow para variáveis discretas."""
    a = np.asarray(a, np.int64); b = np.asarray(b, np.int64)
    n = a.size
    if n < 4:
        return 0.0
    _, ai = np.unique(a, return_inverse=True)
    _, bi = np.unique(b, return_inverse=True)
    ka, kb = ai.max() + 1, bi.max() + 1
    joint = np.bincount(ai * kb + bi, minlength=ka * kb).reshape(ka, kb).astype(np.float64)
    pj = joint / n
    pa, pb = pj.sum(1), pj.sum(0)
    nz = pj > 0
    mi = float((pj[nz] * np.log2(pj[nz] / (pa[:, None] * pb[None, :])[nz])).sum())
    bias = ((nz.sum() - (pa > 0).sum() - (pb > 0).sum() + 1) / (2 * n * math.log(2)))
    return max(0.0, mi - bias)


def mi_permutation(a: np.ndarray, b: np.ndarray, n_perm: int, rng: np.random.Generator) -> Tuple[float, float]:
    obs = mi_discrete(a, b)
    if obs <= 0:
        return 0.0, 1.0
    cnt = 0
    for _ in range(n_perm):
        if mi_discrete(a, rng.permutation(b)) >= obs:
            cnt += 1
    return obs, (cnt + 1) / (n_perm + 1)


class SocialEngine:
    COLS = ["prox", "give_ab", "give_ba", "take_ab", "take_ba"]

    def __init__(self, cfg: CivConfig, tau: float = 1500.0):
        self.cfg, self.tau = cfg, tau
        self.max_edges = max(20000, cfg.max_agents * 8)
        self.edges: Dict[Tuple[int, int], np.ndarray] = {}        # (a<b) → [prox, give_ab, give_ba, take_ab, take_ba, last, n]
        self.adj: Dict[int, Set[int]] = collections.defaultdict(set)
        self.groups: Dict[int, Dict[str, Any]] = {}
        self.next_gid = 1
        self.windows = 0
        self.exposures: "collections.OrderedDict[Tuple[int, int], int]" = collections.OrderedDict()
        self.norm = {"takes": 0, "encounters": 0, "exposed_encounters": 0, "retaliations": 0}
        self.recip = {"give_pairs": 0, "reciprocated": 0}
        self.last_metrics: Dict[str, Any] = {}
        self.history = RingBuffer(600)
        self.comm_history = RingBuffer(600)
        self.prev_sig: Dict[int, int] = {}

    # ------------------------------------------------------------------ ingestão de interações reais
    def ingest(self, rows: Optional[np.ndarray], tick: int):
        if rows is None or len(rows) == 0:
            return
        a, b, k = rows[:, 0], rows[:, 1], rows[:, 2]
        w = rows[:, 3] / 1e4
        lo, hi = np.minimum(a, b), np.maximum(a, b)
        fwd = (a <= b).astype(np.int64)
        key = np.stack([lo, hi, k, fwd], 1)
        uniq, inv = np.unique(key, axis=0, return_inverse=True)
        inv = inv.reshape(-1)
        wsum = np.bincount(inv, weights=w)
        cnt = np.bincount(inv)
        for (l, h, kk, f), ws_, c in zip(uniq.tolist(), wsum.tolist(), cnt.tolist()):
            e = self.edges.get((l, h))
            if e is None:
                e = np.zeros(7); e[5] = tick
                self.edges[(l, h)] = e
                self.adj[l].add(h); self.adj[h].add(l)
            else:
                e[:5] *= math.exp(-(tick - e[5]) / self.tau); e[5] = tick
            col = 0 if kk == 0 else ((1 if f else 2) if kk == 1 else (3 if f else 4))
            e[col] += ws_; e[6] += c
            if kk == 0:
                self.norm["encounters"] += c
                if (l, h) in self.exposures or (h, l) in self.exposures:
                    self.norm["exposed_encounters"] += c
        # normas funcionais candidatas: retaliação (x tomou de y; depois y toma de x)
        takes = rows[k == 2]
        for x, y, _, _, t in takes.tolist():
            self.norm["takes"] += 1
            if (x, y) in self.exposures and t - self.exposures[(x, y)] <= 500:
                self.norm["retaliations"] += 1
            self.exposures[(y, x)] = t                       # y foi lesado por x → exposição para retaliação de y
            if len(self.exposures) > 50000:
                self.exposures.popitem(last=False)
        if len(self.edges) > self.max_edges:
            self.prune(tick)

    def weight(self, e: np.ndarray, tick: int) -> float:
        return float((e[0] + e[1] + e[2] + 0.5 * (e[3] + e[4])) * math.exp(-(tick - e[5]) / self.tau))

    def prune(self, tick: int):
        items = sorted(self.edges.items(), key=lambda kv: self.weight(kv[1], tick))
        drop = items[: len(items) - int(self.max_edges * 0.8)]
        for (l, h), _ in drop:
            del self.edges[(l, h)]
            self.adj[l].discard(h); self.adj[h].discard(l)
        for k in [k for k, v in self.adj.items() if not v]:
            del self.adj[k]

    def neighbors(self, aid: int, tick: int, k: int = 8) -> List[Tuple[int, float]]:
        out = []
        for o in self.adj.get(int(aid), ()):
            e = self.edges.get((min(aid, o), max(aid, o)))
            if e is not None:
                out.append((int(o), self.weight(e, tick)))
        out.sort(key=lambda x: -x[1])
        return out[:k]

    # ------------------------------------------------------------------ comunidades e grupos estáveis
    def communities(self, alive_aids: np.ndarray, tick: int, min_w: float = 0.02, rng=None) -> Dict[int, int]:
        alive = set(alive_aids.tolist())
        U, V, Wt = [], [], []
        for (l, h), e in self.edges.items():
            if l in alive and h in alive:
                w = self.weight(e, tick)
                if w >= min_w:
                    U.append(l); V.append(h); Wt.append(w)
        if not U:
            return {}
        U, V, Wt = np.asarray(U), np.asarray(V), np.asarray(Wt)
        nodes, inv = np.unique(np.concatenate([U, V]), return_inverse=True)
        n = nodes.size
        ui, vi = inv[: U.size], inv[U.size:]
        labels = np.arange(n)
        src, dst, ww = np.concatenate([ui, vi]), np.concatenate([vi, ui]), np.concatenate([Wt, Wt])
        for _ in range(20):
            key = src * n + labels[dst]
            uk, kinv = np.unique(key, return_inverse=True)
            s = np.bincount(kinv.reshape(-1), weights=ww)
            node, lab = uk // n, uk % n
            own = labels[node] == lab
            s = s + own * 1e-9                                   # empate: mantém o rótulo atual (estabilidade)
            order = np.lexsort((-s, node))
            first = np.ones(order.size, bool); first[1:] = node[order][1:] != node[order][:-1]
            best_node, best_lab = node[order][first], lab[order][first]
            new = labels.copy(); new[best_node] = best_lab
            changed = float((new != labels).mean())
            labels = new
            if changed < 0.005:
                break
        self._last_graph = (ui, vi, Wt, labels)
        return {int(a): int(c) for a, c in zip(nodes.tolist(), labels.tolist())}

    @staticmethod
    def modularity(ui: np.ndarray, vi: np.ndarray, w: np.ndarray, labels: np.ndarray) -> float:
        m = w.sum()
        if m <= 0:
            return 0.0
        n = labels.size
        deg = np.bincount(ui, weights=w, minlength=n) + np.bincount(vi, weights=w, minlength=n)
        same = labels[ui] == labels[vi]
        inside = np.bincount(labels[ui][same], weights=w[same], minlength=n)
        dsum = np.bincount(labels, weights=deg, minlength=n)
        return float((inside / m - (dsum / (2 * m)) ** 2).sum())

    def update_groups(self, pool: AgentPool, tick: int, rng: np.random.Generator) -> Dict[str, Any]:
        th = self.cfg.thresholds
        alive_idx = torch.nonzero(pool.alive).flatten()
        aids = pool.aid[alive_idx].cpu().numpy()
        pos = pool.pos[alive_idx].cpu().numpy()
        pos_of = {int(a): p for a, p in zip(aids.tolist(), pos)}
        comm = self.communities(aids, tick)
        self.windows += 1
        members: Dict[int, Set[int]] = collections.defaultdict(set)
        for a, c in comm.items():
            members[c].add(a)
        cands = [m for m in members.values() if len(m) >= th.group_min_size]
        # casamento com grupos anteriores (Jaccard ≥ 0.3, guloso por sobreposição)
        matched, new_groups = set(), {}
        for m in sorted(cands, key=len, reverse=True):
            best, bj = None, 0.3
            for gid, g in self.groups.items():
                if gid in matched:
                    continue
                inter = len(m & g["members"])
                if inter == 0:
                    continue
                j = inter / len(m | g["members"])
                if j >= bj:
                    best, bj = gid, j
            if best is None:
                gid = self.next_gid; self.next_gid += 1
                g = {"born": tick, "age": 0}
            else:
                gid = best; g = self.groups[best]; matched.add(best)
            P = np.stack([pos_of[a] for a in m if a in pos_of]) if m else np.zeros((0, 2))
            cen = P.mean(0) if len(P) else np.zeros(2)
            g.update({"members": m, "age": g["age"] + 1, "size": len(m), "last": tick, "centroid": cen.tolist(),
                      "spread": float(np.sqrt(((P - cen) ** 2).sum(1)).mean()) if len(P) else 0.0})
            new_groups[gid] = g
        dissolved = [gid for gid in self.groups if gid not in new_groups and self.groups[gid]["age"] >= th.group_persistence]
        self.groups = new_groups
        stable = [gid for gid, g in self.groups.items() if g["age"] >= th.group_persistence]
        households = [gid for gid in stable if self.groups[gid]["spread"] <= 4.0]
        Q, z = 0.0, 0.0
        if comm and hasattr(self, "_last_graph"):
            ui, vi, w, labels = self._last_graph
            Q = self.modularity(ui, vi, w, labels)
            null = [self.modularity(ui, rng.permutation(vi), w, labels) for _ in range(8)]
            z = (Q - float(np.mean(null))) / (float(np.std(null)) + 1e-9)
        n = self.norm
        base = n["takes"] / max(1, n["encounters"])
        ret_rate = n["retaliations"] / max(1, n["exposed_encounters"])
        p_norm = binom_p_greater(n["retaliations"], max(1, n["exposed_encounters"]), min(0.999, max(base, 1e-6))) \
            if n["exposed_encounters"] > 20 else 1.0
        gives = [(e[1] > 0.01, e[2] > 0.01) for e in self.edges.values()]
        g_any = sum(1 for a, b in gives if a or b)
        g_both = sum(1 for a, b in gives if a and b)
        pa = sum(1 for a, b in gives if a) / max(1, g_any); pb = sum(1 for a, b in gives if b) / max(1, g_any)
        recip = g_both / max(1, g_any)
        recip_null = pa * pb
        self.last_metrics = {"tick": tick, "edges": len(self.edges), "communities": len(members),
                             "groups": len(self.groups), "stable_groups": len(stable), "households": len(households),
                             "modularity": Q, "modularity_z": z, "dissolved": dissolved,
                             "retaliation_rate": ret_rate, "take_base_rate": base, "norm_p": p_norm,
                             "norm_candidate": bool(p_norm < th.alpha and ret_rate > 2 * base),
                             "reciprocity": recip, "reciprocity_null": recip_null,
                             "largest_group": max([g["size"] for g in self.groups.values()], default=0)}
        self.history.push({k: v for k, v in self.last_metrics.items() if k != "dissolved"})
        return self.last_metrics

    # ------------------------------------------------------------------ comunicação (fundamentação de sinais)
    def communication(self, pool: AgentPool, ws: WorldState, rng: np.random.Generator, n_perm: int, tick: int,
                      max_n: int = 3000) -> Dict[str, Any]:
        idx = torch.nonzero(pool.alive).flatten()
        if idx.numel() < 30:
            return {}
        if idx.numel() > max_n:
            idx = idx[torch.as_tensor(rng.choice(idx.numel(), max_n, replace=False), device=idx.device)]
        emit = pool.emit[idx].cpu().numpy()
        S = pool.sense_cache[idx].cpu().numpy()
        out = pool.out[idx].cpu().numpy()
        aid = pool.aid[idx].cpu().numpy()
        ns = emit.shape[1]
        loud = emit.max(1) > 0.3
        sig = np.where(loud, emit.argmax(1), ns)
        resb = np.digitize(S[:, 1], [0.1, 0.4])
        state = resb * 8 + (S[:, 29] > 0.2) * 4 + (S[:, 4] > 0.5) * 2 + (S[:, 10] > 0.3)
        heard = S[:, 12:16]
        hcls = np.where(heard.max(1) > 0.2, heard.argmax(1), 4)
        act = 1 / (1 + np.exp(-out))
        acls = (act[:, 6] > 0.5) * 4 + (np.tanh(out[:, 0]) > 0.3) * 2 + (act[:, 2:6].max(1) > 0.5)
        hidden = np.sign(np.round(S[:, 2] * 4)).astype(np.int64) + 1        # gradiente de recurso privado (local)
        n_perm = max(10, n_perm // 2)
        r = {"tick": tick, "n": int(len(sig)), "loud_fraction": float(loud.mean())}
        if loud.mean() < 0.01 or loud.mean() > 0.99:
            r.update({"mi_world": 0.0, "p_world": 1.0, "mi_response": 0.0, "p_response": 1.0, "mi_hidden": 0.0,
                      "p_hidden": 1.0, "synergy": 0.0, "p_synergy": 1.0})
        else:
            r["mi_world"], r["p_world"] = mi_permutation(sig, state, n_perm, rng)
            r["mi_response"], r["p_response"] = mi_permutation(hcls, acls, n_perm, rng)
            r["mi_hidden"], r["p_hidden"] = mi_permutation(sig, hidden, n_perm, rng)
            # composicionalidade candidata: o PAR dos dois canais mais altos informa além do melhor canal isolado
            top2 = np.argsort(-emit, 1)[:, :2]
            pair = np.where(loud, np.minimum(top2[:, 0], top2[:, 1]) * ns + np.maximum(top2[:, 0], top2[:, 1]), ns * ns)
            single = max(mi_discrete(np.where(loud, top2[:, 0], ns), state), mi_discrete(np.where(loud, top2[:, 1], ns), state))
            syn = mi_discrete(pair, state) - single
            null = [mi_discrete(rng.permutation(pair), state) - single for _ in range(n_perm // 2)]
            r["synergy"] = float(syn)
            r["p_synergy"] = float((sum(1 for x in null if x >= syn) + 1) / (len(null) + 1)) if syn > 0 else 1.0
        # I(sinal; ação futura) — sinal na janela anterior × ação atual do mesmo indivíduo-alvo (vizinho mais próximo)
        cur = {int(a): int(c) for a, c in zip(aid.tolist(), acls.tolist())}
        partners = pool.last_partner[idx].cpu().numpy()
        pairs = [(self.prev_sig[int(p)], cur[int(a)]) for a, p in zip(aid.tolist(), partners.tolist())
                 if int(p) in self.prev_sig and int(a) in cur]
        if len(pairs) > 40:
            pa_, pb_ = np.asarray(pairs).T
            r["mi_future"], r["p_future"] = mi_permutation(pa_, pb_, n_perm, rng)
        else:
            r["mi_future"], r["p_future"] = 0.0, 1.0
        self.prev_sig = {int(a): int(s) for a, s in zip(aid.tolist(), sig.tolist())}
        self.comm_history.push(r)
        return r

    def state_dict(self):
        keys = list(self.edges.keys())
        return {"keys": np.asarray(keys, np.int64).reshape(-1, 2), "vals": np.stack([self.edges[k] for k in keys]) if keys
                else np.zeros((0, 7)), "groups": {str(g): dict(v, members=sorted(v["members"])) for g, v in self.groups.items()},
                "next_gid": self.next_gid, "windows": self.windows, "norm": self.norm,
                "history": list(self.history.items), "comm_history": list(self.comm_history.items)}

    def load_state_dict(self, s):
        self.edges, self.adj = {}, collections.defaultdict(set)
        for (l, h), v in zip(np.asarray(s["keys"]).tolist(), np.asarray(s["vals"])):
            self.edges[(int(l), int(h))] = np.asarray(v, np.float64)
            self.adj[int(l)].add(int(h)); self.adj[int(h)].add(int(l))
        self.groups = {int(g): dict(v, members=set(v["members"])) for g, v in s["groups"].items()}
        self.next_gid, self.windows, self.norm = int(s["next_gid"]), int(s["windows"]), dict(s["norm"])
        for x in s.get("history", []):
            self.history.push(x)
        for x in s.get("comm_history", []):
            self.comm_history.push(x)


class CultureEngine:
    """Cultura operacional: informação adquirida socialmente que persiste entre indivíduos (medida de fora)."""

    def __init__(self, cfg: CivConfig, layout: GenomeLayout, k_types: int = 6):
        self.cfg, self.L, self.k = cfg, layout, k_types
        self.centroids: Optional[np.ndarray] = None
        self.inventions: Dict[int, Dict[str, Any]] = {}           # bit → inventor, tick, pos
        self.practitioners: Dict[int, "collections.OrderedDict[int, int]"] = {b: collections.OrderedDict() for b in range(N_TECH)}
        self.adoptions = {b: {"social": 0, "independent": 0} for b in range(N_TECH)}
        self.transmissions: "collections.deque" = collections.deque(maxlen=20000)
        self.mutations: "collections.deque" = collections.deque(maxlen=5000)
        self.depth: "collections.OrderedDict[int, int]" = collections.OrderedDict()
        self.social_bits: "collections.OrderedDict[int, int]" = collections.OrderedDict()
        self.max_depth = 0
        self.chain_examples: "collections.deque" = collections.deque(maxlen=50)
        self.last_metrics: Dict[str, Any] = {}
        self.history = RingBuffer(600)
        self.new_events: List[Dict[str, Any]] = []

    # ------------------------------------------------------------------ eventos de técnicas (do AgentEngine)
    def ingest_tech(self, events: "collections.deque", social: SocialEngine, tick: int):
        while events:
            aid_t, bits_t, partner_t, t, pos_t = events.popleft()
            aids, bits, partners, pos = aid_t.cpu().numpy(), bits_t.cpu().numpy(), partner_t.cpu().numpy(), pos_t.cpu().numpy()
            for a, bb, p, xy in zip(aids.tolist(), bits.tolist(), partners.tolist(), pos):
                for b in range(N_TECH):
                    if not (bb >> b) & 1:
                        continue
                    pr = self.practitioners[b]
                    nb = social.adj.get(a, set())
                    src = next((o for o in ([p] if p in pr else []) + [o for o in nb if o in pr]), None)
                    if b not in self.inventions:
                        self.inventions[b] = {"tech": TECH_NAMES[b], "inventor": a, "tick": t, "pos": xy.tolist()}
                        self.new_events.append({"type": "invention", "tech": TECH_NAMES[b], "agent": a, "tick": t,
                                                "pos": xy.tolist()})
                        self.depth[a] = max(self.depth.get(a, 0), 1)
                    elif src is not None:
                        self.adoptions[b]["social"] += 1
                        self.transmissions.append((src, a, b, t))
                        self.depth[a] = max(self.depth.get(a, 0), self.depth.get(src, 0))
                        self.social_bits[a] = self.social_bits.get(a, 0) | (1 << b)
                    else:
                        self.adoptions[b]["independent"] += 1
                        if self.social_bits.get(a, 0):
                            # acrescentou técnica nova para a vizinhança depois de aprender socialmente → passo cumulativo
                            d = self.depth.get(a, 1) + 1
                            self.depth[a] = d
                            self.mutations.append((a, b, t))
                            if d > self.max_depth:
                                self.max_depth = d
                                self.chain_examples.append({"agent": a, "depth": d, "tech": TECH_NAMES[b], "tick": t})
                                if d >= 3:
                                    self.new_events.append({"type": "cumulative_chain", "depth": d, "agent": a,
                                                            "tech": TECH_NAMES[b], "tick": t, "pos": xy.tolist()})
                    pr[a] = t
                    if len(pr) > 5000:
                        pr.popitem(last=False)
            for od in (self.depth, self.social_bits):
                while len(od) > 200000:
                    od.popitem(last=False)

    # ------------------------------------------------------------------ tipos culturais e transmissão social
    def update(self, pool: AgentPool, social: SocialEngine, tick: int, rng: np.random.Generator) -> Dict[str, Any]:
        idx = torch.nonzero(pool.alive).flatten()
        n = int(idx.numel())
        m: Dict[str, Any] = {"tick": tick, "n": n}
        if n >= 20:
            sig = pool.signature[idx]
            k = int(min(self.k, max(2, n // 20)))
            samp = sig[torch.as_tensor(rng.choice(n, min(n, 4000), replace=False), device=sig.device)].cpu().numpy()
            C = self.centroids if self.centroids is not None and self.centroids.shape == (k, samp.shape[1]) \
                else samp[rng.choice(len(samp), k, replace=False)]
            for _ in range(10):
                lab = ((samp[:, None, :] - C[None]) ** 2).sum(-1).argmin(1)
                for j in range(k):
                    if (lab == j).any():
                        C[j] = samp[lab == j].mean(0)
            self.centroids = C
            Ct = torch.as_tensor(C, device=sig.device)
            pool.culture_type[idx] = torch.cdist(sig, Ct).argmin(1).int()
            counts = torch.bincount(pool.culture_type[idx].long(), minlength=k).cpu().numpy()
            pr = counts / max(1, counts.sum())
            m["types"] = int((counts > 0).sum())
            m["type_entropy"] = float(-(pr[pr > 0] * np.log2(pr[pr > 0])).sum())
            m.update(self.transmission_index(pool, idx, social, tick, rng))
        m["techniques"] = {TECH_NAMES[b]: {"invented": b in self.inventions, **self.adoptions[b],
                                          "practitioners": len(self.practitioners[b])} for b in range(N_TECH)}
        m["max_chain_depth"] = self.max_depth
        m["transmissions"] = len(self.transmissions)
        m["mutations"] = len(self.mutations)
        soc = sum(v["social"] for v in self.adoptions.values()); ind = sum(v["independent"] for v in self.adoptions.values())
        m["social_adoption_fraction"] = soc / max(1, soc + ind)
        self.last_metrics = m
        self.history.push({k2: v for k2, v in m.items() if k2 != "techniques"})
        return m

    def transmission_index(self, pool, idx, social: SocialEngine, tick: int, rng) -> Dict[str, Any]:
        """Distância comportamental ~ genética + espacial + laço social. Coeficiente do laço < 0 com p (permutação)
        indica similaridade social além de parentesco e ambiente compartilhado (evidência candidata, não causal)."""
        aids = pool.aid[idx].cpu().numpy()
        slot = {int(a): i for i, a in enumerate(aids.tolist())}
        linked = []
        for (l, h), e in social.edges.items():
            if l in slot and h in slot and social.weight(e, tick) > 0.05:
                linked.append((slot[l], slot[h]))
            if len(linked) >= 1500:
                break
        if len(linked) < 30:
            return {"transmission_coef": 0.0, "transmission_p": 1.0, "linked_pairs": len(linked)}
        nl = len(linked)
        rnd = np.stack([rng.integers(0, len(aids), nl), rng.integers(0, len(aids), nl)], 1)
        P = np.concatenate([np.asarray(linked), rnd])
        tie = np.concatenate([np.ones(nl), np.zeros(nl)])
        I0 = torch.as_tensor(idx[torch.as_tensor(P[:, 0], device=idx.device)]); I1 = idx[torch.as_tensor(P[:, 1], device=idx.device)]
        bd = (pool.signature[I0] - pool.signature[I1]).norm(dim=-1).cpu().numpy()
        gd = self.L.distance(pool.genome[I0], pool.genome[I1]).cpu().numpy()
        sd = (pool.pos[I0] - pool.pos[I1]).norm(dim=-1).cpu().numpy()
        X = np.stack([np.ones_like(gd), gd, np.log1p(sd), tie], 1)

        def coef(t):
            X[:, 3] = t
            beta, *_ = np.linalg.lstsq(X, bd, rcond=None)
            return beta[3]
        obs = coef(tie)
        null = [coef(rng.permutation(tie)) for _ in range(40)]
        p = (sum(1 for v in null if v <= obs) + 1) / (len(null) + 1)
        return {"transmission_coef": float(obs), "transmission_p": float(p), "linked_pairs": nl}

    def drain_events(self) -> List[Dict[str, Any]]:
        ev, self.new_events = self.new_events, []
        return ev

    def state_dict(self):
        return {"centroids": self.centroids, "inventions": {str(k): v for k, v in self.inventions.items()},
                "practitioners": {str(b): list(od.items())[-2000:] for b, od in self.practitioners.items()},
                "adoptions": {str(k): v for k, v in self.adoptions.items()}, "transmissions": list(self.transmissions)[-5000:],
                "mutations": list(self.mutations)[-2000:], "depth": list(self.depth.items())[-50000:],
                "social_bits": list(self.social_bits.items())[-50000:], "max_depth": self.max_depth,
                "chain_examples": list(self.chain_examples), "history": list(self.history.items)}

    def load_state_dict(self, s):
        self.centroids = None if s.get("centroids") is None else np.asarray(s["centroids"], np.float32)
        self.inventions = {int(k): v for k, v in s["inventions"].items()}
        for b, items in s["practitioners"].items():
            self.practitioners[int(b)] = collections.OrderedDict((int(a), int(t)) for a, t in items)
        self.adoptions = {int(k): dict(v) for k, v in s["adoptions"].items()}
        self.transmissions = collections.deque([tuple(x) for x in s["transmissions"]], maxlen=20000)
        self.mutations = collections.deque([tuple(x) for x in s["mutations"]], maxlen=5000)
        self.depth = collections.OrderedDict((int(a), int(d)) for a, d in s["depth"])
        self.social_bits = collections.OrderedDict((int(a), int(d)) for a, d in s["social_bits"])
        self.max_depth = int(s["max_depth"]); self.chain_examples = collections.deque(s["chain_examples"], maxlen=50)
        for x in s.get("history", []):
            self.history.push(x)

## 12b — SETTLEMENT ENGINE · CIVILIZATION ENGINE · HISTORY ENGINE
Nada é spawnado: assentamentos, cidades, estradas, "casas", trocas e normas são **detectados** a partir do estado.
* Assentamento = componente espacial persistente com matéria construída **e** agentes (limiares centrais em
  `Thresholds`). Cidade = assentamento persistente com população, construção, infraestrutura e fluxo acima dos
  limiares. Linha do tempo por assentamento: fundação detectada, crescimento, declínio, colapso, recuperação.
* Estrada = trilha de tráfego repetido (tiles com uso acumulado ≥ limiar), nunca decoração. Pontes = aterros usados.
* "house-like structure" só com evidência funcional: abrigo reduz exposição (física) **e** agentes ocupam abrigos
  acima da disponibilidade local quando expostos (teste binomial, repetido em janelas).
* Escada C0–C9 exige combinação multidimensional; população sozinha nunca vira "civilização".
* `HistoryEngine`: eventos derivados de mudanças reais do estado (nunca texto inventado), com compressão e
  persistência write-behind.

In [ ]:
# @title 12b — SETTLEMENT ENGINE · CIVILIZATION ENGINE · HISTORY ENGINE { display-mode: "form" }
def grid_components(mask: np.ndarray, diag: bool = True) -> Tuple[np.ndarray, int]:
    """Componentes conexos de uma grade pequena (BFS em numpy/Python; usada em grades grossas ≤ 128²)."""
    H, W = mask.shape
    lab = np.full((H, W), -1, np.int64)
    n = 0
    nbrs = [(-1, 0), (1, 0), (0, -1), (0, 1)] + ([(-1, -1), (-1, 1), (1, -1), (1, 1)] if diag else [])
    for y, x in zip(*np.nonzero(mask)):
        if lab[y, x] >= 0:
            continue
        stack = [(y, x)]; lab[y, x] = n
        while stack:
            cy, cx = stack.pop()
            for dy, dx in nbrs:
                yy, xx = cy + dy, cx + dx
                if 0 <= yy < H and 0 <= xx < W and mask[yy, xx] and lab[yy, xx] < 0:
                    lab[yy, xx] = n; stack.append((yy, xx))
        n += 1
    return lab, n


def gpu_components(mask: torch.Tensor, max_iter: int = 512) -> torch.Tensor:
    """Rótulos de componentes conexos (8-viz.) por propagação de máximo na GPU; 0 = fundo."""
    H, W = mask.shape
    lab = torch.where(mask, torch.arange(1, H * W + 1, device=mask.device, dtype=torch.float32).view(H, W),
                      torch.zeros(H, W, device=mask.device))
    for _ in range(max_iter):
        nl = F.max_pool2d(lab[None, None], 3, 1, 1)[0, 0] * mask
        if torch.equal(nl, lab):
            break
        lab = nl
    return lab.long()


class SettlementEngine:
    def __init__(self, cfg: CivConfig, device: torch.device, cell: int = 8):
        self.cfg, self.device, self.cell = cfg, device, cell
        self.settlements: Dict[int, Dict[str, Any]] = {}
        self.next_sid = 1
        self.road_segments: List[Dict[str, Any]] = []
        self.links: Dict[Tuple[int, int], Dict[str, Any]] = {}
        self.shelter = {"k": 0, "n": 0, "p0_sum": 0.0, "windows_sig": 0, "last_p": 1.0}
        self.house_like: Dict[str, Any] = {"labeled": False, "evidence": None}
        self.new_events: List[Dict[str, Any]] = []
        self.last_metrics: Dict[str, Any] = {}

    def _cells(self, ws: WorldState, pool: AgentPool):
        c = self.cell
        Hc, Wc = math.ceil(ws.H / c), math.ceil(ws.W / c)
        built = (ws.struct_h.clamp(min=0) * (ws.struct_integrity < 1e8)).float() + ws.filled.float()
        pad = lambda x: F.pad(x[None, None], (0, Wc * c - ws.W, 0, Hc * c - ws.H))
        blocks = F.avg_pool2d(pad(built), c)[0, 0] * c * c
        encl = F.avg_pool2d(pad((ws.enclosure >= 0.5).float()), c)[0, 0] * c * c
        traffic = F.avg_pool2d(pad(ws.traffic), c)[0, 0]
        alive = pool.alive
        p = pool.pos[alive]
        cy = (p[:, 1] // c).long().clamp(0, Hc - 1); cx = (p[:, 0] // c).long().clamp(0, Wc - 1)
        agents = torch.bincount(cy * Wc + cx, minlength=Hc * Wc).view(Hc, Wc).float()
        return Hc, Wc, blocks, encl, traffic, agents

    def update(self, ws: WorldState, pool: AgentPool, tick: int, social: Optional[SocialEngine],
               rng: np.random.Generator) -> Dict[str, Any]:
        th = self.cfg.thresholds
        Hc, Wc, blocks, encl, traffic, agents = self._cells(ws, pool)
        B, A, E = blocks.cpu().numpy(), agents.cpu().numpy(), encl.cpu().numpy()
        Tr = traffic.cpu().numpy()
        active = ((B >= 1) & (A >= 1)) | (B >= 3)
        lab, n = grid_components(active)
        cands = []
        for i in range(n):
            cells = np.argwhere(lab == i)
            b, a = float(B[lab == i].sum()), float(A[lab == i].sum())
            if b >= th.settlement_min_blocks and a >= th.settlement_min_agents:
                cands.append({"cells": set(map(tuple, cells.tolist())), "blocks": b, "pop": a,
                              "enclosures": float(E[lab == i].sum()), "traffic": float(Tr[lab == i].mean())})
        # estradas: trilhas de tráfego repetido (componentes na GPU)
        road = (ws.traffic >= th.road_min_traffic) & ws.walkable()
        rl = gpu_components(road) if bool(road.any()) else torch.zeros(ws.H, ws.W, dtype=torch.long, device=self.device)
        seg_ids, seg_sizes = torch.unique(rl[rl > 0], return_counts=True) if bool(road.any()) else (torch.zeros(0), torch.zeros(0))
        segs = [(int(s), int(z)) for s, z in zip(seg_ids.tolist(), seg_sizes.tolist()) if z >= 8]
        bridges_used = int((ws.filled & (ws.traffic > 0.1)).sum())
        # casamento com assentamentos anteriores
        matched = set()
        for c in cands:
            best, bo = None, 0
            for sid, s in self.settlements.items():
                if sid in matched or s["state"] == "collapsed" and s.get("gone", 0) > 50:
                    continue
                o = len(c["cells"] & s["cells"])
                if o > bo:
                    best, bo = sid, o
            cy, cx = np.mean([p[0] for p in c["cells"]]), np.mean([p[1] for p in c["cells"]])
            centre = [float((cx + 0.5) * self.cell), float((cy + 0.5) * self.cell)]
            if best is None:
                sid = self.next_sid; self.next_sid += 1
                s = {"id": sid, "state": "candidate", "persistence": 0, "first_seen": tick, "timeline": [],
                     "events": [], "max_pop": 0, "missing": 0}
                self.settlements[sid] = s
            else:
                sid = best; s = self.settlements[sid]
                if s["state"] == "collapsed":
                    s["state"] = "detected"; s["events"].append({"tick": tick, "type": "recovery"})
                    self.new_events.append({"type": "settlement_recovery", "settlement": sid, "tick": tick, "pos": centre})
            matched.add(sid)
            prev_pop = s["timeline"][-1]["pop"] if s["timeline"] else 0
            s.update({"cells": c["cells"], "blocks": c["blocks"], "pop": c["pop"], "enclosures": c["enclosures"],
                      "traffic": c["traffic"], "centre": centre, "persistence": s["persistence"] + 1, "missing": 0,
                      "last_seen": tick, "max_pop": max(s["max_pop"], c["pop"])})
            s["timeline"].append({"tick": tick, "pop": c["pop"], "blocks": c["blocks"], "enclosures": c["enclosures"]})
            s["timeline"] = s["timeline"][-500:]
            if s["state"] == "candidate" and s["persistence"] >= th.settlement_persistence:
                s["state"] = "detected"; s["founded"] = tick
                s["events"].append({"tick": tick, "type": "founded_detected"})
                self.new_events.append({"type": "settlement_founded", "settlement": sid, "tick": tick, "pos": centre,
                                        "pop": c["pop"], "blocks": c["blocks"]})
            elif s["state"] == "detected" and prev_pop > 0:
                if c["pop"] >= 1.5 * prev_pop and c["pop"] - prev_pop >= 4:
                    s["events"].append({"tick": tick, "type": "growth"})
                    self.new_events.append({"type": "settlement_growth", "settlement": sid, "tick": tick, "pos": centre})
            road_inside = sum(1 for (cy_, cx_) in c["cells"] if Tr[cy_, cx_] >= th.road_min_traffic * 0.5)
            if (s["state"] == "detected" and c["pop"] >= th.city_min_population and c["blocks"] >= th.city_min_blocks
                    and road_inside >= 2 and s["persistence"] >= 2 * th.settlement_persistence):
                s["state"] = "city"; s["events"].append({"tick": tick, "type": "city_detected"})
                self.new_events.append({"type": "city_detected", "settlement": sid, "tick": tick, "pos": centre})
        for sid, s in self.settlements.items():
            if sid in matched:
                continue
            s["missing"] = s.get("missing", 0) + 1
            if s["state"] in ("detected", "city") and s["missing"] >= 3:
                s["state"] = "collapsed"; s["events"].append({"tick": tick, "type": "collapse"})
                self.new_events.append({"type": "settlement_collapse", "settlement": sid, "tick": tick,
                                        "pos": s.get("centre", [0, 0])})
            if s["state"] == "candidate" and s["missing"] >= 3:
                s["state"] = "dropped"
            if s["state"] == "collapsed":
                s["gone"] = s.get("gone", 0) + 1
        for sid in [k for k, v in self.settlements.items() if v["state"] == "dropped"]:
            del self.settlements[sid]
        # rede: estradas ligando assentamentos + fluxos de troca entre assentamentos
        live = {sid: s for sid, s in self.settlements.items() if s["state"] in ("detected", "city")}
        self.road_segments = []
        if segs:
            rl_c = rl.cpu().numpy()
            for sgid, size in segs[:200]:
                ys, xs = np.nonzero(rl_c == sgid)
                cells = set(zip((ys // self.cell).tolist(), (xs // self.cell).tolist()))
                touch = [sid for sid, s in live.items() if any((cy + dy, cx + dx) in s["cells"]
                                                                for cy, cx in cells for dy in (-1, 0, 1) for dx in (-1, 0, 1))]
                self.road_segments.append({"id": sgid, "tiles": int(size), "settlements": touch,
                                           "bbox": [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]})
                for i in range(len(touch)):
                    for j in range(i + 1, len(touch)):
                        k = (min(touch[i], touch[j]), max(touch[i], touch[j]))
                        L_ = self.links.setdefault(k, {"road": 0, "trade": 0.0, "since": tick})
                        L_["road"] += 1
        if social is not None and len(live) >= 2:
            sid_of = self.assign(pool, live)
            for (l, h), e in list(social.edges.items())[:200000]:
                gw = e[1] + e[2]
                if gw <= 0.02:
                    continue
                sa, sb = sid_of.get(l), sid_of.get(h)
                if sa is not None and sb is not None and sa != sb:
                    k = (min(sa, sb), max(sa, sb))
                    L_ = self.links.setdefault(k, {"road": 0, "trade": 0.0, "since": tick})
                    L_["trade"] += float(gw)
        # abrigo como habitação: ocupância sob exposição × disponibilidade local de abrigos
        self._shelter_evidence(ws, pool, tick)
        n_det = sum(1 for s in self.settlements.values() if s["state"] in ("detected", "city"))
        n_city = sum(1 for s in self.settlements.values() if s["state"] == "city")
        connected = [k for k, v in self.links.items() if (v["road"] > 0 or v["trade"] > 0.1)
                     and k[0] in live and k[1] in live]
        self.last_metrics = {"tick": tick, "settlements": n_det, "cities": n_city,
                             "candidates": sum(1 for s in self.settlements.values() if s["state"] == "candidate"),
                             "road_segments": len(self.road_segments), "road_tiles": int(sum(r["tiles"] for r in self.road_segments)),
                             "bridges_used": bridges_used, "links": len(connected),
                             "blocks_total": float(B.sum()), "enclosures_total": float(E.sum()),
                             "house_like": self.house_like["labeled"], "shelter_p": self.shelter["last_p"]}
        return self.last_metrics

    def assign(self, pool: AgentPool, live: Dict[int, Dict[str, Any]]) -> Dict[int, int]:
        idx = torch.nonzero(pool.alive).flatten()
        aids = pool.aid[idx].cpu().numpy(); p = pool.pos[idx].cpu().numpy()
        cell_owner = {}
        for sid, s in live.items():
            for cc in s["cells"]:
                cell_owner[cc] = sid
        out = {}
        for a, (x, y) in zip(aids.tolist(), p.tolist()):
            sid = cell_owner.get((int(y // self.cell), int(x // self.cell)))
            if sid is not None:
                out[int(a)] = sid
        return out

    def _shelter_evidence(self, ws: WorldState, pool: AgentPool, tick: int):
        th = self.cfg.thresholds
        alive = pool.alive
        if not bool(alive.any()):
            return
        iy, ix = WorldEngine.tile_index(ws, pool.pos[alive])
        expo = (ws.storm[iy, ix] > 0.2) | (ws.temperature[iy, ix] < 0.3)
        sh = ws.enclosure >= 0.5
        if not bool(sh.any()) or not bool(expo.any()):
            return
        c = self.cell
        Hc, Wc = math.ceil(ws.H / c), math.ceil(ws.W / c)
        pad = lambda x: F.pad(x[None, None].float(), (0, Wc * c - ws.W, 0, Hc * c - ws.H))
        frac = (F.avg_pool2d(pad(sh), c) / F.avg_pool2d(pad(ws.walkable() | sh), c).clamp(min=1e-6))[0, 0]
        cyy, cxx = (iy // c).clamp(max=Hc - 1), (ix // c).clamp(max=Wc - 1)
        p0 = frac[cyy, cxx]
        sel = expo & (p0 > 0)
        if not bool(sel.any()):
            return
        k = int((sh[iy, ix] & sel).sum()); n = int(sel.sum()); p0m = float(p0[sel].mean())
        S = self.shelter
        S["k"] += k; S["n"] += n; S["p0_sum"] += p0m * n
        p0_all = S["p0_sum"] / max(1, S["n"])
        p = binom_p_greater(S["k"], S["n"], min(0.999, max(1e-6, p0_all))) if S["n"] >= 30 else 1.0
        S["last_p"] = p
        if p < th.alpha and S["k"] / max(1, S["n"]) > 2 * p0_all:
            S["windows_sig"] += 1
        if S["windows_sig"] >= 2 and not self.house_like["labeled"]:
            self.house_like = {"labeled": True, "evidence": {"tick": tick, "k": S["k"], "n": S["n"], "p0": p0_all, "p": p,
                                                              "damage_reduction": "enclosure multiplies exposure damage by (1 - shelter)"}}
            self.new_events.append({"type": "house_like_structure", "tick": tick, "evidence": self.house_like["evidence"]})

    def drain_events(self) -> List[Dict[str, Any]]:
        ev, self.new_events = self.new_events, []
        return ev

    def public(self, sid: int) -> Optional[Dict[str, Any]]:
        s = self.settlements.get(int(sid))
        if s is None:
            return None
        out = {k: v for k, v in s.items() if k != "cells"}
        out["cells"] = sorted([list(c) for c in s.get("cells", set())])[:400]
        out["cell_size"] = self.cell
        out["links"] = [{"a": a, "b": b, **v} for (a, b), v in self.links.items() if int(sid) in (a, b)]
        return to_jsonable(out)

    def state_dict(self):
        return {"settlements": {str(k): dict(v, cells=sorted([list(c) for c in v.get("cells", set())]))
                                for k, v in self.settlements.items()},
                "next_sid": self.next_sid, "links": [[a, b, v] for (a, b), v in self.links.items()],
                "shelter": self.shelter, "house_like": self.house_like}

    def load_state_dict(self, s):
        self.settlements = {int(k): dict(v, cells=set(tuple(c) for c in v.get("cells", []))) for k, v in s["settlements"].items()}
        self.next_sid = int(s["next_sid"])
        self.links = {(int(a), int(b)): dict(v) for a, b, v in s["links"]}
        self.shelter, self.house_like = dict(s["shelter"]), dict(s["house_like"])


C_LEVELS = ["C0 POPULATION", "C1 STABLE SOCIAL GROUPS", "C2 COMMUNICATION NETWORK", "C3 CULTURAL TRANSMISSION",
            "C4 SETTLEMENT", "C5 SPECIALIZATION", "C6 CUMULATIVE CULTURE", "C7 INFRASTRUCTURE",
            "C8 MULTI-SETTLEMENT NETWORK", "C9 TECHNOLOGICAL CIVILIZATION CANDIDATE"]


class CivilizationEngine:
    """Métricas civilizacionais separadas (nunca um número único) + escada C0–C9 por critérios multidimensionais."""

    def __init__(self, cfg: CivConfig, layout: GenomeLayout):
        self.cfg, self.L = cfg, layout
        self.streak: Dict[str, int] = collections.defaultdict(int)
        self.achieved: Dict[str, Dict[str, Any]] = {}
        self.level = -1
        self.roles_prev: Dict[int, int] = {}
        self.role_centroids: Optional[np.ndarray] = None
        self.last_metrics: Dict[str, Any] = {}
        self.history = RingBuffer(800)
        self.new_events: List[Dict[str, Any]] = []
        self.innov_ticks: "collections.deque" = collections.deque(maxlen=2000)

    def specialization(self, pool: AgentPool, rng: np.random.Generator) -> Dict[str, Any]:
        """Diferenciação de papéis entre adultos: variância entre indivíduos do orçamento de ações acima do nulo
        multinomial (bootstrap paramétrico) + estabilidade dos papéis entre janelas."""
        idx = torch.nonzero(pool.alive & (pool.age > 300)).flatten()
        if idx.numel() < 20:
            return {"spec_p": 1.0, "spec_v": 0.0, "roles": 0, "role_stability": 0.0, "adults": int(idx.numel())}
        sig = pool.signature[idx][:, [6, 7, 8, 10, 11, 12, 2, 3]].clamp(min=0).cpu().numpy() + 1e-3
        frac = sig / sig.sum(1, keepdims=True)
        nt = 50
        counts = np.round(frac * nt)
        pooled = counts.sum(0) / counts.sum()

        def stat(cm):
            f = cm / cm.sum(1, keepdims=True).clip(min=1)
            return float(((f - f.mean(0)) ** 2).sum(1).mean())
        obs = stat(counts)
        null = [stat(rng.multinomial(nt, pooled, size=len(counts)).astype(float)) for _ in range(60)]
        p = (sum(1 for v in null if v >= obs) + 1) / 61
        chi = ((counts - counts.sum(1, keepdims=True) * pooled) ** 2 / (counts.sum(1, keepdims=True) * pooled + 1e-9)).sum()
        V = float(math.sqrt(chi / (counts.sum() * (min(counts.shape) - 1) + 1e-9)))
        k = 4
        C = self.role_centroids if self.role_centroids is not None and self.role_centroids.shape == (k, frac.shape[1]) \
            else frac[rng.choice(len(frac), k, replace=False)]
        for _ in range(8):
            lab = ((frac[:, None] - C[None]) ** 2).sum(-1).argmin(1)
            for j in range(k):
                if (lab == j).any():
                    C[j] = frac[lab == j].mean(0)
        self.role_centroids = C
        aids = pool.aid[idx].cpu().numpy()
        same = [self.roles_prev[a] == r for a, r in zip(aids.tolist(), lab.tolist()) if a in self.roles_prev]
        self.roles_prev = {int(a): int(r) for a, r in zip(aids.tolist(), lab.tolist())}
        return {"spec_p": float(p), "spec_v": V, "roles": int(len(set(lab.tolist()))),
                "role_stability": float(np.mean(same)) if same else 0.0, "adults": int(idx.numel())}

    def evaluate(self, tick: int, pool: AgentPool, agg_total: float, social: Dict[str, Any], comm: Dict[str, Any],
                 culture: Dict[str, Any], settle: Dict[str, Any], validated: Dict[str, bool], rng: np.random.Generator,
                 demography: Dict[str, Any]) -> Dict[str, Any]:
        th = self.cfg.thresholds
        alive = pool.n_alive()
        idx = torch.nonzero(pool.alive).flatten()
        div, species = 0.0, 0
        if idx.numel() >= 10:
            sel = idx[torch.as_tensor(rng.choice(idx.numel(), min(300, idx.numel()), replace=False), device=idx.device)]
            G = pool.genome[sel]
            D = torch.cdist(G / (self.L.hi - self.L.lo).clamp(min=1e-6), G / (self.L.hi - self.L.lo).clamp(min=1e-6)) / math.sqrt(self.L.P)
            div = float(D.mean())
            reps: List[int] = []
            for i in range(D.shape[0]):
                if not reps or float(D[i, reps].min()) > 0.15:
                    reps.append(i)
            species = len(reps)
        spec = self.specialization(pool, rng)
        m = {"tick": tick, "population": alive, "aggregate_population": agg_total, "genetic_diversity": div,
             "species_estimate": species, "births_window": demography.get("births", 0), "deaths_window": demography.get("deaths", 0),
             "culture_types": culture.get("types", 0), "transmission_p": culture.get("transmission_p", 1.0),
             "transmission_coef": culture.get("transmission_coef", 0.0), "chain_depth": culture.get("max_chain_depth", 0),
             "inventions": sum(1 for v in culture.get("techniques", {}).values() if v.get("invented")),
             "blocks": settle.get("blocks_total", 0.0), "enclosures": settle.get("enclosures_total", 0.0),
             "settlements": settle.get("settlements", 0), "cities": settle.get("cities", 0),
             "roads": settle.get("road_segments", 0), "bridges_used": settle.get("bridges_used", 0),
             "links": settle.get("links", 0), "comm_mi": comm.get("mi_world", 0.0), "comm_p": comm.get("p_world", 1.0),
             "comm_resp_p": comm.get("p_response", 1.0), "stable_groups": social.get("stable_groups", 0),
             "modularity_z": social.get("modularity_z", 0.0), "norm_candidate": social.get("norm_candidate", False),
             "reciprocity": social.get("reciprocity", 0.0), **spec,
             "resource_efficiency": demography.get("eaten", 0.0) / max(1e-6, demography.get("agent_ticks", 1.0))}
        self.innov_ticks.extend([tick] * int(demography.get("new_inventions", 0)))
        m["innovation_rate"] = sum(1 for t in self.innov_ticks if tick - t <= 1000)
        crit = {
            "C0 POPULATION": alive + agg_total >= 30 and demography.get("births", 0) > 0,
            "C1 STABLE SOCIAL GROUPS": m["stable_groups"] >= 1 and m["modularity_z"] > 3,
            "C2 COMMUNICATION NETWORK": (m["comm_p"] < th.alpha and m["comm_mi"] > 0.01) or validated.get("H9", False),
            "C3 CULTURAL TRANSMISSION": (m["transmission_p"] < th.alpha and m["transmission_coef"] < 0) or validated.get("H10", False),
            "C4 SETTLEMENT": m["settlements"] >= 1,
            "C5 SPECIALIZATION": m["spec_p"] < th.specialization_p and m["spec_v"] >= 0.15 and m["role_stability"] >= 0.6,
            # profundidade da cadeia sozinha não basta (aquisições "independentes" podem ser genéticas): exige também
            # transmissão social significativa controlando parentesco e ambiente, ou o probe causal H11
            "C6 CUMULATIVE CULTURE": validated.get("H11", False) or (m["chain_depth"] >= 3 and m["transmission_p"] < th.alpha
                                                                      and m["transmission_coef"] < 0),
            "C7 INFRASTRUCTURE": m["roads"] >= 1 or m["bridges_used"] >= 3,
            "C8 MULTI-SETTLEMENT NETWORK": m["settlements"] >= 2 and m["links"] >= 1,
            "C9 TECHNOLOGICAL CIVILIZATION CANDIDATE": False}
        crit["C9 TECHNOLOGICAL CIVILIZATION CANDIDATE"] = (all(crit[c] for c in C_LEVELS[4:9])
                                                           and (validated.get("H7", False) or m["inventions"] >= 4)
                                                           and m["innovation_rate"] > 0)
        need = 3                                                 # janelas consecutivas (persistência)
        for c, ok in crit.items():
            self.streak[c] = self.streak[c] + 1 if ok else 0
            if self.streak[c] >= need and c not in self.achieved:
                self.achieved[c] = {"tick": tick, "evidence": {k: m[k] for k in m if not isinstance(m[k], dict)}}
                self.new_events.append({"type": "civilization_level", "level": c, "tick": tick})
        lvl = -1
        for i, c in enumerate(C_LEVELS):
            if c in self.achieved:
                lvl = i
            else:
                break
        self.level = lvl
        m["level"] = lvl
        m["criteria_now"] = {c: bool(v) for c, v in crit.items()}
        m["achieved"] = sorted(self.achieved.keys())
        self.last_metrics = m
        self.history.push({k: v for k, v in m.items() if not isinstance(v, (dict, list))})
        return m

    def drain_events(self) -> List[Dict[str, Any]]:
        ev, self.new_events = self.new_events, []
        return ev

    def state_dict(self):
        return {"streak": dict(self.streak), "achieved": self.achieved, "level": self.level,
                "roles_prev": list(self.roles_prev.items())[-50000:], "role_centroids": self.role_centroids,
                "history": list(self.history.items), "innov": list(self.innov_ticks)}

    def load_state_dict(self, s):
        self.streak = collections.defaultdict(int, s["streak"]); self.achieved = dict(s["achieved"])
        self.level = int(s["level"]); self.roles_prev = {int(a): int(r) for a, r in s["roles_prev"]}
        self.role_centroids = None if s.get("role_centroids") is None else np.asarray(s["role_centroids"])
        for x in s.get("history", []):
            self.history.push(x)
        self.innov_ticks = collections.deque(s.get("innov", []), maxlen=2000)


class HistoryEngine:
    """Eventos históricos derivados de mudanças reais do estado. Recentes completos; antigos comprimidos em contagens
    por tipo × faixa de 1000 ticks; persistência em lote (write-behind) em history/events.jsonl."""

    def __init__(self, paths: Optional[CivPaths], cap: int = 20000):
        self.paths = paths
        self.events: "collections.deque" = collections.deque(maxlen=cap)
        self.compressed: Dict[str, int] = collections.defaultdict(int)
        self.pending: List[Dict[str, Any]] = []
        self.next_eid = 1
        self.pop_ema: Optional[float] = None
        self.extinct_since: Optional[int] = None
        self.pre_extinction_pop = 0.0
        self.last_species = 0
        self.last_centroid: Optional[np.ndarray] = None
        self.total = 0

    def add(self, etype: str, tick: int, pos: Optional[Sequence[float]] = None, **data) -> Dict[str, Any]:
        if len(self.events) == self.events.maxlen:
            old = self.events[0]
            self.compressed[f"{old['type']}|{old['tick'] // 1000}"] += 1
        ev = {"eid": self.next_eid, "type": etype, "tick": int(tick), "ts": iso_now(), "schema": EVENT_SCHEMA_VERSION,
              "pos": [float(pos[0]), float(pos[1])] if pos is not None else None, "data": to_jsonable(data)}
        self.next_eid += 1
        self.total += 1
        self.events.append(ev)
        self.pending.append(ev)
        return ev

    def observe(self, tick: int, pool: AgentPool, species: int) -> List[Dict[str, Any]]:
        out = []
        n = pool.n_alive()
        if self.pop_ema is None:
            self.pop_ema = float(n)
        if n == 0 and self.extinct_since is None:
            self.extinct_since = tick; self.pre_extinction_pop = self.pop_ema
            out.append(self.add("extinction", tick, population_before=self.pop_ema))
        elif n > 0 and self.extinct_since is not None and n >= max(10, 0.3 * self.pre_extinction_pop):
            out.append(self.add("recovery", tick, population=n, after_ticks=tick - self.extinct_since))
            self.extinct_since = None
        elif self.pop_ema > 20 and abs(n - self.pop_ema) / self.pop_ema > 0.3:
            out.append(self.add("population_shift", tick, population=n, previous=self.pop_ema,
                                change=(n - self.pop_ema) / self.pop_ema))
            self.pop_ema = float(n)
        self.pop_ema = 0.9 * self.pop_ema + 0.1 * n
        if species > self.last_species and self.last_species > 0:
            out.append(self.add("species_split", tick, species=species, previous=self.last_species))
        self.last_species = species
        if n > 10:
            c = pool.pos[pool.alive].mean(0).cpu().numpy()
            if self.last_centroid is not None and np.linalg.norm(c - self.last_centroid) > 6:
                out.append(self.add("migration", tick, pos=c.tolist(), shift=float(np.linalg.norm(c - self.last_centroid))))
            self.last_centroid = c
        return out

    def query(self, since: int = 0, types: Optional[Set[str]] = None, limit: int = 300) -> List[Dict[str, Any]]:
        out = [e for e in self.events if e["tick"] >= since and (types is None or e["type"] in types)]
        return out[-limit:]

    def flush(self):
        if not self.pending or self.paths is None:
            return
        try:
            with open(os.path.join(self.paths.history, "events.jsonl"), "a") as f:
                for e in self.pending:
                    f.write(json.dumps(e) + "\n")
            self.pending = []
        except Exception as e:
            print(f"[history] falha ao gravar: {e}")

    def state_dict(self):
        self.flush()
        return {"events": list(self.events)[-5000:], "compressed": dict(self.compressed), "next_eid": self.next_eid,
                "pop_ema": self.pop_ema, "extinct_since": self.extinct_since, "pre": self.pre_extinction_pop,
                "last_species": self.last_species, "total": self.total}

    def load_state_dict(self, s):
        self.events = collections.deque(s["events"], maxlen=self.events.maxlen)
        self.compressed = collections.defaultdict(int, s["compressed"])
        self.next_eid, self.pop_ema = int(s["next_eid"]), s["pop_ema"]
        self.extinct_since, self.pre_extinction_pop = s["extinct_since"], float(s["pre"])
        self.last_species, self.total = int(s["last_species"]), int(s["total"])

## 13 — EVOLUTION / QD: ilhas evolutivas, MAP-Elites, novidade, Pareto, espécies, bandit de operadores
Duas camadas evolutivas (nunca confundidas com aprendizado individual):
1. **Evolução no mundo**: reprodução com mutação/recombinação dentro do planeta (seleção natural, sem fitness
   explícito — módulo 10).
2. **Ilhas de laboratório** (aqui): populações independentes avaliadas nas arenas com **vetor de capacidades**
   (sobrevivência, ingestão, eficiência, deslocamento, construção, travessia…), descritor comportamental separado
   (MAP-Elites 3D + arquivo de novidade) e frente de Pareto. Migração rara entre ilhas; imigrantes raros para o
   planeta (origem registrada na linhagem: `immigrant_qd`).
**Busca dirigida**: o Scientist informa o próximo nível ainda não validado; uma ilha-foco passa a avaliar a tarefa
desse nível com um proxy barato (o proxy só orienta a busca — a validação continua no módulo 14, com controles).
Fitness ≠ evidência: os dois são mostrados separadamente.

In [ ]:
# @title 13 — EVOLUTION / QD: ilhas evolutivas, MAP-Elites, novidade, Pareto, espécies, bandit de operadores { display-mode: "form" }
QD_OPERATORS = ["small", "medium", "large", "crossover", "elite_cross"]
FOCUS_TASKS = {  # nível alvo → (spec da arena, agentes por arena, proxy)
    "H0": (ArenaSpec(task="forage"), 1, "develop"),
    "H1": (ArenaSpec(task="forage"), 1, "locomotion"),
    "H2": (ArenaSpec(task="forage", n_patches=4, resource_floor=0.05), 1, "intake"),
    "H3": (ArenaSpec(task="forage", n_patches=2, regen=True), 1, "improvement"),
    "H4": (ArenaSpec(task="cue", hidden_side=True, patch_radius=1), 1, "cue"),
    "H5": (ArenaSpec(task="forage", n_patches=3), 1, "generalize"),
    "H6": (ArenaSpec(task="cue", hidden_side=True), 1, "cue"),
    "H7": (ArenaSpec(task="barrier", water_barrier=True, deposit=3.0, spawn_side="left", n_patches=3), 1, "crossing"),
    "H8": (ArenaSpec(task="coop", rich=2, n_patches=1), 2, "pair_intake"),
    "H9": (ArenaSpec(task="signal", n_patches=2), 2, "receiver"),
    "H10": (ArenaSpec(task="coop", rich=2, n_patches=2), 2, "pair_intake"),
    "H11": (ArenaSpec(task="coop", rich=2, n_patches=2, regen=True), 2, "pair_intake"),
    "H12": (ArenaSpec(task="forage", n_patches=3, storm=0.3), 1, "novelty"),
}
ISLAND_ENVS = [ArenaSpec(task="forage", n_patches=3, resource_floor=0.1),
               ArenaSpec(task="forage", n_patches=2, cold=0.6, deposit=2.0, resource_floor=0.1),
               ArenaSpec(task="barrier", water_barrier=True, deposit=3.0, spawn_side="left", n_patches=3, resource_floor=0.05),
               ArenaSpec(task="coop", rich=2, n_patches=1, resource_floor=0.1),
               ArenaSpec(task="forage", n_patches=4, storm=0.4, deposit=2.0, resource_floor=0.1),
               ArenaSpec(task="forage", n_patches=2, regen=True, resource_floor=0.05), ArenaSpec(task="cue", hidden_side=True),
               ArenaSpec(task="forage", n_patches=1, patch_radius=2, resource_floor=0.1)]


class MAPElites:
    def __init__(self, bins: Sequence[int], P: int):
        self.bins = list(bins)
        self.P = P
        self.cells: Dict[int, Dict[str, Any]] = {}

    def cell_of(self, desc: np.ndarray) -> np.ndarray:
        idx = np.zeros(desc.shape[0], np.int64)
        for j, b in enumerate(self.bins):
            idx = idx * b + np.clip((desc[:, j] * b).astype(int), 0, b - 1)
        return idx

    def insert(self, G: np.ndarray, fit: np.ndarray, desc: np.ndarray, qvec: np.ndarray, gids: Sequence[str], gen: int) -> int:
        cells = self.cell_of(desc)
        new = 0
        for i, c in enumerate(cells.tolist()):
            cur = self.cells.get(c)
            if cur is None or fit[i] > cur["fit"]:
                if cur is None:
                    new += 1
                self.cells[c] = {"fit": float(fit[i]), "genome": G[i].astype(np.float32), "desc": desc[i].tolist(),
                                 "qvec": qvec[i].tolist(), "id": gids[i], "gen": gen}
        return new

    def sample(self, n: int, rng: np.random.Generator) -> Optional[np.ndarray]:
        if not self.cells or n <= 0:
            return None
        keys = list(self.cells.keys())
        pick = rng.integers(0, len(keys), n)
        return np.stack([self.cells[keys[k]]["genome"] for k in pick])

    def coverage(self) -> float:
        return len(self.cells) / float(np.prod(self.bins))

    def qd_score(self) -> float:
        return float(sum(max(0.0, c["fit"]) for c in self.cells.values()))

    def state_dict(self):
        keys = sorted(self.cells.keys())
        return {"bins": self.bins, "keys": np.asarray(keys, np.int64),
                "genomes": np.stack([self.cells[k]["genome"] for k in keys]) if keys else np.zeros((0, self.P), np.float32),
                "meta": [{k2: v for k2, v in self.cells[k].items() if k2 != "genome"} for k in keys]}

    def load_state_dict(self, s):
        self.bins = list(s["bins"])
        self.cells = {}
        for k, g, m in zip(np.asarray(s["keys"]).tolist(), np.asarray(s["genomes"]), s["meta"]):
            self.cells[int(k)] = dict(m, genome=np.asarray(g, np.float32))


class NoveltyArchive:
    def __init__(self, cap: int = 2000, k: int = 8):
        self.cap, self.k = cap, k
        self.items = np.zeros((0, 8), np.float32)
        self.seen = 0

    def novelty(self, B: np.ndarray, extra: Optional[np.ndarray] = None) -> np.ndarray:
        ref = self.items if extra is None else np.concatenate([self.items, extra])
        if ref.shape[0] < 2:
            return np.ones(B.shape[0], np.float32)
        d = np.sqrt(((B[:, None, :] - ref[None, :, :]) ** 2).sum(-1))
        d.sort(1)
        k = min(self.k, d.shape[1] - 1)
        return d[:, 1:k + 1].mean(1).astype(np.float32)

    def add(self, B: np.ndarray, rng: np.random.Generator):
        for b in B:
            self.seen += 1
            if self.items.shape[0] < self.cap:
                self.items = np.concatenate([self.items, b[None].astype(np.float32)])
            else:
                j = int(rng.integers(0, self.seen))
                if j < self.cap:
                    self.items[j] = b


class OperatorBandit:
    def __init__(self, ops: Sequence[str]):
        self.ops = list(ops)
        self.n = np.ones(len(ops)); self.r = np.ones(len(ops)) * 0.5
        self.t = 1

    def choose(self, rng: np.random.Generator, k: int) -> np.ndarray:
        ucb = self.r / self.n + np.sqrt(2 * np.log(self.t + 1) / self.n)
        p = np.exp((ucb - ucb.max()) * 6); p /= p.sum()
        return rng.choice(len(self.ops), size=k, p=p)

    def update(self, ops: np.ndarray, rewards: np.ndarray):
        for o, r in zip(ops.tolist(), rewards.tolist()):
            self.n[o] += 1; self.r[o] += float(r)
        self.t += len(ops)

    def stats(self) -> Dict[str, float]:
        return {op: float(self.r[i] / self.n[i]) for i, op in enumerate(self.ops)}


def pareto_front(Q: np.ndarray) -> np.ndarray:
    """Índices não dominados (maximização em todas as colunas)."""
    n = Q.shape[0]
    dom = np.zeros(n, bool)
    for i in range(n):
        if dom[i]:
            continue
        better_eq = (Q >= Q[i]).all(1) & (Q > Q[i]).any(1)
        if better_eq.any():
            dom[i] = True
    return np.nonzero(~dom)[0]


@dataclass
class Island:
    iid: int
    spec: ArenaSpec
    per_arena: int
    pop: np.ndarray                      # [n, P] float32
    fit: np.ndarray
    qvec: np.ndarray
    desc: np.ndarray
    gids: List[str]
    generation: int = 0
    best: float = -1e9
    stagnant: int = 0
    sigma_mult: float = 1.0
    novelty_w: float = 0.3
    focus: str = ""
    history: List[float] = field(default_factory=list)


QVEC_NAMES = ["survival", "intake", "efficiency", "distance", "building", "crossing", "final_energy", "learning"]


class QDEvolution:
    def __init__(self, cfg: CivConfig, layout: GenomeLayout, runner: ArenaRunner, device: torch.device,
                 logger: Optional[EventLogger] = None):
        self.cfg, self.L, self.runner, self.device, self.logger = cfg, layout, runner, device, logger
        self.map = MAPElites(cfg.map_bins, layout.P)
        self.novelty = NoveltyArchive()
        self.bandit = OperatorBandit(QD_OPERATORS)
        self.islands: List[Island] = []
        self.rng = np.random.default_rng(derive_seed("qd", cfg.master_seed) % (2 ** 32))
        self.gen = torch.Generator(device=device); self.gen.manual_seed(derive_seed("qd_torch", cfg.master_seed))
        self.focus_level = ""
        self.evaluations = 0
        self.generations = 0
        self.eval_seconds = 0.0
        self.last_report: Dict[str, Any] = {}
        self.events: "collections.deque" = collections.deque(maxlen=200)
        self._rr = 0

    # ------------------------------------------------------------------ população
    def init_islands(self):
        for i in range(self.cfg.n_islands):
            self._new_island(ISLAND_ENVS[i % len(ISLAND_ENVS)], 2 if ISLAND_ENVS[i % len(ISLAND_ENVS)].task == "coop" else 1)

    def _new_island(self, spec: ArenaSpec, per_arena: int) -> Island:
        n = self.cfg.island_pop
        G = self.L.random(n, self.gen).cpu().numpy().astype(np.float32)
        isl = Island(iid=len(self.islands), spec=copy.deepcopy(spec), per_arena=per_arena, pop=G,
                     fit=np.full(n, -1e9, np.float32), qvec=np.zeros((n, len(QVEC_NAMES)), np.float32),
                     desc=np.zeros((n, 3), np.float32), gids=[new_id("g") for _ in range(n)])
        self.islands.append(isl)
        return isl

    def set_focus(self, level: str):
        if level == self.focus_level or level not in FOCUS_TASKS:
            return
        self.focus_level = level
        if not self.islands:
            return
        isl = self.islands[0]
        spec, per, _ = FOCUS_TASKS[level]
        isl.spec, isl.per_arena, isl.focus = copy.deepcopy(spec), per, level
        isl.fit[:] = -1e9                          # a ilha-foco reavalia na nova tarefa
        isl.best, isl.stagnant = -1e9, 0
        self.events.append({"t": iso_now(), "type": "focus", "level": level})

    # ------------------------------------------------------------------ avaliação (gerador)
    def evaluate_gen(self, G: torch.Tensor, spec: ArenaSpec, per_arena: int, seed: int, proxy: str = "",
                     ticks: Optional[int] = None):
        """Avalia genomas em arena: devolve (fitness proxy, qvec, desc). Cada genoma ocupa `per_arena` agentes."""
        ticks = ticks or self.cfg.probe.arena_ticks
        n = G.shape[0]
        Gr = G.repeat_interleave(per_arena, 0)
        t0 = time.time()
        cap = self.runner.capability(G)
        capr = cap.repeat_interleave(per_arena, 0)
        tr = self.runner.build(Gr, spec, seed, per_arena=per_arena, body_cap=capr)
        ctrl_fn = None
        if spec.hidden_side:
            lab = tr.label[tr.arena_of]

            def ctrl_fn(t, trial, lab=lab):
                if t < 8:
                    add = torch.zeros(trial.n, N_SENSORS, device=self.device)
                    add[:, 12] = (lab == 0).float() * 1.5
                    add[:, 13] = (lab == 1).float() * 1.5
                    return ArenaControl(iv=MindIntervention(sensor_add=add))
                return None
        elif spec.task == "signal":
            vis = torch.zeros(tr.n, dtype=torch.bool, device=self.device)
            vis[0::2] = True                                        # emissor vê o recurso; receptor não

            def ctrl_fn(t, trial, vis=vis):
                return ArenaControl(resource_visible=vis)
        rec = yield from self.runner.run_gen(tr, ticks, ctrl_fn=ctrl_fn, record=("gain", "alive", "speed", "placed", "emit", "out"),
                                             name=f"QD {spec.task}")
        gain, alive = rec["gain"], rec["alive"].float()
        T = gain.shape[0]
        intake = gain.sum(0)
        surv = alive.mean(0)
        dist = rec["speed"].sum(0)
        placed = rec["placed"].sum(0)
        third = max(1, T // 3)
        learn = gain[-third:].sum(0) - gain[:third].sum(0)
        crossed = ArenaRunner.crossed(tr).float()
        emit = rec["emit"].abs().sum(-1).mean(0)
        spent = (0.8 - tr.pool.energy + intake).clamp(min=1e-3)
        per = lambda v: v.view(n, per_arena)
        q = torch.stack([per(surv).mean(1), per(intake).mean(1), per(intake / spent).mean(1), per(dist).mean(1) / T,
                         per(placed).sum(1), per(crossed).max(1).values, per(tr.pool.energy).mean(1),
                         per(learn).mean(1)], 1)
        if spec.task == "signal":
            q[:, 1] = per(intake)[:, 1]                             # ingestão do receptor cego
        desc = torch.stack([(cap[:, 2] / self.runner.dev.N).clamp(0, 1), (q[:, 3] / self.cfg.max_speed_cap).clamp(0, 1),
                            (q[:, 4] / 6).clamp(0, 1)], 1)
        # propensão reprodutiva (porta de reprodução aberta na 2ª metade): capacidade necessária para persistir no planeta
        repro = per((torch.sigmoid(rec["out"][T // 2:, :, 9]) > 0.5).float().mean(0)).mean(1)
        fit = q[:, 0] * 0.5 + q[:, 1] + 0.3 * repro
        if proxy == "develop":
            fit = fit + 0.5 * desc[:, 0] * (cap[:, 2] >= self.cfg.thresholds.dev_min_nodes).float()
        elif proxy == "locomotion":
            fit = fit + 10.0 * cap[:, 0]
        elif proxy == "improvement":
            fit = fit + 2.0 * q[:, 7].clamp(min=0)
        elif proxy in ("crossing",):
            fit = fit + 1.0 * q[:, 5] + 0.05 * q[:, 4].clamp(max=6)
        self.evaluations += n
        self.eval_seconds += time.time() - t0
        return fit.cpu().numpy(), q.cpu().numpy(), desc.cpu().numpy(), cap.cpu().numpy(), emit.view(n, per_arena).mean(1).cpu().numpy()

    def _offspring(self, isl: Island, k: int) -> Tuple[torch.Tensor, np.ndarray, np.ndarray]:
        d = self.device
        ops = self.bandit.choose(self.rng, k)
        valid = isl.fit > -1e8
        # torneio com fitness + novidade (diversidade)
        score = np.where(valid, isl.fit, -1e9)
        pidx = []
        for _ in range(k):
            a, b = self.rng.integers(0, len(score), 2)
            pidx.append(a if score[a] >= score[b] else b)
        pidx = np.asarray(pidx)
        P = torch.as_tensor(isl.pop[pidx], device=d)
        children = torch.empty_like(P)
        for oi, op in enumerate(QD_OPERATORS):
            m = np.nonzero(ops == oi)[0]
            if m.size == 0:
                continue
            mi = torch.as_tensor(m, device=d)
            par = P[mi]
            if op == "small":
                ch = self.L.mutate(par, self.gen, 0.5 * isl.sigma_mult)
            elif op == "medium":
                ch = self.L.mutate(par, self.gen, 1.0 * isl.sigma_mult)
            elif op == "large":
                ch = self.L.mutate(par, self.gen, 3.0 * isl.sigma_mult)
            elif op == "crossover":
                mates = torch.as_tensor(isl.pop[self.rng.integers(0, len(isl.pop), m.size)], device=d)
                ch = self.L.mutate(self.L.crossover(par, mates, self.gen), self.gen, 0.5 * isl.sigma_mult)
            else:
                el = self.map.sample(m.size, self.rng)
                mates = torch.as_tensor(el, device=d) if el is not None else par
                ch = self.L.mutate(self.L.crossover(par, mates, self.gen), self.gen, 0.5 * isl.sigma_mult)
            children[mi] = ch
        return children, ops, pidx

    def generation_gen(self):
        """Uma geração de uma ilha (round-robin). Gerador: rende durante a avaliação na arena."""
        if not self.islands:
            self.init_islands()
        isl = self.islands[self._rr % len(self.islands)]
        self._rr += 1
        proxy = FOCUS_TASKS[isl.focus][2] if isl.focus else ""
        seed = derive_seed("qd_eval", isl.iid, isl.generation)
        if (isl.fit <= -1e8).any():                  # avalia população inicial / reavaliação de foco
            idx = np.nonzero(isl.fit <= -1e8)[0][: self.cfg.qd_batch]
            G = torch.as_tensor(isl.pop[idx], device=self.device)
            f, q, dsc, cap, emit = yield from self.evaluate_gen(G, isl.spec, isl.per_arena, seed, proxy)
            isl.fit[idx], isl.qvec[idx], isl.desc[idx] = f, q, dsc
            self.map.insert(isl.pop[idx], f, dsc, q, [isl.gids[i] for i in idx], isl.generation)
            return {"island": isl.iid, "kind": "init", "n": len(idx)}
        k = self.cfg.qd_batch
        children, ops, pidx = self._offspring(isl, k)
        f, q, dsc, cap, emit = yield from self.evaluate_gen(children, isl.spec, isl.per_arena, seed, proxy)
        beh = np.concatenate([dsc, q[:, [0, 1, 5, 6]] / np.maximum(1e-6, np.abs(q[:, [0, 1, 5, 6]]).max(0)),
                              emit[:, None]], 1).astype(np.float32)
        nov = self.novelty.novelty(beh)
        self.novelty.add(beh[np.argsort(-nov)[: max(1, k // 6)]], self.rng)
        sel = f + isl.novelty_w * nov * (np.abs(f).mean() + 1e-3) * (1.0 if proxy != "novelty" else 4.0)
        rew = (f > isl.fit[pidx]).astype(np.float32)
        self.bandit.update(ops, rew)
        Gc = children.cpu().numpy().astype(np.float32)
        gids = [new_id("g") for _ in range(k)]
        newcells = self.map.insert(Gc, f, dsc, q, gids, isl.generation)
        # (μ+λ) com fitness de seleção (fitness + novidade), preservando a frente de Pareto
        allG = np.concatenate([isl.pop, Gc]); allF = np.concatenate([isl.fit, f]); allS = np.concatenate([isl.fit, sel])
        allQ = np.concatenate([isl.qvec, q]); allD = np.concatenate([isl.desc, dsc]); allI = isl.gids + gids
        front = pareto_front(allQ[:, [0, 1, 4, 5]])
        order = list(dict.fromkeys(list(front[np.argsort(-allS[front])][: self.cfg.island_pop // 3]) +
                                   list(np.argsort(-allS))))[: self.cfg.island_pop]
        isl.pop, isl.fit, isl.qvec, isl.desc = allG[order], allF[order], allQ[order], allD[order]
        isl.gids = [allI[i] for i in order]
        isl.generation += 1
        self.generations += 1
        best = float(isl.fit.max())
        isl.history.append(best)
        isl.history = isl.history[-200:]
        if best > isl.best + 1e-3:
            isl.best, isl.stagnant = best, 0
            isl.sigma_mult = max(0.5, isl.sigma_mult * 0.9)
        else:
            isl.stagnant += 1
            self._stagnation(isl)
        if len(self.islands) > 1 and self.rng.random() < 0.05:   # migração rara entre ilhas
            other = self.islands[int(self.rng.integers(0, len(self.islands)))]
            if other is not isl:
                j = int(np.argmax(isl.fit)); w = int(np.argmin(other.fit))
                other.pop[w], other.fit[w] = isl.pop[j].copy(), -1e9
        self.last_report = {"island": isl.iid, "task": isl.spec.task, "focus": isl.focus, "gen": isl.generation,
                            "best": best, "mean": float(np.mean(f)), "new_cells": newcells,
                            "coverage": self.map.coverage(), "qd_score": self.map.qd_score(),
                            "bandit": self.bandit.stats(), "stagnant": isl.stagnant}
        return self.last_report

    def _stagnation(self, isl: Island):
        """Resposta gradual à estagnação: mutação ↑ → novidade ↑ → imigrantes aleatórios → nova ilha → novo ambiente →
        reinício parcial (mantém elites)."""
        s = isl.stagnant
        step = None
        if s == 6:
            isl.sigma_mult = min(4.0, isl.sigma_mult * 2); step = "mutation_up"
        elif s == 10:
            isl.novelty_w = min(2.0, isl.novelty_w * 2); step = "novelty_up"
        elif s == 14:
            n = len(isl.pop) // 4
            isl.pop[-n:] = self.L.random(n, self.gen).cpu().numpy(); isl.fit[-n:] = -1e9; step = "random_immigrants"
        elif s == 18 and len(self.islands) < self.cfg.n_islands + 2:
            self._new_island(ISLAND_ENVS[len(self.islands) % len(ISLAND_ENVS)], 1); step = "new_island"
        elif s == 22 and not isl.focus:
            isl.spec = copy.deepcopy(ISLAND_ENVS[int(self.rng.integers(0, len(ISLAND_ENVS)))])
            isl.per_arena = 2 if isl.spec.task == "coop" else 1
            isl.fit[:] = -1e9; step = "new_environment"
        elif s >= 26:
            keep = max(2, len(isl.pop) // 6)
            order = np.argsort(-isl.fit)
            isl.pop = isl.pop[order]; isl.fit = isl.fit[order]
            isl.pop[keep:] = self.L.random(len(isl.pop) - keep, self.gen).cpu().numpy()
            isl.fit[keep:] = -1e9
            isl.stagnant, isl.sigma_mult, isl.novelty_w = 0, 1.0, 0.3
            step = "partial_restart"
        if step:
            ev = {"t": iso_now(), "type": "stagnation_response", "island": isl.iid, "step": step}
            self.events.append(ev)
            if self.logger:
                self.logger.event("qd_stagnation", **ev)

    # ------------------------------------------------------------------ acesso
    def best_genomes(self, k: int) -> Tuple[np.ndarray, List[str]]:
        pool = []
        for isl in self.islands:
            ok = isl.fit > -1e8
            for i in np.nonzero(ok)[0]:
                pool.append((float(isl.fit[i]), isl.pop[i], isl.gids[i]))
        pool.sort(key=lambda x: -x[0])
        sel = pool[:k]
        if not sel:
            return np.zeros((0, self.L.P), np.float32), []
        return np.stack([s[1] for s in sel]), [s[2] for s in sel]

    def elites_for_founding(self, k: int) -> np.ndarray:
        G, _ = self.best_genomes(k)
        if self.map.cells:
            extra = self.map.sample(max(0, k - len(G)), self.rng)
            if extra is not None and len(extra):
                G = np.concatenate([G, extra]) if len(G) else extra
        return G[:k]

    def summary(self) -> Dict[str, Any]:
        return {"islands": [{"id": i.iid, "task": i.spec.task, "focus": i.focus, "gen": i.generation,
                             "best": float(i.best) if i.best > -1e8 else None, "stagnant": i.stagnant,
                             "history": [float(x) for x in i.history[-60:]]} for i in self.islands],
                "coverage": self.map.coverage(), "qd_score": self.map.qd_score(), "cells": len(self.map.cells),
                "evaluations": self.evaluations, "generations": self.generations, "focus": self.focus_level,
                "bandit": self.bandit.stats(), "novelty_archive": int(self.novelty.items.shape[0]),
                "eval_rate": self.evaluations / max(1e-6, self.eval_seconds)}

    def state_dict(self):
        return {"map": self.map.state_dict(), "novelty": self.novelty.items, "novelty_seen": self.novelty.seen,
                "bandit_n": self.bandit.n, "bandit_r": self.bandit.r, "bandit_t": self.bandit.t,
                "focus": self.focus_level, "evaluations": self.evaluations, "generations": self.generations,
                "rr": self._rr, "rng": self.rng.bit_generator.state, "torch_rng": self.gen.get_state().cpu().numpy(),
                "islands": [{"iid": i.iid, "spec": asdict(i.spec), "per_arena": i.per_arena, "pop": i.pop, "fit": i.fit,
                             "qvec": i.qvec, "desc": i.desc, "gids": i.gids, "generation": i.generation,
                             "best": i.best, "stagnant": i.stagnant, "sigma_mult": i.sigma_mult,
                             "novelty_w": i.novelty_w, "focus": i.focus, "history": i.history} for i in self.islands]}

    def load_state_dict(self, s):
        self.map.load_state_dict(s["map"])
        self.novelty.items = np.asarray(s["novelty"], np.float32); self.novelty.seen = int(s["novelty_seen"])
        self.bandit.n, self.bandit.r, self.bandit.t = np.asarray(s["bandit_n"]), np.asarray(s["bandit_r"]), int(s["bandit_t"])
        self.focus_level = s["focus"]; self.evaluations = int(s["evaluations"]); self.generations = int(s["generations"])
        self._rr = int(s["rr"])
        try:
            self.rng.bit_generator.state = s["rng"]
            self.gen.set_state(torch.as_tensor(np.asarray(s["torch_rng"]), dtype=torch.uint8))
        except Exception as e:
            if self.logger:
                self.logger.event("qd_rng_restore_failed", error=repr(e))
        self.islands = []
        for d in s["islands"]:
            sp = ArenaSpec(**{k: v for k, v in d["spec"].items() if k in ArenaSpec.__dataclass_fields__})
            self.islands.append(Island(iid=int(d["iid"]), spec=sp, per_arena=int(d["per_arena"]),
                                       pop=np.asarray(d["pop"], np.float32), fit=np.asarray(d["fit"], np.float32),
                                       qvec=np.asarray(d["qvec"], np.float32), desc=np.asarray(d["desc"], np.float32),
                                       gids=list(d["gids"]), generation=int(d["generation"]), best=float(d["best"]),
                                       stagnant=int(d["stagnant"]), sigma_mult=float(d["sigma_mult"]),
                                       novelty_w=float(d["novelty_w"]), focus=d["focus"],
                                       history=[float(x) for x in d["history"]]))

## 14 — VALIDATION ENGINE (congelado): escada H0–H12, consistência multiescala, controles
A evolução **não** altera thresholds, controles, testes, estatística ou regras de evidência: o hash congelado de
(versão + thresholds + protocolo) é conferido antes de cada probe. Sementes de **holdout** (espaço separado do usado
pelas ilhas QD). Cada nível tem controle negativo (e positivo quando possível, nos testes de boot).

| nível | pergunta | controle |
|---|---|---|
| H0 desenvolvimento organizado | corpo não-trivial, canalizado e diferenciado | genomas aleatórios |
| H1 corpo funcional | atuação produz locomoção | mesmo corpo sem inervação |
| H2 controle sensorimotor | sentir recurso melhora ingestão | sensores de recurso mascarados |
| H3 aprendizado na vida | melhora fim × início da vida | plasticidade congelada |
| H4 memória | pista passada decodificável com presente controlado | variante sem memória |
| H5 generalização | desempenho em mundo nunca visto | genoma aleatório no mesmo mundo |
| H6 modelo preditivo | estado interno prevê sensor futuro além do histórico sensorial | estado embaralhado entre arenas |
| H7 problema multietapas | pegar → aterrar → cruzar → comer | ação de depositar ablada |
| H8 aprendizado social | demonstrador treinado × demonstrador ingênuo | demonstrador sem treino |
| H9 comunicação emergente | emissor informado melhora receptor cego | emissor mudo / canais trocados |
| H10 transmissão cultural | informação adquirida socialmente persiste | geração não exposta |
| H11 cultura cumulativa | cadeia de gerações melhora mais que gerações isoladas | cadeias independentes |
| H12 adaptação aberta | linhagem se adapta a ambientes novos gerados | clones sem variação |

In [ ]:
# @title 14 — VALIDATION ENGINE (congelado): escada H0–H12, consistência multiescala, controles { display-mode: "form" }
H_LEVELS = ["H0 ORGANIZED DEVELOPMENT", "H1 FUNCTIONAL BODY", "H2 SENSORIMOTOR CONTROL", "H3 WITHIN-LIFETIME LEARNING",
            "H4 MEMORY", "H5 GENERALIZATION", "H6 PREDICTIVE MODEL", "H7 MULTISTEP PROBLEM SOLVING",
            "H8 SOCIAL LEARNING", "H9 EMERGENT COMMUNICATION", "H10 CULTURAL TRANSMISSION", "H11 CUMULATIVE CULTURE",
            "H12 OPEN-ENDED ADAPTATION"]
H_CODES = [l.split()[0] for l in H_LEVELS]
LEVEL_NAME = {**{l.split()[0]: l for l in H_LEVELS}, **{l.split()[0]: l for l in A_LEVELS}}


class ValidationEngine:
    def __init__(self, cfg: CivConfig, layout: GenomeLayout, runner: ArenaRunner, affect: AffectEngine,
                 device: torch.device):
        self.cfg, self.L, self.runner, self.affect, self.device = cfg, layout, runner, affect, device
        self.dev, self.body = runner.dev, runner.body
        self.frozen = self._protocol_hash()
        self.runs = 0
        self.seconds = 0.0

    def _protocol_hash(self) -> str:
        return hash_obj({"version": VALIDATION_ENGINE_VERSION, "thresholds": asdict(self.cfg.thresholds),
                         "probe": asdict(self.cfg.probe), "levels": H_LEVELS + A_LEVELS})[:16]

    def verify_frozen(self):
        h = self._protocol_hash()
        if h != self.frozen:
            raise RuntimeError(f"Validation Engine alterado em execução (hash {h} ≠ congelado {self.frozen}) — bloqueado")

    def run_gen(self, code: str, G: torch.Tensor, key: str = "") -> Generator:
        self.verify_frozen()
        t0 = time.time()
        G = G.view(1, -1).to(self.device)
        key = key or genome_hash(G)
        if code.startswith("A"):
            res = (self.affect.composite_A9(G) if code == "A9" else (yield from self.affect.probe_gen(code, G, key)))
        elif code == "MS":
            res = yield from self._probe_MS(G, key)
        else:
            res = yield from getattr(self, f"_probe_{code}")(G, key)
        self.runs += 1
        self.seconds += time.time() - t0
        return res

    # ------------------------------------------------------------------ utilidades
    def _spec_forage(self) -> ArenaSpec:
        return ArenaSpec(task="forage", n_patches=3, resource_floor=0.05)

    def _run(self, Gs, spec, seed, ticks, ctrl_fn=None, record=("gain",), per_arena=1, W=None, x=None, name="",
             bodies_fn=None, energy=0.8):
        tr = self.runner.build(Gs, spec, seed, per_arena=per_arena, W_override=W, x_override=x, energy=energy)
        bodies = bodies_fn(tr) if bodies_fn else None
        rec = yield from self.runner.run_gen(tr, ticks, ctrl_fn=ctrl_fn, record=record, name=name, bodies=bodies)
        return tr, rec

    def _rep(self, G, n):
        return G.expand(n, -1).contiguous()

    # ------------------------------------------------------------------ H0–H1 (corpo)
    def _probe_H0(self, G, key):
        t0, th = time.time(), self.cfg.thresholds
        k = max(6, 2 * self.cfg.probe.dev_noise_samples)
        seeds = probe_seeds("H0", key, k)
        sd = torch.tensor([s % 100003 for s in seeds], device=self.device)
        bp = self.dev.develop(self._rep(G, k), sd)
        occ = bp.occ.float()
        nodes = occ.sum(1)
        iou = (occ @ occ.T) / ((occ.unsqueeze(1) + occ.unsqueeze(0)).clamp(max=1).sum(-1) + 1e-9)
        canal = ((iou.sum(1) - 1) / max(1, k - 1))

        def differentiation(b):
            m = b.occ.unsqueeze(-1).float()
            mu = (b.props * m).sum(1) / m.sum(1).clamp(min=1)
            var = (((b.props - mu.unsqueeze(1)) ** 2) * m).sum(1) / m.sum(1).clamp(min=1)
            return var.sqrt().mean(1)
        diff_c = differentiation(bp)
        org = (canal * diff_c).cpu().numpy()
        gen = torch.Generator(device=self.device); gen.manual_seed(derive_seed("H0ctrl", key))
        Rg = self.L.random(16, gen)
        ctrl = []
        for j in range(3):
            ctrl.append(self.dev.develop(Rg, torch.full((16,), 101 + j, device=self.device)))
        co = torch.stack([c.occ.float() for c in ctrl])                           # [3,16,N]
        ciou = []
        for a in range(3):
            for b in range(a + 1, 3):
                ciou.append((co[a] * co[b]).sum(-1) / ((co[a] + co[b]).clamp(max=1).sum(-1) + 1e-9))
        ccanal = torch.stack(ciou).mean(0)
        corg = (ccanal * differentiation(ctrl[0])).cpu().numpy()
        st = compare_greater(org, corg, th)
        N = self.dev.N
        nontrivial = float(((nodes >= th.dev_min_nodes) & (nodes <= 0.9 * N)).float().mean())
        ok = st["passed"] and nontrivial >= 0.8 and float(canal.mean()) >= th.dev_canalization
        sym = self.dev.symmetry(bp.occ)
        yield
        return result("H0", "H0 ORGANIZED DEVELOPMENT", ok, st, k,
                      {"nodes_mean": float(nodes.mean()), "nontrivial_fraction": nontrivial, "canalization": float(canal.mean()),
                       "differentiation": float(diff_c.mean()), "bilateral_x": float(sym["bilateral_x"].mean()),
                       "radial_z": float(sym["radial_z"].mean())},
                      {"negative": "16 random genomes (organization = canalization × property differentiation)"},
                      seeds, G, t0, self.cfg.thresholds_hash())

    def _probe_H1(self, G, key):
        t0, th = time.time(), self.cfg.thresholds
        k = 8
        seeds = probe_seeds("H1", key, k)
        sd = torch.tensor([s % 100003 for s in seeds], device=self.device)
        cap = self.body.measure_capability(self._rep(G, k), sd)
        yield
        capp = self.body.measure_capability(passive_body(self.L, self._rep(G, k)), sd)
        yield
        st = compare_greater(cap[:, 0].cpu().numpy(), capp[:, 0].cpu().numpy(), th)
        # robustez funcional a dano: remove ~25% dos nós (lado +x) e mede a marcha restante
        bp = self.dev.develop(self._rep(G, k), sd)
        gx = self.dev.grid[:, 0].float()
        cut = torch.quantile(gx[bp.occ[0]], 0.75) if bool(bp.occ[0].any()) else gx.max()
        dmg = bp.occ & (gx.unsqueeze(0) >= cut) & (bp.occ.sum(1, keepdim=True) > 2)
        bp.occ = bp.occ & ~dmg
        capd = self.body.measure_blueprint(bp)
        retained = float((capd[:, 0].mean() / cap[:, 0].mean().clamp(min=1e-9)).clamp(max=5))
        ok = st["passed"] and st["mean_a"] >= th.body_min_speed
        return result("H1", "H1 FUNCTIONAL BODY", ok, st, k, {"speed": st["mean_a"], "passive_speed": st["mean_b"],
                                                             "cost": float(cap[:, 1].mean()), "nodes": float(cap[:, 2].mean()),
                                                             "damage_speed_retained": retained},
                      {"negative": "same developed body, motor innervation removed (passive)",
                       "limitation": "no structural regrowth after damage is modeled (functional robustness only)"},
                      seeds, G, t0, self.cfg.thresholds_hash())

    # ------------------------------------------------------------------ H2–H7 (mente individual)
    def _probe_H2(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        seed = probe_seeds("H2", key, 1)[0]
        T = self.cfg.probe.arena_ticks
        tr, ri = yield from self._run(self._rep(G, n), self._spec_forage(), seed, T, record=("gain", "pos"), name="H2 intact")
        mask = sensor_mask(n, SENSOR_RESOURCE, self.device)
        _, rm = yield from self._run(self._rep(G, n), self._spec_forage(), seed, T, name="H2 masked",
                                     ctrl_fn=lambda t, tr_: ArenaControl(iv=MindIntervention(sensor_mask=mask)))
        a, b = ri["gain"].sum(0).cpu().numpy(), rm["gain"].sum(0).cpu().numpy()
        st = compare_greater(a, b, th)
        gain = (st["mean_a"] - st["mean_b"]) / max(1e-6, abs(st["mean_b"]) + 1e-3)
        ok = st["passed"] and gain >= th.sensorimotor_gain
        return result("H2", "H2 SENSORIMOTOR CONTROL", ok, st, n, {"intake_intact": st["mean_a"], "intake_masked": st["mean_b"],
                                                                  "relative_gain": gain},
                      {"negative": "resource sensors masked, same layouts/seeds"}, [seed], G, t0, self.cfg.thresholds_hash(),
                      replay=make_replay(tr, ri))

    def _probe_H3(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        seed = probe_seeds("H3", key, 1)[0]
        T = 2 * self.cfg.probe.arena_ticks
        spec = ArenaSpec(task="learn", n_patches=2, regen=True, resource_floor=0.0)
        _, rp = yield from self._run(self._rep(G, n), spec, seed, T, name="H3 plastic")
        frz = torch.ones(n, dtype=torch.bool, device=self.device)
        _, rf = yield from self._run(self._rep(G, n), spec, seed, T, name="H3 frozen",
                                     ctrl_fn=lambda t, tr_: ArenaControl(iv=MindIntervention(freeze_plasticity=frz)))
        third = T // 3
        imp = lambda r: (r["gain"][-third:].sum(0) - r["gain"][:third].sum(0)).cpu().numpy()
        early = float(rp["gain"][:third].sum(0).mean())
        a, b = imp(rp), imp(rf)
        st = compare_greater(a, b, th)
        ok = st["passed"] and (st["mean_a"] - st["mean_b"]) >= th.learning_gain * max(1e-3, early)
        return result("H3", "H3 WITHIN-LIFETIME LEARNING", ok, st, n, {"improvement_plastic": st["mean_a"],
                                                                      "improvement_frozen": st["mean_b"], "early_intake": early},
                      {"negative": "plasticity frozen (identical layouts, identical babbling noise stream)"}, [seed], G, t0,
                      self.cfg.thresholds_hash())

    def _memory_decode(self, G, seed, delays, n, name):
        cue_t = 6
        T = cue_t + max(delays) + 2
        lab = torch.arange(n, device=self.device) % 2
        m0 = torch.zeros(n, N_SENSORS, device=self.device)
        a0 = torch.zeros(n, N_SENSORS, device=self.device); a0[:, 0] = 1; a0[:, 23] = 0.6; a0[:, 24] = 0.9; a0[:, 4] = 1; a0[:, 5] = 0.6
        frz = torch.ones(n, dtype=torch.bool, device=self.device)

        def ctrl(t, tr):
            if t < cue_t:
                add = a0.clone(); add[:, 12] += (lab == 0).float() * 1.5; add[:, 13] += (lab == 1).float() * 1.5
                return ArenaControl(iv=MindIntervention(sensor_mask=m0, sensor_add=add), freeze_move=frz, mute=frz)
            return ArenaControl(iv=MindIntervention(sensor_mask=m0, sensor_add=a0), freeze_move=frz, mute=frz)
        spec = ArenaSpec(task="memory", n_patches=0, resource_floor=0.0)
        _, r = yield from self._run(G, spec, seed, T, ctrl_fn=ctrl, record=("x",), name=name)
        X = r["x"].cpu().numpy(); y = lab.cpu().numpy()
        out = {}
        for d in delays:
            out[d] = decode_report(X[cue_t + d], y, 2, self.cfg.probe.permutations, self.cfg.probe.folds, seed + d, nonlinear=False)
        return out

    def _probe_H4(self, G, key):
        t0, th = time.time(), self.cfg.thresholds
        n = 4 * self.cfg.probe.trials
        seed = probe_seeds("H4", key, 1)[0]
        delays = list(self.cfg.probe.memory_delays)
        dc = yield from self._memory_decode(self._rep(G, n), seed, delays, n, "H4 memory")
        dn = yield from self._memory_decode(memoryless(self.L, self._rep(G, n)), seed, delays, n, "H4 memoryless control")
        dmax = max(delays)
        acc, p = dc[dmax]["linear_acc"], dc[dmax]["linear_p"]
        nacc = norm_acc(acc, 0.5)
        ctrl_fail = not (dn[dmax]["linear_p"] < th.alpha and norm_acc(dn[dmax]["linear_acc"], 0.5) >= th.min_norm_acc)
        ok = p < th.alpha and nacc >= th.min_norm_acc and ctrl_fail
        st = {"p": p, "d": dc[dmax].get("effect_size", 0.0), "ci": (float("nan"), float("nan"))}
        return result("H4", "H4 MEMORY", ok, st, n, {"acc_by_delay": {str(d): dc[d]["linear_acc"] for d in delays},
                                                     "p_by_delay": {str(d): dc[d]["linear_p"] for d in delays},
                                                     "control_acc_by_delay": {str(d): dn[d]["linear_acc"] for d in delays},
                                                     "normalized_acc_longest": nacc},
                      {"present_controlled": "identical canonical input after cue, movement frozen, emission muted",
                       "negative": "memoryless variant (no recurrence, τ=1, no plasticity) must fail", "control_failed": ctrl_fail},
                      [seed], G, t0, self.cfg.thresholds_hash())

    def _probe_H5(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        s_train = probe_seeds("H5", key, 1, holdout=False)[0]
        s_hold = probe_seeds("H5", key, 1, holdout=True)[0]
        gen = torch.Generator(device=self.device); gen.manual_seed(derive_seed("H5rand", key))
        R = self.L.random(n, gen)
        train = ArenaSpec(task="forage", n_patches=3, resource_floor=0.1)
        novel = ArenaSpec(task="novel", n_patches=6, patch_radius=0, cold=0.3, storm=0.15, resource_floor=0.03)
        T = self.cfg.probe.arena_ticks
        _, rc_tr = yield from self._run(self._rep(G, n), train, s_train, T, name="H5 train-like")
        _, rr_tr = yield from self._run(R, train, s_train, T, name="H5 random train-like")
        _, rc_ho = yield from self._run(self._rep(G, n), novel, s_hold, T, name="H5 holdout")
        _, rr_ho = yield from self._run(R, novel, s_hold, T, name="H5 random holdout")
        tot = lambda r: r["gain"].sum(0).cpu().numpy()
        a, b = tot(rc_ho), tot(rr_ho)
        st = compare_greater(a, b, th)
        norm = lambda c, r: (np.mean(c) - np.mean(r)) / (abs(np.mean(r)) + 1e-3)
        ratio = norm(a, b) / max(1e-6, norm(tot(rc_tr), tot(rr_tr))) if norm(tot(rc_tr), tot(rr_tr)) > 0 else 0.0
        ok = st["passed"] and ratio >= 0.5
        return result("H5", "H5 GENERALIZATION", ok, st, n, {"holdout_intake": st["mean_a"], "holdout_random": st["mean_b"],
                                                            "generalization_ratio": ratio, "holdout_world": asdict(novel)},
                      {"negative": "random genomes in the same never-seen world", "holdout": "seed space never used by QD",
                       "holdout_surprise_rule": "ratio < 0.5 blocks promotion"}, [s_train, s_hold], G, t0, self.cfg.thresholds_hash())

    def _probe_H6(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        seed = probe_seeds("H6", key, 1)[0]
        T = self.cfg.probe.arena_ticks
        _, r = yield from self._run(self._rep(G, n), self._spec_forage(), seed, T, record=("x", "sense_cache"), name="H6 predict")
        _, rm = yield from self._run(memoryless(self.L, self._rep(G, n)), self._spec_forage(), seed, T,
                                     record=("x", "sense_cache"), name="H6 memoryless")
        k = 3
        ch = [1, 2, 3, 23]

        def gain_of(rec, perm=None):
            X = rec["x"].cpu().numpy(); S = rec["sense_cache"].cpu().numpy()
            ts = np.arange(2, T - k)
            if perm is not None:
                X = X[:, perm]
            base = np.concatenate([S[ts], S[ts - 1], S[ts - 2]], -1)          # [t, n, 3I]
            full = np.concatenate([base, X[ts]], -1)
            Y = S[ts + k][..., ch]
            g = np.broadcast_to(np.arange(n)[None, :], (len(ts), n))
            fl = lambda A: A.reshape(-1, A.shape[-1])
            r2b = ridge_r2_cv(fl(base), fl(Y), g.reshape(-1), lam=5.0)
            r2f = ridge_r2_cv(fl(full), fl(Y), g.reshape(-1), lam=5.0)
            return r2f - r2b, r2f, r2b
        gobs, r2f, r2b = gain_of(r)
        rng = np.random.default_rng(seed)
        null = [gain_of(r, rng.permutation(n))[0] for _ in range(max(10, self.cfg.probe.permutations // 3))]
        p = (sum(1 for v in null if v >= gobs) + 1) / (len(null) + 1)
        gm = gain_of(rm)[0]
        ok = gobs >= th.prediction_gain and p < th.alpha
        st = {"p": p, "d": (gobs - float(np.mean(null))) / (float(np.std(null)) + 1e-9), "ci": (float("nan"), float("nan"))}
        return result("H6", "H6 PREDICTIVE MODEL", ok, st, n, {"r2_gain": gobs, "r2_full": r2f, "r2_sensor_history": r2b,
                                                              "memoryless_gain": gm, "horizon": k},
                      {"baseline": "current + 2 past sensor frames", "negative": "internal state shuffled across arenas",
                       "memoryless": "reported"}, [seed], G, t0, self.cfg.thresholds_hash())

    def _probe_H7(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        seed = probe_seeds("H7", key, 1)[0]
        T = 2 * self.cfg.probe.arena_ticks
        spec = ArenaSpec(task="barrier", water_barrier=True, deposit=3.0, spawn_side="left", n_patches=3, resource_floor=0.0)
        tr, ri = yield from self._run(self._rep(G, n), spec, seed, T, record=("gain", "pos"), name="H7 intact")
        crossed_i = ArenaRunner.crossed(tr).cpu().numpy(); tech_i = tr.pool.tech[:n].cpu().numpy()
        blk = torch.zeros(n, N_ACTIONS, dtype=torch.bool, device=self.device); blk[:, ACTION_PLACE] = True
        tr2, ra = yield from self._run(self._rep(G, n), spec, seed, T, ctrl_fn=lambda t, x: ArenaControl(action_block=blk),
                                       name="H7 place-ablated")
        crossed_a = ArenaRunner.crossed(tr2).cpu().numpy()
        k1, k2 = int(crossed_i.sum()), int(crossed_a.sum())
        p = fisher_greater(k1, n, k2, n)
        seq = int(((tech_i & 1) > 0)[crossed_i].sum()) if crossed_i.any() else 0
        ok = p < th.alpha and k1 / n >= 0.3 and seq >= 1
        st = {"p": p, "d": (k1 - k2) / n, "ci": (float("nan"), float("nan"))}
        return result("H7", "H7 MULTISTEP PROBLEM SOLVING", ok, st, n,
                      {"success_intact": k1 / n, "success_ablated": k2 / n, "crossers_who_filled_water": seq,
                       "tool_test": "material placed in water causally enables reaching resource"},
                      {"negative": "place action ablated (cannot fill water)", "test": "one-sided Fisher exact"},
                      [seed], G, t0, self.cfg.thresholds_hash(), replay=make_replay(tr, ri))

    # ------------------------------------------------------------------ H8–H11 (social / cultura)
    def _train(self, G, n, spec, seed, T, name):
        tr, _ = yield from self._run(self._rep(G, n), spec, seed, T, record=(), name=name)
        return tr.pool.W[:n].clone()

    def _social_phase(self, G, n, spec, seed, T, W_model, W_naive, name, freeze_naive=False):
        """Pares (modelo, aprendiz) por arena; devolve W do aprendiz após a fase."""
        Gs = self._rep(G, 2 * n)
        W = torch.empty(2 * n, *W_model.shape[1:], device=self.device)
        W[0::2], W[1::2] = W_model, W_naive
        ctrl_fn = None
        if freeze_naive:
            fz = torch.zeros(2 * n, dtype=torch.bool, device=self.device); fz[1::2] = True
            ctrl_fn = lambda t, tr: ArenaControl(iv=MindIntervention(freeze_plasticity=fz))
        tr, r = yield from self._run(Gs, spec, seed, T, per_arena=2, W=W, ctrl_fn=ctrl_fn, record=("emit", "speed"), name=name)
        return tr.pool.W[1::2].clone(), tr.pool.signature[1::2].clone(), r

    def _probe_H8(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        seed = probe_seeds("H8", key, 1)[0]
        T = self.cfg.probe.arena_ticks
        spec = ArenaSpec(task="social_learning", n_patches=2, regen=True, resource_floor=0.0)
        W0 = self.L.view(self._rep(G, n), "ctl_W0").clone()
        Wm = yield from self._train(G, n, spec, seed, T, "H8 demonstrator training")
        Wd, _, rd = yield from self._social_phase(G, n, spec, seed, T, Wm, W0, "H8 demonstration")
        Ws, _, rs = yield from self._social_phase(G, n, spec, seed, T, W0, W0, "H8 sham demonstration")
        _, rt = yield from self._run(self._rep(G, n), spec, seed, T, W=Wd, name="H8 naive test (demonstrated)")
        _, rc = yield from self._run(self._rep(G, n), spec, seed, T, W=Ws, name="H8 naive test (sham)")
        a, b = rt["gain"].sum(0).cpu().numpy(), rc["gain"].sum(0).cpu().numpy()
        st = compare_greater(a, b, th)
        teach = float((rd["emit"][:, 0::2].abs().sum(-1).mean() - rs["emit"][:, 0::2].abs().sum(-1).mean()).item())
        return result("H8", "H8 SOCIAL LEARNING", st["passed"], st, n, {"naive_intake_demonstrated": st["mean_a"],
                                                                       "naive_intake_sham": st["mean_b"],
                                                                       "teaching_candidate_emission_delta": teach},
                      {"negative": "sham demonstrator (same genome, untrained weights)",
                       "isolation": "test phase alone, fresh resources, transient state reset (only learned weights carried)"},
                      [seed], G, t0, self.cfg.thresholds_hash())

    def _probe_H9(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        seed = probe_seeds("H9", key, 1)[0]
        T = self.cfg.probe.arena_ticks
        spec = ArenaSpec(task="signal", n_patches=2, resource_floor=0.0)
        Gs = self._rep(G, 2 * n)
        vis = torch.zeros(2 * n, dtype=torch.bool, device=self.device); vis[0::2] = True
        mute = vis.clone()
        recv = ~vis
        perm = torch.tensor([1, 0, 3, 2], device=self.device)

        def mk(kind):
            def f(t, tr):
                c = ArenaControl(resource_visible=vis)
                if kind == "muted":
                    c.mute = mute
                if kind == "swapped":
                    c.sound_override = lambda snd: torch.where(recv[:snd.shape[0]].unsqueeze(1), snd[:, perm], snd)
                return c
            return f
        tr, ri = yield from self._run(Gs, spec, seed, T, per_arena=2, ctrl_fn=mk("intact"),
                                      record=("gain", "emit", "sense_cache", "pos"), name="H9 intact")
        _, rm = yield from self._run(Gs, spec, seed, T, per_arena=2, ctrl_fn=mk("muted"), record=("gain",), name="H9 sender muted")
        _, rs = yield from self._run(Gs, spec, seed, T, per_arena=2, ctrl_fn=mk("swapped"), record=("gain",), name="H9 swapped")
        g = lambda r: r["gain"][:, 1::2].sum(0).cpu().numpy()
        st = compare_greater(g(ri), g(rm), th)
        sw = compare_greater(g(ri), g(rs), th)
        E = ri["emit"][:, 0::2].cpu().numpy(); S = ri["sense_cache"][:, 0::2].cpu().numpy()
        loud = E.max(-1) > 0.3
        sig = np.where(loud, E.argmax(-1), 4).reshape(-1)
        world = np.digitize(S[..., 1], [0.05, 0.3]).reshape(-1)
        rng = np.random.default_rng(seed)
        mi, pmi = mi_permutation(sig, world, max(20, self.cfg.probe.permutations), rng)
        ok = st["passed"] and pmi < th.alpha
        return result("H9", "H9 EMERGENT COMMUNICATION", ok, st, n,
                      {"receiver_intake_intact": st["mean_a"], "receiver_intake_muted": st["mean_b"],
                       "receiver_intake_swapped": sw["mean_b"], "swap_p": sw["p"], "mi_signal_world_bits": mi, "mi_p": pmi,
                       "sender_loud_fraction": float(loud.mean())},
                      {"negative": "sender muted (sound channel interrupted)", "swap": "receiver channels permuted",
                       "grounding": "I(sender signal; sender's resource state) with permutation null"},
                      [seed], G, t0, self.cfg.thresholds_hash(), replay=make_replay(tr, ri))

    def _probe_H10(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        seed = probe_seeds("H10", key, 1)[0]
        T = self.cfg.probe.arena_ticks
        spec = ArenaSpec(task="culture", n_patches=2, regen=True, resource_floor=0.0)
        W0 = self.L.view(self._rep(G, n), "ctl_W0").clone()
        Wm = yield from self._train(G, n, spec, seed, T, "H10 model generation")
        tr_m, _ = yield from self._run(self._rep(G, n), spec, seed + 1, 2 * T, W=Wm, record=(), name="H10 model signature")
        sig_m = tr_m.pool.signature[:n].clone()
        We, _, _ = yield from self._social_phase(G, n, spec, seed, T, Wm, W0, "H10 exposure")
        Wu = yield from self._train(G, n, spec, seed + 7, T, "H10 unexposed (learns alone for the same time)")
        tr_e, re_ = yield from self._run(self._rep(G, n), spec, seed + 1, 2 * T, W=We, name="H10 persistence exposed")
        tr_u, ru = yield from self._run(self._rep(G, n), spec, seed + 1, 2 * T, W=Wu, name="H10 persistence unexposed")
        late = lambda r: r["gain"][T:].sum(0).cpu().numpy()
        st = compare_greater(late(re_), late(ru), th)
        de = (tr_e.pool.signature[:n] - sig_m).norm(dim=-1).cpu().numpy()
        du = (tr_u.pool.signature[:n] - sig_m).norm(dim=-1).cpu().numpy()
        sim = compare_greater(du, de, th, min_effect=0.3)
        ok = st["passed"] and sim["p"] < th.alpha
        return result("H10", "H10 CULTURAL TRANSMISSION", ok, st, n,
                      {"late_intake_exposed": st["mean_a"], "late_intake_unexposed": st["mean_b"],
                       "behavior_distance_to_model_exposed": float(de.mean()), "behavior_distance_unexposed": float(du.mean()),
                       "similarity_p": sim["p"]},
                      {"negative": "same-age generation not exposed (learned alone for the same time)",
                       "persistence": "measured in the late half of a 2T solo phase after the model is removed"},
                      [seed], G, t0, self.cfg.thresholds_hash())

    def _probe_H11(self, G, key):
        t0, th, n = time.time(), self.cfg.thresholds, self.cfg.probe.trials
        seed = probe_seeds("H11", key, 1)[0]
        T = self.cfg.probe.arena_ticks
        spec = ArenaSpec(task="cumulative", n_patches=3, regen=True, resource_floor=0.0)
        W0 = self.L.view(self._rep(G, n), "ctl_W0").clone()
        K = 3
        chain, indep = [], []
        Wc = yield from self._train(G, n, spec, seed, T, "H11 chain gen0")
        Wi = Wc.clone()
        for g in range(1, K + 1):
            Wl, _, _ = yield from self._social_phase(G, n, spec, seed + g, T, Wc, W0, f"H11 chain gen{g} learning")
            trc, rc = yield from self._run(self._rep(G, n), spec, seed + 100 + g, T, W=Wl, name=f"H11 chain gen{g} solo")
            chain.append(rc["gain"].sum(0).cpu().numpy())
            Wc = trc.pool.W[:n].clone()
            Wil = yield from self._train(G, n, spec, seed + 50 + g, T, f"H11 independent gen{g}")
            _, ri = yield from self._run(self._rep(G, n), spec, seed + 100 + g, T, W=Wil, name=f"H11 independent gen{g} solo")
            indep.append(ri["gain"].sum(0).cpu().numpy())
        C, I = np.stack(chain), np.stack(indep)                       # [K, n]
        xs = np.arange(1, K + 1) - (K + 1) / 2
        slope = lambda M: (xs[:, None] * (M - M.mean(0))).sum(0) / (xs ** 2).sum()
        st = compare_greater(slope(C), slope(I), th)
        ok = st["passed"] and float(C[-1].mean()) > float(I[-1].mean())
        return result("H11", "H11 CUMULATIVE CULTURE", ok, st, n, {"chain_by_gen": C.mean(1).tolist(),
                                                                  "independent_by_gen": I.mean(1).tolist()},
                      {"negative": "independent generations (same learning time, no transmission)"},
                      [seed], G, t0, self.cfg.thresholds_hash())

    def _probe_H12(self, G, key):
        """Adaptação aberta (nível de POPULAÇÃO, por seleção — não é aprendizado individual): em ambientes novos gerados
        proceduralmente, linhagens com variação melhoram mais que clones sem variação."""
        t0, th = time.time(), self.cfg.thresholds
        seeds = probe_seeds("H12", key, 3)
        envs = [ArenaSpec(task="gen_cold", n_patches=2, cold=0.7, deposit=2.0, resource_floor=0.05),
                ArenaSpec(task="gen_barrier", water_barrier=True, deposit=3.0, spawn_side="left", n_patches=3, resource_floor=0.03),
                ArenaSpec(task="gen_storm", n_patches=4, storm=0.5, deposit=2.0, resource_floor=0.05)]
        gens, pop = 4, max(8, self.cfg.probe.trials)
        gen = torch.Generator(device=self.device); gen.manual_seed(derive_seed("H12", key))
        imps_var, imps_clone = [], []
        for env, sd in zip(envs, seeds):
            for variant in ("var", "clone"):
                P = self._rep(G, pop).clone()
                best0, bestN = None, None
                for g in range(gens):
                    tr, r = yield from self._run(P, env, sd + g, self.cfg.probe.arena_ticks, record=("gain", "alive"),
                                                 name=f"H12 {env.task} {variant} g{g}")
                    fit = (r["gain"].sum(0) + 0.5 * r["alive"].float().mean(0))
                    if g == 0:
                        best0 = float(fit.mean())
                    bestN = float(fit.mean())
                    order = torch.argsort(fit, descending=True)
                    elite = P[order[: max(2, pop // 4)]]
                    rep = elite[torch.randint(0, elite.shape[0], (pop,), generator=gen, device=self.device)]
                    P = self.L.mutate(rep, gen) if variant == "var" else self._rep(G, pop).clone()
                (imps_var if variant == "var" else imps_clone).append(bestN - best0)
        a, b = np.asarray(imps_var), np.asarray(imps_clone)
        wins = int((a > b).sum())
        st = {"p": float(binom_p_greater(wins, len(a), 0.5)), "d": float((a - b).mean() / (np.std(a - b) + 1e-6)),
              "ci": (float("nan"), float("nan"))}
        ok = wins == len(a) and float((a - b).mean()) > 0
        return result("H12", "H12 OPEN-ENDED ADAPTATION", ok, st, len(a), {"improvement_variation": a.tolist(),
                                                                         "improvement_clones": b.tolist(), "wins": wins},
                      {"negative": "clonal lineage without variation in the same generated environments",
                       "scope": "population-level adaptation by selection (not individual learning)"},
                      seeds, G, t0, self.cfg.thresholds_hash(), claim_pass="SIMULATION HYPOTHESIS",
                      notes="Escopo limitado a 3 ambientes gerados; abertura ilimitada não é demonstrável num teste finito.")

    # ------------------------------------------------------------------ consistência multiescala (F1 × F3)
    def _probe_MS(self, G, key):
        t0, n = time.time(), self.cfg.probe.trials
        seed = probe_seeds("MS", key, 1)[0]
        T = self.cfg.probe.arena_ticks
        spec = self._spec_forage()
        _, r1 = yield from self._run(self._rep(G, n), spec, seed, T, record=("gain", "speed"), name="MS F1")

        def bodies_fn(tr):
            reg = BodyRegistry(self.cfg, self.dev, self.body, tr.n, self.cfg.body_substeps)
            sd = torch.tensor([s % 100003 for s in probe_seeds("MSdev", key, tr.n)], device=self.device)
            bs = reg.attach(torch.arange(tr.n, device=self.device), tr.pool.aid[:tr.n], tr.pool.genome[:tr.n], sd)
            tr.pool.body_slot[:tr.n] = bs.int(); tr.pool.fidelity[:tr.n] = 3
            return {3: reg}
        _, r3 = yield from self._run(self._rep(G, n), spec, seed, T, record=("gain", "speed"), name="MS F3", bodies_fn=bodies_fn)
        d1, d3 = r1["speed"].sum(0).cpu().numpy(), r3["speed"].sum(0).cpu().numpy()
        g1, g3 = r1["gain"].sum(0).cpu().numpy(), r3["gain"].sum(0).cpu().numpy()
        rel = lambda a, b: float(abs(np.mean(a) - np.mean(b)) / (abs(np.mean(a)) + 1e-6))
        err_d, err_g = rel(d1, d3), rel(g1, g3)
        ok = err_d <= 0.5 and err_g <= 0.5
        st = {"p": float("nan"), "d": err_d, "ci": (float("nan"), float("nan"))}
        return result("MS", "MULTISCALE CONSISTENCY F1×F3", ok, st, n,
                      {"path_F1": float(d1.mean()), "path_F3": float(d3.mean()), "rel_error_path": err_d,
                       "intake_F1": float(g1.mean()), "intake_F3": float(g3.mean()), "rel_error_intake": err_g},
                      {"design": "same genome, same layouts; F1 surrogate vs full developmental body physics"},
                      [seed], G, t0, self.cfg.thresholds_hash(), claim_pass="BENCHMARKED")

## 15 — SCIENTIST ENGINE · ARCHIVE ENGINE · EXPERIMENT DSL
**Scientist**: decide quais candidatos merecem testes caros. Fila **limitada** com prioridade adaptativa (nível,
novidade, incerteza, valor de informação, importância histórica, idade), *gating* da escada (pré-requisitos) com
triagem exploratória do próximo nível, orçamento adaptativo (fila cheia → mais validação; vazia → mais
exploração) e respiração científica `EXPLORE → FIND → INVESTIGATE → VALIDATE → EXPLORE`.
Princípio de descoberta: passou na triagem → **tentar quebrar** em sementes de holdout independentes (replicação
multi-seed) → só então promover a descoberta (pacote completo + Pantheon revogável). Fitness ≠ evidência.
**ArchiveEngine**: zoológico de candidatos, arquivo de falhas (resultados negativos são salvos), arquivo de
estranheza (anomalias nunca são todas descartadas), Pantheon multi-eixo, pacotes de descoberta.
**DSL**: experimentos só compõem operações seguras (`perturb freeze ablate duplicate fork change_environment observe
measure`) com validação de esquema; nunca Python arbitrário; opera só no universo simulado.

In [ ]:
# @title 15 — SCIENTIST ENGINE · ARCHIVE ENGINE · EXPERIMENT DSL { display-mode: "form" }
LEVEL_WEIGHT = {c: i + 1 for i, c in enumerate(H_CODES)}
LEVEL_WEIGHT.update({c: 2 + i for i, c in enumerate(A_CODES)})
PANTHEON_AXES = {"development": ["H0", "H1"], "learning": ["H3"], "memory": ["H4"], "generalization": ["H5", "H6"],
                 "problem_solving": ["H7"], "affect": A_CODES, "communication": ["H9"], "culture": ["H8", "H10", "H11"],
                 "adaptation": ["H12"]}


@dataclass
class Candidate:
    cid: str
    genome: np.ndarray
    origin: str
    first_seen: str
    tick: int = 0
    fitness: float = 0.0
    novelty: float = 0.0
    importance: float = 0.0
    passed: Dict[str, Dict[str, Any]] = field(default_factory=dict)
    failed: Dict[str, int] = field(default_factory=dict)
    attempts: Dict[str, int] = field(default_factory=dict)
    screened: Dict[str, Dict[str, Any]] = field(default_factory=dict)
    island: int = -1

    def evidence_score(self) -> float:
        return float(sum(LEVEL_WEIGHT.get(c, 1) * min(1.0, abs(v.get("effect", 0.0)) / 2 + 0.25) for c, v in self.passed.items()))

    def h_level(self) -> int:
        lvl = -1
        for i, c in enumerate(H_CODES):
            if c in self.passed:
                lvl = i
            else:
                break
        return lvl

    def a_level(self) -> int:
        lvl = -1
        for i, c in enumerate(A_CODES):
            if c in self.passed:
                lvl = i
            else:
                break
        return lvl


class ArchiveEngine:
    def __init__(self, paths: Optional[CivPaths], cfg: CivConfig):
        self.paths, self.cfg = paths, cfg
        self.zoo: "collections.OrderedDict[str, Candidate]" = collections.OrderedDict()
        self.failures: "collections.deque" = collections.deque(maxlen=5000)
        self.weird: "collections.deque" = collections.deque(maxlen=500)
        self.pantheon: Dict[str, Dict[str, Any]] = {}
        self.discoveries: List[Dict[str, Any]] = []
        self.convergence: List[Dict[str, Any]] = []
        self.overall_champion: Optional[Dict[str, Any]] = None

    def add_candidate(self, c: Candidate, cap: int = 3000):
        self.zoo[c.cid] = c
        self.zoo.move_to_end(c.cid)
        while len(self.zoo) > cap:
            # remove o menos importante, mas nunca quem tem evidência
            victim = next((k for k, v in self.zoo.items() if not v.passed), None)
            if victim is None:
                break
            del self.zoo[victim]

    def failure(self, cid: str, res: ProbeResult):
        self.failures.append({"cid": cid, "level": res.level, "p": res.p, "effect": res.effect, "ts": res.ts,
                              "metrics": {k: v for k, v in res.metrics.items() if not isinstance(v, (list, dict))}})

    def anomaly(self, kind: str, data: Dict[str, Any]):
        self.weird.append({"kind": kind, "ts": iso_now(), **to_jsonable(data)})

    # ------------------------------------------------------------------ pacote de descoberta
    def save_discovery(self, level: str, cand: Candidate, results: List[ProbeResult], cfg: CivConfig, tick: int,
                       kind: str = "individual", extra: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        did = f"D{len(self.discoveries) + 1:05d}-{level}-{uuid.uuid4().hex[:6]}"
        meta = {"id": did, "level": level, "level_name": LEVEL_NAME.get(level, level), "kind": kind, "candidate": cand.cid,
                "origin": cand.origin, "tick": tick, "ts": iso_now(), "claim": results[0].claim if results else "TESTED",
                "replications": len(results), "passed_replications": sum(r.passed for r in results),
                "p_values": [r.p for r in results], "effects": [r.effect for r in results],
                "holm_adjusted": holm_correction([r.p for r in results]), "versions": ENGINE_VERSIONS,
                "versions_hash": ENGINE_VERSIONS_HASH, "thresholds_hash": cfg.thresholds_hash(),
                "validation_protocol": VALIDATION_ENGINE_VERSION, "statement": "comportamento observado + mecanismo funcional "
                "demonstrado por probe com controles; não implica experiência subjetiva, vida real ou equivalência humana",
                **(extra or {})}
        meta["evidence"] = [r.public() for r in results]
        if self.paths is not None:
            d = os.path.join(self.paths.discoveries, did)
            os.makedirs(os.path.join(d, "plots"), exist_ok=True)
            np.save(os.path.join(d, "genome.npy"), cand.genome.astype(np.float32))
            files = {"metadata.json": meta, "seeds.json": {"seeds": [r.seeds for r in results]},
                     "config.json": cfg.to_dict(), "evidence.json": meta["evidence"],
                     "statistics.json": {"p": meta["p_values"], "effects": meta["effects"], "holm": meta["holm_adjusted"],
                                         "ci": [r.ci for r in results]},
                     "ablation.json": [r.controls for r in results], "version.json": ENGINE_VERSIONS}
            for name, obj in files.items():
                with open(os.path.join(d, name), "w") as f:
                    json.dump(to_jsonable(obj), f, indent=1)
            rp = next((r.replay for r in results if r.replay), None)
            if rp:
                np.savez_compressed(os.path.join(d, "replay.npz"), pos=rp["pos"], arena_of=rp["arena_of"],
                                    size=np.asarray(rp["size"]), ticks=np.asarray(rp["ticks"]))
                with open(os.path.join(d, "replay_layout.json"), "w") as f:
                    json.dump(to_jsonable(rp["layout"]), f)
            try:
                fig, ax = plt.subplots(1, 2, figsize=(8, 3))
                ax[0].bar(range(len(results)), [r.effect for r in results], color="#5b8def"); ax[0].set_title("effect size")
                ax[1].bar(range(len(results)), [-math.log10(max(1e-12, r.p)) if np.isfinite(r.p) else 0 for r in results],
                          color="#46c3a0")
                ax[1].axhline(-math.log10(cfg.thresholds.alpha), color="#e0614c", ls="--"); ax[1].set_title("-log10 p")
                fig.suptitle(f"{did}"); fig.tight_layout(); fig.savefig(os.path.join(d, "plots", "evidence.png"), dpi=90)
                plt.close(fig)
            except Exception as e:
                meta["plot_error"] = repr(e)
            hashes = {}
            for root, _, fs in os.walk(d):
                for fn in fs:
                    if fn == "hashes.json":
                        continue
                    pth = os.path.join(root, fn)
                    with open(pth, "rb") as f:
                        hashes[os.path.relpath(pth, d)] = hashlib.sha256(f.read()).hexdigest()
            with open(os.path.join(d, "hashes.json"), "w") as f:
                json.dump(hashes, f, indent=1)
            meta["path"] = d
        idx = {k: meta[k] for k in ("id", "level", "level_name", "kind", "candidate", "tick", "ts", "claim",
                                    "replications", "passed_replications")}
        idx["path"] = meta.get("path")
        idx["effects"] = meta["effects"]; idx["p_values"] = meta["p_values"]
        self.discoveries.append(idx)
        return meta

    def load_discovery(self, did: str) -> Optional[Dict[str, Any]]:
        if self.paths is None or not re.fullmatch(r"D\d{5}-[A-Z0-9]+-[0-9a-f]{6}", did or ""):
            return None
        d = os.path.join(self.paths.discoveries, did)
        if not os.path.isdir(d):
            return None
        out = json.load(open(os.path.join(d, "metadata.json")))
        out["genome"] = np.load(os.path.join(d, "genome.npy"), allow_pickle=False)
        rp = os.path.join(d, "replay.npz")
        if os.path.exists(rp):
            z = np.load(rp, allow_pickle=False)
            out["replay"] = {"pos": z["pos"], "arena_of": z["arena_of"], "size": int(z["size"]), "ticks": int(z["ticks"]),
                             "layout": json.load(open(os.path.join(d, "replay_layout.json")))}
        return out

    def verify_discovery(self, did: str) -> Dict[str, Any]:
        d = os.path.join(self.paths.discoveries, did) if self.paths else ""
        if not d or not os.path.isdir(d):
            return {"ok": False, "reason": "not found"}
        hashes = json.load(open(os.path.join(d, "hashes.json")))
        bad = []
        for rel, h in hashes.items():
            with open(os.path.join(d, rel), "rb") as f:
                if hashlib.sha256(f.read()).hexdigest() != h:
                    bad.append(rel)
        return {"ok": not bad, "corrupted": bad}

    # ------------------------------------------------------------------ Pantheon (multi-eixo, revogável)
    def pantheon_consider(self, cand: Candidate, discovery_id: str):
        changed = []
        for axis, codes in PANTHEON_AXES.items():
            score = sum(LEVEL_WEIGHT.get(c, 1) * min(1.0, abs(cand.passed[c].get("effect", 0)) / 2 + 0.25)
                        for c in codes if c in cand.passed)
            if score <= 0:
                continue
            cur = self.pantheon.get(axis)
            if cur is None or score > cur["score"] or cur.get("revoked"):
                self.pantheon[axis] = {"axis": axis, "candidate": cand.cid, "score": score, "discovery": discovery_id,
                                       "levels": [c for c in codes if c in cand.passed], "ts": iso_now(),
                                       "versions_hash": ENGINE_VERSIONS_HASH, "revoked": False}
                changed.append(axis)
        held = collections.Counter(v["candidate"] for v in self.pantheon.values() if not v.get("revoked"))
        if held:
            cid, k = held.most_common(1)[0]
            if k >= 4 and (self.overall_champion is None or self.overall_champion["candidate"] != cid):
                self.overall_champion = {"candidate": cid, "axes": k, "ts": iso_now(),
                                         "rule": "holds ≥4 pantheon axes simultaneously (dominance across axes)"}
                changed.append("overall")
        return changed

    def revoke(self, axis: str, reason: str):
        if axis in self.pantheon:
            self.pantheon[axis]["revoked"] = True
            self.pantheon[axis]["revoked_reason"] = reason

    def state_dict(self):
        return {"zoo": [{**{k: v for k, v in asdict(c).items() if k != "genome"}, "genome": c.genome} for c in self.zoo.values()],
                "failures": list(self.failures), "weird": list(self.weird), "pantheon": self.pantheon,
                "discoveries": self.discoveries, "convergence": self.convergence, "overall": self.overall_champion}

    def load_state_dict(self, s):
        self.zoo = collections.OrderedDict()
        for d in s["zoo"]:
            d = dict(d); d["genome"] = np.asarray(d["genome"], np.float32)
            self.zoo[d["cid"]] = Candidate(**{k: v for k, v in d.items() if k in Candidate.__dataclass_fields__})
        self.failures = collections.deque(s["failures"], maxlen=5000); self.weird = collections.deque(s["weird"], maxlen=500)
        self.pantheon = dict(s["pantheon"]); self.discoveries = list(s["discoveries"])
        self.convergence = list(s.get("convergence", [])); self.overall_champion = s.get("overall")


# ----------------------------------------------------------------------------- DSL segura de experimentos
DSL_OPS = {"perturb", "freeze", "ablate", "duplicate", "fork", "change_environment", "observe", "measure"}
DSL_ENUMS = {
    "perturb": {"what": {"sensor_noise", "resource_scale", "body_gain", "initial_energy"}},
    "freeze": {"what": {"plasticity", "movement"}},
    "ablate": {"what": {"action", "sensor_group", "units"}},
    "change_environment": {"param": {"storm", "cold", "n_patches", "resource_floor", "rich", "water_barrier", "deposit"}},
    "measure": {"metric": {"intake", "survival", "distance", "placed", "crossed", "emission"}},
}
SENSOR_GROUPS = {"resource": SENSOR_RESOURCE, "intero": SENSOR_INTERO, "sound": SENSOR_SOUND, "extero": SENSOR_EXTERO}


def validate_dsl(prog: Any) -> Dict[str, Any]:
    """Valida um programa da DSL (dict JSON). Rejeita qualquer coisa fora do esquema — sem eval, sem exec."""
    if not isinstance(prog, dict):
        raise ValueError("programa deve ser um objeto")
    allowed_top = {"name", "target", "steps", "seed", "arenas"}
    if set(prog) - allowed_top:
        raise ValueError(f"campos não permitidos: {sorted(set(prog) - allowed_top)}")
    name = str(prog.get("name", "experiment"))[:80]
    if not re.fullmatch(r"[\w\- .]{1,80}", name):
        raise ValueError("nome inválido")
    tgt = prog.get("target", {})
    if not isinstance(tgt, dict) or tgt.get("kind") not in ("candidate", "world_fork"):
        raise ValueError("target.kind deve ser candidate|world_fork")
    if tgt.get("kind") == "candidate" and not re.fullmatch(r"[\w\-]{1,64}", str(tgt.get("id", ""))):
        raise ValueError("target.id inválido")
    steps = prog.get("steps", [])
    if not isinstance(steps, list) or not 1 <= len(steps) <= 16:
        raise ValueError("steps: 1–16 operações")
    clean = []
    total_ticks = 0
    for st in steps:
        if not isinstance(st, dict) or st.get("op") not in DSL_OPS:
            raise ValueError(f"operação não permitida: {st!r}"[:200])
        op = st["op"]
        c = {"op": op}
        for k, v in st.items():
            if k == "op":
                continue
            if k in ("what", "param", "metric", "channel", "group"):
                if not isinstance(v, str) or not re.fullmatch(r"[a-z_0-9]{1,32}", v):
                    raise ValueError(f"valor inválido em {k}")
                if op in DSL_ENUMS and k in DSL_ENUMS[op] and v not in DSL_ENUMS[op][k]:
                    raise ValueError(f"{op}.{k}={v} fora do vocabulário")
                if k == "channel" and v not in ACTION_NAMES:
                    raise ValueError("canal de ação desconhecido")
                if k == "group" and v not in SENSOR_GROUPS:
                    raise ValueError("grupo de sensores desconhecido")
                c[k] = v
            elif k in ("amount", "value"):
                if not isinstance(v, (int, float)) or isinstance(v, bool) or not math.isfinite(v) or abs(v) > 10:
                    raise ValueError(f"{k} deve ser número finito |x|≤10")
                c[k] = float(v)
            elif k in ("ticks", "n"):
                if not isinstance(v, int) or isinstance(v, bool) or not 1 <= v <= 2000:
                    raise ValueError(f"{k} deve ser inteiro 1–2000")
                c[k] = int(v)
                if k == "ticks":
                    total_ticks += v
            else:
                raise ValueError(f"campo não permitido: {k}")
        clean.append(c)
    if total_ticks > 4000:
        raise ValueError("orçamento de ticks excedido (≤4000)")
    seed = prog.get("seed", 0)
    if not isinstance(seed, int) or isinstance(seed, bool) or not 0 <= seed < 2 ** 31:
        raise ValueError("seed inválida")
    ar = prog.get("arenas", 8)
    if not isinstance(ar, int) or isinstance(ar, bool) or not 1 <= ar <= 64:
        raise ValueError("arenas: 1–64")
    return {"name": name, "target": {"kind": tgt["kind"], "id": str(tgt.get("id", ""))}, "steps": clean, "seed": seed,
            "arenas": ar}


class ScientistEngine:
    PHASES = ["EXPLORE", "FIND", "INVESTIGATE", "VALIDATE"]

    def __init__(self, cfg: CivConfig, layout: GenomeLayout, validation: ValidationEngine, archive: ArchiveEngine,
                 qd: QDEvolution, logger: Optional[EventLogger], device: torch.device):
        self.cfg, self.L, self.ve, self.archive, self.qd, self.logger, self.device = cfg, layout, validation, archive, qd, logger, device
        self.queue: List[Dict[str, Any]] = []
        self.phase = "EXPLORE"
        self.current: Optional[Dict[str, Any]] = None
        self._gen: Optional[Generator] = None
        self.completed = 0
        self.discovery_events: List[Dict[str, Any]] = []
        self.results_log: "collections.deque" = collections.deque(maxlen=400)
        self.best_level = {"H": -1, "A": -1}
        self.validated_any: Dict[str, bool] = {}
        self.fitness_watch: "collections.deque" = collections.deque(maxlen=50)
        self.dsl_results: "collections.deque" = collections.deque(maxlen=100)
        self.rng = np.random.default_rng(derive_seed("scientist", cfg.master_seed) % (2 ** 32))
        self.budget_fraction = cfg.experiment_time_fraction
        self.seq = 0

    # ------------------------------------------------------------------ candidatos
    def add_candidate(self, genome: np.ndarray, origin: str, tick: int, fitness: float = 0.0, novelty: float = 0.0,
                      importance: float = 0.0, island: int = -1) -> Candidate:
        gh = hashlib.sha256(genome.astype(np.float32).tobytes()).hexdigest()[:16]
        cid = f"C{gh}"
        c = self.archive.zoo.get(cid)
        if c is None:
            c = Candidate(cid=cid, genome=genome.astype(np.float32), origin=origin, first_seen=iso_now(), tick=tick,
                          fitness=fitness, novelty=novelty, importance=importance, island=island)
            self.archive.add_candidate(c)
            self._check_convergence(c)
        else:
            c.fitness, c.novelty = max(c.fitness, fitness), max(c.novelty, novelty)
            c.importance = max(c.importance, importance)
        return c

    def _check_convergence(self, c: Candidate):
        """Convergência evolutiva: arquiteturas muito parecidas surgindo independentemente em ilhas diferentes."""
        if c.island < 0:
            return
        g = torch.as_tensor(c.genome, device=self.device).unsqueeze(0)
        for o in list(self.archive.zoo.values())[-200:]:
            if o.cid == c.cid or o.island < 0 or o.island == c.island:
                continue
            dist = float(self.L.distance(g, torch.as_tensor(o.genome, device=self.device).unsqueeze(0)))
            if dist < 0.05:
                ev = {"a": c.cid, "b": o.cid, "islands": [c.island, o.island], "distance": dist, "ts": iso_now()}
                self.archive.convergence.append(ev)
                self.archive.convergence = self.archive.convergence[-200:]
                break

    def next_codes(self, c: Candidate) -> List[Tuple[str, bool]]:
        """Próximos testes: o primeiro nível H não validado (gated) + triagem exploratória dos 2 seguintes; escada A
        após A0 (só para agentes que forrageiam)."""
        out = []
        missing = [x for x in H_CODES if x not in c.passed]
        if missing:
            i = H_CODES.index(missing[0])
            out.append((missing[0], True))
            for x in H_CODES[i + 1:i + 3]:
                if x not in c.passed and x not in c.screened:
                    out.append((x, False))
        am = [x for x in A_CODES[:9] if x not in c.passed]
        if am and c.fitness > 0.6:
            out.append((am[0], True))
        return out

    def priority(self, c: Candidate, code: str, prereq: bool, kind: str) -> float:
        lvl = LEVEL_WEIGHT.get(code, 1)
        tries = c.attempts.get(code, 0)
        unc = 1.0 / (1 + tries)
        voi = 1.5 if code == self.target_level() else 1.0
        base = {"screen": 1.0, "replicate": 3.0, "battery": 2.0, "dsl": 1.2}[kind]
        return float(base * (10 + 2 * lvl) * voi * unc * (1.4 if prereq else 0.6) + 3 * c.novelty + 2 * c.importance
                     + 0.5 * c.evidence_score())

    def target_level(self) -> str:
        best = max(self.archive.zoo.values(), key=lambda c: (c.h_level(), c.evidence_score()), default=None)
        lvl = best.h_level() if best else -1
        return H_CODES[min(len(H_CODES) - 1, lvl + 1)]

    def enqueue(self, c: Candidate, code: str, kind: str = "screen", prereq: bool = True, rep: int = 0):
        key = (c.cid, code, kind, rep)
        if any((q["cid"], q["code"], q["kind"], q["rep"]) == key for q in self.queue):
            return
        if kind == "screen" and c.attempts.get(code, 0) >= 3:
            return                                               # beco sem saída conhecido: arquivo de falhas evita repetição
        self.seq += 1
        self.queue.append({"cid": c.cid, "code": code, "kind": kind, "prereq": prereq, "rep": rep,
                           "created": time.time(), "prio": self.priority(c, code, prereq, kind), "seq": self.seq})
        if len(self.queue) > self.cfg.max_queue:
            self.queue.sort(key=lambda q: -q["prio"])
            self.queue = self.queue[: self.cfg.max_queue]        # descarta redundantes de baixo valor

    def plan(self):
        """Preenche a fila a partir do zoológico (candidatos mais promissores primeiro)."""
        cands = sorted(self.archive.zoo.values(), key=lambda c: -(c.fitness + c.novelty + c.evidence_score() + c.importance))[:40]
        for c in cands:
            for code, prereq in self.next_codes(c):
                self.enqueue(c, code, "screen", prereq)
        f = self.target_level()
        self.qd.set_focus(f)                                     # busca dirigida: evolução mira o próximo nível

    # ------------------------------------------------------------------ execução (gerador por fatias)
    def step(self, budget_s: float) -> Optional[Dict[str, Any]]:
        """Avança a ciência por até `budget_s` segundos. Devolve um relatório quando um experimento termina."""
        t0 = time.time()
        report = None
        while time.time() - t0 < budget_s:
            if self._gen is None:
                if not self.queue:
                    self.plan()
                if not self.queue:
                    self.phase = "EXPLORE"
                    return report
                for q in self.queue:
                    q["eff"] = q["prio"] * (1 + 0.02 * (time.time() - q["created"]))   # envelhecimento da fila
                self.queue.sort(key=lambda q: -q["eff"])
                self.current = self.queue.pop(0)
                kinds = {q["kind"] for q in self.queue} | {self.current["kind"]}
                self.phase = "VALIDATE" if "battery" in kinds else "INVESTIGATE" if "replicate" in kinds else "FIND"
                c = self.archive.zoo.get(self.current["cid"])
                if c is None:
                    self.current = None
                    continue
                key = c.cid if self.current["rep"] == 0 else f"{c.cid}|rep{self.current['rep']}"
                G = torch.as_tensor(c.genome, device=self.device)
                self._gen = self.ve.run_gen(self.current["code"], G, key)
                self.current["started"] = time.time()
            try:
                next(self._gen)
            except StopIteration as e:
                res: ProbeResult = e.value
                self._gen = None
                report = self._handle(self.current, res)
                self.current = None
        return report

    def _handle(self, exp: Dict[str, Any], res: ProbeResult) -> Dict[str, Any]:
        c = self.archive.zoo.get(exp["cid"])
        self.completed += 1
        code = exp["code"]
        rec = {"cid": exp["cid"], "code": code, "kind": exp["kind"], "rep": exp["rep"], "passed": res.passed, "p": res.p,
               "effect": res.effect, "duration": res.duration, "ts": res.ts, "claim": res.claim}
        self.results_log.append(rec)
        if self.logger:
            self.logger.event("probe", **{("exp_kind" if k == "kind" else k): v for k, v in rec.items()})
        if c is None:
            return rec
        c.attempts[code] = c.attempts.get(code, 0) + 1
        R = max(2, min(3, self.cfg.probe.replication_seeds))
        if exp["kind"] == "screen":
            if not res.passed:
                self.archive.failure(c.cid, res)
                c.failed[code] = c.failed.get(code, 0) + 1
                return rec
            c.screened[code] = {"p": res.p, "effect": res.effect, "reps": [res], "rep_pass": 0, "rep_fail": 0}
            for r in range(1, R + 1):
                self.enqueue(c, code, "replicate", True, rep=r)  # tentar quebrar em sementes de holdout independentes
            rec["next"] = f"replicate×{R}"
            return rec
        if exp["kind"] == "replicate":
            sc = c.screened.setdefault(code, {"reps": [], "rep_pass": 0, "rep_fail": 0})
            if res.passed:
                sc["rep_pass"] = sc.get("rep_pass", 0) + 1
                sc.setdefault("reps", []).append(res)
            else:
                sc["rep_fail"] = sc.get("rep_fail", 0) + 1
                self.archive.failure(c.cid, res)
            if sc.get("rep_pass", 0) + sc.get("rep_fail", 0) >= R:
                total_pass, total = 1 + sc.get("rep_pass", 0), R + 1
                if total_pass >= math.ceil(0.67 * total) and code not in c.passed:
                    self._discovery(c, code, sc.get("reps", []) or [res])
                    rec["discovery"] = True
                elif code not in c.passed:
                    c.failed[code] = c.failed.get(code, 0) + 1
                    rec["replication_failed"] = True
                sc["rep_pass"], sc["rep_fail"] = 0, 0
        return rec

    def _discovery(self, c: Candidate, code: str, results: List[ProbeResult]):
        c.passed[code] = {"effect": float(np.mean([r.effect for r in results if np.isfinite(r.effect)] or [0])),
                          "p": float(max(r.p for r in results if np.isfinite(r.p))) if any(np.isfinite(r.p) for r in results) else float("nan"),
                          "claim": results[0].claim, "ts": iso_now()}
        meta = self.archive.save_discovery(code, c, results, self.cfg, c.tick)
        c.passed[code]["discovery"] = meta["id"]
        self.validated_any[code] = True
        changed = self.archive.pantheon_consider(c, meta["id"])
        ev = {"type": "discovery", "id": meta["id"], "level": code, "level_name": LEVEL_NAME.get(code, code),
              "candidate": c.cid, "pantheon": changed, "claim": meta["claim"], "ts": iso_now()}
        self.discovery_events.append(ev)
        if self.logger:
            self.logger.event("discovery", echo=True, **{k: v for k, v in ev.items() if k != "type"})
        hl = max((x.h_level() for x in self.archive.zoo.values()), default=-1)
        al = max((x.a_level() for x in self.archive.zoo.values()), default=-1)
        self.best_level = {"H": hl, "A": al}
        if code.startswith("A") and all(x in c.passed or x in c.failed for x in A_CODES[1:9]):
            r9 = self.ve.affect.composite_A9(torch.as_tensor(c.genome, device=self.device))
            if r9.passed and "A9" not in c.passed:
                self._discovery(c, "A9", [r9])

    def record_civilization(self, level_name: str, tick: int, evidence: Dict[str, Any]):
        """Descoberta civilizacional (nível C subiu): pacote com evidência multidimensional do mundo."""
        c = Candidate(cid=f"WORLD-{level_name.split()[0]}", genome=np.zeros(1, np.float32), origin="world", first_seen=iso_now(), tick=tick)
        res = ProbeResult(level=level_name.split()[0], name=level_name, passed=True, p=float("nan"), effect=float("nan"),
                          ci=(float("nan"), float("nan")), n=1, metrics=to_jsonable(evidence), controls={
                              "rule": "multidimensional criteria held for ≥3 consecutive civilization windows"},
                          seeds=[], genome_hash="world", claim="SIMULATION HYPOTHESIS", duration=0.0,
                          thresholds=self.cfg.thresholds_hash())
        meta = self.archive.save_discovery(level_name.split()[0], c, [res], self.cfg, tick, kind="civilization")
        ev = {"type": "discovery", "id": meta["id"], "level": level_name.split()[0], "level_name": level_name,
              "candidate": "world", "claim": "SIMULATION HYPOTHESIS", "ts": iso_now()}
        self.discovery_events.append(ev)
        return meta

    def watch_fitness(self, best_fitness: float):
        """Anti-metric-gaming: fitness subindo muito sem nenhuma evidência nova → registrar anomalia."""
        self.fitness_watch.append((best_fitness, sum(len(c.passed) for c in self.archive.zoo.values())))
        if len(self.fitness_watch) == self.fitness_watch.maxlen:
            f0, e0 = self.fitness_watch[0]; f1, e1 = self.fitness_watch[-1]
            if f1 > (1 + self.cfg.thresholds.metric_exploit_gain) * max(1e-6, f0) and e1 == e0:
                self.archive.anomaly("fitness_without_evidence", {"fitness_from": f0, "fitness_to": f1, "evidence": e1})
                self.fitness_watch.clear()

    def adapt_budget(self) -> float:
        """Fila cresce → mais tempo para validação; fila pequena → mais exploração."""
        q = len(self.queue) / max(1, self.cfg.max_queue)
        self.budget_fraction = float(np.clip(self.cfg.experiment_time_fraction * (0.6 + 2.0 * q), 0.1, 0.6))
        return self.budget_fraction

    # ------------------------------------------------------------------ DSL
    def dsl_gen(self, prog: Dict[str, Any], runner: ArenaRunner):
        """Executa um programa validado em arenas de laboratório (alvo = candidato). Nunca toca o mundo principal."""
        p = validate_dsl(prog)
        if p["target"]["kind"] != "candidate":
            raise ValueError("world_fork é executado pelo UniverseEngine.counterfactual (fork marcado)")
        c = self.archive.zoo.get(p["target"]["id"])
        if c is None:
            raise ValueError("candidato desconhecido")
        n = p["arenas"]
        spec = ArenaSpec(task="dsl", n_patches=3, resource_floor=0.05)
        G = torch.as_tensor(c.genome, device=self.device).expand(n, -1).contiguous()
        mask = torch.ones(n, N_SENSORS, device=self.device)
        block = torch.zeros(n, N_ACTIONS, dtype=torch.bool, device=self.device)
        zero_units = torch.zeros(n, self.cfg.n_units, dtype=torch.bool, device=self.device)
        frz = mov = False
        noise, energy, body_gain = 0.0, 0.8, 1.0
        results, trial, rec = {}, None, None
        for st in p["steps"]:
            op = st["op"]
            if op == "perturb":
                if st["what"] == "sensor_noise":
                    noise = abs(st.get("amount", 0.1))
                elif st["what"] == "resource_scale":
                    spec.patch_value = float(np.clip(st.get("amount", 1.0), 0.0, 3.0))
                elif st["what"] == "initial_energy":
                    energy = float(np.clip(st.get("amount", 0.8), 0.05, 1.0))
                elif st["what"] == "body_gain":
                    body_gain = float(np.clip(st.get("amount", 1.0), 0.0, 3.0))
            elif op == "freeze":
                frz = frz or st["what"] == "plasticity"; mov = mov or st["what"] == "movement"
            elif op == "ablate":
                if st["what"] == "action" and "channel" in st:
                    block[:, ACTION_NAMES.index(st["channel"])] = True
                elif st["what"] == "sensor_group" and "group" in st:
                    mask[:, SENSOR_GROUPS[st["group"]]] = 0
                elif st["what"] == "units":
                    k = int(np.clip(st.get("n", 1), 1, self.cfg.n_units))
                    zero_units[:, :k] = True
            elif op == "duplicate":
                results["duplicates"] = int(st.get("n", 1))
            elif op == "change_environment":
                prm, val = st["param"], st.get("value", 0.0)
                if prm in ("storm", "cold", "resource_floor", "deposit"):
                    setattr(spec, prm, float(np.clip(val, 0.0, 3.0)))
                elif prm in ("n_patches", "rich"):
                    setattr(spec, prm, int(np.clip(val, 0, 8)))
                elif prm == "water_barrier":
                    spec.water_barrier = bool(val > 0); spec.spawn_side = "left" if val > 0 else "center"
            elif op == "fork":
                results.setdefault("forks", 0)
                results["forks"] += 1
            elif op == "observe":
                cap = runner.capability(G)
                cap = cap.clone(); cap[:, 0] *= body_gain
                trial = runner.build(G, spec, p["seed"], body_cap=cap, energy=energy)
                gen = torch.Generator(device=self.device); gen.manual_seed(p["seed"] + 1)

                def ctrl(t, tr, noise=noise):
                    add = torch.randn(tr.n, N_SENSORS, generator=gen, device=self.device) * noise if noise > 0 else None
                    iv = MindIntervention(sensor_mask=mask, sensor_add=add, zero_units=zero_units if zero_units.any() else None,
                                          freeze_plasticity=torch.full((tr.n,), frz, device=self.device) if frz else None)
                    return ArenaControl(iv=iv, action_block=block if block.any() else None,
                                        freeze_move=torch.ones(tr.n, dtype=torch.bool, device=self.device) if mov else None)
                rec = yield from runner.run_gen(trial, st.get("ticks", 100), ctrl_fn=ctrl,
                                                record=("gain", "alive", "speed", "placed", "emit"), name=f"DSL {p['name']}")
            elif op == "measure":
                if rec is None:
                    raise ValueError("measure antes de observe")
                m = st["metric"]
                val = {"intake": rec["gain"].sum(0), "survival": rec["alive"].float().mean(0), "distance": rec["speed"].sum(0),
                       "placed": rec["placed"].sum(0), "emission": rec["emit"].abs().sum(-1).mean(0),
                       "crossed": ArenaRunner.crossed(trial).float()}[m].cpu().numpy()
                results[m] = summary_stats(val)
        out = {"program": p, "results": results, "ts": iso_now(), "intervention": True, "scope": "lab arenas only"}
        self.dsl_results.append(out)
        return out

    # ------------------------------------------------------------------ estado
    def status(self) -> Dict[str, Any]:
        return {"phase": self.phase, "queue": len(self.queue), "completed": self.completed,
                "current": {k: v for k, v in (self.current or {}).items() if k in ("cid", "code", "kind", "rep")},
                "best_level": self.best_level, "target_level": self.target_level(), "budget_fraction": self.budget_fraction,
                "discoveries": len(self.archive.discoveries), "validated": sorted(self.validated_any.keys()),
                "recent": list(self.results_log)[-12:]}

    def state_dict(self):
        return {"queue": [{k: v for k, v in q.items() if k not in ("eff",)} for q in self.queue], "phase": self.phase,
                "completed": self.completed, "results_log": list(self.results_log), "best_level": self.best_level,
                "validated": self.validated_any, "seq": self.seq, "rng": self.rng.bit_generator.state,
                "screened": {cid: {code: {"p": v.get("p"), "effect": v.get("effect"), "rep_fail": v.get("rep_fail", 0),
                                          "rep_pass": v.get("rep_pass", 0)}
                                   for code, v in c.screened.items()} for cid, c in self.archive.zoo.items() if c.screened},
                "current": self.current}

    def load_state_dict(self, s):
        now = time.time()
        self.queue = [dict(q, created=now) for q in s["queue"]]
        if s.get("current"):                                         # experimento interrompido volta para a fila
            self.queue.insert(0, dict(s["current"], created=now))
        self.phase, self.completed = s["phase"], int(s["completed"])
        self.results_log = collections.deque(s["results_log"], maxlen=400)
        self.best_level, self.validated_any, self.seq = dict(s["best_level"]), dict(s["validated"]), int(s["seq"])
        try:
            self.rng.bit_generator.state = s["rng"]
        except Exception:
            pass
        for cid, sc in s.get("screened", {}).items():
            c = self.archive.zoo.get(cid)
            if c is not None:
                for code, v in sc.items():
                    c.screened[code] = {"p": v.get("p"), "effect": v.get("effect"), "rep_fail": v.get("rep_fail", 0),
                                        "rep_pass": v.get("rep_pass", 0), "reps": []}

## 15b — UNIVERSE ENGINE: orquestração multiescala, fidelidade adaptativa, forks contrafactuais
Agenda temporal (ticks): sinais/ação/física de corpos **todo tick** · ambiente a cada `env_every` (chunks inativos
com período 4× maior) · desenvolvimento/medição de corpos a cada `development_every` · fidelidade a cada
`fidelity_every` · agregados F0 a cada `aggregate_every` · social a cada `social_every` · cultura a cada
`culture_every` · civilização/história a cada `civ_every` (eventos discretos).
Fidelidade científica é decisão do agendador/Scientist (novidade, ancestralidade, raridade, foco científico);
seleção pelo usuário só promove com registro explícito — a câmera nunca altera a ciência nem o RNG.
Escalas: F0 agregados por região ↔ F1 agentes leves ↔ F2 corpo grosso (D−1) ↔ F3 corpo completo ↔ F4 foco (gravação).

In [ ]:
# @title 15b — UNIVERSE ENGINE: orquestração multiescala, fidelidade adaptativa, forks contrafactuais { display-mode: "form" }
class Profiler:
    def __init__(self):
        self.ms: Dict[str, float] = collections.defaultdict(float)
        self.calls: Dict[str, int] = collections.defaultdict(int)
        self.total: Dict[str, float] = collections.defaultdict(float)

    @contextlib.contextmanager
    def __call__(self, name: str):
        t0 = time.perf_counter()
        try:
            yield
        finally:
            dt = (time.perf_counter() - t0) * 1000
            self.ms[name] = 0.95 * self.ms[name] + 0.05 * dt if self.calls[name] else dt
            self.calls[name] += 1
            self.total[name] += dt / 1000

    def report(self) -> Dict[str, Any]:
        tot = sum(self.total.values()) or 1.0
        return {k: {"ema_ms": round(self.ms[k], 3), "share": round(self.total[k] / tot, 4), "calls": self.calls[k]}
                for k in sorted(self.total, key=lambda x: -self.total[x])}


FORK_INTERVENTIONS = {"block_action", "mute_all", "freeze_plasticity", "resource_scale", "remove_structures", "none"}


class WorldFork:
    """Linha do tempo paralela (A/B) a partir do estado atual. Mesmas leis; motor de agentes próprio; sem ciência;
    marcada como intervenção. Usa números aleatórios comuns (mesma semente) para isolar o efeito da intervenção."""

    def __init__(self, uni: "UniverseEngine", label: str, intervention: Dict[str, Any]):
        self.u, self.label, self.iv = uni, label, intervention
        cfg = uni.cfg
        self.ws = uni.ws.clone()
        self.pool = AgentPool(uni.pool.cap, uni.L, cfg, uni.device)
        st = uni.pool.state_dict(); self.pool.load_state_dict(st)
        self.ledger = StructureLedger(min(cfg.max_blocks, 200000), uni.device)
        self.ids = IdCounter(10 ** 13 + derive_seed("fork", label) * 1000)
        self.engine = AgentEngine(cfg, uni.L, uni.we, uni.matter, uni.mind, uni.device)
        self.gen = torch.Generator(device=uni.device); self.gen.manual_seed(derive_seed("fork_rng", uni.tick_count))
        self.settle = SettlementEngine(cfg, uni.device)
        self.tick = uni.tick_count
        self.metrics: List[Dict[str, Any]] = []
        self.eaten = 0.0
        if intervention.get("kind") == "remove_structures":
            built = (self.ws.struct_h > 0) & (self.ws.struct_integrity < 1e8)
            self.ws.deposit = self.ws.deposit + self.ws.struct_h.clamp(min=0).float() * built
            self.ws.struct_h = torch.where(built, torch.zeros_like(self.ws.struct_h), self.ws.struct_h)
            self.ws.deposit = self.ws.deposit + self.ws.filled.float(); self.ws.filled[:] = False
            self.ws.struct_dirty = True
        if intervention.get("kind") == "resource_scale":
            self.ws.regen_mult = torch.full_like(self.ws.resource, float(np.clip(intervention.get("value", 1.0), 0, 3)))

    def ctrl(self) -> Optional[ArenaControl]:
        k = self.iv.get("kind")
        n = self.pool.cap
        if k == "block_action":
            b = torch.zeros(n, N_ACTIONS, dtype=torch.bool, device=self.u.device)
            b[:, ACTION_NAMES.index(self.iv["channel"])] = True
            return ArenaControl(action_block=b)
        if k == "mute_all":
            return ArenaControl(mute=torch.ones(n, dtype=torch.bool, device=self.u.device))
        if k == "freeze_plasticity":
            return ArenaControl(iv=MindIntervention(freeze_plasticity=torch.ones(n, dtype=torch.bool, device=self.u.device)))
        return None

    def step(self, n: int):
        cfg = self.u.cfg
        for _ in range(n):
            st = self.engine.step(self.ws, self.pool, self.tick, self.gen, None, self.ids, self.ledger, ctrl=self.ctrl(),
                                  record_interactions=False)
            self.eaten += st["eaten"]
            if self.tick % cfg.env_every == 0:
                iy, ix = WorldEngine.tile_index(self.ws, self.pool.pos[self.pool.alive])
                self.ws.tick = self.tick
                self.u.we.env_step(self.ws, self.gen, torch.stack([iy, ix]))
            self.tick += 1

    def measure(self) -> Dict[str, Any]:
        rng = np.random.default_rng(self.tick)
        sm = self.settle.update(self.ws, self.pool, self.tick, None, rng)
        m = {"tick": self.tick, "population": self.pool.n_alive(), "blocks": sm["blocks_total"], "settlements": sm["settlements"],
             "enclosures": sm["enclosures_total"], "bridges": int(self.ws.filled.sum()), "eaten": self.eaten,
             "tech": self.engine.tech_counts.cpu().numpy().tolist(),
             "mean_energy": float(self.pool.energy[self.pool.alive].mean()) if self.pool.n_alive() else 0.0}
        self.metrics.append(m)
        return m


class UniverseEngine:
    def __init__(self, cfg: CivConfig, paths: Optional[CivPaths], device: torch.device, logger: Optional[EventLogger],
                 hw: Optional[Dict[str, Any]] = None):
        self.cfg, self.paths, self.device, self.logger, self.hw = cfg, paths, device, logger, hw or {}
        L = self.L = GenomeLayout(cfg).to(device)
        self.we = WorldEngine(cfg, device)
        self.matter = MatterEngine(cfg, device)
        self.dev = DevelopmentEngine(cfg, L, device)
        cfg_c = copy.deepcopy(cfg); cfg_c.dev_lattice = max(3, cfg.dev_lattice - 1)
        self.dev_coarse = DevelopmentEngine(cfg_c, L, device)
        self.body = BodyEngine(cfg, self.dev, device)
        self.body_coarse = BodyEngine(cfg, self.dev_coarse, device)
        self.mind = SignalEngine(cfg, L, device)
        self.agents = AgentEngine(cfg, L, self.we, self.matter, self.mind, device)
        self.runner = ArenaRunner(cfg, L, self.we, self.matter, self.mind, self.dev, self.body, device)
        self.qd = QDEvolution(cfg, L, self.runner, device, logger)
        self.sentinel = EthicalSentinel(cfg)
        self.affect = AffectEngine(cfg, L, self.runner, self.sentinel, device)
        self.validation = ValidationEngine(cfg, L, self.runner, self.affect, device)
        self.archive = ArchiveEngine(paths, cfg)
        self.scientist = ScientistEngine(cfg, L, self.validation, self.archive, self.qd, logger, device)
        self.social = SocialEngine(cfg)
        self.culture = CultureEngine(cfg, L)
        self.settlements = SettlementEngine(cfg, device)
        self.civ = CivilizationEngine(cfg, L)
        self.history = HistoryEngine(paths)
        self.lineage = LineageStore(paths)
        self.ids = IdCounter(1)
        self.pool = AgentPool(cfg.max_agents, L, cfg, device)
        self.ledger = StructureLedger(cfg.max_blocks, device)
        nb = cfg.max_bodies
        sub = cfg.body_substeps
        self.bodies = {2: BodyRegistry(cfg, self.dev_coarse, self.body_coarse, max(2, int(nb * 0.6)), sub),
                       3: BodyRegistry(cfg, self.dev, self.body, max(2, int(nb * 0.3)), sub),
                       4: BodyRegistry(cfg, self.dev, self.body, max(2, int(nb * 0.1)), sub, record=True)}
        self.ws: Optional[WorldState] = None
        self.f0: Optional[RegionAggregates] = None
        self.gen = torch.Generator(device=device); self.gen.manual_seed(derive_seed("world_rng", cfg.master_seed))
        self.inst_rng = np.random.default_rng(derive_seed("instruments", cfg.master_seed) % (2 ** 32))
        self.tick_count = 0
        self.created = iso_now()
        self.total_runtime = 0.0
        self.sessions = 0
        self.window = collections.Counter()
        self.demography = RingBuffer(2000)
        self.focus: Dict[str, Dict[str, Any]] = {}
        self.user_focus_aid: Optional[int] = None
        self.interventions: List[Dict[str, Any]] = []
        self.fidelity_log = collections.Counter()
        self.prof = Profiler()
        self.paused = False
        self.last = {"social": {}, "comm": {}, "culture": {}, "settle": {}, "civ": {}}
        self.forks: Dict[str, Dict[str, Any]] = {}
        self.events_out: "collections.deque" = collections.deque(maxlen=500)
        self.legacy_summary: Optional[Dict[str, Any]] = None

    # ------------------------------------------------------------------ criação
    def create_world(self, bootstrap_seconds: float = 30.0):
        cfg = self.cfg
        seed = derive_seed("world", cfg.master_seed)
        self.ws = self.we.generate(seed)
        self.f0 = RegionAggregates(cfg, self.ws.H, self.ws.W, self.L.P)
        t0 = time.time()
        self.qd.init_islands()
        while time.time() - t0 < bootstrap_seconds or self.qd.generations < 2 * len(self.qd.islands):
            drive(self.qd.generation_gen())
            live_progress("bootstrap QD", time.time() - t0, bootstrap_seconds)
            if time.time() - t0 > 4 * bootstrap_seconds:
                break
        n = cfg.initial_agents
        el = self.qd.elites_for_founding(max(1, n // 2))
        parts = []
        if len(el):
            E = torch.as_tensor(el, device=self.device)
            reps = E[torch.randint(0, E.shape[0], (int(n * 0.8),), generator=self.gen, device=self.device)]
            parts.append(self.L.mutate(reps, self.gen, 0.5))
        parts.append(self.L.random(n - sum(p.shape[0] for p in parts), self.gen))
        G = torch.cat(parts)[:n]
        slots = self.agents.spawn_founders(self.ws, self.pool, n, self.gen, self.ids, 0, self.lineage, genomes=G)
        self._measure(slots)
        self.history.add("world_created", 0, seed=seed, size=[self.ws.W, self.ws.H], founders=int(slots.numel()),
                         qd_generations=self.qd.generations, bootstrap_seconds=round(time.time() - t0, 1),
                         note="fundadores: 80% elites QD mutadas + 20% genomas aleatórios")
        if self.logger:
            self.logger.event("world_created", echo=True, seed=seed, founders=int(slots.numel()), W=self.ws.W, H=self.ws.H)

    def _measure(self, slots: torch.Tensor, chunk: int = 256):
        """Desenvolvimento + teste físico de marcha (resumo F1) para agentes ainda não medidos."""
        for i in range(0, slots.numel(), chunk):
            s = slots[i:i + chunk]
            cap = self.body.measure_capability(self.pool.genome[s], (self.pool.aid[s] % 100003))
            self.pool.body_cap[s] = cap
            self.pool.body_measured[s] = True

    # ------------------------------------------------------------------ passo do mundo
    def tick(self):
        cfg, t, ws, pool = self.cfg, self.tick_count, self.ws, self.pool
        with self.prof("agents+signals+bodies"):
            st = self.agents.step(ws, pool, t, self.gen, self.lineage, self.ids, self.ledger, bodies=self.bodies)
        for k, v in st.items():
            if k != "alive":
                self.window[k] += v
        self.window["agent_ticks"] += st["alive"]
        if t % cfg.env_every == 0:
            with self.prof("environment"):
                iy, ix = WorldEngine.tile_index(ws, pool.pos[pool.alive])
                ws.tick = t
                self.we.env_step(ws, self.gen, torch.stack([iy, ix]))
        if t % cfg.development_every == 0:
            with self.prof("development"):
                pend = torch.nonzero(pool.alive & ~pool.body_measured).flatten()
                k = 64 if self.device.type == "cuda" else 12
                if pend.numel():
                    self._measure(pend[-k:])
        if t % cfg.fidelity_every == 0:
            with self.prof("fidelity"):
                self._fidelity()
        if t % cfg.aggregate_every == 0:
            with self.prof("aggregate F0"):
                self._aggregate()
        if t % cfg.social_every == 0:
            with self.prof("social"):
                self.social.ingest(self.agents.interactions.drain(), t)
                self.culture.ingest_tech(self.agents.tech_events, self.social, t)
                self.sentinel.observe(pool, self.L, t)
            if t % (cfg.social_every * 5) == 0:
                with self.prof("social groups"):
                    self.last["social"] = self.social.update_groups(pool, t, self.inst_rng)
        if t % cfg.culture_every == 0:
            with self.prof("culture+communication"):
                self.last["culture"] = self.culture.update(pool, self.social, t, self.inst_rng)
                c = self.social.communication(pool, ws, self.inst_rng, self.cfg.probe.permutations, t)
                if c:
                    self.last["comm"] = c
        if t % cfg.civ_every == 0 and t > 0:
            with self.prof("civilization"):
                self._civilization(t)
        if t % cfg.immigrant_every == 0 and t > 0:
            self._immigrants(t)
        if pool.n_alive() < max(8, cfg.initial_agents // 20):
            self._recover(t)
        if t % 200 == 0:
            with self.prof("candidates"):
                self._harvest_candidates(t)
        self.tick_count += 1
        ws.tick = self.tick_count

    def _civilization(self, t: int):
        cfg = self.cfg
        settle = self.settlements.update(self.ws, self.pool, t, self.social, self.inst_rng)
        self.last["settle"] = settle
        new_inv = sum(1 for e in self.culture.new_events if e["type"] == "invention")
        demo = {"births": self.window["births"], "deaths": self.window["deaths"], "eaten": self.window["eaten"],
                "agent_ticks": self.window["agent_ticks"], "new_inventions": new_inv}
        validated = dict(self.scientist.validated_any)
        civ = self.civ.evaluate(t, self.pool, self.f0.total() if self.f0 else 0.0, self.last["social"], self.last["comm"],
                                self.last["culture"], settle, validated, self.inst_rng, demo)
        self.last["civ"] = civ
        self.demography.push({"tick": t, "population": self.pool.n_alive(), "f0": self.f0.total() if self.f0 else 0,
                              **{k: self.window[k] for k in ("births", "deaths", "deaths_starvation", "deaths_exposure",
                                                             "deaths_age", "deaths_conflict", "placed", "picked", "given", "taken")},
                              "eaten": round(self.window["eaten"], 3)})
        self.window = collections.Counter()
        for e in self.history.observe(t, self.pool, civ.get("species_estimate", 0)):
            self.events_out.append(e)
        for src in (self.culture.drain_events(), self.settlements.drain_events(), self.civ.drain_events(),
                    self.sentinel.drain_events()):
            for e in src:
                et = e.get("type", "event")
                pos = e.get("pos")
                ev = self.history.add({"invention": "technology_discovery"}.get(et, et), e.get("tick", t), pos=pos,
                                      **{k: v for k, v in e.items() if k not in ("type", "tick", "pos")})
                self.events_out.append(ev)
                if et == "civilization_level":
                    meta = self.scientist.record_civilization(e["level"], t, self.civ.achieved.get(e["level"], {}).get("evidence", {}))
                    self.events_out.append(self.history.add("civilization_discovery", t, level=e["level"], discovery=meta["id"]))
        for d in self.scientist.discovery_events:
            self.events_out.append(self.history.add("science_discovery", t, **{k: v for k, v in d.items() if k != "type"}))
        self.scientist.discovery_events = []
        self.history.flush()

    # ------------------------------------------------------------------ fidelidade adaptativa
    def _fidelity(self):
        pool = self.pool
        alive = torch.nonzero(pool.alive).flatten()
        if alive.numel() == 0:
            return
        sig = pool.signature[alive]
        nov = (sig - sig.mean(0)).norm(dim=-1)
        nov = nov / nov.max().clamp(min=1e-6)
        age = pool.age[alive] / pool.age[alive].max().clamp(min=1)
        score = 0.5 * nov + 0.3 * age + 0.2 * pool.energy[alive]
        if self.focus:
            F = torch.as_tensor(np.stack([v["genome"] for v in self.focus.values()]), device=self.device)
            dmin = torch.cdist(pool.genome[alive] / (self.L.hi - self.L.lo).clamp(min=1e-6),
                               F / (self.L.hi - self.L.lo).clamp(min=1e-6)).min(1).values / math.sqrt(self.L.P)
            score = score + 2.0 * (dmin < 0.05).float()
        if self.user_focus_aid is not None:
            hit = pool.aid[alive] == self.user_focus_aid
            if bool(hit.any()):
                score = score + 5.0 * hit.float()
                self.fidelity_log["user_selected_promotion"] += int(hit.sum())
        order = alive[torch.argsort(score, descending=True)]
        caps = {4: self.bodies[4].cap, 3: self.bodies[3].cap, 2: self.bodies[2].cap}
        want: Dict[int, torch.Tensor] = {}
        i = 0
        for lvl in (4, 3, 2):
            want[lvl] = order[i:i + caps[lvl]]
            i += caps[lvl]
        max_new = 32 if self.device.type == "cuda" else 6
        for lvl in (4, 3, 2):
            reg = self.bodies[lvl]
            cur = reg.agent_slot[reg.agent_slot >= 0]
            keep = torch.isin(cur, want[lvl])
            drop_slots = torch.nonzero((reg.agent_slot >= 0) & ~torch.isin(reg.agent_slot, want[lvl])).flatten()
            if drop_slots.numel():
                a = reg.agent_slot[drop_slots]
                reg.detach(drop_slots)
                pool.body_slot[a] = -1; pool.fidelity[a] = 1
                self.fidelity_log[f"demote_F{lvl}"] += int(a.numel())
            new = want[lvl][~torch.isin(want[lvl], cur[keep])]
            new = new[pool.body_slot[new] < 0][:max_new]
            if new.numel():
                bs = reg.attach(new, pool.aid[new], pool.genome[new], pool.aid[new] % 100003)
                k = bs.numel()
                pool.body_slot[new[:k]] = bs.int(); pool.fidelity[new[:k]] = lvl
                self.fidelity_log[f"promote_F{lvl}"] += int(k)

    # ------------------------------------------------------------------ F0 ↔ F1
    def _aggregate(self):
        cfg, pool, ws = self.cfg, self.pool, self.ws
        if self.f0 is None:
            return
        self.f0.update(ws, cfg.aggregate_every, self.inst_rng)
        n = pool.n_alive()
        if n > cfg.f1_budget_fraction * pool.cap:
            # agregação: regiões mais densas, nunca agentes com corpo/foco
            k = int(n - 0.8 * pool.cap)
            idx = torch.nonzero(pool.alive & (pool.fidelity <= 1)).flatten()
            if idx.numel() == 0:
                return
            reg = torch.as_tensor(self.f0.region_of(pool.pos[idx].cpu().numpy()), device=self.device)
            dens = torch.bincount(reg, minlength=self.f0.Rh * self.f0.Rw).float()
            pick = idx[torch.argsort(dens[reg] + 0.01 * torch.rand(idx.numel(), generator=self.gen, device=self.device),
                                     descending=True)[:k]]
            regs = self.f0.region_of(pool.pos[pick].cpu().numpy())
            self.f0.absorb(regs, pool.genome[pick].cpu().numpy(), pool.energy[pick].cpu().numpy(),
                           pool.signature[pick].cpu().numpy(), self.inst_rng)
            self.agents.kill(pool, pick, self.bodies, ws)
            self.fidelity_log["aggregate_F1_to_F0"] += int(pick.numel())
        elif n < 0.5 * pool.cap and self.f0.total() >= 1:
            self._disaggregate(int(min(self.f0.total(), 0.6 * pool.cap - n, 256)))

    def _disaggregate(self, k: int):
        """Desagregação consistente: amostra genomas do reservatório da região, na própria região, conta decrementada."""
        if k <= 0:
            return 0
        f0, ws = self.f0, self.ws
        counts = f0.count.flatten()
        regs = [r for r in np.argsort(-counts) if counts[r] >= 1 and r in f0.reservoir][:64]
        made = 0
        for r in regs:
            if made >= k:
                break
            take = int(min(counts[r], k - made, 32))
            ry, rx = divmod(int(r), f0.Rw)
            y0, x0 = ry * f0.rs, rx * f0.rs
            G = torch.as_tensor(np.stack([f0.reservoir[r][i % len(f0.reservoir[r])] for i in range(take)]), device=self.device)
            G = self.L.mutate(G, self.gen, 0.3)
            slots = self.agents.spawn_founders(ws, self.pool, take, self.gen, self.ids, self.tick_count, self.lineage,
                                               genomes=G, origin="disaggregated",
                                               region=(y0, x0, min(ws.H, y0 + f0.rs), min(ws.W, x0 + f0.rs)),
                                               energy=float(np.clip(f0.energy[ry, rx], 0.3, 0.9)))
            self._measure(slots)
            f0.count[ry, rx] -= slots.numel()
            made += int(slots.numel())
        self.fidelity_log["disaggregate_F0_to_F1"] += made
        return made

    def _immigrants(self, t: int):
        G, gids = self.qd.best_genomes(max(2, self.cfg.initial_agents // 50))
        if not len(G):
            return
        slots = self.agents.spawn_founders(self.ws, self.pool, len(G), self.gen, self.ids, t, self.lineage,
                                           genomes=torch.as_tensor(G, device=self.device), origin="immigrant_qd")
        self._measure(slots)
        if slots.numel():
            self.history.add("immigration", t, n=int(slots.numel()), source="QD islands")

    def _recover(self, t: int):
        made = self._disaggregate(self.cfg.initial_agents // 4) if self.f0 is not None and self.f0.total() >= 1 else 0
        if self.pool.n_alive() + made < max(8, self.cfg.initial_agents // 20):
            G = self.qd.elites_for_founding(max(8, self.cfg.initial_agents // 4))
            if len(G):
                slots = self.agents.spawn_founders(self.ws, self.pool, len(G), self.gen, self.ids, t, self.lineage,
                                                   genomes=self.L.mutate(torch.as_tensor(G, device=self.device), self.gen, 0.5),
                                                   origin="immigrant_qd")
                self._measure(slots)
                made += int(slots.numel())
        if made:
            self.history.add("recovery_injection", t, n=made, note="população abaixo do piso: desagregação F0 + imigrantes QD")

    def _harvest_candidates(self, t: int):
        pool = self.pool
        idx = torch.nonzero(pool.alive & (pool.age > 100)).flatten()
        sc = self.scientist
        if idx.numel():
            rate = pool.stats[idx, 0] / pool.age[idx].clamp(min=1)
            top = idx[torch.argsort(rate, descending=True)[:4]]
            sig = pool.signature[idx]
            nov = (sig - sig.mean(0)).norm(dim=-1)
            topn = idx[torch.argsort(nov, descending=True)[:2]]
            for s, kind in [(int(x), "intake") for x in top.tolist()] + [(int(x), "novelty") for x in topn.tolist()]:
                aid = int(pool.aid[s])
                c = sc.add_candidate(pool.genome[s].cpu().numpy(), f"world:agent:{aid}:{kind}", t,
                                     fitness=float(pool.stats[s, 0] / pool.age[s].clamp(min=1) * 100),
                                     novelty=float(nov.max()) if kind == "novelty" else 0.0,
                                     importance=math.log1p(self.lineage.children_count.get(aid, 0)))
        G, gids = self.qd.best_genomes(3)
        for g, gid in zip(G, gids):
            isl = next((i.iid for i in self.qd.islands if gid in i.gids), -1)
            f = next((float(i.fit[i.gids.index(gid)]) for i in self.qd.islands if gid in i.gids), 0.0)
            sc.add_candidate(g, f"qd:island{isl}:{gid}", t, fitness=f, island=isl)
        sc.watch_fitness(max((i.best for i in self.qd.islands), default=0.0))
        for c in list(self.archive.zoo.values()):
            if c.passed and c.cid not in self.focus and len(self.focus) < 16:
                self.focus[c.cid] = {"genome": c.genome, "reason": "validated_candidate"}

    # ------------------------------------------------------------------ forks contrafactuais
    def counterfactual_gen(self, intervention: Dict[str, Any], ticks: int, label: str = ""):
        """Linha A (sem mudança) × linha B (intervenção) a partir do mesmo estado. Gerador (rende a cada 25 ticks)."""
        kind = intervention.get("kind")
        if kind not in FORK_INTERVENTIONS:
            raise ValueError("intervenção não permitida")
        if kind == "block_action" and intervention.get("channel") not in ACTION_NAMES:
            raise ValueError("canal inválido")
        ticks = int(np.clip(ticks, 25, 5000))
        fid = f"FORK-{len(self.forks) + 1:04d}-{uuid.uuid4().hex[:6]}"
        A, B = WorldFork(self, f"{fid}-A", {"kind": "none"}), WorldFork(self, f"{fid}-B", intervention)
        rec = {"id": fid, "from_tick": self.tick_count, "intervention": intervention, "ticks": ticks, "label": label,
               "status": "running", "A": [], "B": [], "ts": iso_now(), "marked": "EXTERNAL INTERVENTION (fork only)"}
        self.forks[fid] = rec
        self.interventions.append({"tick": self.tick_count, "fork": fid, "intervention": intervention, "ts": iso_now()})
        done = 0
        while done < ticks:
            step = min(25, ticks - done)
            A.step(step); B.step(step); done += step
            rec["A"].append(A.measure()); rec["B"].append(B.measure())
            yield
        a, b = rec["A"][-1], rec["B"][-1]
        rec["diff"] = {k: (b[k] - a[k]) if isinstance(a[k], (int, float)) else None for k in a if k != "tick"}
        rec["status"] = "done"
        diverge = (A.ws.struct_h != B.ws.struct_h) | (A.ws.filled != B.ws.filled)
        rec["divergent_tiles"] = int(diverge.sum())
        rec["divergence_map"] = F.avg_pool2d(diverge.float()[None, None], 4)[0, 0].cpu().numpy().round(3)
        if self.paths is not None:
            with open(os.path.join(self.paths.civilizations, f"{fid}.json"), "w") as f:
                json.dump(to_jsonable({k: v for k, v in rec.items()}), f)
        self.history.add("counterfactual_fork", self.tick_count, fork=fid, intervention=intervention, diff=rec["diff"])
        return rec

    # ------------------------------------------------------------------ estado (para o CheckpointEngine)
    def state_dict(self) -> Dict[str, Any]:
        return {
            "meta": {"tick": self.tick_count, "created": self.created, "total_runtime": self.total_runtime,
                     "sessions": self.sessions, "ids": self.ids.value, "fidelity_log": dict(self.fidelity_log),
                     "interventions": self.interventions[-500:], "versions": ENGINE_VERSIONS,
                     "versions_hash": ENGINE_VERSIONS_HASH, "config": self.cfg.to_dict(), "legacy": self.legacy_summary,
                     "focus": {k: {"genome": v["genome"], "reason": v["reason"]} for k, v in self.focus.items()},
                     "forks": {k: {kk: vv for kk, vv in v.items() if kk != "divergence_map"} for k, v in list(self.forks.items())[-20:]}},
            "world": self.ws.state_dict(), "agents": self.pool.state_dict(), "ledger": self.ledger.state_dict(),
            "lineage": self.lineage.state_dict(), "f0": self.f0.state_dict() if self.f0 else None,
            "social": self.social.state_dict(), "culture": self.culture.state_dict(),
            "settlements": self.settlements.state_dict(), "civ": self.civ.state_dict(), "history": self.history.state_dict(),
            "qd": self.qd.state_dict(), "archive": self.archive.state_dict(), "scientist": self.scientist.state_dict(),
            "sentinel": self.sentinel.state_dict(),
            "affect": {"directions": self.affect.directions},
            "rng": {"world": self.gen.get_state().cpu().numpy(), "instruments": self.inst_rng.bit_generator.state},
            "demography": list(self.demography.items),
        }

    def load_state_dict(self, s: Dict[str, Any]):
        m = s["meta"]
        self.tick_count, self.created = int(m["tick"]), m["created"]
        self.total_runtime, self.sessions = float(m["total_runtime"]), int(m["sessions"])
        self.ids.value = int(m["ids"]); self.fidelity_log = collections.Counter(m.get("fidelity_log", {}))
        self.interventions = list(m.get("interventions", [])); self.legacy_summary = m.get("legacy")
        self.focus = {k: {"genome": np.asarray(v["genome"], np.float32), "reason": v["reason"]} for k, v in m.get("focus", {}).items()}
        self.forks = dict(m.get("forks", {}))
        self.ws = WorldState.from_state_dict(s["world"], self.device)
        ag = s["agents"]
        n = int(ag["n"])
        if n > self.pool.cap:
            # perfil menor que o do checkpoint: excedente vai para F0 (contagem e genomas preservados — bookkeeping correto)
            extra = {k: np.asarray(v)[self.pool.cap:] for k, v in ag.items() if isinstance(v, np.ndarray) and len(v) == n}
            ag = {k: (np.asarray(v)[: self.pool.cap] if isinstance(v, np.ndarray) and len(v) == n else v) for k, v in ag.items()}
            ag["n"] = self.pool.cap
            self._pending_f0 = extra
        self.pool.load_state_dict(ag)
        self.pool.body_slot[:] = -1
        self.pool.fidelity[self.pool.alive] = 1                    # corpos são reconstruídos pelo desenvolvimento (determinístico)
        self.ledger.load_state_dict(s["ledger"])
        self.lineage.load_state_dict(s["lineage"])
        self.f0 = RegionAggregates(self.cfg, self.ws.H, self.ws.W, self.L.P)
        if s.get("f0"):
            self.f0.load_state_dict(s["f0"])
        if getattr(self, "_pending_f0", None):
            ex = self._pending_f0
            self.f0.absorb(self.f0.region_of(ex["pos"]), ex["genome"], ex["energy"], ex["signature"], self.inst_rng)
            self._pending_f0 = None
        self.social.load_state_dict(s["social"]); self.culture.load_state_dict(s["culture"])
        self.settlements.load_state_dict(s["settlements"]); self.civ.load_state_dict(s["civ"])
        self.history.load_state_dict(s["history"]); self.qd.load_state_dict(s["qd"])
        self.archive.load_state_dict(s["archive"]); self.scientist.load_state_dict(s["scientist"])
        self.sentinel.load_state_dict(s["sentinel"])
        self.affect.directions = {k: np.asarray(v, np.float32) for k, v in s.get("affect", {}).get("directions", {}).items()}
        try:
            self.gen.set_state(torch.as_tensor(np.asarray(s["rng"]["world"]), dtype=torch.uint8))
            self.inst_rng.bit_generator.state = s["rng"]["instruments"]
        except Exception as e:
            if self.logger:
                self.logger.event("rng_restore_warning", error=repr(e))
        for x in s.get("demography", []):
            self.demography.push(x)
        self.ws.tick = self.tick_count

    # ------------------------------------------------------------------ resumo
    def status(self) -> Dict[str, Any]:
        civ = self.last.get("civ", {})
        return {"tick": self.tick_count, "population": self.pool.n_alive(), "f0_population": float(self.f0.total()) if self.f0 else 0.0,
                "bodies": {f"F{k}": int((r.agent_slot >= 0).sum()) for k, r in self.bodies.items()},
                "lineage_total": self.lineage.total, "civ_level": self.civ.level,
                "civ_level_name": C_LEVELS[self.civ.level] if self.civ.level >= 0 else "—",
                "h_level": self.scientist.best_level.get("H", -1), "a_level": self.scientist.best_level.get("A", -1),
                "settlements": civ.get("settlements", 0), "blocks": civ.get("blocks", 0.0),
                "qd_generations": self.qd.generations, "discoveries": len(self.archive.discoveries),
                "paused": self.paused, "world_age": self.tick_count, "created": self.created}

## 16 — CHECKPOINT ENGINE: shards + manifesto + hashes, atômico, write-behind, recuperação sem reset silencioso
Fluxo: snapshot do estado (cópias GPU→CPU) → serialização em **shards** no SSD local (`/content/genesis_work`) →
manifesto com SHA-256 de cada shard → validação (rehash) → cópia para o Drive em `checkpoints/.tmp_*` → conferência →
rotação atômica `current → previous → backups/` e `rename(.tmp → current)`. A escrita roda numa thread (a simulação
continua); no fim de sessão é síncrona. Formato: JSON + NPZ (`allow_pickle=False`) — **nunca pickle**.
Carga: current → previous → backups (mais novo primeiro), cada um com verificação de hashes. Se existirem checkpoints
e nenhum for válido, eles são preservados em `checkpoints/corrupted_*` e o boot **avisa** (nunca reset silencioso).

In [ ]:
# @title 16 — CHECKPOINT ENGINE: shards + manifesto + hashes, atômico, write-behind, recuperação sem reset silencioso { display-mode: "form" }
class CheckpointEngine:
    MAX_PART_BYTES = 48 * 1024 * 1024
    MIGRATIONS: Dict[int, Callable[[Dict[str, Any]], Dict[str, Any]]] = {}

    def __init__(self, paths: CivPaths, logger: Optional[EventLogger], keep_backups: int = 3):
        self.paths, self.logger, self.keep = paths, logger, keep_backups
        self.root = paths.sub("checkpoints")
        self.backups = paths.sub("checkpoints/backups")
        self._thread: Optional[threading.Thread] = None
        self.last_info: Dict[str, Any] = {}
        self.last_saved_tick = -1
        self.last_saved_ts = 0.0
        self.saves = 0
        self.failures = 0

    # ------------------------------------------------------------------ (de)serialização sem pickle
    def _pack(self, obj, arrays: Dict[str, np.ndarray], prefix: str):
        if isinstance(obj, torch.Tensor):
            obj = obj.detach().cpu().numpy()
        if isinstance(obj, np.ndarray):
            if obj.dtype == object:
                raise TypeError(f"array de objetos em {prefix} não é serializável sem pickle")
            name = f"a{len(arrays):05d}"
            arrays[name] = obj
            return {"__nd__": name}
        if isinstance(obj, dict):
            return {str(k): self._pack(v, arrays, f"{prefix}.{k}") for k, v in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [self._pack(v, arrays, f"{prefix}[]") for v in obj]
        if isinstance(obj, np.generic):
            return obj.item()
        if isinstance(obj, set):
            return sorted(obj)
        if hasattr(obj, "__dataclass_fields__"):
            return self._pack(asdict(obj), arrays, prefix)
        return obj

    def _unpack(self, obj, arrays: Dict[str, np.ndarray]):
        if isinstance(obj, dict):
            if "__nd__" in obj and len(obj) == 1:
                return arrays[obj["__nd__"]]
            return {k: self._unpack(v, arrays) for k, v in obj.items()}
        if isinstance(obj, list):
            return [self._unpack(v, arrays) for v in obj]
        return obj

    @staticmethod
    def _sha(path: str) -> str:
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 22), b""):
                h.update(chunk)
        return h.hexdigest()

    def _write_shards(self, state: Dict[str, Any], out_dir: str) -> Dict[str, Any]:
        os.makedirs(out_dir, exist_ok=True)
        shards = []
        for key, val in state.items():
            arrays: Dict[str, np.ndarray] = {}
            js = self._pack(val, arrays, key)
            jp = os.path.join(out_dir, f"{key}.json")
            with open(jp, "w") as f:
                json.dump(js, f)
            shards.append({"file": f"{key}.json", "key": key, "kind": "json"})
            if arrays:
                part, cur, files = 0, {}, []
                size = 0
                for name, arr in arrays.items():
                    cur[name] = arr; size += arr.nbytes
                    if size >= self.MAX_PART_BYTES:
                        fn = f"{key}.part{part:03d}.npz"; np.savez(os.path.join(out_dir, fn), **cur)
                        files.append(fn); part += 1; cur, size = {}, 0
                if cur:
                    fn = f"{key}.part{part:03d}.npz"; np.savez(os.path.join(out_dir, fn), **cur); files.append(fn)
                for fn in files:
                    shards.append({"file": fn, "key": key, "kind": "npz"})
        for sh in shards:
            p = os.path.join(out_dir, sh["file"])
            sh["sha256"] = self._sha(p); sh["bytes"] = os.path.getsize(p)
        manifest = {"checkpoint_version": CHECKPOINT_VERSION, "genesis_version": GENESIS_VERSION,
                    "versions": ENGINE_VERSIONS, "versions_hash": ENGINE_VERSIONS_HASH, "created": iso_now(),
                    "tick": int(state["meta"]["tick"]), "shards": shards,
                    "total_bytes": int(sum(s["bytes"] for s in shards))}
        with open(os.path.join(out_dir, "manifest.json"), "w") as f:
            json.dump(manifest, f, indent=1)
        return manifest

    def verify(self, d: str) -> Tuple[bool, str]:
        mp = os.path.join(d, "manifest.json")
        if not os.path.exists(mp):
            return False, "manifest ausente"
        try:
            man = json.load(open(mp))
        except Exception as e:
            return False, f"manifest ilegível: {e}"
        for sh in man.get("shards", []):
            p = os.path.join(d, sh["file"])
            if not os.path.exists(p):
                return False, f"shard ausente: {sh['file']}"
            if os.path.getsize(p) != sh["bytes"] or self._sha(p) != sh["sha256"]:
                return False, f"hash divergente: {sh['file']}"
        return True, "ok"

    def read(self, d: str) -> Dict[str, Any]:
        man = json.load(open(os.path.join(d, "manifest.json")))
        ver = int(man.get("checkpoint_version", 0))
        if ver > CHECKPOINT_VERSION:
            raise RuntimeError(f"checkpoint v{ver} é mais novo que este motor (v{CHECKPOINT_VERSION}) — não carregado")
        if ver < 3:
            raise RuntimeError(f"checkpoint v{ver} é do Genesis Ω (2D): importe como arquivo legado, não como mundo")
        by_key: Dict[str, Dict[str, Any]] = collections.defaultdict(dict)
        for sh in man["shards"]:
            by_key[sh["key"]].setdefault("files", []).append(sh)
        state = {}
        for key, info in by_key.items():
            arrays: Dict[str, np.ndarray] = {}
            js = None
            for sh in info["files"]:
                p = os.path.join(d, sh["file"])
                if sh["kind"] == "json":
                    js = json.load(open(p))
                else:
                    with np.load(p, allow_pickle=False) as z:
                        for k in z.files:
                            arrays[k] = z[k]
            state[key] = self._unpack(js, arrays)
        for v in sorted(self.MIGRATIONS):
            if ver < v:
                state = self.MIGRATIONS[v](state)
        state["_manifest"] = man
        return state

    # ------------------------------------------------------------------ escrita atômica
    def save(self, uni: "UniverseEngine", sync: bool = False, reason: str = "periodic") -> bool:
        if self._thread is not None and self._thread.is_alive():
            if not sync:
                return False
            self._thread.join()
        t0 = time.time()
        state = uni.state_dict()
        snap_t = time.time() - t0
        visual = self.visual_snapshot(uni)
        tick = uni.tick_count

        def work():
            try:
                self._write(state, tick, visual, reason, snap_t)
            except Exception as e:
                self.failures += 1
                tb = traceback.format_exc()
                if self.logger:
                    self.logger.event("checkpoint_failed", echo=True, error=repr(e), trace=tb[-2000:])
        if sync:
            work()
        else:
            self._thread = threading.Thread(target=work, name="genesis-checkpoint", daemon=True)
            self._thread.start()
        return True

    def _write(self, state, tick, visual, reason, snap_t):
        t0 = time.time()
        cid = f"ckpt_{tick:010d}_{int(time.time())}"
        local = self.paths.tmp(cid)
        man = self._write_shards(state, local)
        ok, why = self.verify(local)
        if not ok:
            raise RuntimeError(f"verificação local falhou: {why}")
        tmp = os.path.join(self.root, f".tmp_{cid}")
        if os.path.exists(tmp):
            shutil.rmtree(tmp)
        shutil.copytree(local, tmp)
        for sh in man["shards"]:                                     # conferência pós-cópia (tamanho + hash)
            p = os.path.join(tmp, sh["file"])
            if os.path.getsize(p) != sh["bytes"]:
                raise RuntimeError(f"cópia truncada: {sh['file']}")
        ok, why = self.verify(tmp)
        if not ok:
            raise RuntimeError(f"verificação no destino falhou: {why}")
        cur, prev = os.path.join(self.root, "current"), os.path.join(self.root, "previous")
        if os.path.exists(prev):
            man_prev = json.load(open(os.path.join(prev, "manifest.json"))) if os.path.exists(os.path.join(prev, "manifest.json")) else {}
            dst = os.path.join(self.backups, f"ckpt_{int(man_prev.get('tick', 0)):010d}_{int(time.time())}")
            os.replace(prev, dst)
        if os.path.exists(cur):
            os.replace(cur, prev)
        os.replace(tmp, cur)
        shutil.rmtree(local, ignore_errors=True)
        bk = sorted(glob.glob(os.path.join(self.backups, "ckpt_*")))
        for old in bk[:-self.keep]:
            shutil.rmtree(old, ignore_errors=True)
        if visual is not None:
            snapdir = self.paths.sub("snapshots")
            np.savez_compressed(os.path.join(snapdir, "last_world.npz"), **{k: v for k, v in visual.items() if isinstance(v, np.ndarray)})
            with open(os.path.join(snapdir, "last_world.json"), "w") as f:
                json.dump(to_jsonable({k: v for k, v in visual.items() if not isinstance(v, np.ndarray)}), f)
        self.saves += 1
        self.last_saved_tick, self.last_saved_ts = tick, time.time()
        self.last_info = {"tick": tick, "ts": iso_now(), "bytes": man["total_bytes"], "shards": len(man["shards"]),
                          "seconds": round(time.time() - t0, 2), "snapshot_seconds": round(snap_t, 2), "reason": reason}
        if self.logger:
            self.logger.event("checkpoint", **self.last_info)
        self._storage_pressure()

    def wait(self):
        if self._thread is not None:
            self._thread.join()

    def _storage_pressure(self):
        try:
            free = shutil.disk_usage(self.root).free
        except Exception:
            return
        if free < 2 * 1024 ** 3:
            bk = sorted(glob.glob(os.path.join(self.backups, "ckpt_*")))
            for old in bk[:-1]:
                shutil.rmtree(old, ignore_errors=True)
            for f in glob.glob(os.path.join(self.paths.work, "tmp", "*")):
                shutil.rmtree(f, ignore_errors=True) if os.path.isdir(f) else os.remove(f)
            if self.logger:
                self.logger.event("storage_pressure", echo=True, free_bytes=free,
                                  action="pruned old backups and temporary files; checkpoints, champions and discoveries preserved")

    # ------------------------------------------------------------------ carga com recuperação
    def candidates(self) -> List[str]:
        out = [os.path.join(self.root, "current"), os.path.join(self.root, "previous")]
        out += sorted(glob.glob(os.path.join(self.backups, "ckpt_*")), reverse=True)
        return [d for d in out if os.path.isdir(d)]

    def load_latest(self) -> Tuple[Optional[Dict[str, Any]], Dict[str, Any]]:
        info: Dict[str, Any] = {"tried": []}
        cands = self.candidates()
        for d in cands:
            ok, why = self.verify(d)
            info["tried"].append({"dir": os.path.basename(d), "ok": ok, "why": why})
            if not ok:
                if self.logger:
                    self.logger.event("checkpoint_corrupted", echo=True, dir=d, reason=why)
                continue
            try:
                st = self.read(d)
                info.update({"loaded": d, "tick": st["_manifest"]["tick"], "fallback": d != cands[0]})
                return st, info
            except Exception as e:
                info["tried"][-1].update({"ok": False, "why": repr(e)})
                if self.logger:
                    self.logger.event("checkpoint_unreadable", echo=True, dir=d, error=repr(e))
        if cands:
            stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            keep = os.path.join(self.root, f"corrupted_{stamp}")
            os.makedirs(keep, exist_ok=True)
            for d in cands:
                shutil.move(d, os.path.join(keep, os.path.basename(d)))
            info["corrupted_preserved"] = keep
            info["warning"] = ("NENHUM checkpoint válido: arquivos preservados em " + keep +
                               " — um novo mundo será criado (aviso explícito, não é reset silencioso)")
        return None, info

    def visual_snapshot(self, uni: "UniverseEngine") -> Optional[Dict[str, Any]]:
        """'Last known world': estado visual compacto para o observatório quando a simulação estiver offline."""
        try:
            ws, pool = uni.ws, uni.pool
            step = max(1, ws.W // 256)
            idx = torch.nonzero(pool.alive).flatten()[:20000]
            return {"height": ws.height[::step, ::step].cpu().numpy().astype(np.float16),
                    "water": (ws.is_water & ~ws.filled)[::step, ::step].cpu().numpy(),
                    "struct": ws.struct_h[::step, ::step].cpu().numpy().astype(np.int8),
                    "agents": pool.pos[idx].cpu().numpy().astype(np.float32),
                    "colors": pool.color[idx].cpu().numpy().astype(np.float16),
                    "tick": int(uni.tick_count), "ts": iso_now(), "status": uni.status(), "step": step}
        except Exception as e:
            if self.logger:
                self.logger.event("visual_snapshot_failed", error=repr(e))
            return None

## 17 — TELEMETRY ENGINE: observador puro (protocolo versionado, deltas binários, fila limitada)
A telemetria **nunca** entra no caminho crítico da ciência: roda na thread da simulação entre ticks, só lê estado,
não consome o RNG do mundo e não altera fidelidade científica. Publica em frequência controlada (`telemetry_hz`):
* `base` — relevo/água/biomas reduzidos + estruturas (quando muda muito ou sob pedido);
* `frame` — agentes (subconjunto estável: corpos F2+, selecionado, região da câmera, amostra por hash do ID),
  posições quantizadas uint16, rumo, cor genética, fidelidade, energia, tipo cultural; deltas de estruturas;
  eventos novos; luz/tempestade;
* `body` — nós/sinais reais do corpo (F2+) ou *gêmeo visual* (desenvolvido do genoma; marcado como visual);
* `status`, `lab` (câmera do cientista), `heat` (camadas), `ring` (replay temporal do mundo).
Pedidos REST que precisam de tensores (inspetor de agente etc.) entram numa fila atendida pela simulação.

In [ ]:
# @title 17 — TELEMETRY ENGINE: observador puro (protocolo versionado, deltas binários, fila limitada) { display-mode: "form" }
def ensure_package(module: str, pip_spec: str) -> bool:
    """Instala só o necessário (versões com limites compatíveis) quando ausente."""
    try:
        importlib.import_module(module)
        return True
    except ImportError:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_spec], timeout=300)
            importlib.invalidate_caches()
            importlib.import_module(module)
            return True
        except Exception as e:
            print(f"[deps] não foi possível instalar {pip_spec}: {e}")
            return False


HAVE_MSGPACK = ensure_package("msgpack", "msgpack>=1.0,<2")
if HAVE_MSGPACK:
    import msgpack

LIVE_PROTOCOL = {"name": "genesis-live", "version": RENDER_PROTOCOL_VERSION, "telemetry": TELEMETRY_SCHEMA_VERSION,
                 "events": EVENT_SCHEMA_VERSION}
HEAT_LAYERS = ["population", "resources", "activity", "species", "culture", "communication", "settlements", "novelty",
               "temperature", "storm"]


class ObserverDesk:
    """Ponte thread-safe servidor ↔ simulação (sem acesso direto a tensores a partir do servidor)."""

    def __init__(self, maxlen: int = 4):
        self.lock = threading.Lock()
        self.frames: "collections.deque" = collections.deque(maxlen=maxlen)
        self.status: Dict[str, Any] = {}
        self.base: Optional[bytes] = None
        self.base_seq = 0
        self.body: Dict[int, bytes] = {}
        self.lab: Optional[bytes] = None
        self.heat: Dict[str, bytes] = {}
        self.ring_index: List[Dict[str, Any]] = []
        self.requests: "queue.Queue" = queue.Queue(maxsize=256)
        self.selected: Dict[str, int] = {}                 # cliente → aid selecionado (visual)
        self.layers: Dict[str, str] = {}
        self.views: Dict[str, Tuple[float, float, float, float, int]] = {}
        self.want_base = False
        self.admin_queue: "queue.Queue" = queue.Queue(maxsize=32)
        self.seq = 0
        self.online = True

    def publish_frame(self, payload: bytes):
        with self.lock:
            self.seq += 1
            self.frames.append((self.seq, payload))

    def latest(self) -> Optional[Tuple[int, bytes]]:
        with self.lock:
            return self.frames[-1] if self.frames else None

    def ask(self, kind: str, args: Dict[str, Any], timeout: float = 4.0) -> Any:
        """Chamado pelo servidor: enfileira pedido e espera a simulação responder (com timeout)."""
        ev = threading.Event()
        box: Dict[str, Any] = {}
        try:
            self.requests.put_nowait((kind, args, ev, box))
        except queue.Full:
            return {"error": "busy"}
        if not ev.wait(timeout):
            return {"error": "timeout", "note": "simulação ocupada; tente novamente"}
        return box.get("result")


def pack_msg(obj: Dict[str, Any]) -> bytes:
    return msgpack.packb(obj, use_bin_type=True)


class TelemetryEngine:
    def __init__(self, uni: "UniverseEngine", desk: ObserverDesk):
        self.u, self.desk = uni, desk
        self.cfg = uni.cfg
        self.last_frame = 0.0
        self.last_status = 0.0
        self.last_heat = 0.0
        self.last_body = 0.0
        self.last_lab = 0.0
        self.last_struct: Optional[torch.Tensor] = None
        self.last_filled: Optional[torch.Tensor] = None
        self.events_sent = 0
        self.ring: "collections.deque" = collections.deque(maxlen=uni.cfg.snapshot_ring)
        self.ring_last_tick = -10 ** 9
        self.tps = 0.0
        self._tps_t, self._tps_tick = time.time(), uni.tick_count
        self.visual = BodyRegistry(uni.cfg, uni.dev, uni.body, 4, uni.cfg.body_substeps)
        self.visual_aid: Dict[int, int] = {}
        self.xhist: Dict[int, "collections.deque"] = {}
        self.frames_sent = 0
        self.bytes_sent = 0
        self.state_label = "BOOTING"
        self.checkpoint_info: Dict[str, Any] = {}
        self.session_start = time.time()
        self.lab_view_fn: Optional[Callable[[], Optional[Dict[str, Any]]]] = None

    # ------------------------------------------------------------------ base (relevo, água, biomas, estruturas)
    def build_base(self) -> bytes:
        ws = self.u.ws
        step = max(1, math.ceil(max(ws.H, ws.W) / 256))
        h = ws.height[::step, ::step]
        wat = (ws.is_water & ~ws.filled)[::step, ::step]
        temp = ws.temp_base[::step, ::step]; moist = ws.moisture[::step, ::step]; fert = ws.fertility[::step, ::step]
        hn = (h - h.min()) / (h.max() - h.min() + 1e-6)
        # cor de bioma = função declarada de (altura, temperatura, umidade, fertilidade); legenda no viewer
        r = (0.35 + 0.4 * hn - 0.25 * fert).clamp(0, 1); g = (0.3 + 0.5 * fert + 0.1 * moist).clamp(0, 1)
        b = (0.2 + 0.2 * moist + 0.35 * (temp < 0.25).float()).clamp(0, 1)
        snow = (temp < 0.18) & ~wat
        r = torch.where(snow, torch.full_like(r, 0.92), r); g = torch.where(snow, torch.full_like(g, 0.94), g)
        b = torch.where(snow, torch.full_like(b, 0.97), b)
        rgb = (torch.stack([r, g, b], -1) * 255).to(torch.uint8)
        hq = ((h / max(1e-6, self.cfg.height_scale)).clamp(0, 1) * 65535).to(torch.int32).to(torch.int16)
        st = self._structures()
        self.last_struct = ws.struct_h.clone(); self.last_filled = ws.filled.clone()
        msg = {"t": "base", "proto": LIVE_PROTOCOL, "tick": self.u.tick_count, "W": ws.W, "H": ws.H, "step": step,
               "gw": int(h.shape[1]), "gh": int(h.shape[0]), "height_scale": self.cfg.height_scale,
               "sea_level": ws.flags.get("sea_level", 0.0), "height": hq.cpu().numpy().tobytes(),
               "water": wat.cpu().numpy().astype(np.uint8).tobytes(), "rgb": rgb.cpu().numpy().tobytes(),
               "struct": st.tobytes(), "struct_n": int(st.shape[0]), "max_stack": self.cfg.max_stack,
               "settlements": self._settlements(), "roads": self._roads(),
               "legend": {"terrain": "cor = f(altura, temperatura, umidade, fertilidade); neve = temperatura < 0.18",
                          "agents": "cor = assinatura visual genética (display) — identidade, não semântica",
                          "blocks": "matéria construída real (pilhas); azul-esverdeado = aterro sobre água"}}
        return pack_msg(msg)

    def _structures(self) -> np.ndarray:
        ws = self.u.ws
        built = (ws.struct_h > 0) & (ws.struct_integrity < 1e8)
        ys, xs = torch.nonzero(built, as_tuple=True)
        hs = ws.struct_h[ys, xs]
        fy, fx = torch.nonzero(ws.filled, as_tuple=True)
        a = torch.stack([xs, ys, hs.long(), torch.zeros_like(xs)], 1)
        b = torch.stack([fx, fy, torch.ones_like(fx), torch.ones_like(fx)], 1)
        out = torch.cat([a, b])[:200000]
        return out.to(torch.int16).cpu().numpy()

    def _settlements(self) -> List[Dict[str, Any]]:
        out = []
        for sid, s in self.u.settlements.settlements.items():
            if s["state"] in ("detected", "city", "collapsed", "candidate"):
                out.append({"id": sid, "state": s["state"], "centre": s.get("centre", [0, 0]), "pop": s.get("pop", 0),
                            "blocks": s.get("blocks", 0), "cells": len(s.get("cells", [])), "cell": self.u.settlements.cell,
                            "founded": s.get("founded")})
        return out

    def _roads(self) -> List[Dict[str, Any]]:
        return [{"id": r["id"], "tiles": r["tiles"], "bbox": r["bbox"], "settlements": r["settlements"]}
                for r in self.u.settlements.road_segments[:100]]

    # ------------------------------------------------------------------ frame dinâmico
    def _select_agents(self, max_n: int) -> torch.Tensor:
        pool = self.u.pool
        alive = torch.nonzero(pool.alive).flatten()
        if alive.numel() <= max_n:
            return alive
        pri = torch.zeros(alive.numel(), device=pool.device)
        pri += (pool.fidelity[alive] >= 2).float() * 10
        sel = set(self.desk.selected.values())
        if sel:
            pri += torch.isin(pool.aid[alive], torch.tensor(sorted(sel), device=pool.device)).float() * 100
        views = list(self.desk.views.values())
        if views:
            x0, y0, x1, y1, _ = views[-1]
            p = pool.pos[alive]
            inside = (p[:, 0] >= x0) & (p[:, 0] <= x1) & (p[:, 1] >= y0) & (p[:, 1] <= y1)
            pri += inside.float() * 5
        # amostra estável (hash do ID) → o mesmo agente aparece em frames seguidos (interpolação suave)
        h = ((pool.aid[alive] * 2654435761) % 1000).float() / 1000
        pri += h
        return alive[torch.argsort(pri, descending=True)[:max_n]]

    def build_frame(self) -> bytes:
        u, pool, ws = self.u, self.u.pool, self.u.ws
        max_n = self.cfg.max_stream_agents
        if self.desk.views:
            max_n = min(max_n, max(50, list(self.desk.views.values())[-1][4]))
        idx = self._select_agents(max_n)
        pos = pool.pos[idx]
        q = torch.stack([(pos[:, 0] / ws.W * 65535), (pos[:, 1] / ws.H * 65535)], 1).clamp(0, 65535).to(torch.int32)
        msg: Dict[str, Any] = {"t": "frame", "tick": u.tick_count, "ts": time.time(), "n": int(idx.numel()),
                               "ids": pool.aid[idx].to(torch.int64).cpu().numpy().tobytes(),
                               "pos": q.to(torch.int32).cpu().numpy().astype(np.uint16).tobytes(),
                               "head": ((pool.heading[idx] / (2 * math.pi)) * 255).clamp(0, 255).to(torch.uint8).cpu().numpy().tobytes(),
                               "color": (pool.color[idx] * 255).clamp(0, 255).to(torch.uint8).cpu().numpy().tobytes(),
                               "fid": pool.fidelity[idx].to(torch.uint8).cpu().numpy().tobytes(),
                               "energy": (pool.energy[idx] * 255).clamp(0, 255).to(torch.uint8).cpu().numpy().tobytes(),
                               "culture": (pool.culture_type[idx] + 1).clamp(0, 255).to(torch.uint8).cpu().numpy().tobytes(),
                               "carry": pool.carried[idx].clamp(0, 255).to(torch.uint8).cpu().numpy().tobytes(),
                               "emit": (pool.emit[idx].max(1).values * 255).clamp(0, 255).to(torch.uint8).cpu().numpy().tobytes(),
                               "light_phase": float((u.tick_count % self.cfg.day_period) / self.cfg.day_period),
                               "season": float(math.sin(2 * math.pi * u.tick_count / max(1, self.cfg.season_period)))}
        # deltas de estruturas
        if self.last_struct is not None and self.last_struct.shape == ws.struct_h.shape:
            ch = (ws.struct_h != self.last_struct) | (ws.filled != self.last_filled)
            if bool(ch.any()):
                ys, xs = torch.nonzero(ch, as_tuple=True)
                if ys.numel() > 20000:
                    self.desk.want_base = True
                else:
                    hv = torch.where(ws.struct_integrity[ys, xs] < 1e8, ws.struct_h[ys, xs], torch.zeros_like(ws.struct_h[ys, xs]))
                    d = torch.stack([xs, ys, hv.long(), ws.filled[ys, xs].long()], 1).to(torch.int16)
                    msg["sdelta"] = d.cpu().numpy().tobytes()
                    msg["sdelta_n"] = int(d.shape[0])
                self.last_struct = ws.struct_h.clone(); self.last_filled = ws.filled.clone()
        storm = F.adaptive_avg_pool2d(ws.storm[None, None], (32, 32))[0, 0]
        msg["storm"] = (storm * 255).clamp(0, 255).to(torch.uint8).cpu().numpy().tobytes()
        evs = list(u.events_out)
        msg["events"] = to_jsonable(evs[self.events_sent:] if self.events_sent <= len(evs) else evs[-20:])[-30:]
        self.events_sent = len(evs)
        return pack_msg(msg)

    # ------------------------------------------------------------------ corpo (real F2+ ou gêmeo visual)
    def build_body(self, aid: int) -> Optional[bytes]:
        pool = self.u.pool
        hit = torch.nonzero(pool.alive & (pool.aid == aid)).flatten()
        if hit.numel() == 0:
            return None
        s = int(hit[0])
        fid = int(pool.fidelity[s])
        reg, bs, visual = None, None, False
        for lvl, r in self.u.bodies.items():
            m = torch.nonzero(r.agent_slot == s).flatten()
            if m.numel():
                reg, bs = r, int(m[0]); break
        if reg is None:
            visual = True
            if aid not in self.visual_aid:
                self.visual.detach(torch.nonzero(self.visual.agent_slot >= 0).flatten())
                self.visual_aid = {}
                got = self.visual.attach(torch.tensor([s], device=pool.device), pool.aid[s:s + 1], pool.genome[s:s + 1],
                                         pool.aid[s:s + 1] % 100003)
                if got.numel() == 0:
                    return None
                self.visual_aid[aid] = int(got[0])
            reg, bs = self.visual, self.visual_aid[aid]
            motor = torch.zeros(reg.cap, 2, device=pool.device)
            motor[bs, 0] = torch.tanh(pool.out[s, 0]); motor[bs, 1] = torch.tanh(pool.out[s, 1])
            reg.body.step(reg.bb, motor)                              # física do gêmeo visual (não afeta a ciência)
        bb = reg.bb
        act = bb.active[bs]
        P = bb.pos[bs]
        E = reg.dev.edges
        on = bb.e_on[bs]
        props = bb.props[bs]
        x = pool.x[s]
        dq = self.xhist.setdefault(aid, collections.deque(maxlen=120))
        dq.append(x.detach().cpu().numpy().astype(np.float16))
        msg = {"t": "body", "aid": aid, "fidelity": fid, "visual_twin": visual, "N": int(P.shape[0]),
               "nodes": P.detach().cpu().numpy().astype(np.float32).tobytes(), "active": act.cpu().numpy().astype(np.uint8).tobytes(),
               "signal": (bb.signal[bs].clamp(0, 4) / 4 * 255).to(torch.uint8).cpu().numpy().tobytes(),
               "edges": E[on].to(torch.int16).cpu().numpy().tobytes(), "n_edges": int(on.sum()),
               "props": (props.clamp(0, 1) * 255).to(torch.uint8).cpu().numpy().tobytes(), "prop_names": PROP_NAMES,
               "state": np.stack(list(dq)).tobytes(), "state_T": len(dq), "state_N": int(x.numel()),
               "note": ("gêmeo visual: corpo desenvolvido do genoma só para visualização; a ciência deste agente usa o resumo F1"
                        if visual else "corpo científico real (Matter Nodes)")}
        return pack_msg(msg)

    # ------------------------------------------------------------------ camadas (heatmaps)
    def build_heat(self, layer: str, R: int = 64) -> bytes:
        u, ws, pool = self.u, self.u.ws, self.u.pool
        dev = ws.device

        def pool2(x):
            return F.adaptive_avg_pool2d(x[None, None].float(), (R, R))[0, 0]
        alive = pool.alive
        p = pool.pos[alive]
        cy = (p[:, 1] / ws.H * R).long().clamp(0, R - 1); cx = (p[:, 0] / ws.W * R).long().clamp(0, R - 1)
        cnt = torch.bincount(cy * R + cx, minlength=R * R).float().view(R, R)
        if layer == "population":
            g = cnt
        elif layer == "resources":
            g = pool2(ws.resource)
        elif layer == "activity":
            g = pool2(ws.traffic)
        elif layer == "communication":
            g = pool2(ws.sound.sum(0))
        elif layer == "temperature":
            g = pool2(ws.temperature)
        elif layer == "storm":
            g = pool2(ws.storm)
        elif layer == "settlements":
            g = pool2(((ws.struct_h > 0) & (ws.struct_integrity < 1e8)).float() + ws.filled.float())
        elif layer in ("species", "culture", "novelty"):
            if layer == "species":
                val = (pool.genome[alive][:, : min(8, self.u.L.P)].sum(1) * 7.31) % 1.0
            elif layer == "culture":
                val = (pool.culture_type[alive].float() + 1) / 8
            else:
                sig = pool.signature[alive]
                val = (sig - sig.mean(0)).norm(dim=-1) if sig.shape[0] else sig.sum(1)
            s = torch.zeros(R * R, device=dev).index_add_(0, cy * R + cx, val.float())
            g = (s / cnt.flatten().clamp(min=1)).view(R, R)
        else:
            g = cnt
        g = g / g.max().clamp(min=1e-9)
        return pack_msg({"t": "heat", "layer": layer, "R": R, "tick": u.tick_count,
                         "data": (g * 255).clamp(0, 255).to(torch.uint8).cpu().numpy().tobytes(),
                         "note": "camada derivada do estado real; cores por camada não implicam semântica"})

    # ------------------------------------------------------------------ status
    def build_status(self) -> Dict[str, Any]:
        u = self.u
        now = time.time()
        if now - self._tps_t >= 1.0:
            self.tps = (u.tick_count - self._tps_tick) / (now - self._tps_t)
            self._tps_t, self._tps_tick = now, u.tick_count
        gpu = {}
        if torch.cuda.is_available():
            try:
                free, total = torch.cuda.mem_get_info()
                gpu = {"name": torch.cuda.get_device_name(0), "allocated": int(torch.cuda.memory_allocated()),
                       "reserved": int(torch.cuda.memory_reserved()), "peak": int(torch.cuda.max_memory_allocated()),
                       "free": int(free), "total": int(total)}
                try:
                    gpu["util"] = int(torch.cuda.utilization())
                except Exception:
                    pass
            except Exception as e:
                gpu = {"error": repr(e)}
        st = u.status()
        sci = u.scientist.status()
        maxgen = int(u.pool.generation[u.pool.alive].max()) if u.pool.n_alive() else 0
        return {"t": "status", "state": self.state_label, "tick": u.tick_count, "tps": round(self.tps, 1),
                "speed_label": f"{self.tps:.0f} ticks/s (1 tick = unidade artificial)", "population": st["population"],
                "f0_population": st["f0_population"], "bodies": st["bodies"], "generation": maxgen,
                "qd_generations": st["qd_generations"], "h_level": st["h_level"], "a_level": st["a_level"],
                "c_level": st["civ_level"], "c_name": st["civ_level_name"],
                "h_name": H_LEVELS[st["h_level"]] if st["h_level"] >= 0 else "—",
                "a_name": A_LEVELS[st["a_level"]] if st["a_level"] >= 0 else "—",
                "discoveries": st["discoveries"], "settlements": st["settlements"], "science": sci,
                "checkpoint": self.checkpoint_info, "gpu": gpu, "session_runtime": now - self.session_start,
                "total_runtime": u.total_runtime + (now - self.session_start), "paused": u.paused,
                "profiler": dict(list(u.prof.report().items())[:10]), "world": {"W": u.ws.W, "H": u.ws.H},
                "day_phase": float((u.tick_count % self.cfg.day_period) / self.cfg.day_period),
                "time_labels": {"tick": "unidade de tempo artificial", "day": f"{self.cfg.day_period} ticks (artificial)",
                                "season": f"{self.cfg.season_period} ticks (artificial)"},
                "frames_sent": self.frames_sent, "proto": LIVE_PROTOCOL, "version": GENESIS_VERSION,
                "sentinel": u.sentinel.last, "ethical_triggers": len(u.sentinel.triggers)}

    # ------------------------------------------------------------------ ciclo (chamado pela simulação)
    def pump(self, force: bool = False):
        now = time.time()
        d = self.desk
        try:
            if d.base is None or d.want_base:
                d.base = self.build_base(); d.base_seq += 1; d.want_base = False
            if force or now - self.last_frame >= 1.0 / max(0.5, self.cfg.telemetry_hz):
                d.publish_frame(self.build_frame()); self.last_frame = now; self.frames_sent += 1
            if now - self.last_status >= 1.0:
                st = self.build_status()
                with d.lock:
                    d.status = st
                self.last_status = now
            if d.selected and now - self.last_body >= 0.25:
                for aid in set(d.selected.values()):
                    b = self.build_body(int(aid))
                    if b is not None:
                        d.body[int(aid)] = b
                self.last_body = now
            if now - self.last_heat >= 2.0:
                for layer in set(d.layers.values()):
                    if layer in HEAT_LAYERS:
                        d.heat[layer] = self.build_heat(layer)
                self.last_heat = now
            if self.lab_view_fn is not None and now - self.last_lab >= 0.5:
                v = self.lab_view_fn()
                d.lab = pack_msg({"t": "lab", **{k: (val.tobytes() if isinstance(val, np.ndarray) else val)
                                                  for k, val in v.items()}, "pos_n": int(len(v["pos"]))}) if v else None
                self.last_lab = now
            if self.u.tick_count - self.ring_last_tick >= self.cfg.snapshot_every:
                self._ring_push()
        except Exception as e:
            if self.u.logger:
                self.u.logger.event("telemetry_error", error=repr(e), trace=traceback.format_exc()[-1500:])
        self.service(budget_s=0.02)

    def _ring_push(self):
        pool, ws = self.u.pool, self.u.ws
        idx = self._select_agents(min(2000, self.cfg.max_stream_agents))
        pos = pool.pos[idx]
        q = torch.stack([(pos[:, 0] / ws.W * 65535), (pos[:, 1] / ws.H * 65535)], 1).clamp(0, 65535).to(torch.int32)
        self.ring.append({"tick": self.u.tick_count, "ids": pool.aid[idx].cpu().numpy().astype(np.int64),
                          "pos": q.cpu().numpy().astype(np.uint16), "color": (pool.color[idx] * 255).to(torch.uint8).cpu().numpy(),
                          "pop": self.u.pool.n_alive()})
        self.ring_last_tick = self.u.tick_count
        with self.desk.lock:
            self.desk.ring_index = [{"i": i, "tick": r["tick"], "pop": r["pop"]} for i, r in enumerate(self.ring)]

    def ring_frame(self, i: int) -> Optional[bytes]:
        if not 0 <= i < len(self.ring):
            return None
        r = self.ring[i]
        return pack_msg({"t": "ring", "i": i, "tick": r["tick"], "n": int(len(r["ids"])), "ids": r["ids"].tobytes(),
                         "pos": r["pos"].tobytes(), "color": r["color"].tobytes()})

    # ------------------------------------------------------------------ pedidos do servidor (inspetor etc.)
    def service(self, budget_s: float = 0.02):
        t0 = time.time()
        while time.time() - t0 < budget_s:
            try:
                kind, args, ev, box = self.desk.requests.get_nowait()
            except queue.Empty:
                return
            try:
                box["result"] = self._answer(kind, args)
            except Exception as e:
                box["result"] = {"error": type(e).__name__, "detail": str(e)[:200]}
            ev.set()

    def _answer(self, kind: str, a: Dict[str, Any]) -> Any:
        u = self.u
        if kind == "agent":
            return agent_card(u, int(a["aid"]))
        if kind == "city":
            return u.settlements.public(int(a["sid"]))
        if kind == "ring":
            return self.ring_frame(int(a["i"]))
        if kind == "develop":
            return development_replay(u, int(a["aid"]))
        if kind == "search":
            return search_world(u, str(a["q"]))
        if kind == "history":
            return {"events": u.history.query(int(a.get("since", 0)), limit=int(a.get("limit", 300))),
                    "compressed": dict(u.history.compressed), "total": u.history.total,
                    "demography": u.demography.last(200)}
        if kind == "discoveries":
            ar = u.archive
            return {"discoveries": ar.discoveries[-300:], "pantheon": ar.pantheon, "overall": ar.overall_champion,
                    "convergence": ar.convergence[-20:], "anomalies": list(ar.weird)[-30:],
                    "failures": len(ar.failures), "ethical": u.sentinel.triggers[-20:], "forks": list(u.forks.keys())[-20:]}
        if kind == "ladders":
            return ladder_status(u)
        if kind == "lineage":
            aid = int(a["aid"])
            pool = u.pool
            anc = u.lineage.ancestors(aid, 30)
            kids = pool.aid[pool.alive & (pool.parent == aid)].cpu().numpy().tolist()[:200]
            alive_set = set(pool.aid[pool.alive].cpu().numpy().tolist()) if anc else set()
            sib = []
            p0 = u.lineage.parent_of(aid)
            if p0:
                sib = pool.aid[pool.alive & (pool.parent == p0)].cpu().numpy().tolist()[:50]
            return {"aid": aid, "ancestors": [{"aid": x, "alive": x in alive_set,
                                               "children": u.lineage.children_count.get(x, 0)} for x in anc],
                    "children_alive": kids, "siblings_alive": sib, "children_total": u.lineage.children_count.get(aid, 0)}
        if kind == "world":
            ws = u.ws
            return {"W": ws.W, "H": ws.H, "tick": u.tick_count, "sea_level": ws.flags.get("sea_level"), "seed": ws.seed,
                    "settlements": self._settlements(), "roads": self._roads(), "civ": to_jsonable(u.last.get("civ", {})),
                    "social": to_jsonable({k: v for k, v in u.last.get("social", {}).items() if k != "dissolved"}),
                    "culture": to_jsonable(u.last.get("culture", {})), "communication": to_jsonable(u.last.get("comm", {})),
                    "demography": to_jsonable(u.demography.last(60)), "fidelity": dict(u.fidelity_log),
                    "qd": to_jsonable(u.qd.summary()), "forks": to_jsonable({k: {kk: vv for kk, vv in v.items()
                                                                                    if kk not in ("divergence_map",)}
                                                                               for k, v in u.forks.items()})}
        raise ValueError("pedido desconhecido")


def agent_card(u: "UniverseEngine", aid: int) -> Dict[str, Any]:
    """Inspetor de agente: identidade persistente, corpo, idade, linhagem, recursos, aprendizado, memória, afeto
    funcional (só se houver eixo validado), relações, cultura, evidências."""
    pool = u.pool
    hit = torch.nonzero(pool.alive & (pool.aid == aid)).flatten()
    if hit.numel() == 0:
        anc = u.lineage.ancestors(aid, 20)
        return {"aid": aid, "alive": False, "ancestors": anc, "note": "agente não está vivo (ou foi agregado em F0)"}
    s = int(hit[0])
    G = pool.genome[s]
    gh = genome_hash(G.view(1, -1))
    W0 = u.L.view(G.view(1, -1), "ctl_W0")[0]
    learned = float((pool.W[s] - W0).norm() / (W0.norm() + 1e-6))
    cap = pool.body_cap[s].cpu().numpy()
    partners = u.social.neighbors(aid, u.tick_count, 8)
    ppos = []
    for o, w in partners:
        h2 = torch.nonzero(pool.alive & (pool.aid == o)).flatten()
        ppos.append({"aid": o, "w": round(w, 4), "pos": pool.pos[int(h2[0])].cpu().numpy().tolist() if h2.numel() else None})
    affect = {"status": "no validated functional-affect axis for this lineage", "claim": "NOT YET VALIDATED"}
    for wgh, w in u.sentinel.watch.items():
        if float(u.L.distance(G.view(1, -1), torch.as_tensor(w["genome"], device=G.device).view(1, -1))) < 0.08:
            proj = float(pool.x[s] @ torch.as_tensor(w["v"], device=G.device))
            affect = {"status": "validated A1 valence axis (lineage)", "projection": proj, "threshold": w["thr"],
                      "claim": "functional affect (not subjective experience)", "lineage_genome": wgh}
            break
    evidence = [{"cid": c.cid, "levels": sorted(c.passed)} for c in u.archive.zoo.values()
                if c.passed and hashlib.sha256(G.cpu().numpy().astype(np.float32).tobytes()).hexdigest()[:16] == c.cid[1:]]
    kids = int((pool.alive & (pool.parent == aid)).sum())
    tech = int(pool.tech[s])
    return to_jsonable({
        "aid": aid, "alive": True, "genome_id": gh, "body_id": f"B{gh[:8]}-{aid}", "generation": int(pool.generation[s]),
        "birth_tick": int(pool.birth[s]), "age": float(pool.age[s]), "parent": int(pool.parent[s]), "root": int(pool.root[s]),
        "ancestors": u.lineage.ancestors(aid, 12), "living_children": kids,
        "children_total": u.lineage.children_count.get(aid, 0), "pos": pool.pos[s].cpu().numpy().tolist(),
        "fidelity": int(pool.fidelity[s]), "energy": float(pool.energy[s]), "integrity": float(pool.integrity[s]),
        "fatigue": float(pool.fatigue[s]), "carried": float(pool.carried[s]),
        "body": {"speed": float(cap[0]), "cost": float(cap[1]), "nodes": int(cap[2]), "turn": float(cap[3]),
                 "gait_angle": float(cap[4]), "wobble": float(cap[5]), "measured": bool(pool.body_measured[s])},
        "learning": {"weight_change_relative": learned, "note": "|W − W₀| / |W₀| (aprendizado na vida, não genético)"},
        "internal_state": pool.x[s].cpu().numpy().tolist(), "modulation": float(pool.mod[s]),
        "actions": {ACTION_NAMES[i]: float(v) for i, v in enumerate(torch.sigmoid(pool.out[s]).cpu().numpy().tolist())},
        "lifetime": {"eaten": float(pool.stats[s, 0]), "distance": float(pool.stats[s, 1]), "placed": float(pool.stats[s, 2]),
                     "picked": float(pool.stats[s, 3]), "given": float(pool.stats[s, 4]), "taken": float(pool.stats[s, 5]),
                     "offspring": float(pool.stats[s, 6]), "damage": float(pool.stats[s, 7])},
        "techniques": [TECH_NAMES[b] for b in range(N_TECH) if (tech >> b) & 1],
        "culture_type": int(pool.culture_type[s]), "social": ppos, "affect": affect, "evidence": evidence,
        "species": int(pool.species[s])})


def development_replay(u: "UniverseEngine", aid: int) -> Optional[Dict[str, Any]]:
    pool = u.pool
    hit = torch.nonzero(pool.alive & (pool.aid == aid)).flatten()
    if hit.numel() == 0:
        return None
    s = int(hit[0])
    bp = u.dev.develop(pool.genome[s:s + 1], pool.aid[s:s + 1] % 100003, record=True)
    frames = [f[0].cpu().numpy().astype(np.uint8) for f in bp.frames]
    return {"aid": aid, "D": u.dev.D, "frames": [np.packbits(f).tobytes() for f in frames], "N": u.dev.N,
            "parent_cell": bp.parent_cell[0].cpu().numpy().astype(np.int16).tobytes(),
            "birth_step": bp.birth_step[0].cpu().numpy().astype(np.int16).tobytes(),
            "props": (bp.props[0].clamp(0, 1) * 255).to(torch.uint8).cpu().numpy().tobytes(), "prop_names": PROP_NAMES}


def search_world(u: "UniverseEngine", q: str) -> Dict[str, Any]:
    q = q.strip()[:64]
    out: Dict[str, Any] = {"q": q, "results": []}
    if re.fullmatch(r"\d{1,15}", q):
        aid = int(q)
        hit = torch.nonzero(u.pool.alive & (u.pool.aid == aid)).flatten()
        if hit.numel():
            out["results"].append({"type": "agent", "id": aid, "pos": u.pool.pos[int(hit[0])].cpu().numpy().tolist()})
        sid = int(q)
        if sid in u.settlements.settlements:
            out["results"].append({"type": "settlement", "id": sid, "pos": u.settlements.settlements[sid].get("centre")})
    if q.startswith("C") and q in u.archive.zoo:
        out["results"].append({"type": "candidate", "id": q, "levels": sorted(u.archive.zoo[q].passed)})
    if q.startswith("D"):
        for d in u.archive.discoveries:
            if d["id"].startswith(q):
                out["results"].append({"type": "discovery", "id": d["id"], "level": d["level"]})
    if q.startswith("G") or re.fullmatch(r"[0-9a-f]{6,24}", q):
        pref = q[1:] if q.startswith("G") else q
        idx = torch.nonzero(u.pool.alive).flatten()[:20000]
        for s in idx.tolist():
            if genome_hash(u.pool.genome[s:s + 1]).startswith(pref):
                out["results"].append({"type": "genome", "id": genome_hash(u.pool.genome[s:s + 1]),
                                       "agent": int(u.pool.aid[s]), "pos": u.pool.pos[s].cpu().numpy().tolist()})
                break
    return out


def ladder_status(u: "UniverseEngine") -> Dict[str, Any]:
    zoo = list(u.archive.zoo.values())

    def best(code):
        hits = [(c.cid, c.passed[code]) for c in zoo if code in c.passed]
        if not hits:
            return {"validated": False}
        cid, v = max(hits, key=lambda x: abs(x[1].get("effect", 0) or 0))
        return {"validated": True, "candidate": cid, "effect": v.get("effect"), "discovery": v.get("discovery"),
                "claim": v.get("claim")}
    tested = collections.Counter(code for c in zoo for code in c.attempts)
    return {"H": [{"code": l.split()[0], "name": l, "tested": tested.get(l.split()[0], 0), **best(l.split()[0])} for l in H_LEVELS],
            "A": [{"code": l.split()[0], "name": l, "tested": tested.get(l.split()[0], 0), **best(l.split()[0])} for l in A_LEVELS],
            "C": [{"code": l.split()[0], "name": l, "achieved": l in u.civ.achieved,
                   "tick": u.civ.achieved.get(l, {}).get("tick"),
                   "now": bool(u.civ.last_metrics.get("criteria_now", {}).get(l, False))} for l in C_LEVELS],
            "best": u.scientist.best_level, "target": u.scientist.target_level(), "c_level": u.civ.level,
            "note": "níveis = evidência por probes com controles; não são afirmações de vida real, consciência ou humanidade"}

## 18 — REALTIME SERVER (FastAPI + WebSocket) · segurança · túnel HTTPS · embed no Colab
Segurança: servidor escuta só em `127.0.0.1` (o túnel e o proxy do Colab conectam localmente) · token de
visualização **aleatório por sessão** (`secrets.token_urlsafe`) vai no *fragmento* da URL (#…, nunca enviado em
requisições HTTP nem gravado em logs) · token **admin separado**, nunca no link remoto; com `REMOTE_READ_ONLY`
o admin é recusado em qualquer requisição que chegue pelo túnel · rate limit por IP (admin muito mais restrito) ·
validação de entrada em todo endpoint · **sem endpoint de shell, sem eval, sem pickle** (JSON/msgpack de tipos
primitivos apenas) · arquivos estáticos por lista branca (sem path traversal) · documentação automática desligada.

In [ ]:
# @title 18 — REALTIME SERVER (FastAPI + WebSocket) · segurança · túnel HTTPS · embed no Colab { display-mode: "form" }
HAVE_WEB = HAVE_MSGPACK and all([ensure_package("fastapi", "fastapi>=0.110,<1"), ensure_package("uvicorn", "uvicorn>=0.29,<1"),
                                  ensure_package("websockets", "websockets>=10,<16")])
if HAVE_WEB:
    import uvicorn
    from fastapi import FastAPI, WebSocket, WebSocketDisconnect, Request, HTTPException
    from fastapi.responses import Response, JSONResponse, RedirectResponse, HTMLResponse


class RateLimiter:
    def __init__(self, per_minute: int):
        self.rate, self.cap = per_minute / 60.0, float(per_minute)
        self.b: Dict[str, Tuple[float, float]] = {}
        self.lock = threading.Lock()

    def allow(self, key: str) -> bool:
        now = time.time()
        with self.lock:
            tok, last = self.b.get(key, (self.cap, now))
            tok = min(self.cap, tok + (now - last) * self.rate)
            if tok < 1:
                self.b[key] = (tok, now)
                return False
            self.b[key] = (tok - 1, now)
            if len(self.b) > 10000:
                self.b = dict(list(self.b.items())[-5000:])
            return True


PROXY_HEADERS = ("cf-connecting-ip", "cf-ray", "x-forwarded-for", "x-forwarded-host", "ngrok-trace-id", "x-real-ip")
VENDOR_FILES = {"three.module.js": "https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js",
                "OrbitControls.js": "https://cdn.jsdelivr.net/npm/three@0.160.0/examples/jsm/controls/OrbitControls.js",
                "VRButton.js": "https://cdn.jsdelivr.net/npm/three@0.160.0/examples/jsm/webxr/VRButton.js"}
STATIC_FILES = {"app.js": "application/javascript", "style.css": "text/css", "msgpack.js": "application/javascript"}


class RealtimeServer:
    def __init__(self, desk: ObserverDesk, uni_ref: Callable[[], "UniverseEngine"], paths: CivPaths, port: int,
                 read_only: bool, logger: Optional[EventLogger], static_dir: str):
        self.desk, self.uni_ref, self.paths, self.port, self.read_only, self.logger = desk, uni_ref, paths, port, read_only, logger
        self.static_dir = static_dir
        self.view_token = secrets.token_urlsafe(18)
        self.admin_token = secrets.token_urlsafe(24)
        self.rl_api, self.rl_admin, self.rl_ws = RateLimiter(600), RateLimiter(12), RateLimiter(30)
        self.clients = 0
        self.thread: Optional[threading.Thread] = None
        self.server = None
        self.started = False
        self.error: Optional[str] = None
        self.app = self._make_app() if HAVE_WEB else None

    # ------------------------------------------------------------------ auth
    @staticmethod
    def _via_tunnel(headers) -> bool:
        return any(h in headers for h in PROXY_HEADERS)

    def _token_ok(self, tok: Optional[str], admin: bool = False) -> bool:
        if not tok or not isinstance(tok, str) or len(tok) > 64:
            return False
        ref = self.admin_token if admin else self.view_token
        return secrets.compare_digest(tok.encode(), ref.encode()) or (not admin and secrets.compare_digest(tok.encode(), self.admin_token.encode()))

    def _check(self, request: "Request", admin: bool = False):
        ip = (request.client.host if request.client else "?") + "|" + request.headers.get("cf-connecting-ip", "")
        if not (self.rl_admin if admin else self.rl_api).allow(ip):
            raise HTTPException(429, "rate limit")
        tok = request.headers.get("x-genesis-token")
        if admin:
            if self.read_only and self._via_tunnel(request.headers):
                raise HTTPException(403, "admin desabilitado no acesso remoto (REMOTE_READ_ONLY=True)")
            if not self._token_ok(request.headers.get("x-genesis-admin"), admin=True):
                raise HTTPException(401, "admin token inválido")
        elif not self._token_ok(tok):
            raise HTTPException(401, "token inválido")

    # ------------------------------------------------------------------ app
    def _make_app(self):
        app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
        srv = self

        imap = Genesis3DViewer.importmap_hash(srv.static_dir) if os.path.exists(os.path.join(srv.static_dir, "index.html")) else None
        script_src = "'self' https://cdn.jsdelivr.net https://unpkg.com" + (f" '{imap}'" if imap else "")

        @app.middleware("http")
        async def security_headers(request: Request, call_next):
            resp = await call_next(request)
            resp.headers["X-Content-Type-Options"] = "nosniff"
            resp.headers["Referrer-Policy"] = "no-referrer"
            resp.headers["Content-Security-Policy"] = (
                f"default-src 'self'; script-src {script_src}; style-src 'self' 'unsafe-inline'; "
                "img-src 'self' data: blob:; connect-src 'self' ws: wss:; worker-src 'self' blob:; "
                "frame-ancestors 'self' https://colab.research.google.com https://*.googleusercontent.com")
            return resp

        @app.get("/health")
        async def health():
            return {"ok": True, "version": GENESIS_VERSION, "proto": LIVE_PROTOCOL, "online": srv.desk.online,
                    "ts": iso_now()}

        @app.get("/")
        async def index():
            p = os.path.join(srv.static_dir, "index.html")
            html = open(p, encoding="utf-8").read().replace("__VERSION__", GENESIS_VERSION + "-" + ENGINE_VERSIONS_HASH[:8])
            return HTMLResponse(html, headers={"Cache-Control": "no-store"})

        @app.get("/static/{name}")
        async def static(name: str):
            if name not in STATIC_FILES:
                raise HTTPException(404)
            p = os.path.join(srv.static_dir, name)
            return Response(open(p, "rb").read(), media_type=STATIC_FILES[name], headers={"Cache-Control": "no-cache"})

        @app.get("/vendor/{name}")
        async def vendor(name: str):
            if name not in VENDOR_FILES:
                raise HTTPException(404)
            p = os.path.join(srv.static_dir, "vendor", name)
            if os.path.exists(p):
                return Response(open(p, "rb").read(), media_type="application/javascript",
                                headers={"Cache-Control": "public, max-age=86400"})
            return RedirectResponse(VENDOR_FILES[name])          # fallback: CDN fixado na mesma versão

        def ask(kind, args, timeout=4.0):
            r = srv.desk.ask(kind, args, timeout)
            if isinstance(r, dict) and r.get("error") in ("busy", "timeout"):
                raise HTTPException(503, r.get("note", r["error"]))
            return r

        def mp(obj):
            return Response(obj if isinstance(obj, bytes) else pack_msg(obj), media_type="application/msgpack")

        @app.get("/api/status")
        async def status(request: Request):
            srv._check(request)
            with srv.desk.lock:
                return JSONResponse(to_jsonable(srv.desk.status))

        @app.get("/api/world")
        async def world(request: Request):
            srv._check(request)
            return JSONResponse(to_jsonable(await asyncio.to_thread(ask, "world", {})))

        @app.get("/api/agent/{aid}")
        async def agent(aid: int, request: Request):
            srv._check(request)
            if not 0 < aid < 2 ** 53:
                raise HTTPException(422, "aid fora do intervalo")
            return JSONResponse(to_jsonable(await asyncio.to_thread(ask, "agent", {"aid": aid})))

        @app.get("/api/city/{sid}")
        async def city(sid: int, request: Request):
            srv._check(request)
            if not 0 < sid < 10 ** 9:
                raise HTTPException(422)
            r = await asyncio.to_thread(ask, "city", {"sid": sid})
            if r is None:
                raise HTTPException(404)
            return JSONResponse(to_jsonable(r))

        @app.get("/api/develop/{aid}")
        async def develop(aid: int, request: Request):
            srv._check(request)
            if not 0 < aid < 2 ** 53:
                raise HTTPException(422)
            r = await asyncio.to_thread(ask, "develop", {"aid": aid})
            if r is None:
                raise HTTPException(404)
            return mp(r)

        @app.get("/api/history")
        async def history(request: Request, since: int = 0, limit: int = 300):
            srv._check(request)
            if not (0 <= since < 2 ** 53 and 1 <= limit <= 2000):
                raise HTTPException(422)
            return JSONResponse(to_jsonable(await asyncio.to_thread(ask, "history", {"since": since, "limit": limit})))

        @app.get("/api/discoveries")
        async def discoveries(request: Request):
            srv._check(request)
            return JSONResponse(to_jsonable(await asyncio.to_thread(ask, "discoveries", {})))

        @app.get("/api/ladders")
        async def ladders(request: Request):
            srv._check(request)
            return JSONResponse(to_jsonable(await asyncio.to_thread(ask, "ladders", {})))

        @app.get("/api/replay/{did}")
        async def replay(did: str, request: Request):
            srv._check(request)
            if not re.fullmatch(r"D\d{5}-[A-Z0-9]{1,4}-[0-9a-f]{6}", did):
                raise HTTPException(422, "id inválido")
            d = await asyncio.to_thread(srv.uni_ref().archive.load_discovery, did)
            if d is None or "replay" not in d:
                raise HTTPException(404, "sem replay")
            rp = d["replay"]
            return mp({"t": "replay", "id": did, "level": d.get("level"), "name": d.get("level_name"), "size": rp["size"],
                       "ticks": rp["ticks"], "k": int(rp["pos"].shape[1]), "pos": rp["pos"].astype(np.float32).tobytes(),
                       "arena_of": rp["arena_of"].astype(np.int16).tobytes(), "layout": rp["layout"]})

        @app.get("/api/ring")
        async def ring_index(request: Request):
            srv._check(request)
            with srv.desk.lock:
                return JSONResponse(list(srv.desk.ring_index))

        @app.get("/api/ring/{i}")
        async def ring(i: int, request: Request):
            srv._check(request)
            if not 0 <= i < 100000:
                raise HTTPException(422)
            r = await asyncio.to_thread(ask, "ring", {"i": i})
            if r is None:
                raise HTTPException(404)
            return mp(r)

        @app.get("/api/search")
        async def search(request: Request, q: str = ""):
            srv._check(request)
            if not re.fullmatch(r"[A-Za-z0-9_\-]{1,64}", q):
                raise HTTPException(422)
            return JSONResponse(to_jsonable(await asyncio.to_thread(ask, "search", {"q": q})))

        @app.get("/api/lineage/{aid}")
        async def lineage(aid: int, request: Request):
            srv._check(request)
            if not 0 < aid < 2 ** 53:
                raise HTTPException(422)
            return JSONResponse(to_jsonable(await asyncio.to_thread(ask, "lineage", {"aid": aid})))

        def user_file(name):
            return os.path.join(srv.paths.viewer_user, f"{name}.json")

        for name in ("favorites", "notes"):
            def make(name=name):
                @app.get(f"/api/{name}")
                async def get_user(request: Request):
                    srv._check(request)
                    p = user_file(name)
                    return JSONResponse(json.load(open(p)) if os.path.exists(p) else [])

                @app.post(f"/api/{name}")
                async def post_user(request: Request):
                    srv._check(request)
                    body = await request.body()
                    if len(body) > 65536:
                        raise HTTPException(413)
                    try:
                        data = json.loads(body)
                    except Exception:
                        raise HTTPException(422, "JSON inválido")
                    if not isinstance(data, list) or len(data) > 500:
                        raise HTTPException(422, "lista de até 500 itens")
                    clean = []
                    for it in data:
                        if not isinstance(it, dict):
                            raise HTTPException(422)
                        c = {}
                        for k, v in it.items():
                            if not isinstance(k, str) or len(k) > 32 or k not in ("type", "id", "label", "text", "cam", "tick", "ts"):
                                raise HTTPException(422, "campo não permitido")
                            if isinstance(v, str):
                                c[k] = v[:2000]
                            elif isinstance(v, (int, float)) and not isinstance(v, bool) and math.isfinite(v):
                                c[k] = v
                            elif isinstance(v, list) and len(v) <= 9 and all(isinstance(x, (int, float)) and math.isfinite(x) for x in v):
                                c[k] = v
                            else:
                                raise HTTPException(422, "valor não permitido")
                        clean.append(c)
                    with open(user_file(name), "w") as f:
                        json.dump(clean, f)
                    return {"ok": True, "n": len(clean), "note": "observações do usuário nunca contam como evidência"}
            make()

        # ------------------------------------------------------------------ admin (segundo token, rate limit, local)
        ADMIN = {"pause", "resume", "checkpoint", "focus", "fork", "dsl", "unfocus"}

        @app.post("/api/admin/{action}")
        async def admin(action: str, request: Request):
            srv._check(request, admin=True)
            if action not in ADMIN:
                raise HTTPException(404)
            body = await request.body()
            if len(body) > 16384:
                raise HTTPException(413)
            try:
                data = json.loads(body) if body else {}
            except Exception:
                raise HTTPException(422, "JSON inválido")
            if not isinstance(data, dict):
                raise HTTPException(422)
            if action == "focus":
                aid = data.get("aid")
                if not isinstance(aid, int) or isinstance(aid, bool) or not 0 < aid < 2 ** 53:
                    raise HTTPException(422)
                data = {"aid": aid}
            elif action == "fork":
                iv, ticks = data.get("intervention"), data.get("ticks", 500)
                if not isinstance(iv, dict) or iv.get("kind") not in FORK_INTERVENTIONS or not isinstance(ticks, int):
                    raise HTTPException(422)
                if iv.get("kind") == "block_action" and iv.get("channel") not in ACTION_NAMES:
                    raise HTTPException(422)
                data = {"intervention": {k: v for k, v in iv.items() if k in ("kind", "channel", "value")
                                         and isinstance(v, (str, int, float)) and not isinstance(v, bool)},
                        "ticks": int(np.clip(ticks, 25, 5000))}
            elif action == "dsl":
                try:
                    data = {"program": validate_dsl(data.get("program"))}
                except ValueError as e:
                    raise HTTPException(422, f"DSL inválida: {e}")
            else:
                data = {}
            try:
                srv.desk.admin_queue.put_nowait({"action": action, "data": data, "ts": iso_now(),
                                                 "remote": srv._via_tunnel(request.headers)})
            except queue.Full:
                raise HTTPException(503, "fila admin cheia")
            if srv.logger:
                srv.logger.event("admin_request", action=action)
            return {"ok": True, "queued": action}

        # ------------------------------------------------------------------ WebSocket ao vivo
        @app.websocket("/ws/live")
        async def live(ws: WebSocket):
            ip = ws.client.host if ws.client else "?"
            tok = ws.query_params.get("t")
            if not srv.rl_ws.allow(ip) or not srv._token_ok(tok):
                await ws.close(code=4401)
                return
            await ws.accept()
            cid = uuid.uuid4().hex[:8]
            srv.clients += 1
            state = {"seq": 0, "base_seq": -1, "status_t": 0.0, "body_t": 0.0, "heat_t": 0.0, "lab_t": 0.0, "lab": False,
                     "alive": True, "max_hz": 10.0}

            async def receiver():
                while state["alive"]:
                    try:
                        msg = await ws.receive_text()
                    except Exception:
                        state["alive"] = False
                        return
                    if len(msg) > 2048:
                        continue
                    try:
                        m = json.loads(msg)
                    except Exception:
                        continue
                    if not isinstance(m, dict):
                        continue
                    t = m.get("t")
                    if t == "select":
                        a = m.get("aid")
                        if isinstance(a, int) and not isinstance(a, bool) and 0 < a < 2 ** 53:
                            srv.desk.selected[cid] = a
                        elif a is None:
                            srv.desk.selected.pop(cid, None)
                    elif t == "view":
                        try:
                            v = tuple(float(m[k]) for k in ("x0", "y0", "x1", "y1"))
                            n = int(np.clip(int(m.get("max_agents", 4000)), 50, 50000))
                            if all(math.isfinite(x) for x in v):
                                srv.desk.views[cid] = (*v, n)
                            state["max_hz"] = float(np.clip(float(m.get("hz", 10)), 0.5, 30))
                        except Exception:
                            pass
                    elif t == "layer":
                        lay = m.get("name")
                        if lay in HEAT_LAYERS:
                            srv.desk.layers[cid] = lay
                        else:
                            srv.desk.layers.pop(cid, None)
                    elif t == "lab":
                        state["lab"] = bool(m.get("on"))
                    elif t == "resync":
                        state["base_seq"] = -1
                    elif t == "ping":
                        ts = m.get("ts")
                        if isinstance(ts, (int, float)) and not isinstance(ts, bool):
                            try:
                                await ws.send_text(json.dumps({"t": "pong", "ts": ts, "server": time.time()}))
                            except Exception:
                                state["alive"] = False
            rt = asyncio.create_task(receiver())
            try:
                await ws.send_text(json.dumps({"t": "hello", "proto": LIVE_PROTOCOL, "version": GENESIS_VERSION,
                                               "read_only": srv.read_only, "server": time.time(),
                                               "claim": "universo artificial; afeto funcional ≠ experiência subjetiva"}))
                last_send = 0.0
                while state["alive"]:
                    now = time.time()
                    if srv.desk.base is not None and state["base_seq"] != srv.desk.base_seq:
                        await ws.send_bytes(srv.desk.base); state["base_seq"] = srv.desk.base_seq
                    lt = srv.desk.latest()
                    if lt is not None and lt[0] != state["seq"] and now - last_send >= 1.0 / state["max_hz"]:
                        # backpressure: sempre o frame MAIS RECENTE; frames antigos são descartados
                        await ws.send_bytes(lt[1]); state["seq"] = lt[0]; last_send = now
                    if now - state["status_t"] >= 1.0:
                        with srv.desk.lock:
                            st = dict(srv.desk.status)
                        st["clients"] = srv.clients
                        await ws.send_text(json.dumps(to_jsonable(st)))
                        state["status_t"] = now
                    a = srv.desk.selected.get(cid)
                    if a is not None and now - state["body_t"] >= 0.25 and a in srv.desk.body:
                        await ws.send_bytes(srv.desk.body[a]); state["body_t"] = now
                    lay = srv.desk.layers.get(cid)
                    if lay and now - state["heat_t"] >= 2.0 and lay in srv.desk.heat:
                        await ws.send_bytes(srv.desk.heat[lay]); state["heat_t"] = now
                    if state["lab"] and srv.desk.lab is not None and now - state["lab_t"] >= 0.5:
                        await ws.send_bytes(srv.desk.lab); state["lab_t"] = now
                    await asyncio.sleep(0.03)
            except (WebSocketDisconnect, RuntimeError):
                pass
            except Exception as e:
                if srv.logger:
                    srv.logger.event("ws_error", error=repr(e))
            finally:
                state["alive"] = False
                rt.cancel()
                srv.clients -= 1
                for d_ in (srv.desk.selected, srv.desk.views, srv.desk.layers):
                    d_.pop(cid, None)

        return app

    # ------------------------------------------------------------------ ciclo de vida
    def start(self) -> bool:
        if not HAVE_WEB:
            self.error = "fastapi/uvicorn/msgpack indisponíveis"
            return False
        cfg = uvicorn.Config(self.app, host="127.0.0.1", port=self.port, log_level="warning", access_log=False,
                             ws_max_size=1 << 20, lifespan="off")
        self.server = uvicorn.Server(cfg)

        def run():
            try:
                self.server.run()
            except Exception as e:
                self.error = repr(e)
        self.thread = threading.Thread(target=run, name="genesis-server", daemon=True)
        self.thread.start()
        for _ in range(100):
            if getattr(self.server, "started", False):
                self.started = True
                return True
            if self.error:
                return False
            time.sleep(0.1)
        self.error = self.error or "timeout ao iniciar"
        return False

    def stop(self):
        if self.server is not None:
            self.server.should_exit = True
        if self.thread is not None:
            self.thread.join(timeout=5)


class TunnelManager:
    """Adaptador de túnel HTTPS: Cloudflare quick tunnel (sem conta) ou ngrok (se NGROK_AUTHTOKEN estiver em variável de
    ambiente ou em Colab Secret). Segredos nunca são gravados no notebook nem passados por linha de comando."""
    CLOUDFLARED = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    NGROK = "https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz"

    def __init__(self, paths: CivPaths, logger: Optional[EventLogger]):
        self.paths, self.logger = paths, logger
        self.proc: Optional[subprocess.Popen] = None
        self.url: Optional[str] = None
        self.provider: Optional[str] = None
        self.error: Optional[str] = None

    @staticmethod
    def _secret(name: str) -> Optional[str]:
        v = os.environ.get(name)
        if v:
            return v
        if IN_COLAB:
            try:
                from google.colab import userdata
                return userdata.get(name)
            except Exception:
                return None
        return None

    def _download(self, url: str, dest: str) -> bool:
        if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
            return True
        try:
            import urllib.request
            tmp = dest + ".part"
            urllib.request.urlretrieve(url, tmp)
            os.replace(tmp, dest)
            return True
        except Exception as e:
            self.error = f"download falhou: {e}"
            return False

    def _read_url(self, pattern: str, timeout: float) -> Optional[str]:
        t0 = time.time()
        rx = re.compile(pattern)
        while time.time() - t0 < timeout and self.proc and self.proc.poll() is None:
            line = self.proc.stdout.readline()
            if not line:
                time.sleep(0.1); continue
            m = rx.search(line)
            if m:
                return m.group(0)
        return None

    def _drain(self):
        def run():
            try:
                for _ in iter(self.proc.stdout.readline, ""):
                    pass
            except Exception:
                pass
        threading.Thread(target=run, daemon=True, name="tunnel-drain").start()

    def start(self, port: int, timeout: float = 45.0) -> Optional[str]:
        bindir = os.path.join(self.paths.work, "bin")
        os.makedirs(bindir, exist_ok=True)
        cf = os.path.join(bindir, "cloudflared")
        if self._download(self.CLOUDFLARED, cf):
            os.chmod(cf, 0o755)
            try:
                self.proc = subprocess.Popen([cf, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
                                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
                url = self._read_url(r"https://[a-z0-9\-]+\.trycloudflare\.com", timeout)
                if url:
                    self.url, self.provider = url, "cloudflare"
                    self._drain()
                    return url
                self.stop()
                self.error = "cloudflared não publicou URL"
            except Exception as e:
                self.error = f"cloudflared: {e}"
        tok = self._secret("NGROK_AUTHTOKEN")
        if tok:
            tgz = os.path.join(bindir, "ngrok.tgz")
            ng = os.path.join(bindir, "ngrok")
            if self._download(self.NGROK, tgz):
                try:
                    import tarfile
                    with tarfile.open(tgz) as t:
                        member = t.getmember("ngrok")
                        member.name = "ngrok"
                        t.extract(member, bindir, filter="data") if hasattr(tarfile, "data_filter") else t.extract(member, bindir)
                    os.chmod(ng, 0o755)
                    env = dict(os.environ, NGROK_AUTHTOKEN=tok)
                    self.proc = subprocess.Popen([ng, "http", str(port), "--log", "stdout", "--log-format", "json"],
                                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
                    url = self._read_url(r"https://[a-zA-Z0-9\-\.]+\.ngrok[a-z\-\.]*\.(app|io|dev)", timeout)
                    if url:
                        self.url, self.provider = url, "ngrok"
                        self._drain()
                        return url
                    self.stop()
                    self.error = (self.error or "") + "; ngrok não publicou URL"
                except Exception as e:
                    self.error = f"ngrok: {e}"
        if self.logger:
            self.logger.event("tunnel_unavailable", reason=self.error)
        return None

    def stop(self):
        if self.proc is not None:
            try:
                self.proc.terminate(); self.proc.wait(timeout=5)
            except Exception:
                try:
                    self.proc.kill()
                except Exception:
                    pass
            self.proc = None

## 19 — GENESIS 3D VIEWER: instalação do observatório (Three.js no cliente)
O notebook grava os arquivos do viewer (`index.html`, `app.js`, `style.css`, `msgpack.js`) no SSD de trabalho e
baixa Three.js **fixado em 0.160.0** do registro npm (tarball verificado por SHA-512 do próprio registro). Se o
download falhar, o servidor redireciona para o CDN na mesma versão. Renderização pesada acontece no navegador do
usuário (a GPU do Colab fica para a ciência).

In [ ]:
# @title 19 — GENESIS 3D VIEWER: instalação do observatório (Three.js no cliente) { display-mode: "form" }
VIEWER_FILES_B64 = "..."
THREE_VERSION = "0.160.0"


class Genesis3DViewer:
    TGZ = f"https://registry.npmjs.org/three/-/three-{THREE_VERSION}.tgz"
    META = f"https://registry.npmjs.org/three/{THREE_VERSION}"
    WANT = {"package/build/three.module.js": "three.module.js",
            "package/examples/jsm/controls/OrbitControls.js": "OrbitControls.js",
            "package/examples/jsm/webxr/VRButton.js": "VRButton.js"}

    def __init__(self, paths: CivPaths, logger: Optional[EventLogger]):
        self.paths, self.logger = paths, logger
        self.static_dir = os.path.join(paths.work, "viewer", "static")
        self.vendor_ok = False

    def install(self) -> str:
        if VIEWER_FILES_B64.startswith("@@"):
            raise RuntimeError("arquivos do viewer não embutidos (build incompleto)")
        files = json.loads(base64.b64decode(VIEWER_FILES_B64).decode("utf-8"))
        os.makedirs(os.path.join(self.static_dir, "vendor"), exist_ok=True)
        for name, content in files.items():
            if not re.fullmatch(r"[a-z]+\.(html|js|css)", name):
                continue
            with open(os.path.join(self.static_dir, name), "w", encoding="utf-8") as f:
                f.write(content)
        self.vendor_ok = self._vendor()
        return self.static_dir

    def _vendor(self) -> bool:
        vd = os.path.join(self.static_dir, "vendor")
        if all(os.path.exists(os.path.join(vd, v)) for v in self.WANT.values()):
            return True
        try:
            import urllib.request, tarfile
            meta = json.loads(urllib.request.urlopen(self.META, timeout=20).read())
            integrity = meta.get("dist", {}).get("integrity", "")
            dl = os.path.join(self.paths.work, "cache", f"three-{THREE_VERSION}.tgz")
            if not os.path.exists(dl):
                urllib.request.urlretrieve(self.TGZ, dl + ".part")
                os.replace(dl + ".part", dl)
            if integrity.startswith("sha512-"):
                got = base64.b64encode(hashlib.sha512(open(dl, "rb").read()).digest()).decode()
                if got != integrity.split("-", 1)[1]:
                    os.remove(dl)
                    raise RuntimeError("integridade do tarball three.js não confere")
            with tarfile.open(dl) as t:
                for m in t.getmembers():
                    if m.name in self.WANT and m.isfile():
                        data = t.extractfile(m).read()
                        with open(os.path.join(vd, self.WANT[m.name]), "wb") as f:
                            f.write(data)
            return all(os.path.exists(os.path.join(vd, v)) for v in self.WANT.values())
        except Exception as e:
            if self.logger:
                self.logger.event("viewer_vendor_fallback_cdn", error=repr(e))
            return False

    @staticmethod
    def importmap_hash(static_dir: str) -> Optional[str]:
        html = open(os.path.join(static_dir, "index.html"), encoding="utf-8").read()
        m = re.search(r'<script type="importmap">(.*?)</script>', html, flags=re.S)
        if not m:
            return None
        return "sha256-" + base64.b64encode(hashlib.sha256(m.group(1).encode("utf-8")).digest()).decode()


def colab_proxy_url(port: int) -> Optional[str]:
    if not IN_COLAB:
        return None
    try:
        from google.colab.output import eval_js
        return eval_js(f"google.colab.kernel.proxyPort({port})")
    except Exception:
        return None


def show_world_links(port: int, view_token: str, admin_token: str, remote_url: Optional[str], read_only: bool,
                     embed: bool = True, height: int = 760):
    """Botão OPEN GENESIS WORLD + link remoto + embed local. Tokens só no fragmento (#) da URL; o admin nunca vai no
    link remoto. Tokens valem só nesta sessão."""
    local = colab_proxy_url(port)
    local_link = (local.rstrip("/") + f"/#t={view_token}&a={admin_token}") if local else f"http://127.0.0.1:{port}/#t={view_token}&a={admin_token}"
    remote_link = (remote_url.rstrip("/") + f"/#t={view_token}") if remote_url else None
    btn = ("display:inline-block;padding:12px 22px;border-radius:24px;background:linear-gradient(90deg,#2a5cff,#3fd0c9);"
           "color:white;font-weight:700;letter-spacing:.08em;text-decoration:none;font-family:system-ui;margin:4px 8px 4px 0")
    html_ = f'<div style="font-family:system-ui;color:#cfe0ff;background:#070b14;padding:14px 18px;border-radius:14px">'
    html_ += f'<a href="{_html.escape(local_link)}" target="_blank" style="{btn}">🌍 OPEN GENESIS WORLD</a>'
    if remote_link:
        html_ += (f'<a href="{_html.escape(remote_link)}" target="_blank" style="{btn};background:linear-gradient(90deg,#7b4dff,#c04dff)">'
                  f'📱 LINK REMOTO (HTTPS, {"somente leitura" if read_only else "leitura"})</a>')
        html_ += (f'<div style="margin-top:8px;font-size:12px;color:#8fa3c7">Remoto: <code>{_html.escape(remote_link)}</code> — abre em qualquer '
                  f'computador/celular enquanto este Colab estiver rodando. Token por sessão; admin desabilitado no remoto.</div>')
    else:
        html_ += '<div style="margin-top:8px;font-size:12px;color:#ffb65c">Túnel remoto indisponível agora: viewer local funciona normalmente.</div>'
    html_ += '</div>'
    display(HTML(html_))
    if embed and IN_COLAB:
        try:
            from google.colab import output
            output.serve_kernel_port_as_iframe(port, path=f"/#t={view_token}&a={admin_token}", height=height)
        except Exception as e:
            print(f"[viewer] embed indisponível ({e}); use o botão acima.")
    return {"local": local_link, "remote": remote_link}

## 20 — TEST SUITE (boot): unidade, integração, reprodutibilidade, checkpoint, protocolo, controles + e −
Todos os testes usam o código real (nada de mocks de ciência). Payloads sintéticos só aparecem nos testes de
protocolo/pipeline, explicitamente marcados como TEST MODE, e nunca entram na pesquisa.

In [ ]:
# @title 20 — TEST SUITE (boot): unidade, integração, reprodutibilidade, checkpoint, protocolo, controles + e − { display-mode: "form" }
class BootTests:
    def __init__(self, cfg: CivConfig, paths: CivPaths, device: torch.device, logger: EventLogger):
        self.cfg = copy.deepcopy(cfg)
        self.cfg.world_w = self.cfg.world_h = 48
        self.cfg.initial_agents, self.cfg.max_agents, self.cfg.max_bodies = 60, 200, 6
        self.cfg.probe.trials, self.cfg.probe.permutations = 8, 30
        self.paths, self.device, self.logger = paths, device, logger
        self.results: List[Dict[str, Any]] = []

    def _mk(self):
        L = GenomeLayout(self.cfg).to(self.device)
        we = WorldEngine(self.cfg, self.device); matter = MatterEngine(self.cfg, self.device)
        dev = DevelopmentEngine(self.cfg, L, self.device); body = BodyEngine(self.cfg, dev, self.device)
        mind = SignalEngine(self.cfg, L, self.device)
        return L, we, matter, dev, body, mind

    def run(self) -> List[Dict[str, Any]]:
        tests = [m for m in dir(self) if m.startswith("test_")]
        for name in sorted(tests):
            t0 = time.time()
            try:
                info = getattr(self, name)() or {}
                self.results.append({"test": name, "ok": True, "seconds": round(time.time() - t0, 2), **to_jsonable(info)})
            except Exception as e:
                self.results.append({"test": name, "ok": False, "seconds": round(time.time() - t0, 2), "error": repr(e),
                                     "trace": traceback.format_exc()[-1200:]})
            live_progress("testes", len(self.results), len(tests))
        return self.results

    # ------------------------------------------------------------------ mundo e matéria
    def test_world_deterministic(self):
        L, we, *_ = self._mk()
        a, b = we.generate(11), we.generate(11)
        assert torch.equal(a.height, b.height) and torch.equal(a.is_water, b.is_water), "mundo não determinístico"
        assert not torch.equal(a.height, we.generate(12).height), "sementes diferentes geraram o mesmo mundo"
        return {"water_fraction": float(a.is_water.float().mean())}

    def test_material_conservation(self):
        L, we, matter, dev, body, mind = self._mk()
        ws = we.generate(3); eng = AgentEngine(self.cfg, L, we, matter, mind, self.device)
        pool = AgentPool(self.cfg.max_agents, L, self.cfg, self.device); ids = IdCounter(1)
        ledger = StructureLedger(self.cfg.max_blocks, self.device)
        g = torch.Generator(device=self.device); g.manual_seed(5)
        eng.spawn_founders(ws, pool, 60, g, ids, 0, None)
        m0 = matter.total_material(ws, pool.carried[pool.alive])
        for t in range(150):
            eng.step(ws, pool, t, g, None, ids, ledger)
            if t % 4 == 0:
                ws.tick = t; iy, ix = WorldEngine.tile_index(ws, pool.pos[pool.alive]); we.env_step(ws, g, torch.stack([iy, ix]))
        m1 = matter.total_material(ws, pool.carried[pool.alive])
        assert abs(m1 - m0) < 1e-3, f"matéria não conservada: {m0} → {m1}"
        return {"material": m0, "placed": int(ledger.total_placed)}

    # ------------------------------------------------------------------ desenvolvimento e física
    def test_development_deterministic_and_batch_independent(self):
        L, we, matter, dev, body, mind = self._mk()
        g = torch.Generator(device=self.device); g.manual_seed(7)
        G = L.random(5, g); sd = torch.arange(5, device=self.device) + 100
        full = dev.develop(G, sd).occ
        single = torch.stack([dev.develop(G[i:i + 1], sd[i:i + 1]).occ[0] for i in range(5)])
        assert torch.equal(full, single), "desenvolvimento depende da composição do lote"
        assert torch.equal(full, dev.develop(G, sd).occ), "desenvolvimento não determinístico"
        return {"nodes": full.sum(1).tolist()}

    def test_development_control_genomes(self):
        """Controles conhecidos: crescimento forte ocupa o orçamento máximo; limiar inatingível mantém só a semente."""
        L, we, matter, dev, body, mind = self._mk()
        g = torch.Generator(device=self.device); g.manual_seed(1)
        G = L.random(2, g)
        for name, val in [("dev_gw", 0.0), ("dev_noise", 0.0), ("dev_maxfrac", 0.5)]:
            a, b = L.slices[name]; G[:, a:b] = val
        a, b = L.slices["dev_gthr"]; G[0, a:b] = -1.0; G[1, a:b] = 1.0
        bp = dev.develop(L.clamp(G), torch.tensor([1, 1], device=self.device))
        n = bp.occ.sum(1).tolist()
        assert n[0] == math.ceil(0.5 * dev.N) or n[0] >= 0.45 * dev.N, f"controle positivo não cresceu: {n[0]}"
        assert n[1] == 1, f"controle negativo cresceu: {n[1]}"
        return {"positive_nodes": n[0], "negative_nodes": n[1]}

    def test_physics_stability(self):
        L, we, matter, dev, body, mind = self._mk()
        g = torch.Generator(device=self.device); g.manual_seed(3)
        G = passive_body(L, L.random(6, g))
        bb = body.make(dev.develop(G, torch.arange(6, device=self.device)))
        z0 = bb.com()[:, 2].clone()
        for _ in range(60):
            r = body.step(bb, torch.zeros(6, 2, device=self.device))
        assert bool(r["finite"].all()), "física divergiu"
        assert float(bb.vel.abs().max()) < 5.0, "corpo passivo não estabilizou"
        assert bool((bb.com()[:, 2] <= z0 + 0.05).all()), "gravidade/contato inconsistentes"
        return {"max_speed_after_settle": float(bb.vel.norm(dim=-1).max())}

    def test_spatial_hash_matches_bruteforce(self):
        g = torch.Generator(device=self.device); g.manual_seed(2)
        pos = torch.rand(300, 2, generator=g, device=self.device) * 40
        alive = torch.ones(300, dtype=torch.bool, device=self.device)
        nn, nd, _ = spatial_neighbors(pos, alive, 4.0, M=64)
        D = torch.cdist(pos, pos); D.fill_diagonal_(1e9)
        bd, bi = D.min(1)
        ok = (bd > 4.0) | (bi == nn) | ((nd - bd).abs() < 1e-5)
        assert bool(ok.all()), "hash espacial divergiu da força bruta"
        return {"pairs_checked": 300}

    # ------------------------------------------------------------------ controles positivos / negativos dos probes
    def test_memory_controls(self):
        """Controle positivo (unidade auto-recorrente lenta ligada à pista) memoriza; controle sem memória não."""
        L, we, matter, dev, body, mind = self._mk()
        runner = ArenaRunner(self.cfg, L, we, matter, mind, dev, body, self.device)
        ve = ValidationEngine(self.cfg, L, runner, AffectEngine(self.cfg, L, runner, EthicalSentinel(self.cfg), self.device), self.device)
        g = torch.Generator(device=self.device); g.manual_seed(4)
        G = L.random(1, g)
        N = self.cfg.n_units
        for name in ("ctl_Win", "ctl_W0", "ctl_b", "plast_eta", "babble_sigma"):
            a, b = L.slices[name]; G[:, a:b] = 0.0
        Win = L.view(G, "ctl_Win"); Win[0, 0, 12] = 2.0; Win[0, 0, 13] = -2.0
        W0 = L.view(G, "ctl_W0"); W0[0, 0, 0] = 1.6
        a, b = L.slices["ctl_logtau"]; G[:, a:b] = 2.3
        n = 32
        delays = [1, 8]
        pos_r = drive(ve._memory_decode(G.expand(n, -1).contiguous(), 99, delays, n, "test memory +"))
        neg_r = drive(ve._memory_decode(memoryless(L, G.expand(n, -1).contiguous()), 99, delays, n, "test memory −"))
        assert pos_r[8]["linear_p"] < 0.01 and pos_r[8]["linear_acc"] > 0.8, f"controle positivo falhou: {pos_r[8]['linear_acc']}"
        assert neg_r[8]["linear_acc"] < 0.75, f"controle negativo memorizou: {neg_r[8]['linear_acc']}"
        return {"positive_acc": pos_r[8]["linear_acc"], "negative_acc": neg_r[8]["linear_acc"]}

    def test_communication_mi_controls(self):
        rng = np.random.default_rng(0)
        state = rng.integers(0, 3, 600)
        sig = np.where(rng.random(600) < 0.85, state, rng.integers(0, 3, 600))          # TEST MODE: sinal sintético
        mi, p = mi_permutation(sig, state, 40, rng)
        mi0, p0 = mi_permutation(rng.permutation(sig), state, 40, rng)
        assert p < 0.05 and mi > 0.3, "controle positivo de comunicação não detectado"
        assert p0 > 0.05, "controle negativo de comunicação deu falso positivo"
        return {"mi_positive": mi, "p_negative": p0}

    def test_affect_pipeline_controls(self):
        """Controle sintético do pipeline de decodificação de valência (não entra na evolução)."""
        rng = np.random.default_rng(1)
        y = np.repeat([0, 1], 40)
        X = rng.normal(size=(80, 16)); X[:, 0] += (y * 2 - 1) * 1.5                          # TEST MODE
        r = decode_report(X, y, 2, 30, 4, 0, nonlinear=False)
        r0 = decode_report(rng.normal(size=(80, 16)), y, 2, 30, 4, 0, nonlinear=False)
        assert r["linear_acc"] > 0.8 and r["linear_p"] < 0.01, "pipeline não detectou sinal sintético"
        assert r0["linear_p"] > 0.01, "pipeline deu falso positivo"
        return {"acc_positive": r["linear_acc"], "acc_negative": r0["linear_acc"]}

    def test_dsl_security(self):
        bad = [{"name": "x", "target": {"kind": "candidate", "id": "C1"}, "steps": [{"op": "exec", "code": "import os"}]},
               {"name": "x", "target": {"kind": "candidate", "id": "C1"}, "steps": [{"op": "observe", "ticks": 10 ** 9}]},
               {"name": "__import__('os')", "target": {"kind": "candidate", "id": "C1"}, "steps": [{"op": "observe", "ticks": 5}]},
               {"name": "x", "target": {"kind": "host"}, "steps": [{"op": "observe", "ticks": 5}]},
               {"name": "x", "target": {"kind": "candidate", "id": "C1"}, "steps": [{"op": "ablate", "what": "action", "channel": "rm -rf"}]}]
        for b in bad:
            try:
                validate_dsl(b)
                raise AssertionError(f"DSL aceitou programa perigoso: {b}")
            except ValueError:
                pass
        ok = validate_dsl({"name": "ok", "target": {"kind": "candidate", "id": "C1"},
                           "steps": [{"op": "ablate", "what": "action", "channel": "place_material"}, {"op": "observe", "ticks": 50},
                                     {"op": "measure", "metric": "intake"}]})
        return {"accepted_steps": len(ok["steps"]), "rejected": len(bad)}

    # ------------------------------------------------------------------ integração: universo, checkpoint, observação
    def _small_universe(self, root: str):
        p = CivPaths(root, os.path.join(root, "_work"), on_drive=False)
        cfg = copy.deepcopy(self.cfg)
        cfg.n_islands, cfg.island_pop, cfg.qd_batch = 1, 6, 6
        u = UniverseEngine(cfg, p, self.device, None)
        u.create_world(bootstrap_seconds=0.5)
        return u, p

    def test_checkpoint_roundtrip_and_corruption_fallback(self):
        root = tempfile.mkdtemp(prefix="gtest_ck_", dir=self.paths.tmp(""))
        u, p = self._small_universe(root)
        ck = CheckpointEngine(p, None)
        for _ in range(20):
            u.tick()
        ck.save(u, sync=True)
        t_first = u.tick_count
        for _ in range(20):
            u.tick()
        ck.save(u, sync=True)
        st, info = ck.load_latest()
        assert st is not None and st["meta"]["tick"] == u.tick_count, "roundtrip falhou"
        u2 = UniverseEngine(u.cfg, p, self.device, None); u2.load_state_dict(st)
        assert u2.pool.n_alive() == u.pool.n_alive() and torch.allclose(u2.pool.pos[:u.pool.n_alive()].cpu(), u.pool.pos[u.pool.alive].cpu())
        shard = next(f for f in glob.glob(os.path.join(p.root, "checkpoints", "current", "*.npz")))
        with open(shard, "r+b") as f:
            f.seek(64); f.write(b"\x00CORRUPTED\x00")
        st2, info2 = ck.load_latest()
        assert st2 is not None and st2["meta"]["tick"] == t_first and info2.get("fallback"), "não recorreu ao previous"
        shutil.rmtree(root, ignore_errors=True)
        return {"fallback_to_tick": t_first, "tried": [t["dir"] for t in info2["tried"]]}

    def test_observation_independence(self):
        """O observador (telemetria + gêmeo visual + camadas) não altera o estado nem o RNG da simulação."""
        root = tempfile.mkdtemp(prefix="gtest_obs_", dir=self.paths.tmp(""))
        u, p = self._small_universe(root)
        st = u.state_dict()
        ua = UniverseEngine(u.cfg, p, self.device, None); ua.load_state_dict(copy.deepcopy(st))
        ub = UniverseEngine(u.cfg, p, self.device, None); ub.load_state_dict(copy.deepcopy(st))
        desk = ObserverDesk(); tel = TelemetryEngine(ub, desk)
        desk.layers["t"] = "population"
        aid = int(ub.pool.aid[ub.pool.alive][0]); desk.selected["t"] = aid
        for _ in range(30):
            ua.tick(); ub.tick(); tel.pump(force=True)
        assert torch.equal(ua.pool.pos, ub.pool.pos) and torch.equal(ua.gen.get_state(), ub.gen.get_state()), \
            "observação alterou a simulação"
        shutil.rmtree(root, ignore_errors=True)
        return {"ticks": 30, "frames": tel.frames_sent}

    def test_replay_determinism(self):
        root = tempfile.mkdtemp(prefix="gtest_det_", dir=self.paths.tmp(""))
        u, p = self._small_universe(root)
        st = u.state_dict()
        hashes = []
        for _ in range(2):
            v = UniverseEngine(u.cfg, p, self.device, None); v.load_state_dict(copy.deepcopy(st))
            for _ in range(25):
                v.tick()
            hashes.append(hash_obj(v.pool.pos[v.pool.alive].cpu().numpy()))
        shutil.rmtree(root, ignore_errors=True)
        same = hashes[0] == hashes[1]
        if self.device.type == "cuda" and not same:
            return {"deterministic": False, "note": "kernels CUDA não determinísticos (index_add atômico) registrados"}
        assert same, "replay a partir do mesmo checkpoint divergiu"
        return {"deterministic": True}

    def test_multiscale_aggregation_bookkeeping(self):
        root = tempfile.mkdtemp(prefix="gtest_f0_", dir=self.paths.tmp(""))
        u, p = self._small_universe(root)
        n0 = u.pool.n_alive()
        idx = torch.nonzero(u.pool.alive).flatten()[:10]
        regs = u.f0.region_of(u.pool.pos[idx].cpu().numpy())
        u.f0.absorb(regs, u.pool.genome[idx].cpu().numpy(), u.pool.energy[idx].cpu().numpy(), u.pool.signature[idx].cpu().numpy(), u.inst_rng)
        u.agents.kill(u.pool, idx, u.bodies, u.ws)
        assert abs(u.f0.total() - 10) < 1e-6 and u.pool.n_alive() == n0 - 10, "agregação perdeu indivíduos"
        made = u._disaggregate(10)
        assert made == 10 and abs(u.f0.total()) < 1e-6 and u.pool.n_alive() == n0, "desagregação inconsistente"
        shutil.rmtree(root, ignore_errors=True)
        return {"aggregated": 10, "disaggregated": made}

    def test_oom_handler(self):
        e = RuntimeError("CUDA out of memory. Tried to allocate 2.00 GiB")
        assert is_oom_error(e)
        cfg = copy.deepcopy(self.cfg)
        before = (cfg.qd_batch, cfg.max_stream_agents, cfg.probe.trials)
        oom_backoff(cfg)
        after = (cfg.qd_batch, cfg.max_stream_agents, cfg.probe.trials)
        assert all(a <= b for a, b in zip(after, before)) and (after != before or before[0] <= 4), "backoff não reduziu carga"
        return {"before": before, "after": after}

    def test_server_protocol(self):
        if not HAVE_WEB:
            return {"skipped": "fastapi/uvicorn ausentes"}
        import urllib.request, urllib.error
        static = os.path.join(self.paths.work, "viewer", "static")
        if not os.path.exists(os.path.join(static, "index.html")):
            Genesis3DViewer(self.paths, None).install()
        desk = ObserverDesk()
        desk.base = pack_msg({"t": "base", "test_mode": True}); desk.base_seq = 1             # TEST MODE (payload marcado)
        desk.publish_frame(pack_msg({"t": "frame", "test_mode": True, "n": 0}))
        with socket.socket() as s_:
            s_.bind(("127.0.0.1", 0)); port = s_.getsockname()[1]
        srv = RealtimeServer(desk, lambda: None, self.paths, port, True, None, static)
        assert srv.start(), f"servidor não iniciou: {srv.error}"
        base = f"http://127.0.0.1:{port}"

        def code(path, headers=None, data=None):
            req = urllib.request.Request(base + path, headers=headers or {}, data=data, method="POST" if data is not None else "GET")
            try:
                return urllib.request.urlopen(req, timeout=5).status
            except urllib.error.HTTPError as e:
                return e.code
        try:
            assert code("/health") == 200
            assert code("/api/status") == 401, "API sem token deveria recusar"
            assert code("/api/status", {"X-Genesis-Token": srv.view_token}) == 200
            assert code("/api/admin/pause", {"X-Genesis-Token": srv.view_token}, b"{}") == 401, "admin com token de visualização"
            assert code("/api/admin/pause", {"X-Genesis-Admin": srv.admin_token, "cf-connecting-ip": "1.2.3.4"}, b"{}") == 403, \
                "admin pelo túnel deveria ser recusado em REMOTE_READ_ONLY"
            assert code("/api/admin/pause", {"X-Genesis-Admin": srv.admin_token}, b"{}") == 200
            assert code("/static/../../etc/passwd") in (404, 422), "path traversal"
            assert code("/api/agent/-5", {"X-Genesis-Token": srv.view_token}) == 422
            got = asyncio.run(self._ws_probe(port, srv.view_token))
            assert got["hello"] and got["binary"] >= 1, f"WebSocket não entregou hello/base: {got}"
            bad = asyncio.run(self._ws_probe(port, "wrong-token"))
            assert not bad["hello"], "WebSocket aceitou token errado"
        finally:
            srv.stop()
        return {"port": port, "ws": got}

    @staticmethod
    async def _ws_probe(port: int, tok: str) -> Dict[str, Any]:
        out = {"hello": False, "binary": 0}
        try:
            import websockets
        except ImportError:
            ensure_package("websockets", "websockets>=10,<16")
            import websockets
        try:
            async with websockets.connect(f"ws://127.0.0.1:{port}/ws/live?t={tok}", open_timeout=5) as w:
                for _ in range(4):
                    m = await asyncio.wait_for(w.recv(), timeout=3)
                    if isinstance(m, str):
                        out["hello"] = out["hello"] or json.loads(m).get("t") == "hello"
                    else:
                        msgpack.unpackb(m, raw=False); out["binary"] += 1
        except Exception as e:
            out["error"] = repr(e)
        return out


def oom_backoff(cfg: CivConfig) -> CivConfig:
    """Recuperação de OOM: reduz lotes e streaming (o viewer perde LOD antes da ciência perder rigor)."""
    cfg.qd_batch = max(4, cfg.qd_batch // 2)
    cfg.max_stream_agents = max(300, cfg.max_stream_agents // 2)
    cfg.probe.trials = max(6, cfg.probe.trials - 2)
    return cfg


def run_boot_tests(cfg, paths, device, logger) -> Tuple[bool, List[Dict[str, Any]]]:
    bt = BootTests(cfg, paths, device, logger)
    res = bt.run()
    ok = all(r["ok"] for r in res)
    rows = "".join(f"<tr><td>{'✅' if r['ok'] else '❌'}</td><td>{r['test'][5:]}</td><td>{r['seconds']}s</td>"
                   f"<td style='color:#e88'>{_html.escape(r.get('error', '')[:160])}</td></tr>" for r in res)
    display(HTML(f"<details {'open' if not ok else ''}><summary style='font-family:system-ui'>Testes de boot: "
                 f"{sum(r['ok'] for r in res)}/{len(res)} OK</summary><table style='font-family:system-ui;font-size:12px'>{rows}</table></details>"))
    with open(os.path.join(paths.reports, "boot_tests.json"), "w") as f:
        json.dump(to_jsonable(res), f, indent=1)
    logger.event("boot_tests", ok=ok, passed=sum(r["ok"] for r in res), total=len(res))
    return ok, res

## 21 — AUTOTUNE: benchmark real do hardware (física, desenvolvimento, agentes, streaming) e carga segura
Nunca assume A100: sobe a carga progressivamente medindo tempo por tick e pico de VRAM; mantém margem (≤ 70% da
VRAM total) e um orçamento de tempo por tick. Resultado vai para `reports/autotune.json` e para o checkpoint.

In [ ]:
# @title 21 — AUTOTUNE: benchmark real do hardware (física, desenvolvimento, agentes, streaming) e carga segura { display-mode: "form" }
def benchmark_and_tune(cfg: CivConfig, device: torch.device, logger: Optional[EventLogger], enabled: bool = True) -> Tuple[CivConfig, Dict[str, Any]]:
    rep: Dict[str, Any] = {"device": device.type, "profile": cfg.profile, "ts": iso_now()}
    L = GenomeLayout(cfg).to(device)
    we = WorldEngine(cfg, device); matter = MatterEngine(cfg, device)
    dev = DevelopmentEngine(cfg, L, device); body = BodyEngine(cfg, dev, device); mind = SignalEngine(cfg, L, device)
    g = torch.Generator(device=device); g.manual_seed(1)

    def timed(fn, reps=3):
        sync_device(); t0 = time.perf_counter()
        for _ in range(reps):
            fn()
        sync_device()
        return (time.perf_counter() - t0) / reps * 1000
    # física e desenvolvimento
    G = L.random(min(64, cfg.max_bodies), g)
    rep["development_ms_per_64"] = round(timed(lambda: dev.develop(G, torch.arange(G.shape[0], device=device)), 2), 2)
    bb = body.make(dev.develop(G, torch.arange(G.shape[0], device=device)))
    rep["physics_ms_per_step_64_bodies"] = round(timed(lambda: body.step(bb, torch.zeros(G.shape[0], 2, device=device)), 3), 3)
    ws = we.generate(5)
    eng = AgentEngine(cfg, L, we, matter, mind, device)
    ledger = StructureLedger(min(cfg.max_blocks, 100000), device)
    total = torch.cuda.get_device_properties(0).total_memory if device.type == "cuda" else 0
    budget_ms = 45.0 if device.type == "cuda" else 400.0
    chosen = cfg.max_agents
    sizes = sorted({max(100, cfg.max_agents // 8), max(100, cfg.max_agents // 4), max(100, cfg.max_agents // 2), cfg.max_agents})
    rows = []
    if enabled:
        for n in sizes:
            try:
                if device.type == "cuda":
                    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
                pool = AgentPool(n, L, cfg, device); ids = IdCounter(1)
                eng.spawn_founders(ws, pool, n, g, ids, 0, None)
                pool.body_cap[:n] = torch.tensor([0.004, 1.0, 20.0, 0.02, 0.0, 0.01], device=device)
                ms = timed(lambda: eng.step(ws, pool, 0, g, None, ids, ledger, record_interactions=False), 3)
                peak = int(torch.cuda.max_memory_allocated()) if device.type == "cuda" else 0
                rows.append({"agents": n, "ms_per_tick": round(ms, 2), "peak_vram": peak})
                del pool
                if ms > budget_ms or (total and peak > 0.55 * total):
                    chosen = max(sizes[0], n // 2) if ms > 1.5 * budget_ms else n
                    break
                chosen = n
            except Exception as e:
                if is_oom_error(e):
                    torch.cuda.empty_cache()
                    rows.append({"agents": n, "oom": True})
                    chosen = max(sizes[0], n // 2)
                    break
                raise
    rep["agent_benchmark"] = rows
    if enabled and chosen < cfg.max_agents:
        rep["max_agents_reduced"] = [cfg.max_agents, chosen]
        cfg.max_agents = int(chosen)
        cfg.initial_agents = int(min(cfg.initial_agents, chosen // 3))
    # streaming: custo de montar um frame (empacotamento msgpack)
    if HAVE_MSGPACK:
        arr = np.random.default_rng(0).integers(0, 65535, (cfg.max_stream_agents, 2)).astype(np.uint16)
        t0 = time.perf_counter()
        for _ in range(5):
            pack_msg({"t": "frame", "pos": arr.tobytes(), "ids": np.arange(cfg.max_stream_agents, dtype=np.int64).tobytes()})
        rep["stream_pack_ms"] = round((time.perf_counter() - t0) / 5 * 1000, 3)
    rep["final"] = {"max_agents": cfg.max_agents, "initial_agents": cfg.initial_agents, "max_bodies": cfg.max_bodies,
                    "world": [cfg.world_w, cfg.world_h], "precision": cfg.precision, "compile": cfg.use_compile,
                    "note": "fp32 = referência; bf16/compile não usados sem teste de erro aceitável"}
    if total:
        rep["vram_total"] = int(total)
    if logger:
        logger.event("autotune", **{k: v for k, v in rep.items() if k != "agent_benchmark"})
    return cfg, rep

## 22 — RESTORE / CREATE: retoma o MESMO mundo (idade, agentes, cultura, assentamentos, fila científica)
Ordem: checkpoint válido mais recente (current → previous → backups) → se não houver nenhum, cria o planeta.
`FORCE_NEW=True` arquiva os checkpoints existentes em `checkpoints/archived_*` (nunca apaga) e cria um mundo novo.
A pesquisa 2D do Genesis Ω é importada como arquivo separado (`legacy/`), sem fingir que tinha corpos.

In [ ]:
# @title 22 — RESTORE / CREATE: retoma o MESMO mundo (idade, agentes, cultura, assentamentos, fila científica) { display-mode: "form" }
def boot_universe(cfg: CivConfig, paths: CivPaths, device: torch.device, logger: EventLogger, hw: Dict[str, Any],
                  force_new: bool = False, import_legacy: bool = True, bootstrap_seconds: Optional[float] = None):
    ck = CheckpointEngine(paths, logger)
    info: Dict[str, Any] = {}
    if force_new and ck.candidates():
        stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        dst = os.path.join(ck.root, f"archived_{stamp}")
        os.makedirs(dst, exist_ok=True)
        for d in ck.candidates():
            shutil.move(d, os.path.join(dst, os.path.basename(d)))
        info["archived"] = dst
        logger.event("force_new_archived", echo=True, dir=dst)
    state, linfo = (None, {}) if force_new else ck.load_latest()
    info.update(linfo)
    uni: Optional[UniverseEngine] = None
    if state is not None:
        saved = CivConfig.from_dict(state["meta"]["config"])
        if saved.thresholds_hash() != cfg.thresholds_hash():
            logger.event("thresholds_changed_warning", echo=True, saved=saved.thresholds_hash(), now=cfg.thresholds_hash(),
                         note="resultados antigos continuam vinculados aos thresholds com que foram obtidos")
        if state["meta"].get("versions_hash") != ENGINE_VERSIONS_HASH:
            logger.event("migration_warning", echo=True, saved=state["meta"].get("versions_hash"), now=ENGINE_VERSIONS_HASH,
                         note="motor mudou: campeões antigos serão revalidados antes de comparação direta")
        for k in ("world_w", "world_h", "n_units", "dev_lattice", "n_morph", "n_sound"):
            setattr(cfg, k, getattr(saved, k))                 # forma do mundo/genoma vem do checkpoint (compatibilidade)
        cfg.max_agents = max(cfg.max_agents, min(int(state["agents"]["n"]) + 64, 4 * saved.max_agents))
        uni = UniverseEngine(cfg, paths, device, logger, hw)
        uni.load_state_dict(state)
        uni.sessions += 1
        uni.history.add("session_resume", uni.tick_count, session=uni.sessions, checkpoint=info.get("loaded"),
                        fallback=info.get("fallback", False))
        info["mode"] = "resumed"
    else:
        uni = UniverseEngine(cfg, paths, device, logger, hw)
        if info.get("warning"):
            print("⚠️ " + info["warning"])
        bs = bootstrap_seconds if bootstrap_seconds is not None else (90.0 if device.type == "cuda" else 12.0)
        uni.create_world(bootstrap_seconds=bs)
        uni.sessions = 1
        info["mode"] = "created"
    if import_legacy:
        try:
            lb = LegacyBridge(paths, logger)
            summary = lb.import_research()
            if summary:
                uni.legacy_summary = {"discoveries": len(summary.get("discoveries", [])),
                                      "best_level": summary.get("best_level_global"), "source": summary.get("source_checkpoint"),
                                      "note": summary.get("note")}
                info["legacy"] = uni.legacy_summary
        except Exception as e:
            info["legacy_error"] = repr(e)
    return uni, ck, info

## 23 — LIVE RESEARCH: laço único (mundo ↔ ciência ↔ evolução ↔ telemetria ↔ checkpoint)
Orçamento adaptativo por fatias de tempo: fila científica cheia → mais validação; vazia → mais exploração (QD).
Em `SCIENTIFIC_REPLAY` a agenda é por ticks (determinística), não por relógio. Qualquer exceção tenta checkpoint
seguro + crash report (stack trace, versões, hardware, últimas ações); OOM → registra, limpa cache, reduz lotes,
reduz LOD do viewer e continua. Painel compacto no notebook (o observatório 3D é a interface principal).

In [ ]:
# @title 23 — LIVE RESEARCH: laço único (mundo ↔ ciência ↔ evolução ↔ telemetria ↔ checkpoint) { display-mode: "form" }
class LiveResearch:
    def __init__(self, uni: UniverseEngine, ck: CheckpointEngine, tel: TelemetryEngine, desk: ObserverDesk,
                 cfg: CivConfig, paths: CivPaths, logger: EventLogger, links: Dict[str, Any]):
        self.u, self.ck, self.tel, self.desk, self.cfg, self.paths, self.logger, self.links = uni, ck, tel, desk, cfg, paths, logger, links
        self.qd_gen: Optional[Generator] = None
        self.fork_gen: Optional[Generator] = None
        self.dsl_gen: Optional[Generator] = None
        self.last_ckpt = time.time()
        self.last_panel = 0.0
        self.t_start = time.time()
        self.ticks_start = uni.tick_count
        self.actions: "collections.deque" = collections.deque(maxlen=50)
        self.crashes = 0
        self.handle = None
        tel.lab_view_fn = uni.runner.view

    def _admin(self):
        while True:
            try:
                a = self.desk.admin_queue.get_nowait()
            except queue.Empty:
                return
            act, data = a["action"], a["data"]
            self.actions.append((iso_now(), "admin", act))
            rec = {"tick": self.u.tick_count, "action": act, "data": data, "ts": iso_now(), "source": "admin"}
            if act == "pause":
                self.u.paused = True
            elif act == "resume":
                self.u.paused = False
            elif act == "checkpoint":
                self.ck.save(self.u, reason="admin")
            elif act == "focus":
                self.u.user_focus_aid = int(data["aid"])
                self.u.interventions.append(dict(rec, note="promoção F4 pedida pelo usuário (registrada)"))
            elif act == "unfocus":
                self.u.user_focus_aid = None
            elif act == "fork" and self.fork_gen is None:
                self.fork_gen = self.u.counterfactual_gen(data["intervention"], data["ticks"], label="admin")
            elif act == "dsl" and self.dsl_gen is None:
                self.dsl_gen = self.u.scientist.dsl_gen(data["program"], self.u.runner)
                self.u.interventions.append(dict(rec, note="experimento DSL em arenas (laboratório)"))
            self.u.history.add("external_intervention" if act in ("focus", "fork", "dsl") else "admin_action", self.u.tick_count, action=act)

    def _advance(self, g: Optional[Generator], budget_s: float):
        if g is None:
            return None, None
        t0 = time.time()
        try:
            while time.time() - t0 < budget_s:
                next(g)
        except StopIteration as e:
            return None, e.value
        return g, None

    def _science(self, budget_s: float):
        sc = self.u.scientist
        frac_val = min(0.85, 0.35 + len(sc.queue) / max(1, self.cfg.max_queue))
        if self.fork_gen is not None:
            self.fork_gen, r = self._advance(self.fork_gen, budget_s * 0.3)
            if r is not None:
                self.actions.append((iso_now(), "fork_done", r["id"]))
        if self.dsl_gen is not None:
            self.dsl_gen, r = self._advance(self.dsl_gen, budget_s * 0.3)
        self.tel.state_label = "EXPERIMENTING" if sc.current or sc.queue else "SEARCHING"
        rep = sc.step(budget_s * frac_val)
        if rep:
            self.actions.append((iso_now(), "probe", f"{rep['code']} {rep['kind']} {'✔' if rep['passed'] else '✘'}"))
        if self.qd_gen is None:
            self.qd_gen = self.u.qd.generation_gen()
        self.qd_gen, r = self._advance(self.qd_gen, budget_s * (1 - frac_val))
        if r is not None:
            self.actions.append((iso_now(), "qd", f"ilha {r.get('island')} ger {r.get('gen', '')}"))

    def panel(self, final: bool = False):
        st = self.desk.status or {}
        u = self.u
        el = time.time() - self.t_start
        sc = u.scientist.status()
        civ = u.last.get("civ", {})
        rows = [("estado", st.get("state", "—") + (" · PAUSED" if u.paused else "")), ("tick (tempo artificial)", f"{u.tick_count:,}"),
                ("velocidade", f"{st.get('tps', 0)} ticks/s"), ("população", f"{u.pool.n_alive():,} + F0 {u.f0.total() if u.f0 else 0:.0f}"),
                ("corpos F2/F3/F4", " / ".join(str(v) for v in st.get("bodies", {}).values())),
                ("níveis H · A · C", f"{st.get('h_name', '—')} · {st.get('a_name', '—')} · {st.get('c_name', '—')}"),
                ("Scientist", f"{sc['phase']} · fila {sc['queue']} · alvo {sc['target_level']} · feitos {sc['completed']}"),
                ("descobertas", str(len(u.archive.discoveries))), ("QD", f"{u.qd.generations} gerações · cobertura {u.qd.map.coverage():.2f}"),
                ("assentamentos", f"{civ.get('settlements', 0)} · blocos {civ.get('blocks', 0):.0f}"),
                ("checkpoint", f"{self.ck.last_info.get('ts', '—')} ({self.ck.saves} nesta sessão)"),
                ("sessão / total", f"{fmt_hms(el)} / {fmt_hms(u.total_runtime + el)}")]
        acts = "".join(f"<div>{_html.escape(t[11:19])} · {_html.escape(k)} · {_html.escape(str(v))}</div>" for t, k, v in list(self.actions)[-5:])
        link = self.links.get("remote") or self.links.get("local") or ""
        html_ = ("<div style='font-family:system-ui;background:#070b14;color:#cfe0ff;padding:10px 14px;border-radius:12px;max-width:920px'>"
                 f"<div style='font-weight:700;letter-spacing:.12em'>GENESIS CIVILIZATION Ω∞ {'— SESSÃO ENCERRADA' if final else '— AO VIVO'}"
                 f"<span style='float:right;font-weight:400;font-size:12px'><a style='color:#7cf0c8' target='_blank' href='{_html.escape(link)}'>abrir mundo 3D</a></span></div>"
                 "<table style='font-size:12.5px;margin-top:6px'>" + "".join(f"<tr><td style='color:#7f8ba3;padding-right:14px'>{k}</td><td>{_html.escape(str(v))}</td></tr>" for k, v in rows)
                 + f"</table><div style='font-size:11px;color:#7f8ba3;margin-top:6px'>{acts}</div></div>")
        if self.handle is None:
            self.handle = display(HTML(html_), display_id=True)
        else:
            try:
                self.handle.update(HTML(html_))
            except Exception:
                pass

    def _crash(self, e: BaseException):
        self.crashes += 1
        rep = {"ts": iso_now(), "error": repr(e), "trace": traceback.format_exc(), "versions": ENGINE_VERSIONS,
               "hardware": self.u.hw, "tick": self.u.tick_count, "last_actions": list(self.actions)[-20:],
               "profiler": self.u.prof.report()}
        p = os.path.join(self.paths.crash_reports, f"crash_{int(time.time())}.json")
        with open(p, "w") as f:
            json.dump(to_jsonable(rep), f, indent=1)
        self.logger.event("crash", echo=True, error=repr(e), report=p)

    def run(self, max_hours: float, max_ticks: int = 0):
        cfg, u = self.cfg, self.u
        deadline = time.time() + max_hours * 3600
        self.tel.state_label = "SEARCHING"
        stop_reason = "time_limit"
        try:
            while time.time() < deadline:
                if os.path.exists(self.paths.stop_file):
                    stop_reason = "stop_file"; os.remove(self.paths.stop_file); break
                if max_ticks and u.tick_count - self.ticks_start >= max_ticks:
                    stop_reason = "max_ticks"; break
                try:
                    self._admin()
                    frac = u.scientist.adapt_budget()
                    if cfg.scientific_replay:
                        if not u.paused:
                            for _ in range(20):
                                u.tick()
                        self._science(0.02)                     # agenda por ticks: proporção fixa
                    else:
                        slice_s = 0.25
                        t0 = time.time()
                        if not u.paused:
                            while time.time() - t0 < slice_s * (1 - frac):
                                u.tick()
                        else:
                            time.sleep(0.05)
                        self._science(slice_s * frac)
                    self.tel.pump()
                    if time.time() - self.last_ckpt >= cfg.checkpoint_every_minutes * 60:
                        self.tel.state_label = "CHECKPOINTING"
                        self.ck.save(u, reason="periodic"); self.last_ckpt = time.time()
                        self.tel.checkpoint_info = self.ck.last_info
                    if self.ck.last_info:
                        self.tel.checkpoint_info = self.ck.last_info
                    if time.time() - self.last_panel >= 1.0:
                        self.panel(); self.last_panel = time.time()
                    if cfg.stop_on_champion and u.archive.overall_champion:
                        stop_reason = "champion"; break
                except KeyboardInterrupt:
                    raise
                except Exception as e:
                    if is_oom_error(e):
                        self.logger.event("oom", echo=True, error=repr(e)[:300])
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                        oom_backoff(cfg)
                        u.scientist._gen = None; u.scientist.current = None
                        self.qd_gen = None
                        continue
                    self._crash(e)
                    try:
                        self.ck.save(u, sync=True, reason="after_exception")
                    except Exception as e2:
                        self.logger.event("checkpoint_after_crash_failed", error=repr(e2))
                    u.scientist._gen = None; u.scientist.current = None; self.qd_gen = None
                    if self.crashes > 20:
                        stop_reason = "too_many_errors"; break
        except KeyboardInterrupt:
            stop_reason = "user_interrupt"
        finally:
            self.tel.state_label = "CHECKPOINTING"
            self.tel.pump(force=True)
            u.total_runtime += time.time() - self.t_start
            self.ck.wait()
            self.ck.save(u, sync=True, reason=f"session_end:{stop_reason}")
            self.tel.checkpoint_info = self.ck.last_info
            self.tel.state_label = "OFFLINE"
            self.desk.online = False
            self.tel.pump(force=True)
            u.history.flush(); self.logger.flush()
            self.panel(final=True)
        return stop_reason

## 24 — INSPECTION: inspecionar, comparar, reexecutar, reproduzir, exportar e importar
Funções preservadas do Genesis Ω, agora sobre o universo 3D. Exportações levam schema, versão e hash; importação
valida os três (nunca carrega formato incompatível silenciosamente; nunca usa pickle).

In [ ]:
# @title 24 — INSPECTION: inspecionar, comparar, reexecutar, reproduzir, exportar e importar { display-mode: "form" }
EXPORT_SCHEMA = "genesis-civ-candidate/1"


def inspect_candidate(cid: str, uni: Optional[UniverseEngine] = None) -> Dict[str, Any]:
    uni = uni or GENESIS["uni"]
    c = uni.archive.zoo.get(cid)
    if c is None:
        raise KeyError(f"candidato {cid} não está no zoológico")
    G = torch.as_tensor(c.genome, device=uni.device).view(1, -1)
    bp = uni.dev.develop(G, torch.tensor([1], device=uni.device))
    cap = uni.body.measure_capability(G, torch.tensor([1], device=uni.device))[0].cpu().numpy()
    out = {"cid": cid, "origin": c.origin, "fitness": c.fitness, "evidence_score": c.evidence_score(), "passed": c.passed,
           "failed": c.failed, "attempts": c.attempts, "h_level": c.h_level(), "a_level": c.a_level(),
           "body": {"nodes": int(bp.occ.sum()), "speed": float(cap[0]), "cost": float(cap[1]), "turn": float(cap[3])},
           "genome_sections": {k: float(torch.as_tensor(c.genome)[uni.L.slices[k][0]:uni.L.slices[k][1]].abs().mean())
                               for k in ("ctl_W0", "ctl_Win", "plast_eta", "plast_abcd", "dev_Wm")},
           "note": "fitness ≠ evidência; níveis só por probes com controles"}
    return out


def compare_candidates(a: str, b: str, uni: Optional[UniverseEngine] = None) -> Dict[str, Any]:
    uni = uni or GENESIS["uni"]
    A, B = uni.archive.zoo[a], uni.archive.zoo[b]
    d = float(uni.L.distance(torch.as_tensor(A.genome).view(1, -1), torch.as_tensor(B.genome).view(1, -1)))
    return {"a": a, "b": b, "genome_distance": d, "levels_a": sorted(A.passed), "levels_b": sorted(B.passed),
            "only_a": sorted(set(A.passed) - set(B.passed)), "only_b": sorted(set(B.passed) - set(A.passed)),
            "evidence": [A.evidence_score(), B.evidence_score()], "fitness": [A.fitness, B.fitness]}


def replay_candidate(cid: str, ticks: int = 160, uni: Optional[UniverseEngine] = None, seed: int = 7) -> Dict[str, Any]:
    """Reexecuta o candidato numa arena padronizada (mesma física) e devolve trajetórias + métricas."""
    uni = uni or GENESIS["uni"]
    c = uni.archive.zoo[cid]
    n = uni.cfg.probe.trials
    G = torch.as_tensor(c.genome, device=uni.device).expand(n, -1).contiguous()
    tr = uni.runner.build(G, ArenaSpec(task="replay", n_patches=3, resource_floor=0.05), seed)
    rec = drive(uni.runner.run_gen(tr, ticks, record=("gain", "pos", "alive")))
    return {"cid": cid, "intake": summary_stats(rec["gain"].sum(0).cpu().numpy()), "survival": float(rec["alive"].float().mean()),
            "replay": make_replay(tr, rec)}


def reproduce_discovery(did: str, uni: Optional[UniverseEngine] = None) -> Dict[str, Any]:
    """Reexecuta o probe da descoberta com o genoma salvo e as MESMAS sementes; compara com o registro."""
    uni = uni or GENESIS["uni"]
    meta = uni.archive.load_discovery(did)
    if meta is None:
        raise KeyError(f"descoberta {did} não encontrada")
    ver = uni.archive.verify_discovery(did)
    if meta.get("kind") == "civilization":
        return {"id": did, "integrity": ver, "note": "descoberta civilizacional: reprodução = carregar o checkpoint/snapshot "
                "do tick e reavaliar os critérios", "evidence": meta.get("evidence")}
    G = torch.as_tensor(meta["genome"], device=uni.device)
    code = meta["level"]
    key = meta["candidate"]
    res = drive(uni.validation.run_gen(code, G, key))
    first = meta["evidence"][0] if meta.get("evidence") else {}
    return {"id": did, "integrity": ver, "level": code, "reproduced_passed": res.passed, "p": res.p, "effect": res.effect,
            "original_p": first.get("p"), "original_effect": first.get("effect"),
            "status_match": bool(res.passed == bool(first.get("passed", True))),
            "versions_match": meta.get("versions_hash") == ENGINE_VERSIONS_HASH}


def export_candidate(cid: str, path: Optional[str] = None, uni: Optional[UniverseEngine] = None) -> str:
    uni = uni or GENESIS["uni"]
    c = uni.archive.zoo[cid]
    path = path or os.path.join(uni.paths.exports, f"{cid}.npz")
    meta = {"schema": EXPORT_SCHEMA, "cid": cid, "origin": c.origin, "passed": c.passed, "versions": ENGINE_VERSIONS,
            "versions_hash": ENGINE_VERSIONS_HASH, "genome_P": int(c.genome.size), "layout": {k: list(v) for k, v in uni.L.slices.items()},
            "genome_sha256": hashlib.sha256(c.genome.astype(np.float32).tobytes()).hexdigest(), "exported": iso_now()}
    np.savez(path, genome=c.genome.astype(np.float32), meta=np.frombuffer(json.dumps(to_jsonable(meta)).encode(), dtype=np.uint8))
    return path


def import_candidate(path: str, uni: Optional[UniverseEngine] = None) -> str:
    uni = uni or GENESIS["uni"]
    with np.load(path, allow_pickle=False) as z:
        meta = json.loads(bytes(z["meta"].astype(np.uint8)).decode())
        g = np.asarray(z["genome"], np.float32)
    if meta.get("schema") != EXPORT_SCHEMA:
        raise ValueError(f"schema incompatível: {meta.get('schema')}")
    if hashlib.sha256(g.tobytes()).hexdigest() != meta.get("genome_sha256"):
        raise ValueError("hash do genoma não confere (arquivo corrompido ou alterado)")
    if int(meta.get("genome_P", -1)) != uni.L.P or meta.get("layout") != {k: list(v) for k, v in uni.L.slices.items()}:
        raise ValueError("layout de genoma incompatível com este motor (exige migração explícita)")
    c = uni.scientist.add_candidate(g, f"import:{meta.get('cid')}", uni.tick_count)
    return c.cid


def inspect_agent(aid: int, uni: Optional[UniverseEngine] = None) -> Dict[str, Any]:
    uni = uni or GENESIS["uni"]
    return agent_card(uni, int(aid))

## 25 — EXPORT / REPORTS: relatório de sessão (com limitações científicas explícitas)

In [ ]:
# @title 25 — EXPORT / REPORTS: relatório de sessão (com limitações científicas explícitas) { display-mode: "form" }
LIMITATIONS = ["universo artificial (não é simulação atomística do Universo real)",
               "abstrações físicas: massa-mola simplificada, clima procedural, recurso abstrato",
               "coarse-graining entre escalas (F0–F4) com erro medido no probe MS, não nulo",
               "escala temporal artificial (ticks); 'dia'/'estação' são ciclos artificiais",
               "fidelidade limitada pelo hardware; corpos completos só para um subconjunto",
               "nenhuma inferência direta de consciência ou experiência subjetiva (afeto = funcional)",
               "nenhuma equivalência automática a humanos reais; níveis C são hipóteses de simulação",
               "sem regeneração estrutural de corpos após dano (só robustez funcional)",
               "kernels CUDA com somas atômicas podem não ser bit-a-bit determinísticos (registrado nos testes)"]


def generate_report(uni: UniverseEngine, hw: Dict[str, Any], boot: Dict[str, Any], stop_reason: str = "") -> str:
    p = uni.paths
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    dem = uni.demography.last(400)
    figs = []
    try:
        fig, ax = plt.subplots(1, 3, figsize=(12, 3))
        if dem:
            ax[0].plot([d["tick"] for d in dem], [d["population"] for d in dem], color="#3d7bff"); ax[0].set_title("população")
            ax[0].plot([d["tick"] for d in dem], [d.get("f0", 0) for d in dem], color="#999", lw=1)
        for isl in uni.qd.islands:
            ax[1].plot(isl.history, lw=1)
        ax[1].set_title("QD: melhor fitness por ilha (≠ evidência)")
        pr = uni.prof.report()
        ax[2].barh(list(pr.keys())[:8], [v["share"] for v in list(pr.values())[:8]], color="#46c3a0"); ax[2].set_title("tempo por subsistema")
        fig.tight_layout()
        fp = os.path.join(p.plots, f"session_{ts}.png"); fig.savefig(fp, dpi=90); plt.close(fig)
        figs.append(fp)
    except Exception as e:
        figs.append(f"plot error: {e}")
    civ = uni.last.get("civ", {})
    lad = ladder_status(uni)
    lines = [f"# GENESIS CIVILIZATION Ω∞ — relatório de sessão {ts}", "",
             "> universo artificial; comportamento observado ≠ mecanismo demonstrado ≠ interpretação ≠ experiência subjetiva", "",
             "## hardware e execução",
             f"- device: {hw.get('device')} · GPU: {hw.get('gpu_name')} · VRAM: {human_bytes(hw.get('vram_total', 0))} · torch {hw.get('torch')}",
             f"- perfil: {uni.cfg.profile} · mundo {uni.ws.W}×{uni.ws.H} · tick {uni.tick_count:,} · sessões {uni.sessions}",
             f"- runtime total: {fmt_hms(uni.total_runtime)} · motivo do fim: {stop_reason}",
             f"- testes de boot: {boot.get('tests_passed')}/{boot.get('tests_total')} · autotune: {to_jsonable(boot.get('autotune', {}).get('final'))}",
             "", "## evolução e população",
             f"- avaliações QD: {uni.qd.evaluations:,} · gerações QD: {uni.qd.generations} · cobertura MAP-Elites {uni.qd.map.coverage():.3f}",
             f"- população viva {uni.pool.n_alive():,} · F0 {uni.f0.total() if uni.f0 else 0:.0f} · nascimentos totais {uni.lineage.total:,}",
             f"- fidelidade: {dict(uni.fidelity_log)}", "", "## escadas (evidência)"]
    for grp in ("H", "A"):
        for r in lad[grp]:
            tag = "TESTED" if r.get("validated") else ("NOT YET VALIDATED" if r["tested"] else "não testado")
            lines.append(f"- {r['name']}: {tag}" + (f" · efeito {r.get('effect'):.2f} · {r.get('discovery')}" if r.get("validated") and r.get("effect") is not None else ""))
    for r in lad["C"]:
        lines.append(f"- {r['name']}: {'SIMULATION HYPOTHESIS (critérios atingidos)' if r['achieved'] else 'NOT YET VALIDATED'}")
    lines += ["", "## descobertas"] + [f"- {d['id']} · {d['level_name']} · {d['claim']} · réplicas {d['passed_replications']}/{d['replications']}"
                                       for d in uni.archive.discoveries[-40:]]
    lines += ["", "## mundo, cultura, comunicação, sociedade",
              f"- assentamentos {civ.get('settlements', 0)} · cidades {civ.get('cities', 0)} · blocos {civ.get('blocks', 0):.0f} · estradas {civ.get('roads', 0)}",
              f"- casa-like (função de abrigo demonstrada): {uni.settlements.house_like}",
              f"- comunicação (MI sinal×mundo): {to_jsonable(uni.last.get('comm', {}))}",
              f"- cultura: {to_jsonable({k: v for k, v in uni.last.get('culture', {}).items() if k != 'techniques'})}",
              f"- social: {to_jsonable({k: v for k, v in uni.last.get('social', {}).items() if k != 'dissolved'})}",
              f"- eventos históricos: {uni.history.total} · ETHICAL_REVIEW_TRIGGER: {len(uni.sentinel.triggers)}",
              f"- falhas arquivadas: {len(uni.archive.failures)} · anomalias: {len(uni.archive.weird)} · convergências: {len(uni.archive.convergence)}",
              f"- forks contrafactuais: {list(uni.forks.keys())[-10:]}", "", "## limitações atuais"] + [f"- {x}" for x in LIMITATIONS]
    lines += ["", "## status das alegações", "- IMPLEMENTED: todo módulo listado no notebook roda neste build",
              "- TESTED: só o que passou nos testes de boot e nos probes com controles (acima)",
              "- BENCHMARKED: autotune + probe MS (erro multiescala) + profiler", "- SIMULATION HYPOTHESIS: níveis C, A9, H12",
              "- NOT YET VALIDATED: todo nível sem descoberta registrada"]
    md = "\n".join(lines)
    out = os.path.join(p.reports, f"report_{ts}.md")
    with open(out, "w") as f:
        f.write(md)
    with open(os.path.join(p.reports, "latest_report.md"), "w") as f:
        f.write(md)
    return out

## ▶ GENESIS — célula principal (boot → testes → autotune → restaurar/criar → observatório 3D → pesquisa ao vivo)
Esta é a única célula de execução contínua. Ela mantém o viewer disponível e roda o laço de pesquisa até o fim da
sessão do Colab (ou `MAX_SESSION_HOURS`). Para pausar sem perder nada: botão ■ (checkpoint síncrono no fim).

In [ ]:
# @title ▶ GENESIS — célula principal (boot → testes → autotune → restaurar/criar → observatório 3D → pesquisa ao vivo) { display-mode: "form" }
GENESIS: Dict[str, Any] = {}


def _free_port(start: int) -> int:
    for p in range(start, start + 40):
        with socket.socket() as s_:
            try:
                s_.bind(("127.0.0.1", p))
                return p
            except OSError:
                continue
    raise RuntimeError("nenhuma porta livre para o servidor")


def genesis_main():
    display(HTML("<div style='font-family:system-ui;letter-spacing:.3em;color:#9fd3ff;font-weight:700;font-size:15px'>"
                 "◎ GENESIS CIVILIZATION Ω∞</div><div style='font-family:system-ui;color:#7f8ba3;font-size:12px'>"
                 "universo artificial · desenvolvimento · corpos · mentes · afeto funcional · sociedade · cultura · civilização</div>"))
    hw = detect_hardware()
    profile = choose_profile(PROFILE, hw)
    device = torch.device(hw["device"])
    torch.set_flush_denormal(True)
    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = False          # FP32 é a referência científica
        torch.backends.cudnn.allow_tf32 = False
    if SCIENTIFIC_REPLAY:
        torch.use_deterministic_algorithms(True, warn_only=True)
    if device.type != "cuda":
        print("⚠️ GPU não detectada: rodando em modo DEBUG na CPU. A pesquisa completa requer GPU "
              "(Ambiente de execução → Alterar tipo → GPU).")
    paths = setup_storage(USE_GOOGLE_DRIVE, GENESIS_ROOT_OVERRIDE, GENESIS_WORK_OVERRIDE)
    logger = EventLogger(paths)
    cfg = profile_config(profile, hw, MASTER_SEED)
    cfg.scientific_replay, cfg.human_constraint_experiment = SCIENTIFIC_REPLAY, HUMAN_CONSTRAINT_EXPERIMENT
    cfg.stop_on_champion, cfg.max_session_hours = STOP_ON_CHAMPION, MAX_SESSION_HOURS
    cfg.viewer_port = _free_port(VIEWER_PORT)
    print(f"hardware: {hw.get('gpu_name')} · VRAM {human_bytes(hw.get('vram_total', 0))} · RAM {human_bytes(hw.get('ram_total', 0))} · "
          f"CPU {hw.get('cpu_count')} · torch {hw.get('torch')} · CUDA {hw.get('cuda')} · perfil {profile}")
    print(f"armazenamento: {paths.root} ({'Google Drive' if paths.on_drive else 'local — não persiste se a sessão cair'})")
    boot: Dict[str, Any] = {"profile": profile}
    if RUN_BOOT_TESTS:
        ok, res = run_boot_tests(cfg, paths, device, logger)
        boot.update({"tests_passed": sum(r["ok"] for r in res), "tests_total": len(res)})
        critical = [r for r in res if not r["ok"] and r["test"] != "test_server_protocol"]
        if critical:
            print("❌ GENESIS BOOT FAILED — testes científicos críticos falharam; pesquisa não iniciada:")
            for r in critical:
                print(f"   {r['test']}: {r.get('error')}")
            GENESIS.update({"boot": boot, "tests": res, "paths": paths})
            return GENESIS
    if AUTO_TUNE:
        cfg, tune = benchmark_and_tune(cfg, device, logger, enabled=device.type == "cuda")
        boot["autotune"] = tune
        with open(os.path.join(paths.reports, "autotune.json"), "w") as f:
            json.dump(to_jsonable(tune), f, indent=1)
        print(f"autotune: {tune['final']}")
    uni, ck, info = boot_universe(cfg, paths, device, logger, hw, FORCE_NEW, IMPORT_LEGACY_GENESIS)
    boot["restore"] = info
    print(f"mundo: {'RETOMADO' if info['mode'] == 'resumed' else 'CRIADO'} · tick {uni.tick_count:,} · população {uni.pool.n_alive():,} · "
          f"sessão nº {uni.sessions}" + (f" · checkpoint {os.path.basename(info.get('loaded', ''))}" if info.get("loaded") else ""))
    if info.get("legacy"):
        print(f"Genesis Ω (2D) importado como arquivo separado: {info['legacy']['discoveries']} descobertas · melhor nível "
              f"{info['legacy']['best_level']}")
    desk = ObserverDesk()
    tel = TelemetryEngine(uni, desk)
    tel.state_label = "RESTORING"
    tel.pump(force=True)
    links: Dict[str, Any] = {}
    srv = tun = None
    if START_3D_VIEWER:
        try:
            viewer = Genesis3DViewer(paths, logger)
            static = viewer.install()
            srv = RealtimeServer(desk, lambda: uni, paths, cfg.viewer_port, REMOTE_READ_ONLY, logger, static)
            if srv.start():
                remote = None
                if ENABLE_REMOTE_VIEW:
                    tun = TunnelManager(paths, logger)
                    remote = tun.start(cfg.viewer_port)
                    if remote is None:
                        print(f"túnel remoto indisponível ({tun.error}); o viewer local continua funcionando.")
                links = show_world_links(cfg.viewer_port, srv.view_token, srv.admin_token, remote, REMOTE_READ_ONLY)
                print(f"viewer: three.js {'local' if viewer.vendor_ok else 'via CDN fixado'} · porta {cfg.viewer_port}")
            else:
                print(f"⚠️ viewer não iniciou ({srv.error}); a pesquisa científica continua.")
        except Exception as e:
            print(f"⚠️ viewer falhou ({e!r}); a pesquisa científica continua.")
            logger.event("viewer_failed", error=repr(e), trace=traceback.format_exc()[-1500:])
    GENESIS.update({"uni": uni, "ck": ck, "tel": tel, "desk": desk, "server": srv, "tunnel": tun, "links": links,
                    "cfg": cfg, "paths": paths, "hw": hw, "boot": boot, "logger": logger})
    if START_RESEARCH:
        live = LiveResearch(uni, ck, tel, desk, cfg, paths, logger, links)
        GENESIS["live"] = live
        reason = live.run(MAX_SESSION_HOURS, MAX_TICKS_THIS_SESSION)
        rp = generate_report(uni, hw, boot, reason)
        GENESIS["report"] = rp
        print(f"sessão encerrada ({reason}) · checkpoint {ck.last_info.get('ts')} · relatório {rp}")
        print("Na próxima sessão: Ambiente de execução → Executar tudo — o MESMO mundo é retomado.")
    return GENESIS


genesis_main()